<a href="https://colab.research.google.com/github/drperezt/GCP_devs/blob/main/Finance_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**¡¡¡¡ WARNING ❗❗❗❗** **⚠**

☣ ☝

**Este colab ***NUNCA*** ❌❗ debe ejecutarse todo (completo), solo la sección que se requiera, algunas secciones se ejecutan solo una vez durante el ejercicio, realizar su ejecución en el momento equivocado generará errores ❌❗ para los usuarios.**


# **Bibliotecas**
❗❗❗Siempre ejecutar al inicio

In [ ]:
!pip install xlsxwriter
# ! sudo dpkg-reconfigure locales

# ! sudo apt-get install xclip

In [ ]:
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from google.colab import drive
from google.auth import default
from google.api_core.exceptions import BadRequest, GoogleAPICallError
from google.api_core import exceptions
import openpyxl
from openpyxl import load_workbook
from openpyxl import Workbook
from openpyxl.utils import FORMULAE
from babel.dates import format_date
import locale
import bigframes.pandas as bpd
import pandas as pd
import numpy as np
from google.colab import files
import shutil
import os
import re
# import xlsxwriter
import csv
from IPython.display import display, Javascript
from datetime import date
from google.cloud.bigquery.enums import EntityTypes
import unicodedata
from decimal import Decimal, getcontext

import time
from datetime import datetime
import pytz
timezone = pytz.timezone('America/Mexico_City')

# import xclip#pyperclip

auth.authenticate_user()
import gspread
from oauth2client.client import GoogleCredentials

from google.colab import userdata


creds, _ = default()
gc = gspread.authorize(creds)

bpd.options.bigquery.project = userdata.get('GCP_PROJECT_ID')
drive.mount('/content/drive')

project_id = userdata.get('GCP_PROJECT_ID')
client = bigquery.Client(project = project_id)


def strip_accents(text):
    """
    Removes accents from a string using unicodedata.
    """
    # Normalize the string to NFD (Canonical Decomposition) form
    normalized_text = unicodedata.normalize('NFD', text)
    # Filter out combining characters (accents)
    stripped_text = "".join(c for c in normalized_text if not unicodedata.combining(c))
    return stripped_text

# **Parámetros**
❗❗❗Siempre ejecutar al inicio

In [ ]:
##Las tablas base se pueden editar en:
#   https://docs.google.com/spreadsheets/d/1eGLkHvdTx4I1ymFWVCy5v1bKiohCKOtu3OLH2dDqBj0/edit?gid=0#gid=0$0

##MESESPRES ES PARA CONCEPTOS, PERIODOS PARA CALCULOS DE BASE

### Se tienen dos modelos de presupuesto, revisión al actual y ajuste al actual con definición al siguiente

# DIMENSION6pres = "ME_PPTO"
DIMENSION6pres = "REV"

##Para realizar pruebas, se genera en QAS, y en PRO es lo que generan los usuarios (deBONO_ESP ser)

## Definir parámetros,
# version son las versiones a realizar conforme al DIMENSION6 de presupuesto (DIMENSION6pres)

ss = gc.open_by_url("https://docs.google.com/spreadsheets/d/1eGLkHvdTx4I1ymFWVCy5v1bKiohCKOtu3OLH2dDqBj0/edit?gid=0#gid=0$0")
ssh = ss.worksheet("CONFIG")

inflacionplan = float(ssh.acell('B2').value.split("%")[0])/100 # inflación
aniobase = int(ssh.acell('B1').value) # es el actual (que se esté presupuestando)
porc_inc = float(ssh.acell('B3').value.split("%")[0])/100 # porc_inc es el porcentaje de incremento de nómina
mesbaser = 3 #int(ssh.acell('B4').value) # es el mes último de reales para históricos (este sirve para RACUM)
mesbasehc = 3 #int(ssh.acell('B5').value) # es el mes base, para replicar nómina
mesinc = int(ssh.acell('B6').value) # es el mes de incremento de nómina anual general
mesinc_vales = int(ssh.acell('B7').value) # es el mes de incremento de vales de despensa
porc_inc_vales = float(ssh.acell('B8').value.split("%")[0])/100
porc_inc_vuniform = float(ssh.acell('B9').value.split("%")[0])/100
porc_inc_sig_anio = float(ssh.acell('B10').value.split("%")[0])/100 # incremento de sueldos del siguiente año de presupuesto, sirve para BONO BONO_ESP
meses_bono = float(ssh.acell('B11').value)
mes_captura = int(ssh.acell('B12').value)
direcciones = ssh.acell('B15').value.split(",")
porc_inc_seg_anio = float(ssh.acell('B16').value.split("%")[0])/100 # incremento de sueldos del 2 año posterior al base, sirve para BONO BONO_ESP PLAN
direcciones_nomina = ssh.acell('B17').value.split(",")
mes_inc_valesali = int(ssh.acell('B18').value)
inflacion_actual = float(ssh.acell('B19').value.split("%")[0])/100 # inflación para cálculos del ejercicio actual
divisiones_mne = ssh.acell('B20').value.split(",") # modelos que utilizaran nómina
isn_cdmx = float(ssh.acell('B21').value.split("%")[0])/100 # porcentaje ISN CDMX
MATRICIALES_CFO = ssh.acell('B22').value.split(",") # modelos que utilizaran nómina

# strippeMATRICIALES_CFOd_list = [s.strip() for s in original_list]

variablequotation = '"""'

for f in range(0,len(direcciones)) :
  direcciones[f] = direcciones[f].strip()

for f in range(0,len(direcciones_nomina)) :
  direcciones_nomina[f] = direcciones_nomina[f].strip()

vales_via_planta = float(ssh.acell('B13').value)
vales_via_planta_2 = float(ssh.acell('B14').value)

datebaser = date(aniobase,mesbaser,1)
monthname = format_date(datebaser, "MMMM", locale='es').upper()

dateformat = '%y-%m-%d %H:%M:%S.%f'

if DIMENSION6pres == "ME_PPTO":
  params = {"aniobase": aniobase, "mesbaser": mesbaser, "mesbasehc": mesbasehc, "mes_inc": mesinc, "mes_inc_vales": mesinc_vales,
            "porc_inc_vuniform": str(porc_inc_vuniform), "porc_inc_vales": str(porc_inc_vales),
            "porc_inc": str(porc_inc), "version":['PPTO_V2','PPTO_V0']}
  aniopres = str(params['aniobase']+1)
  carpeta = "1. Presupuesto"

else:
  params = {"aniobase": aniobase, "mesbaser": mesbaser, "mesbasehc": mesbasehc, "mes_inc": mesinc, "mes_inc_vales": mesinc_vales,
            "porc_inc_vuniform": str(porc_inc_vuniform), "porc_inc_vales": str(porc_inc_vales),
            "porc_inc": str(porc_inc), "version": ['PPTO_V1']}
  aniopres = str(params['aniobase'])
  carpeta = "2. Revisión"

if DIMENSION6pres == "ME_PPTO":
  versionprimmeses = 'Reales'
  aniocarpeta = aniobase +1
else:
  versionprimmeses = 'PPTO'
  aniocarpeta = aniobase

rutaexcelventas = ""
rutaventas = f"/"
rutaepm = f""
rutaNegocio10 = f""
rutaeeff = f""

datasetbases = "financial_budgeting"#DIMENSION6pres + aniopres ## Esto cambiará a "PRESUPUESTO_CORRIENTE"

datasetbases_reportes_bi = "REPORTES_BI"

project_datasetbases = f"{client.project}.{datasetbases}"
project_datasetreportesbi = f"{client.project}.{datasetbases_reportes_bi}"

project_datasetparametros = f"{client.project}.PARAMETROS_PRESUPUESTO"
project_datasetdatosmaestros = f"{client.project}.datos_maestros"

current_month = datetime.now().month

aniobasehc = aniobase

if mesbasehc >= current_month :
  aniobasehc = aniobase-1
  val_incmes_nom = f" {mesinc} > {mesbasehc} AND {aniobase} = {aniobasehc} "
else:
  aniobasehc = aniobase
  val_incmes_nom = f" {mesinc} > {mesbasehc}"

filtromesescalculonom = f"(T2.Version = '{params['version'][0]}' AND T2.mes > {mesbaser}) OR (T2.Version <> '{params['version'][0]}')"

caseajustemescaptura = f"CASE WHEN T2.MES < {mes_captura} THEN {mes_captura} ELSE T2.MES END"

In [ ]:
###Antes se hacía un dataframe por "segmento de la base, por mejores prácticas de Gobierno de Datos, se cambia"
# datasetbases_cn = DIMENSION6pres + aniopres + "_CN"
# datasetbases_cedulas = DIMENSION6pres + aniopres + "_CEDULAS"
# datasetbases_ventas = DIMENSION6pres + aniopres + "_VENTAS"
# datasetbases_archcarg = DIMENSION6pres + aniopres + "_ARCHIVOS_CARGA"
# datasetbases_Negocio9 = DIMENSION6pres + aniopres + "_CEDULAS_Negocio9"
# datasetbases_nomina = DIMENSION6pres + aniopres + "_NOMINA"
# project_datasetbasescn = f"{client.project}.{datasetbases_cn}"
# project_datasetbasescedulas = f"{client.project}.{datasetbases_cedulas}"
# project_datasetbasesventas = f"{client.project}.{datasetbases_ventas}"
# project_datasetbasesarchcarg = f"{client.project}.{datasetbases_archcarg}"
# project_datasetNegocio8 = f"{client.project}.{datasetbases_Negocio9}"
# project_datasetnomina = f"{client.project}.{datasetbases_nomina}"

# datasetss = [datasetbases,datasetbases_cn,datasetbases_cedulas,datasetbases_ventas,datasetbases_archcarg,
#              datasetbases_reportes_bi,datasetbases_Negocio9,datasetbases_nomina]
# projectsdatasets = [project_datasetbases,project_datasetbasescn,project_datasetbasescedulas,
#                     project_datasetbasesventas,project_datasetbasesarchcarg,project_datasetreportesbi,
#                     project_datasetNegocio8,project_datasetnomina]

⭕  CREACION DE DATASETS
(deprecated)

In [ ]:
# for i in range(0,len(datasetss)) :
#   try:
#     dataset = client.get_dataset(datasetss[i])
#     print(f"BigQuery dataset {projectsdatasets[i]} exists\n")

#   except:
#     print(f"BigQuery dataset {projectsdatasets[i]} doesn't exist, so creating it\n")
#     dataset = client.create_dataset(bigquery.Dataset(projectsdatasets[i]))
#     print("dataset created")

# **Tablas**
❗❗❗Siempre ejecutar al inicio

In [ ]:
ER = f"{project_id}.data_warehouse_finance.accounting_journal"
###Datos maestros

CEBES = f"{project_datasetdatosmaestros}.CEBES_CATALOG"
CECOS = f"{project_datasetdatosmaestros}.CECO_CATALOG"
JERARCUENTAPPTO = f"{project_datasetparametros}.PAR_JERARQUIA_GENERAL"
NOMINAPOSICION = f"{project_datasetparametros}.PAR_NOMINA_POSICION_CUENTA"
# CUENTAS = f"{client.project}.mus_qas_drv_datos_maestros.Cuentas"
CUENTAS = f"{project_datasetdatosmaestros}.jerarquiacuentas"
CUENTAS_SBC = f"{project_datasetparametros}.CUENTAS_ISN"
PERIODOS = f"{project_datasetparametros}.MESES_COMPLETOS"
MESESPRES = f"{project_datasetparametros}.PAR_MESESPRES"
PARPERIODOS = f"{project_datasetparametros}.PAR_PERIODOS"
POSICIONES = f"{project_datasetparametros}.POSICIONES"
CUENTAS_NOM = f"{project_datasetparametros}.PAR_CUENTAS_NOMINA" ### MAPEO CUENTA PROVISION + CUENTA REAL
CUENTAS_CARGA_INICIAL = f"{project_datasetparametros}.PAR_CUENTAS_CARGA_INICIAL"
CUENTAS_DM = f"{project_datasetdatosmaestros}.Cuentas_DM"
DIMENSION5S_CORP = f"{project_datasetparametros}.DIMENSION5CACIONES_CORP"
CUENTAS_PRO_ADC= f"{project_datasetparametros}.PAR_CUENTAS_PRORRATEO_UNIDADES_NEGOCIO"
CUENTAS_REGLAS_NOM = f"{project_datasetparametros}.MODELO_NOMINA_REGLAS_CUENTAS"  ####
CUENTAS_CED_FISOS = f"{project_datasetparametros}.PAR_CUENTAS_CED_ESPECIALS"
CUENTAS_CED_INM_ADC = f"{project_datasetparametros}.PAR_CUENTAS_CED_ESPECIALS2"
PAR_BONO_ESP = f"{project_datasetparametros}.PAR_BONO"
CECOS_MATRICIALES_CFO = f"{project_datasetparametros}.MATRICIALES_CFO"

###PARAMETROS Y CATALOGOS INICIALES

tabla_params = f"{project_datasetbases}.PARAMETROS"
tabla_pargen = f"{project_datasetbases}.PARAMETROS_GENERALES"

tablareglasnom = f"{project_datasetparametros}.MODELO_NOMINA_REGLAS_CUENTAS"
tablavales = f"{project_datasetbases}.VALES_DESPENSA"
tablavales_un = f"{project_datasetbases}.VALES_UNIFORME"
headcountnew = f"{project_id}.human_capital.HC_2026_2"
tabladistpres = f"{project_datasetbases}.DISTPRESTACIONES"
tablapres_disanioanterior = f'''{project_datasetbases}.DISTRIBUCION_ANIO_ANTERIOR'''
tablapres_disanioactual = f'''{project_datasetbases}.DISTRIBUCION_ANIO_ACTUAL'''

##MODELO NOMINA ETZ
nomina_headcountbase = f"{project_datasetbases}.HEADCOUNT_BASE"
nomina_detalle = f"{project_datasetbases}.MODELO_ETZ_BASE"
nomina_cecos_posicion = f"{project_datasetbases}.CECOS_POSICION"
nomina_parametros = f"{project_datasetbases}.PARAMETROS_NOMINA"
nomina_tabulador = f"{project_datasetbases}.TABULADOR"
nomina_DIMENSION5caciones_estados = f"{project_datasetbases}.DIMENSION5CACIONES_ESTADOS"
nomina_isn = f"{project_datasetbases}.ISN"
nomina_mapeo_cuentas = f"{project_datasetparametros}.PAR_CUENTAS_MODELO_NOMINA"
sp_carga_posicion_ceco = f"{project_datasetbases}.MODELO_NOMINA_Negocio9_sp"
tabla_captura_hc_Negocio9 = f"{project_datasetbases}.CAPTURA_HC_Negocio9"
areapers =f"{project_datasetparametros}.AREA_PERSONAL"

tabla_hc_total = f"{project_datasetbases}.Posiciones_Presupuestales_HC"

###PARA REPORTE

base_hist_Negocio9 = f"{project_datasetreportesbi}.BASEP_HISTORICOS_Negocio9"
base_primerosmesesrealesnom = f"{project_datasetbases}.Primeros_meses"

###CALCULOS INICIALES

base_hist = f"{project_datasetbases}.BASEP_HISTORICOS"
base_hist_acum = f"{project_datasetbases}.BASEP_HISTORICOS_ACUM"

tablanomina = f"{project_datasetbases}.BASEP_NOMINA_BASE"
tablanomina_vales = f"{project_datasetbases}.BASEP_NOMINA_VALES"
tablabonobase = f"{project_datasetbases}.BASEP_BONOBASE"

tablanompres = f"{project_datasetbases}.BASEP_NOMINA_PRESTACIONES"  ##DEPRECATED
tablacargapresrep = f'''{tablanompres}_replicas'''
tablacargapresccero = f'''{tablanompres}_crecimientocero'''
tablacargapresccero_sinout = f'''{tablanompres}_crecimientocero_sinout'''
tablacargapresminc = f'''{tablanompres}_mesinc'''
tablacargaporc = f'''{tablanompres}_porc''' #porcentajes ultimos 12 meses
tablacargaporc_ant = f'''{tablanompres}_porc_ant''' #porcentajes
tablacargaptugar = f'''{tablanompres}_ptugar'''
tablacargapinfl = f'''{tablanompres}_inflacion'''
tablanompres_isn = f'''{tablanompres}_isn'''
tablanompres_anioantinf = f'''{tablanompres}_anioantinf'''
tablanom_diff = f"{project_datasetbases}.BASEP_NOMINA_DIFF"

tablaCARGA_INICIAL = f"{project_datasetbases}.BASEP_CARGA_INICIAL"

base_presupuesta_id = f"{project_datasetbases}.BASEP_FORECAST"
forecast_sscc = f'''{base_presupuesta_id}_SSCC'''

forecast_nomascara = f'''{base_presupuesta_id}_NOMASCARA'''


Validacion_Cuotas = f"{project_datasetbases}.Validacion_Cuotas"

Validacion_Prorrateo = f"{project_datasetbases}.Validacion_Prorrateo"

Validacion_Dist_Inmob = f"{project_datasetbases}.Validacion_Dist_Inmob"

###TABLAS CONCEPTOS DE NEGOCIO

tabla_base_hc = f'''{project_datasetbases}.HC_CN'''
tabla_base_cn = f'''{project_datasetbases}.CN'''
tabla_base_gencn = f'''{project_datasetbases}.GEN'''
tabla_base_gercn = f'''{project_datasetbases}.GER'''
vista_prorrycuot = f'''{project_datasetbases}.Resultados_Prorrateo'''
cedulablancaid = f'''{project_datasetbases}.CEDULAS'''

###TABLAS DESTINO STORED PROCEDURE BASE PRESUPUESTAL

nomina_calculo = f"{project_datasetbases}.BASE_MODELO_NOMINA_FIJO" ## MODELO ETZ SE HACE POR SP

tabla_atributos_calc_inic = f'''{project_datasetbases}.CALCULOS_INCIALES'''
tabla_base_cn_sp = f'''{project_datasetbases}.SP_CN'''
tabla_base_gencn_sp = f'''{project_datasetbases}.SP_GEN'''
tabla_base_gercn_sp = f'''{project_datasetbases}.SP_GER'''
tabla_base_hccn_sp = f'''{project_datasetbases}.SP_CNHC'''
tabla_base__mne_hccn_sp = f'''{project_datasetbases}.SP_MNE_CNHC'''
sp_CEDULAS = f'''{project_datasetbases}.SP_CEDULAS'''
base_presupuestal_sscc = f"{project_datasetbases}.{'BASEP_FORECAST_SSCC_PLAN'}"   ###PENDIENTE
nomina_etz_cn = f"{project_datasetbases}.SP_CNHC_ETZ" ## MODELO ETZ SE HACE POR SP
nomina_etz_cn_detalle = f"{project_datasetbases}.SP_CNHC_DETALLE_ETZ" ## MODELO ETZ SE HACE POR SP
tabla_base_prorrateo = f"{project_datasetbases}.SP_Resultados_Prorrateo" ## MODELO ETZ SE HACE POR SP

cedulas_id = f'''{project_datasetbases}.vCEDULAS'''
cedulas_Negocio9_id = f'''{project_datasetbases}.vCEDULAS_Negocio9'''
ventas_id = f'''{project_datasetbases}.vVENTAS'''
calc_id = f'''{project_datasetbases}.vCALCULOS'''

backupk_raw_data = f'''{project_datasetbases}.BACKUP_RAW_DATA'''

tabla_externos = f'''{project_datasetbases}.Carga_Externos_REV26'''

##BASES FINALES PARA REPORTEO

vista_forecast = f'''{project_datasetbases}.BASE_PRESUPUESTAL_COMPLETA_MES'''
log_audit_base = f'''{project_datasetbases}.LOG_AUDITORIA_BASE_PRESUPUESTAL'''
vista_consolidacion = f'''{project_datasetbases}.BASE_CONSOLIDADA'''

bbreport = f'''{project_datasetbases}.vBUILDING_BLOCK'''
hcreport = f'''{project_datasetbases}.vHC_r'''

tablahistventas = f"{project_datasetreportesbi}.Ventas_Historicos"
ventasreport = f'''{project_datasetreportesbi}.vVentas_r'''
erreport_Negocio8bi = f'''{project_datasetreportesbi}.vNegocio9'''

headcount_historicos = f'''{project_datasetbases}.HC_BASE_REPORTE'''

hcreportcalc = f'''{project_datasetbases}.vHC_r_calc'''

####Negocio8
divisiones_Negocio9 = ['Negocio8', 'Negocio9','Negocio10']
matriz_datosmaestros_Negocio9 = f"{project_datasetreportesbi}.Negocio9_Matriz_DIMENSION5caciones"
jerarquia_int_Negocio9 = f'''{project_datasetreportesbi}.JERARQUIA_Negocio9_INT'''
jerarquia_HFT_Negocio9 = f'''{project_datasetreportesbi}.Jerarquia_Negocio9_HFT'''
jerarquia_intsub_Negocio9 = f'''{project_datasetreportesbi}.Jerarquia_Negocio9_Int_subtotales'''
jerarquia_HFTvista_Negocio9 = f'''{project_datasetreportesbi}.Jerarquia_Expec_Negocio9_VISTA'''
ventas_Negocio9_trans = f'''{project_datasetbases}.VENTAS_Negocio9'''

Negocio9_onetimers = f'''{project_datasetreportesbi}.Negocio9_ONE_TIMERS'''

Negocio9_cedulas_sp = f'''{project_datasetreportesbi}.SP_Negocio9_CEDULAS'''

###CARGAS FUERA DE ARCHIVO

tablacarga1 =  f'''{project_datasetbases}.Carga_1'''
tablacarga2 =  f'''{project_datasetbases}.Carga_2'''
tablacarga3 =  f'''{project_datasetbases}.Carga_3'''
tablacargaxxxxxx = f'''{project_datasetbases}.XXXXX'''

##STORED PROCEDURES

stored_procedure_conceptos = f'''{project_datasetbases}.CONCEPTOS_sp''' #CON CECO
stored_procedure_cn_gen = f'''{project_datasetbases}.CN_GEN_sp''' # GENERALES HOJA ER
stored_procedure_cn_ger = f'''{project_datasetbases}.CN_GER_sp''' # SIN CECO
stored_procedure_cnhc = f'''{project_datasetbases}.CONCEPTOSHC_sp'''
stored_procedure_cedulas = f'''{project_datasetbases}.CEDULAS_sp'''
stored_procedure_DIFF_PRORRATEO = f'''{project_datasetbases}.DIFF_PRORRATEO_sp''' #???
stored_procedure_SSCC_PLAN = f'''{project_datasetbases}.SSCC_PLAN_sp''' #??? INCONCLUSO
stored_procedure_atributos = f'''{project_datasetbases}.ATRIBUTOS_sp'''
stored_procedure_MODELO_NOM_ETZ = f'''{project_datasetbases}.MODELO_NOM_ETZ_sp'''
stored_procedure_MODELO_NOM_ETZ_cn = f'''{project_datasetbases}.MODELO_NOM_ETZ_cn_sp'''
stored_procedure_BACKUP_BASE = f'''{project_datasetbases}.BASE_BACKUP_PRESUPUESTAL'''
stored_procedure_BACKUP_BASE_raw = f'''{project_datasetbases}.BACKUP_BASES_RAW'''

stored_procedure_Negocio9 = f'''{project_datasetbases}.CEDULASNegocio9_sp'''
storedprocedure_Negocio9_historicos = f'''{project_datasetbases}.HISTORICOS_Negocio9_sp'''
stored_procedure_nomina_Negocio9 = f"{project_datasetbases}.MODELO_NOMINA_Negocio9_sp"
stored_procedure_Negocio9_mascara = f"{project_datasetbases}.Negocio9_MASCARA"

#TABLAS STORED PROCEDURE ATRIBUTOS
stored_procedure_nominaprimerosmeses = f'''{base_primerosmesesrealesnom}_sp'''

In [ ]:
parametros_tabla = pd.DataFrame([monthname])
parametros_tabla.columns = ['MES_BASE_RACUM']

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    parametros_tabla, tabla_params, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tabla_params)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:
mesescompleto = [1,2,3,4,5,6,7,8,9,10,11,12]
mesescompleto2 = ["ENERO","FEBRERO","MARZO","ABRIL","MAYO","JUNIO","JULIO","AGOSTO","SEPTIEMBRE","OCTUBRE","NOVIEMBRE","DICIEMBRE"]
mesescompletos = {
    'Periodo': mesescompleto,
    'NOMBRE': mesescompleto2
}
mesescompleto_pd = pd.DataFrame(mesescompletos)
# mesescompleto_pd = mesescompleto_pd.rename(columns={mesescompleto_pd.columns[0]:"Periodo" })

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    mesescompleto_pd, PERIODOS, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(PERIODOS)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:
posiciones = ['Base','Director Corporativo','Director','Subdirector','Gerente','Coordinador','Jefatura','Personal Gral']
# mesescompleto2 = ["ENERO","FEBRERO","MARZO","ABRIL","MAYO","JUNIO","JULIO","AGOSTO","SEPTIEMBRE","OCTUBRE","NOVIEMBRE","DICIEMBRE"]
# mesescompletos = [mesescompleto,mesescompleto2]
posiciones_pd = pd.DataFrame(posiciones)
posiciones_pd = posiciones_pd.rename(columns={posiciones_pd.columns[0]:"Posiciones" })

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    posiciones_pd, POSICIONES, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(POSICIONES)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:
parametros_generales = [ [
"inflacionplan" ,
"inflacion_actual",
"aniobase" ,
"porc_inc" ,
"mesbaser" ,
"mesbasehc" ,
"mesinc" ,
"mesinc_vales" ,
"porc_inc_vales" ,
"porc_inc_vuniform" ,
"porc_inc_sig_anio" ,
"meses_bono" ,
"mes_captura" ,
"direcciones" ,
"porc_inc_seg_anio" ,
"direcciones_nomina",
"vales_via_planta",
"vales_via_planta_2",
"primera_version",
"versiones",
"divisiones_mne",
"isn_cdmx"
]
,
[
inflacionplan ,
inflacion_actual,
aniobase ,
porc_inc ,
mesbaser ,
mesbasehc ,
mesinc ,
mesinc_vales ,
porc_inc_vales ,
porc_inc_vuniform ,
porc_inc_sig_anio ,
meses_bono ,
mes_captura ,
', '.join(direcciones) ,
porc_inc_seg_anio ,
', '.join(direcciones_nomina),
  vales_via_planta,
  vales_via_planta_2,
  params['version'][0],
  # ', '.join(
      params['version'],
      # ),
  ', '.join(divisiones_mne),
isn_cdmx
  ]
]

parametros_generales_pd = pd.DataFrame(parametros_generales)
parametros_generales_pd.columns = parametros_generales_pd.loc[0,parametros_generales_pd.columns]
parametros_generales_pd = parametros_generales_pd[1:]


# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    parametros_generales_pd, tabla_pargen, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tabla_pargen)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

# **Permisos**

⭐ Asignación de permisos a datasets

(deprecated, Gobierno de datos)

In [ ]:
# ####PERMISOS
# # * "userByEmail" -- A single user or service account. For example "fred@example.com"
# # * "groupByEmail" -- A group of users. For example "example@googlegroups.com"

# email = "grupo google"

# sso = ss.worksheet("OWNERS")

# ownersdf = pd.DataFrame(sso.get("A1:A"+str(ssh.row_count)))
# owners = ownersdf[0].tolist()

# for i in range(0,len(datasetss)) :
#   print(datasetss[i])
#   dataset = client.get_dataset(datasetss[i])
#   # Set role to a one of the "Basic roles for datasets https://cloud.google.com/bigquery/docs/access-control-basic-roles#dataset-basic-roles

#   role = "READER"

#   entity_type = EntityTypes.GROUP_BY_EMAIL

#   entries = list(dataset.access_entries)
#   entries.append(
#       bigquery.AccessEntry(
#           role=role,
#           entity_type=entity_type,
#           entity_id=email,
#       )
#   )
#   dataset.access_entries = entries

#   dataset = client.update_dataset(dataset, ["access_entries"])  # Make an API request.
#   print(
#       "Updated dataset '{}' with modified user permissions.".format(projectsdatasets[i])
#   )

#   for j in range(0,len(owners)):
#     entity_type = EntityTypes.USER_BY_EMAIL
#     entries = list(dataset.access_entries)
#     entries.append(
#         bigquery.AccessEntry(
#             role="OWNER",
#             entity_type=entity_type,
#             entity_id=owners[j],
#         )
#     )
#     dataset.access_entries = entries

#     dataset = client.update_dataset(dataset, ["access_entries"])  # Make an API request.
#     print(
#         "Updated dataset '{}' with modified owner permissions.".format(projectsdatasets[i])
#     )

In [ ]:
# email = 'grupo google'

# for i in range(0,len(datasetss)) :
#   print(datasetss[i])
#   dataset = client.get_dataset(datasetss[i])
#   # Set role to a one of the "Basic roles for datasets https://cloud.google.com/bigquery/docs/access-control-basic-roles#dataset-basic-roles

#   role = "WRITER"

#   if "CN" in datasetss[i] or "CEDULAS" in datasetss[i] or "NOMINA" in datasetss[i] :
#     entity_type = EntityTypes.GROUP_BY_EMAIL

#     entries = list(dataset.access_entries)
#     entries.append(
#         bigquery.AccessEntry(
#             role=role,
#             entity_type=entity_type,
#             entity_id=email,
#         )
#     )
#     dataset.access_entries = entries

#     dataset = client.update_dataset(dataset, ["access_entries"])  # Make an API request.
#     print(
#         "Updated dataset '{}' with modified user permissions.".format(projectsdatasets[i])
#     )

#   else:
#     continue



In [ ]:
# email = "grupo google"


# for i in range(0,len(datasetss)) :
#     print(datasetss[i])
#     dataset = client.get_dataset(datasetss[i])

#     dataset.access_entries = [
#       entry for entry in dataset.access_entries
#       if entry.entity_id != email
#       ]

#     if "CN" in datasetss[i] or "CEDULAS" in datasetss[i] :


#         try:
#             # Update just the `access_entries` property of the dataset.
#             dataset = client.update_dataset(
#                 dataset,
#                 ["access_entries"],
#             )

#             # Notify user that the API call was successful.
#             full_dataset_id = f"{dataset.project}.{dataset.dataset_id}"
#             print(f"Revoked dataset access for '{email}' to ' dataset '{full_dataset_id}.'")

#         except :
#             print(
#                 f"Dataset '{dataset.dataset_id}' was modified remotely before this update. "
#                 "Fetch the latest version and retry."
#             )
#     else :
#         continue


# 🎁 **SENTENCIA REPROCESAR "UNA PARTE"**

In [ ]:
# filtro = f""" ( T0.CECO IN UNNEST ({MATRICIALES_CFO}) AND
# T0.DIMENSION1 NOT IN UNNEST ({direcciones}) )
# """


# filtro = f"""
# (
#   (T0.CECO IN UNNEST ({MATRICIALES_CFO})  AND
#   T0.DIMENSION1 NOT IN UNNEST ({direcciones})
#   )
# # OR
# #   (T0.DIMENSION1 IN UNNEST ({direcciones})
# #   )
# )
# """

filtro = f" T0.DIMENSION1 IS NOT NULL" ### MUEVE TOOOODOOOOO

#### TRANSFORMA NOMBRE DE CAMPOS ER A DM

filtroDM = filtro.replace("DIMENSION2",'DIMENSION2')
filtroDM = filtroDM.replace("DIMENSION3",'DIMENSION3')
filtroDM = filtroDM.replace("T0.CECO",'CECO')
filtroDM = filtroDM.replace("T0.DIMENSION1",'DIMENSION1')

filtroDM_d = filtro.replace("DIMENSION2","COALESCE(CECOS.DIMENSION2,CEBES.DIMENSION2)")
filtroDM_d = filtroDM_d.replace("DIMENSION3","COALESCE(CECOS.DIMENSION3,CEBES.DIMENSION3)")
filtroDM_d = filtroDM_d.replace("DIMENSION1","COALESCE(CECOS.DIMENSION1,CEBES.DIMENSION1)")

###PARA BORRAR DE LA TABLA RAW DATA, SOLO TIENE CECOS
scipt_delete = f"""
CREATE OR REPLACE TEMP TABLE filtered_ids AS (
  SELECT CECO as id, 'CECO' as type FROM `{CECOS}`
  WHERE {filtroDM}
  UNION ALL
  SELECT CEBE as id, 'CEBE' as type FROM `{CEBES}`
  WHERE {filtroDM.replace("CECO",'--CECO')}
);
"""




# ⛪ Históricos 🆗

⭐ Periodos cerrados y acumulados del ejercio anterior

❗ Incluye primeros meses reales nómina

In [ ]:
##ESTE ES UNA PRUEBA PARA EL CCOE

##### SCRIPT

queryhistoricos = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{base_hist}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {base_hist}

WITH BASE AS (
SELECT
CECO,CEBE,
T0.Cuenta, Ejercicio, Periodo, VERSION, 'HISTORICOS' as `ORIGEN`, SUM(IMPORTE)as IMPORTE

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA

WHERE (
##REALES ULTIMOS DOS EJERCICIOS CERRADOS
  ( T0.VERSION = 'Reales' AND (T0.Ejercicio IN ({aniobase}-1,{aniobase}-2)) ) OR
##PRESUPUESTO Y ME DEL ULTIMO EJERCICIO
( T0.VERSION IN ('PPTO', 'PPTO_V2') AND (T0.Ejercicio IN ({aniobase}-1)) ) OR
##PRESPUESTO DEL EJERCICIO ACTUAL VERSION PPTO
(  Ejercicio = {aniobase}  AND T0.Version = 'PPTO' )
)

AND {filtro}

GROUP BY 1,2,3,4,5,6,7 )

SELECT *

FROM BASE

WHERE IMPORTE <> 0

'''

client.query(queryhistoricos).result()
print("Creado ",base_hist)

In [ ]:
######ACUMULADOS EJERCICIO ACTUAL

##### SCRIPT

query_hacum = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{base_hist_acum}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {base_hist_acum}
(
WITH BASE AS (

##ACUM DEL EJERCICIO ACTUAL

SELECT

CECO,CEBE,
Cuenta, Ejercicio, Periodo,
CASE WHEN T0.VERSION = 'Reales' THEN 'RACUM'
    WHEN T0.Version= 'PPTO' THEN 'PACUM' END as `Version`,
'HISTORICOS_ACUM' as `ORIGEN`, SUM(IMPORTE)as IMPORTE

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0

WHERE (T0.VERSION IN ('Reales','PPTO') AND (T0.Ejercicio IN ({aniobase})) )
AND {filtro}

GROUP BY 1,2,3,4,5,6,7
),

CREACION AS (
SELECT CUENTA, CECO, CEBE,
Ejercicio||"_"||CASE WHEN T1.Periodo = 1 THEN 'ENERO'
     WHEN T1.Periodo = 2 THEN 'FEBRERO'
     WHEN T1.Periodo = 3 THEN 'MARZO'
     WHEN T1.Periodo = 4 THEN 'ABRIL'
     WHEN T1.Periodo = 5 THEN 'MAYO'
     WHEN T1.Periodo = 6 THEN 'JUNIO'
     WHEN T1.Periodo = 7 THEN 'JULIO'
     WHEN T1.Periodo = 8 THEN 'AGOSTO'
     WHEN T1.Periodo = 9 THEN 'SEPTIEMBRE'
     WHEN T1.Periodo = 10 THEN 'OCTUBRE'
     WHEN T1.Periodo = 11 THEN 'NOVIEMBRE'
     WHEN T1.Periodo = 12 THEN 'DICIEMBRE'
     END as Ejercicio ,
Version, ORIGEN,

CASE WHEN T0.Periodo <= T1.Periodo THEN Importe ELSE 0 END as IMPORTE

FROM BASE T0
CROSS JOIN {PERIODOS} T1
)

SELECT CECO,CEBE,
Cuenta, Ejercicio, 0 as Periodo, VERSION, `ORIGEN`, SUM(IMPORTE)as IMPORTE

FROM CREACION

WHERE IMPORTE <> 0

GROUP BY 1,2,3,4,5,6,7
)
'''
client.query(query_hacum).result()
print("Creado ",base_hist_acum)

❗❗❗❗❗❗❗   Históricos Negocio8

In [ ]:
query_hNegocio9 = f'''CREATE OR REPLACE TABLE {base_hist_Negocio9} AS
(
##REALES ULTIMOS DOS EJERCICIOS CERRADOS

SELECT T0.`Cuenta`, `Ejercicio`,
T0.DIMENSION1, T0.DIMENSION2 as DIMENSION2,
CASE WHEN Marca IN ('YYYYYY','ZZZZZZ') THEN 'AAAAAA' ELSE DIMENSION5 END ||
INITCAP(T0.DIMENSION1) as IDUB,
T0.Periodo,
T0.VERSION,
'HISTORICOS' as `ORIGEN`, FUNCION
,SUM(IMPORTE)as IMPORTE

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA

WHERE T0.VERSION = 'Reales' AND (T0.Ejercicio IN ({aniobase}-1,{aniobase}-2,{aniobase}-3)) AND DIMENSION1 IN UNNEST ({divisiones_Negocio9})
GROUP BY 1,2,3,4,5,6,7,8,9

UNION ALL

##PRESUPUESTO Y ME DEL ULTIMO EJERCICIO

SELECT T0.`Cuenta`, `Ejercicio`,
T0.DIMENSION1, T0.DIMENSION2 as DIMENSION2,
CASE WHEN Marca IN ('YYYYYY','ZZZZZZ') THEN 'AAAAAA' ELSE DIMENSION5 END ||
INITCAP(T0.DIMENSION1) as IDUB,
T0.Periodo,
T0.VERSION,
'HISTORICOS' as `ORIGEN`, FUNCION
,SUM(IMPORTE)as IMPORTE

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA

WHERE T0.VERSION IN ('PPTO', 'PPTO_V2') AND (T0.Ejercicio IN ({aniobase}-1)) AND DIMENSION1 IN UNNEST ({divisiones_Negocio9})

GROUP BY 1,2,3,4,5,6,7,8,9

UNION ALL

##EJERCICIO ACTUAL

SELECT T0.`Cuenta`, `Ejercicio`,
T0.DIMENSION1, T0.DIMENSION2 as DIMENSION2,
CASE WHEN Marca IN ('YYYYYY','ZZZZZZ') THEN 'AAAAAA' ELSE DIMENSION5  END ||
INITCAP(T0.DIMENSION1) as IDUB,
T0.Periodo,
T0.VERSION,
'HISTORICOS' as `ORIGEN`, FUNCION
,SUM(IMPORTE)as IMPORTE

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA

WHERE T0.VERSION IN ('Reales','PPTO','PPTO_V0') AND (T0.Ejercicio IN ({aniobase})) AND DIMENSION1 IN UNNEST ({divisiones_Negocio9})

GROUP BY 1,2,3,4,5,6,7,8,9
)
'''

client.query(query_hNegocio9).result()
print("Creado ",base_hist_Negocio9)

# ⚡ **Calculos Iniciales**

# Sueldos ☝NÓMINA (GASTOS DE PERSONAL)

❗❗❗ Cuidado en el reproceso, afecta FORECAST

⭐ Cuando se modifique sueldos, se debe volver a ejecutar prestaciones (con su diferencia) y BONO_ESP, porque se realiza sobre Sueldos


Base mes elegido se replica a los meses subsecuentes.

Incremento anual (porcentaje indicado, en mes idicado)

Noviembre Aguinaldo, enero prima vacional (duplica el valor de cuenta Liverflex) --se deben quitar montos de rendimientos

Agosto vale de uniforme confidencial se provisiona de enero a julio se registra en vales de despensa

Vales de despensa montos específicos

Renta gasolina y mantenimiento cada de negocio lo presupuesta

QUITAR RECLA Y PROVISION DE RENDIMIENTOS DE LA 41 EN FEBRERO PARA EL "ARRASTRE"
Paso 1 descargar info de controlling
Paso 2 cargar en tabla para unirlo.
Paso 3 actualizar código.

In [ ]:
##### SCRIPT

query_nomina = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablanomina}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablanomina}

(
SELECT
  T1.Cuenta_Reporte AS Cuenta,
  T0.CeCo,
  T0.CeBe,

  CASE
    WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase}
    ELSE {aniobase} +1
  END AS Ejercicio,

  CAST(
    T2.MES
     AS INT64) AS Periodo,
  T2.Version,

  {mesinc} AS MES_INC,
  {porc_inc} AS PORC_INC,
  T0.Periodo AS MES_BASE,

  (
    CASE
      WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1')
        AND T2.MES < T0.PERIODO
        THEN 0
      WHEN T2.MES < {mesinc}
           AND T2.VERSION = 'PPTO_V0'
        THEN (1 + {porc_inc})

      WHEN T2.MES >= {mesinc}
           AND T2.VERSION IN ('PPTO_V2','PPTO_V1')
        THEN (1 + {porc_inc})

      WHEN T2.MES >= {mesinc}
           AND T2.VERSION = 'PPTO_V0'
        THEN (1 + {porc_inc}) * (1 + {porc_inc})

      ELSE 1
    END
  )
  *
  CASE
    WHEN T1.Cuenta_Reporte = '51000041'
         AND T2.MES IN (1,11)
         AND T0.PERIODO NOT IN (1,11) ---EL MES BASE ES ENERO O NOVIEMBRE
      THEN 2
    WHEN T1.Cuenta_Reporte = '51000041'
         AND T2.MES NOT IN (1,11)
         AND T0.PERIODO IN (1,11) ---EL MES BASE ES ENERO O NOVIEMBRE
      THEN 0.5
    ELSE 1
  END
  *
  SUM(IFNULL(T0.Importe, 0)) AS Importe,

  'NOM_SUE' AS ORIGEN

FROM {ER} T0

LEFT JOIN {CUENTAS_NOM} T1
  ON T0.Cuenta = T1.Cuenta_origen

CROSS JOIN (
  SELECT *
  FROM {MESESPRES}
  WHERE Version IN UNNEST({params['version']})
) T2

WHERE T1.Clasificacion LIKE '%SUELDOS%'
  AND T0.VERSION = 'Reales'
  AND T0.Ejercicio = {aniobasehc}
  AND T0.Periodo = {mesbasehc}
  AND T2.MES >= {mesbasehc}
  AND {filtro}

GROUP BY
  T0.CeCo,
  T0.CeBe,
  T2.MES,
  T2.Version,
  T1.Cuenta_Reporte,
  T0.Periodo

UNION ALL
-------------------------------------------------------

SELECT
  T1.Cuenta_Reporte AS Cuenta,
  T0.CeCo,
  T0.CeBe,

  SAFE_CAST(T0.Ejercicio AS INT64) as Ejercicio,

  SAFE_CAST(T0.Periodo AS INT64) as PERIODO,
  '{params['version'][0]}' as Version,

  {mesinc} AS MES_INC,
  {porc_inc} AS PORC_INC,
  T0.Periodo AS MES_BASE,

  SUM(IFNULL(T0.Importe, 0)) AS Importe,

  'NOM_SUE' AS ORIGEN

FROM {ER} T0

LEFT JOIN {CUENTAS_NOM} T1
  ON T0.Cuenta = T1.Cuenta_origen

WHERE T1.Clasificacion LIKE '%SUELDOS%'
  AND T0.VERSION = 'Reales'
  AND T0.Ejercicio = {aniobasehc}
  AND T0.Periodo < {mesbasehc}
  AND {filtro}

GROUP BY
  1,2,3,4,5,6,7,8,9
)
'''
client.query(query_nomina).result()
print("Creado ",tablanomina)

# Vales de Despensa☝NÓMINA (GASTOS DE PERSONAL)
❗❗❗ Cuidado en el reproceso, afecta FORECAST

Depende de headcount compartido por people analytics


In [ ]:
#CARGA VALES DE DESPENSA
rutavales = "/content/drive/Shareddrives/XXXXX"
archivo_vales = "Matriz  Vales de despensa"

vales_excel = pd.read_excel(rutavales+archivo_vales+".xlsx", sheet_name='Matriz', header=6, dtype=str)

vales_excel = vales_excel.loc[:,vales_excel.columns != 'Texto]
vales_excel = pd.melt(vales_excel, id_vars=['DivP'], value_vars=['1C', '1H', '1D', '1E', '1F', '1G', '1V', '2G', '2V', '4G',
       '4V', '4J', 'Especiales 1', 'Especiales 2'])
vales_excel = vales_excel.rename(columns={'DivP':'DIMENSION5','variable':'DIMENSION_CH','value':'Importe_vales'})
vales_excel['Importe_vales'] = vales_excel['Importe_vales'].replace("*",0).astype(float)
vales_excel['DIMENSION5'] = vales_excel['DIMENSION5'].str.zfill(4)



# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    vales_excel, tablavales, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tablavales)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:
#CARGA VALES DE UNIFORME
rutavales_un = "/content/drive/Shareddrives/XXXXX/"
archivo_vales_un = "Beneficios 2025"

vales_un_excel = pd.read_excel(rutavales_un+archivo_vales_un+".xlsx", sheet_name='Beneficios 2025', header=32, dtype=str)

vales_un_excel.columns

vales_un_excel = vales_un_excel.loc[:,['Unnamed: 1','Monto']]
vales_un_excel = vales_un_excel.rename(columns={'Unnamed: 1':'Posicion'})


# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    vales_un_excel, tablavales_un, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tablavales_un)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:
#CARGA VALES DE DESPENSA V2
rutavales = "/content/drive/Shareddrives/XXXXX/"
archivo_vales = "VALES_V2"
tablavales2_actual= f'''{tablavales}_actual'''

vales_excel = pd.read_excel(rutavales+archivo_vales+".xlsx", sheet_name='Matriz', header=5, dtype=str)

vales_excel = vales_excel.loc[:,vales_excel.columns != 'Texto de división de personal']

vales_excel.columns
vales_excel = pd.melt(vales_excel, id_vars=['DivP'], value_vars=['1C.1', '1H.1', '1D.1',
       '1E.1', '1F.1', '1G.1', '1V.1', '2G.1', '3G.1', '2V.1', '3V.1', '4V.1',
       '5V.1', '4J.1', 'Especiales 1.1', 'Especiales 2.1'])
vales_excel = vales_excel.rename(columns={'DivP':'DIMENSION5','variable':'DIMENSION_CH','value':'Importe_vales'})
vales_excel['DIMENSION_CH'] = vales_excel['DIMENSION_CH'].str.split(".").str[0]
vales_excel['Importe_vales'] = vales_excel['Importe_vales'].replace("*",0).astype(float)
vales_excel['DIMENSION5'] = vales_excel['DIMENSION5'].str.zfill(4)



# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    vales_excel, tablavales2_actual, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tablavales2_actual)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:
#CARGA VALES DE DESPENSA V2
rutavales = "/content/drive/Shareddrives/XXXXXXX/"
archivo_vales = "VALES_DESPENSA_22"
tablavales2= f'''{tablavales}_2'''

vales_excel = pd.read_excel(rutavales+archivo_vales+".xlsx", sheet_name='Matriz', header=5, dtype=str)

vales_excel = vales_excel.loc[:,vales_excel.columns != 'Texto']

vales_excel.columns
vales_excel = pd.melt(vales_excel, id_vars=['DivP'], value_vars=['1C', '1H', '1D', '1E', '1F', '1G', '1V', '2G', '3G', '2V',
       '3V', '4V', '5V', '4J', 'Especiales 1', 'Especiales 2'])
vales_excel = vales_excel.rename(columns={'DivP':'DIMENSION5','variable':'AREA_PERSONAL','value':'Importe_vales'})
vales_excel['Importe_vales'] = vales_excel['Importe_vales'].replace("*",0).astype(float)
vales_excel['DIMENSION5'] = vales_excel['DIMENSION5'].str.zfill(4)



# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    vales_excel, tablavales2, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tablavales2)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

In [ ]:

##### SCRIPT

# filtro = filtro.replace("T0",'T7')
# filtroDM = filtroDM.replace("T0",'T7')

query_vales = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablanomina_vales}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablanomina_vales}

WITH CALCULO AS (
SELECT "51000032" as Cuenta, T7.CeCo, T7.CeBe,

CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio`,

CAST(T2.MES AS INT64) as `Periodo`,

T2.Version,
{mesinc_vales} as `MES_INC`,
0 as `PORC_INC`, 0 as `MES_BASE`,

COUNT(T5.num_per) as Personas, T5.AREA_PERSONAL,

T5.Posici__n as POSICION,

--VALES UNIFORME
IFNULL(CASE WHEN (T7.DIMENSION1 = 'Negocio11' AND T2.MES <= 7 --AND T5.Posici__n IN ('Subdirector','Director','Director Corporativo') )
OR (T2.MES <= 7 ) )
          THEN COUNT(T5.num_per) * CAST(T6.MONTO AS INT64)/7/1.16
ELSE 0 END,0) --VALES DE UNIFORME***** ABRIR POR HEADCOUNT POSICION --
+
--VALES DE DESPENSA
IFNULL(COUNT(T5.num_per) *
CASE WHEN T2.MES >= {mesinc_vales} THEN (1+{porc_inc_vales}) ELSE 1 END * -- INCREMENTO
--CASE WHEN DATE_DIFF(DATE(CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END, SAFE_CAST(T2.MES AS INT64),1), T1.FECHA , MONTH) <=3  THEN
     -- CASE WHEN T5.GRUPO_PERSONAL IN ('D','C') THEN {vales_via_planta}
     --     WHEN T5.AREA_PERSONAL IN ('2G','2V') THEN {vales_via_planta_2}
     -- END
    -- ELSE
    T8.Importe_vales
    -- END
,0)
as Importe,
IFNULL(

  CASE WHEN T2.MES >= {mesinc_vales} THEN (1+{porc_inc_vales}) ELSE 1 END * -- INCREMENTO
  --CASE WHEN DATE_DIFF(DATE(CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END, SAFE_CAST(T2.MES AS INT64),1), T1.FECHA , MONTH) <=3  THEN
       -- CASE WHEN T5.GRUPO_PERSONAL IN ('D','C') THEN {vales_via_planta}
       --    WHEN T5.AREA_PERSONAL IN ('2G','2V') THEN {vales_via_planta_2}
       -- END
      --ELSE
      T8.Importe_vales
      -- END

  ,0) AS Importe_vales,
IFNULL(CAST(T6.MONTO AS INT64)/1.16,0) AS MONTO

, 'NOM_VAL' as `ORIGEN`

FROM {headcountnew} T5
LEFT JOIN `{project_id}.mus_qas_drv_datos_maestros.HEADCOUNTS` T1 ON T5.num_per=CAST(T1.Numero_empleado AS INT64)
INNER JOIN {CECOS} T7 ON REPEAT("0",10-LENGTH(CAST(T5.CECO AS STRING)))||T5.CECO=T7.CECO
LEFT JOIN {tablavales_un} T6 ON
        CASE WHEN T5.Posici__n IN ('Director Corporativo', 'Director','Subdirector')
             THEN 'Director / Subdirector' ELSE T5.Posici__n END = T6.POSICION
LEFT JOIN {tablavales} T8 ON T5.AREA_PERSONAL=T8.AREA_PERSONAL AND
                            T1.Div_per=T8.DIMENSION5

##SE REPLICA PARA TODOS LOS MESES A PRESUPUESTAR
CROSS JOIN (
  SELECT *
  FROM `{MESESPRES}`
  WHERE Version IN UNNEST ({params['version']})
  ) T2

WHERE extract(month from T5.anio) =
{mesbasehc}
 AND T2.MES > {mesbasehc}

AND {filtro.replace("T0.","T7.")}

GROUP BY 2,3,4,5,6,T2.MES,T6.MONTO,T8.Importe_vales, T5.AREA_PERSONAL,T5.Posici__n,T5.GRUPO_PERSONAL,T7.DIMENSION1, T1.FECHA

)

SELECT * EXCEPT(IMPORTE,PERSONAS), SUM(PERSONAS) as PERSONAS, SUM(IMPORTE) as IMPORTE

FROM CALCULO

GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14

UNION ALL

SELECT Cuenta,CeCo,CeBe,
SAFE_CAST(EJERCICIO as INT64) as Ejercicio,
SAFE_CAST(Periodo as INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as MES_INC,
0 as PORC_INC,
0 as MES_BASE,
"" as AREA_PERSONAL,
"" as POSICION,
0 as Importe_vales,
0 as MONTO,
'NOM_VAL' as ORIGEN,
0 as PERSONAS,
IMPORTE

FROM {ER}

WHERE CUENTA = '51000032' and ejercicio = {aniobase} and version = 'Reales' and PERIODO <= {mesbaser}

'''
client.query(query_vales).result()
print("Creado ",tablanomina_vales)

# Base Nómina Prestaciones☝NÓMINA (GASTOS DE PERSONAL)
❗❗❗ Cuidado en el reproceso, afecta FORECAST

⭐ Se calcula a partir de Nómina, si se corre nómina, debe ejecutarse también este rubro

♦ El cálculo de prestaciones toma el sueldo mensual, y le aplica el porcentaje del último año (prestación / sueldo)

Se realiza en dos partes, primero calcula los porcentajes, la segunda celda lo carga a BQ y la tercera realiza el cálculo de prestaciones y lo carga

QUITAR DE 2024 REALES LOS 40 MDP DE LA 41 -PARA CALCULO % PRESTACIONES
Paso 1 cargar los montos de diferencia en una tabla, para poder unirlos a los históricos.
Paso 2 actualizar código.


Generar diferencia anual vs PLAN 4+PPTO_V1 8 (en mayo)

Generar comparativo vales vs PLAN

Iterativo, primer nivel CECO, si no existe DIR ARE, si no existe DIR COR, si no existe DIV NEG; no contempla vales de despensa.

**CALCULO PRESTACIONES PF ANTERIOR**

⭐ Nuevo cálculo de prestaciones.

Reglas en: link de la política


❗❗❗❗ **Porcentajes anio anterior**

Cálculo de porcentajes base año anterior

In [ ]:
# ##### SCRIPT

# ########DIFICIL

# # ----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
# # {scipt_delete}

# # DELETE FROM `{tablapres_disanioanterior}` t
# # WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
# #    OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

# query_anioant = f'''

# ---UNA VEZ BORRADOS, INSERTAR EL CALCULO

# INSERT INTO {tablapres_disanioanterior}

# WITH

# BPC AS (
#   SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion
#   FROM `{ER}` T0
#   INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
#   LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen
#   LEFT JOIN {DIMENSION5S_CORP} DIMENSION5S_CORP ON T0..DIMENSION5=DIMENSION5S_CORP.DIMENSION5 AND DIMENSION5S_CORP.CONSIDERAR = '' AND T0.CUENTA IN ('51000053','51000408') -- SOLO PARA ISN

#   WHERE Ejercicio IN ( {aniobase-1} ) AND VERSION IN ('Reales') AND
#   (
#     Clasificacion LIKE '%SUELDO%'
#     OR
#     REGLA3 = 'PORCENTAJE REAL ULTIMO AÑO FLAT BASE ANUAL'
#     )
#     AND {filtro}
#     AND DIMENSION5S_CORP.DIMENSION5 IS NULL
# ),

# BASE_PRESTACIONES_CECO_CEBE AS
# (
#   SELECT Cuenta, CeCo, CeBe, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE UPPER(REGLA3) LIKE ('%PORCENTAJE%') --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, CeCo, CeBe
# ),

# ##AGRUPACION MONTO CUENTA CECO CEBE -SUELDOS

#   BASE_SUELDO_CECO_CEBE_1 AS
#   (
#     SELECT CeCo, CeBe, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY CeCo, CeBe


#   ),

#     BASE_SUELDO_CECO_CEBE AS
#   (
#     SELECT T0.CeCo, T0.CeBe, SUM(T0.Importe) as `Importe`

#     FROM BASE_SUELDO_CECO_CEBE_1 t0

#     GROUP BY T0.CeCo, T0.CeBe
#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_CECO AS (

#     SELECT T10.Cuenta, T10.CeCo, T10.CeBe,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDO_CECO_CEBE T11
#     LEFT JOIN BASE_PRESTACIONES_CECO_CEBE T10 ON T10.CeCo=T11.CeCo AND T10.CeBe=T11.CeBe

# ),

# ##DISTRIBUCION AREA

# ##AGRUPACION MONTO MES CUENTA DIMENSION3 -PRESTACIONES

#   BASE_PRESTACIONES_DIMENSION3 AS

#   (SELECT Cuenta, DIMENSION3, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%PRESTACIONES%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, DIMENSION3 ),

# ##AGRUPACION MONTO CUENTA CECO CEBE -PRESTACIONES

#   BASE_SUELDOS_DIMENSION3 AS
#   (
#     SELECT DIMENSION3, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY DIMENSION3

#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_DIMENSION3 AS (

#     SELECT T10.Cuenta, T10.DIMENSION3,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDOS_DIMENSION3 T11
#     LEFT JOIN BASE_PRESTACIONES_DIMENSION3 T10 ON T10.DIMENSION3=T11.DIMENSION3

# ),

# ##DISTRIBUCION CORP

# ##AGRUPACION MONTO MES CUENTA DIMENSION2 -PRESTACIONES

#   BASE_PRESTACIONES_DIMENSION2 AS

# (SELECT Cuenta, DIMENSION2, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%PRESTACIONES%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, DIMENSION2),

# ##AGRUPACION MONTO CUENTA CECO CEBE -PRESTACIONES

#   BASE_SUELDOS_DIMENSION2 AS
#   (
#     SELECT DIMENSION2, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY DIMENSION2

#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_DIMENSION2 AS (

#   SELECT T10.Cuenta, T10.DIMENSION2,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDOS_DIMENSION2 T11
#     LEFT JOIN BASE_PRESTACIONES_DIMENSION2 T10 ON T10.DIMENSION2=T11.DIMENSION2

# ),

# ##DISTRIBUCION DIVISION DE NEGOCIO

# ##AGRUPACION MONTO MES CUENTA DIMENSION1 -PRESTACIONES

#   BASE_PRESTACIONES_DIMENSION1 AS

# (SELECT Cuenta, DIMENSION1, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%PRESTACIONES%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, DIMENSION1 ),

# ##AGRUPACION MONTO CUENTA CECO CEBE -PRESTACIONES

#   BASE_SUELDOS_DIMENSION1 AS
#   (
#     SELECT DIMENSION1, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY DIMENSION1

#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_DIMENSION1 AS (
#     SELECT T10.Cuenta, T10.DIMENSION1,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDOS_DIMENSION1 T11
#     LEFT JOIN BASE_PRESTACIONES_DIMENSION1 T10 ON T10.DIMENSION1=T11.DIMENSION1
# ),

# --- UNIR LAS DISTRIBUCIONES

#   DISTRIBUCIONES AS (
#   SELECT Cuenta, CeCo, CeBe, '' as DIR_V, "CECO" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_CECO
#   UNION ALL
#   SELECT Cuenta, '' as CeCo, '' as CeBe, DIMENSION3 as DIR_V, "DIMENSION3" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_DIMENSION3
#   UNION ALL
#   SELECT Cuenta, '' as CeCo, '' as CeBe, DIMENSION2 as DIR_V, "DIMENSION2" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_DIMENSION2
#   UNION ALL
#   SELECT Cuenta, '' as CeCo, '' as CeBe, DIMENSION1 as DIR_V, "DIMENSION1" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_DIMENSION1

# )

# SELECT *
# FROM DISTRIBUCIONES
# WHERE IFNULL(CUENTA,"")<>""
# '''

# client.query(query_anioant).result()
# print("Creado ",tablapres_disanioanterior)

❗❗❗❗ **Porcentajes roll year**

Cálculo de porcentajes base año actual, meses disponibles + los que faltan del año anterior

In [ ]:

# query = f'''
# DECLARE CECOS STRING;
# DECLARE CEBES STRING;
# DECLARE DELETESENTENCE STRING;

# ----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS

# {scipt_delete}

# SET DELETESENTENCE = FORMAT("""

# DELETE FROM {tablapres_disanioactual}
# WHERE (CUENTA LIKE '4%%' AND CEBE IN (%s)) OR (CUENTA LIKE '5%%' AND CECO IN (%s)) OR ifnull(CECO,"") = "" ------no aplica para cecos
# """,
# CEBES,CECOS)
# ;

# EXECUTE IMMEDIATE DELETESENTENCE;

# ---UNA VEZ BORRADOS, INSERTAR EL CALCULO

# INSERT INTO {tablapres_disanioactual}

# WITH

# BPC AS (
#   SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion
#   FROM `{ER}` T0
#   INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
#   LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen
#   LEFT JOIN {DIMENSION5S_CORP} DIMENSION5S_CORP ON T0.DIMENSION5=DIMENSION5S_CORP.DIMENSION5 AND DIMENSION5S_CORP.CONSIDERAR = '' AND T0.CUENTA IN ('51000053','51000408') -- SOLO PARA ISN

#   WHERE (
#     (Ejercicio IN ( {aniobasehc} ) AND Periodo <= {mesbaser}) --AÑO ACTUAL
#     OR
#     (Ejercicio IN ( {aniobasehc-1} ) AND Periodo > {mesbaser}) --AÑO ANTERIOR
#     )
#   AND VERSION IN ('Reales')
#   AND (Clasificacion LIKE '%SUELDO%' OR REGLA3 IN
#     ('PORCENTAJE REAL BASE POR DEFINIR_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)',
#      'PORCENTAJE REAL BASE PRIMER SEMESTRE ACTUAL_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)')
#      )
#   AND DIMENSION5S_CORP.DIMENSION5 IS NULL
#   AND {filtro}
# ),

# BASE_PRESTACIONES_CECO_CEBE AS
# (
#   SELECT Cuenta, CeCo, CeBe, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE UPPER(REGLA3) LIKE ('%PORCENTAJE%') --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, CeCo, CeBe
# ),

# ##AGRUPACION MONTO CUENTA CECO CEBE -SUELDOS

#   BASE_SUELDO_CECO_CEBE_1 AS
#   (
#     SELECT CeCo, CeBe, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY CeCo, CeBe

#   ),

#     BASE_SUELDO_CECO_CEBE AS
#   (
#     SELECT T0.CeCo, T0.CeBe, SUM(T0.Importe) as `Importe`

#     FROM BASE_SUELDO_CECO_CEBE_1 t0

#     GROUP BY T0.CeCo, T0.CeBe
#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_CECO AS (

#     SELECT T10.Cuenta, T10.CeCo, T10.CeBe,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDO_CECO_CEBE T11
#     LEFT JOIN BASE_PRESTACIONES_CECO_CEBE T10 ON T10.CeCo=T11.CeCo AND T10.CeBe=T11.CeBe

# ),

# ##DISTRIBUCION AREA

# ##AGRUPACION MONTO MES CUENTA DIMENSION3 -PRESTACIONES

#   BASE_PRESTACIONES_DIMENSION3 AS

#   (SELECT Cuenta, DIMENSION3, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%PRESTACIONES%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, DIMENSION3 ),

# ##AGRUPACION MONTO CUENTA CECO CEBE -PRESTACIONES

#   BASE_SUELDOS_DIMENSION3 AS
#   (
#     SELECT DIMENSION3, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY DIMENSION3
#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_DIMENSION3 AS (

#     SELECT T10.Cuenta, T10.DIMENSION3,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDOS_DIMENSION3 T11
#     LEFT JOIN BASE_PRESTACIONES_DIMENSION3 T10 ON T10.DIMENSION3=T11.DIMENSION3

# ),

# ##DISTRIBUCION CORP

# ##AGRUPACION MONTO MES CUENTA DIMENSION2 -PRESTACIONES

#   BASE_PRESTACIONES_DIMENSION2 AS

# (SELECT Cuenta, DIMENSION2, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%PRESTACIONES%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, DIMENSION2),

# ##AGRUPACION MONTO CUENTA CECO CEBE -PRESTACIONES

#   BASE_SUELDOS_DIMENSION2 AS
#   (
#     SELECT DIMENSION2, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY DIMENSION2
#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_DIMENSION2 AS (

#   SELECT T10.Cuenta, T10.DIMENSION2,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDOS_DIMENSION2 T11
#     LEFT JOIN BASE_PRESTACIONES_DIMENSION2 T10 ON T10.DIMENSION2=T11.DIMENSION2

# ),

# ##DISTRIBUCION DIVISION DE NEGOCIO

# ##AGRUPACION MONTO MES CUENTA DIMENSION1 -PRESTACIONES

#   BASE_PRESTACIONES_DIMENSION1 AS

# (SELECT Cuenta, DIMENSION1, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%PRESTACIONES%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY Cuenta, DIMENSION1 ),

# ##AGRUPACION MONTO CUENTA CECO CEBE -PRESTACIONES

#   BASE_SUELDOS_DIMENSION1 AS
#   (
#     SELECT DIMENSION1, SUM(Importe) as `Importe`

#   FROM BPC

#   WHERE Clasificacion LIKE '%SUELDO%'
#     --AND Importe > 0 --SOLO POSITIVOS
#   GROUP BY DIMENSION1
#   ),

# ##CALCULAR DISTRIBUCION PRESTACIONES ANUAL

#   DISTRIBUCION_ANIO_DIMENSION1 AS (
#     SELECT T10.Cuenta, T10.DIMENSION1,

#     CASE WHEN ROUND(T11.`Importe`,0) = 0 THEN 0 ELSE
#     T10.Importe/T11.Importe END as `Importe`, T11.Importe`SUELDOS`, T10.Importe`PRESTACION`

#     FROM BASE_SUELDOS_DIMENSION1 T11
#     LEFT JOIN BASE_PRESTACIONES_DIMENSION1 T10 ON T10.DIMENSION1=T11.DIMENSION1
# ),

# --- UNIR LAS DISTRIBUCIONES

#   DISTRIBUCIONES AS (
#   SELECT Cuenta, CeCo, CeBe, '' as DIR_V, "CECO" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_CECO
#   UNION ALL
#   SELECT Cuenta, '' as CeCo, '' as CeBe, DIMENSION3 as DIR_V, "DIMENSION3" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_DIMENSION3
#   UNION ALL
#   SELECT Cuenta, '' as CeCo, '' as CeBe, DIMENSION2 as DIR_V, "DIMENSION2" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_DIMENSION2
#   UNION ALL
#   SELECT Cuenta, '' as CeCo, '' as CeBe, DIMENSION1 as DIR_V, "DIMENSION1" as `DIMENSION6`, Importe
#   FROM DISTRIBUCION_ANIO_DIMENSION1

# )

# SELECT *
# FROM DISTRIBUCIONES
# WHERE IFNULL(CUENTA,"")<>""

# '''

# client.query(query).result()
# print("Creado ",tablapres_disanioactual)

❗❗❗ ⭐ PORCENTAJE REAL ULTIMO AÑO FLAT BASE ANUAL

In [ ]:
##### SCRIPT

querydist = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargaporc_ant}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablacargaporc_ant}

WITH BASE_SUELDOS_PR AS (
  SELECT T0.CeCo, T0.CeBe, T0.`Ejercicio`, T0.Periodo, T0.Version, T1.DIMENSION1, T1.DIMENSION2, T1.DIMENSION3, SUM(T0.Importe) as `Importe`
  FROM {tablanomina} T0
  --ATRIBUTOS CECO
  LEFT JOIN `{CECOS}` T1 ON T0.CECO=T1.CECO
  WHERE {filtro.replace("T0","T1")}
  GROUP BY T0.Ejercicio, T0.Periodo, T0.Version, T0.CeCo, T0.CeBe, T1.DIMENSION1, T1.DIMENSION2, T1.DIMENSION3
  ),

  CECOS AS (

-- DISTRIBUCION CECOS
  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN PRESTACIONES
  INNER JOIN {tablapres_disanioanterior} T12
    ON T11.CECO=T12.CECO

),

  AREA AS (
  -- DISTRIBUCION DIR AREA

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN {tablapres_disanioanterior} T12
    ON T11.DIMENSION3=T12.DIR_V AND DIMENSION6 = "DIMENSION3"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO

  WHERE CECOS.CUENTA IS NULL
  ),

  CORP AS (
  -- DISTRIBUCION DIR CORP

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN {tablapres_disanioanterior} T12
    ON T11.DIMENSION2=T12.DIR_V AND DIMENSION6 = "DIMENSION2"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA
   ON T11.DIMENSION3=AREA.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL
  ),

 DIV AS (
  -- DISTRIBUCION DIV NEG

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  LEFT JOIN {tablapres_disanioanterior} T12
    ON T11.DIMENSION1=T12.DIR_V AND DIMENSION6 = "DIMENSION1"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA
   ON T11.DIMENSION3=AREA.DIR_V
  LEFT JOIN CORP
   ON T11.DIMENSION2=CORP.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL AND CORP.CUENTA IS NULL
 ),

 FINAL AS (

 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CECOS
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM AREA
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CORP
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM DIV
  )

SELECT Cuenta, CeCo, CeBe, Ejercicio,
SAFE_CAST(CASE WHEN PERIODO <= {mesbaser} THEN {mes_captura} ELSE PERIODO END AS INT64) as Periodo,
Version,
SAFE_CAST(Sueldos AS NUMERIC) AS Sueldos,
SAFE_CAST(PORC_PREST AS NUMERIC) AS PORC_PREST, DIMENSION6, 0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE, SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc_ant" as ORIGEN

FROM FINAL

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST(PERIODO AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc_ant" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc} AND
  (
    REGLA3 = 'PORCENTAJE REAL ULTIMO AÑO FLAT BASE ANUAL'
    )
    AND {filtro}

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST({mes_captura} AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) *-1 AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc_ant" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
  AND REGLA3 = 'PORCENTAJE REAL ULTIMO AÑO FLAT BASE ANUAL'
    AND {filtro}

;
'''

client.query(querydist).result()
print("Creado ",tablacargaporc_ant)


❗❗❗❗ Crecimiento Cero

Tomar los últimos 12 meses reales disponibles + presupuesto --- se puede hacer en base Forecast


In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargapresccero}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablacargapresccero}

WITH
BPC AS (
  SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion
  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen
  WHERE Ejercicio IN ( {aniobase} ) AND
    (
      (Periodo <= {mesbaser} AND VERSION IN ('Reales')) OR (Periodo > {mesbaser} AND VERSION IN ('PPTO'))
      )
      AND REGLA3 = 'CRECIMIENTO CERO'
    AND {filtro}
),

FINAL AS (

SELECT Cuenta, CeCo, CeBe,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio`,
PERIODO as `Periodo`,
T2.Version,
0 as `MES_INC`,
0.0 as `PORC_INC`,
{mesbaser} as `MES_BASE`,

---INCREMENTO
IFNULL(T0.Importe,0)
as `Importe`

, 'NOM_PRES_CCERO' as `ORIGEN`

FROM BPC T0
CROSS JOIN (
  SELECT DISTINCT VERSION
  FROM `{MESESPRES}`
  WHERE Version IN UNNEST ({params['version']})
  ) T2

--WHERE (T2.Version = '{params['version'][0]}' AND PERIODO > {mesbaser}) OR (T2.Version <> '{params['version'][0]}')
)

SELECT Cuenta, CeCo, CeBe, Ejercicio, SAFE_CAST(PERIODO AS INT64) as Periodo, Version,
SAFE_CAST(0.0 AS NUMERIC) as Sueldos, SAFE_CAST(0.0 AS NUMERIC) as PORC_PREST, '' as DIMENSION6, 0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE, SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_crecimientocero" as ORIGEN

FROM FINAL


'''
client.query(query).result()
print("Creado ",tablacargapresccero)

❗❗❗❗ Crecimiento Cero sin Outliers

Identificar los outliers y quitarselos.
---


*   51000000 NOMINA Y
*   51000000 NOMINA X

In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargapresccero_sinout}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

'''
client.query(query).result()
print("Borrado: ",tablacargapresccero_sinout)


query =f""" WITH
BPC AS (
  SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion
  FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T1 ON T0.Cuenta=T1.Cuenta_origen
  WHERE Ejercicio IN ( {aniobase} ) AND
    (
      (Periodo <= {mesbaser} AND VERSION IN ('Reales')) OR (Periodo > {mesbaser} AND VERSION IN ('PPTO'))
      )
      AND REGLA3 = 'CRECIMIENTO CERO SIN OUTLIERS'
      AND {filtro}
)

SELECT Cuenta, CeCo, CeBe,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio_Final`,
PERIODO as `Periodo`,
T2.Version as Version_final,
0 as `MES_INC`,
0.0 as `PORC_INC`,
{mesbaser} as `MES_BASE`,

---INCREMENTO
IFNULL(T0.Importe,0)
as `Importe`,
T0.Version,
T0.Ejercicio
, 'NOM_PRES_CCERO_SO' as `ORIGEN`

FROM BPC T0
CROSS JOIN (
  SELECT DISTINCT VERSION
  FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_MESESPRES`
  WHERE Version IN UNNEST ({params['version']})
  ) T2

--WHERE (T2.Version = '{params['version'][0]}' AND PERIODO > {mesbaser}) OR (T2.Version <> '{params['version'][0]}')
"""

query_out = f"""
SELECT *, ((Q1 + Q3)/2) AS Importe_bien
 FROM `{project_id}.data_warehouse_finance.tabla_outliers`
WHERE Cuenta IN ('51000011','51000056')
"""


BASE_OUTLIERS = bpd.read_gbq(query_out)
BASEP_NOMINA_PRESTACIONES = bpd.read_gbq(query)

#-------------------------------------------------------------------------------
# Convierte a pandas clásico
df_outliers = BASE_OUTLIERS.to_pandas()
df_presta = BASEP_NOMINA_PRESTACIONES.to_pandas()

# Ajustar los nombres de columnas si no coinciden exactamente
llaves = ['Cuenta', 'CeCo', 'CeBe', 'Periodo', 'Ejercicio']

# 1. Merge de ambos DataFrames por las llaves (como ya lo tienes)
df_merged = df_presta.merge(
    df_outliers[llaves + ['Importe_bien']],
    on=llaves,
    how='left'
)

# 2. Sobreescribe el Importe solo cuando Importe_bien no sea nulo
df_merged['Importe'] = df_merged['Importe_bien'].combine_first(df_merged['Importe'])

# 3. Elimina columnas auxiliares
# df_merged = df_merged.drop(columns=['Importe_bien'])

# Forza 'Importe' a float
df_merged['Importe'] = df_merged['Importe'].fillna(0)

# 4. Ajustes finales (renames, drops)
df_merged = df_merged.drop(columns=['Ejercicio', 'Version'])
df_merged.rename(columns={'Ejercicio_Final': 'Ejercicio'}, inplace=True)
df_merged.rename(columns={'Version_final': 'Version'}, inplace=True)
#-------------------------------------------------------------------------------

#CARGA A BIGQUERY

#BASEP_NOMINA_PRESTACIONES_pd = pd.DataFrame(BASEP_NOMINA_PRESTACIONES)
BASEP_NOMINA_PRESTACIONES_pd = df_merged.copy()
# BASEP_NOMINA_PRESTACIONES_pd['SUELDOS'] = 0
BASEP_NOMINA_PRESTACIONES_pd = BASEP_NOMINA_PRESTACIONES_pd.rename(columns={"Importe_bien":"SUELDOS"})
BASEP_NOMINA_PRESTACIONES_pd['SUELDOS'] = BASEP_NOMINA_PRESTACIONES_pd['SUELDOS'].fillna(0).astype(str).apply(Decimal)
BASEP_NOMINA_PRESTACIONES_pd['PORC_PREST'] = 0
BASEP_NOMINA_PRESTACIONES_pd['PORC_PREST'] = BASEP_NOMINA_PRESTACIONES_pd['SUELDOS'].astype(str).apply(Decimal)
# BASEP_NOMINA_PRESTACIONES_pd['DIMENSION6'] = 'OULIER'
BASEP_NOMINA_PRESTACIONES_pd.loc[BASEP_NOMINA_PRESTACIONES_pd['SUELDOS'] != 0,"DIMENSION6"]  = "Outlier"
BASEP_NOMINA_PRESTACIONES_pd['DIMENSION6'] = BASEP_NOMINA_PRESTACIONES_pd['DIMENSION6'].fillna("No Outlier")

BASEP_NOMINA_PRESTACIONES_pd = BASEP_NOMINA_PRESTACIONES_pd[["Cuenta","CeCo","CeBe","Ejercicio","Periodo","Version","SUELDOS","PORC_PREST",'DIMENSION6',"MES_INC","PORC_INC","MES_BASE","Importe","ORIGEN"]]
BASEP_NOMINA_PRESTACIONES_pd['Periodo'] = BASEP_NOMINA_PRESTACIONES_pd['Periodo'].astype(int)
BASEP_NOMINA_PRESTACIONES_pd['ORIGEN'] = 'NOM_PRE_cecimientocero_sinout'

BASEP_NOMINA_PRESTACIONES_pd["Importe"] = (
    BASEP_NOMINA_PRESTACIONES_pd["Importe"]
    .astype(str)              # critical: avoid float artifacts
    .apply(Decimal)
)

#BASEP_NOMINA_PRESTACIONES_pd.columns = BASEP_NOMINA_PRESTACIONES.columns


# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    BASEP_NOMINA_PRESTACIONES_pd, tablacargapresccero_sinout, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tablacargapresccero_sinout)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

#df_merged.loc[(df_merged['Cuenta'] == '51000056')&(df_merged['CeCo'] == '8200031003')]
print("Cargada :",tablacargapresccero_sinout)

❗❗❗❗ Inflación, distribución mensual

Reales actuales (últimos 12 meses )  * inflación

In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargapinfl}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablacargapinfl}

WITH
BPC AS (
  SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion
  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen
  WHERE
    ((Periodo <= {mesbaser} AND VERSION IN ('Reales') AND Ejercicio = {aniobase}) OR (Periodo > {mesbaser} AND VERSION IN ('Reales') AND Ejercicio = {aniobase-1}) )
      AND REGLA3 = 'INFLACION - MENSUAL'
      AND {filtro}
),

FINAL AS (

SELECT Cuenta, CeCo, CeBe,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio`,
PERIODO as `Periodo`,
T2.Version,
0 as `MES_INC`,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {inflacion_actual} ELSE {inflacionplan} END as `PORC_INC`,
{mesbaser} as `MES_BASE`,

---INFLACION
(1 + CASE WHEN PERIODO <= {mesbaser} THEN 0 WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {inflacion_actual} ELSE {inflacionplan} END )*
IFNULL(T0.Importe,0)
as `Importe`

, 'NOM_PRES_INFLACION' as `ORIGEN`

FROM BPC T0
CROSS JOIN (
  SELECT DISTINCT VERSION
  FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_MESESPRES`
  WHERE Version IN UNNEST ({params['version']})
  ) T2
)

SELECT Cuenta, CeCo, CeBe, Ejercicio,
SAFE_CAST(PERIODO AS INT64) as Periodo, Version,
SAFE_CAST(0.0 AS NUMERIC) as Sueldos,
SAFE_CAST(0.0 AS NUMERIC) as PORC_PREST,
'' as DIMENSION6, 0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_inflacion" as ORIGEN FROM FINAL

'''

client.query(query).result()
print("Creado ",tablacargapinfl)

⭐ **Replica año anterior + inflación**

In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablanompres_anioantinf}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablanompres_anioantinf}

WITH
BPC AS (
  SELECT T0.* EXCEPT(CUENTA,PERIODO,IMPORTE), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion,
  IMPORTE,
  CASE WHEN PERIODO <= {mesbaser} AND EJERCICIO = {aniobase-1} THEN {mes_captura} ELSE PERIODO END AS PERIODO
  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen
  WHERE
    VERSION IN ('Reales') AND (Ejercicio = {aniobase-1} OR (EJERCICIO = {aniobase} AND PERIODO <= {mesbaser}))
      AND REGLA3 = 'REAL AÑO ANTERIOR + INFLACION'
      AND {filtro}
),

AJUSTE AS (
  SELECT * EXCEPT(IMPORTE,PERIODO), IMPORTE * -1 as IMPORTE , {mes_captura} as PERIODO

  FROM BPC

  WHERE EJERCICIO = {aniobase}
),

FINAL AS (

SELECT Cuenta, CeCo, CeBe,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio`,
PERIODO as `Periodo`,
T2.Version,
0 as `MES_INC`,
CASE WHEN CUENTA IN ('51000058','51000409') THEN
  CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {inflacion_actual} ELSE {inflacionplan} END
    WHEN CUENTA IN ('51000059') THEN
    0.056 END
   as `PORC_INC`,
1 as `MES_BASE`,

---INFLACION
(1 +
  CASE WHEN EJERCICIO = {aniobase} THEN 0
    WHEN CUENTA IN ('51000058','51000409') THEN
      CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {inflacion_actual} ELSE {inflacionplan} END
        WHEN CUENTA IN ('51000059') THEN
        0.056 END
 )*
IFNULL(T0.Importe,0)
as `Importe`

, 'NOM_PRES_AÑO_ANTERIOR_INFLACION' as `ORIGEN`

FROM (SELECT * FROM BPC UNION ALL SELECT * FROM AJUSTE) T0
CROSS JOIN (
  SELECT DISTINCT VERSION
  FROM `{MESESPRES}`
  WHERE Version IN UNNEST ({params['version']})
  ) T2

)

SELECT Cuenta, CeCo, CeBe, Ejercicio, SAFE_CAST(PERIODO AS INT64) as Periodo, Version, SAFE_CAST(0.0 AS NUMERIC) as Sueldos,
SAFE_CAST(0.0 AS NUMERIC) as PORC_PREST, '' as DIMENSION6, MES_INC, PORC_INC, MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE, ORIGEN
FROM FINAL

'''

client.query(query).result()
print("Creado ",tablanompres_anioantinf)

❗❗❗❗ Mensual con incremento

Reales actuales + presupuesto * incremento nómina

In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargapresminc}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablacargapresminc}

WITH
BPC AS (
  SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio = {aniobase} AND
    ((Periodo <= {mesbaser} AND VERSION IN ('Reales')) OR (Periodo > {mesbaser} AND VERSION IN ('PPTO') ) )
      AND REGLA3 = 'MENSUAL + INCREMENTO'
      AND {filtro}
),

FINAL AS (

SELECT Cuenta, CeCo, CeBe,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio`,
PERIODO as `Periodo`,
T2.Version,
0 as `MES_INC`,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {porc_inc} ELSE {porc_inc_sig_anio} END as `PORC_INC`,
{mesbaser} as `MES_BASE`,

---INFLACION
(1 + CASE WHEN PERIODO <= {mesbaser} THEN 0 WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {porc_inc} ELSE {porc_inc_sig_anio} END )*
IFNULL(T0.Importe,0)
as `Importe`

, 'NOM_PRES_MES_INC' as `ORIGEN`

FROM BPC T0
CROSS JOIN (
  SELECT DISTINCT VERSION
  FROM `{MESESPRES}`
  WHERE Version IN UNNEST ({params['version']})
  ) T2

WHERE CUENTA NOT IN ('')

)

SELECT Cuenta, CeCo, CeBe, Ejercicio,
SAFE_CAST(PERIODO AS INT64) as Periodo, Version,
SAFE_CAST(0.0 AS NUMERIC) as Sueldos,
SAFE_CAST(0.0 AS NUMERIC) as PORC_PREST, '' as DIMENSION6,
0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,

"NOM_PRE_mesinc" as ORIGEN

FROM FINAL

'''

client.query(query).result()
print("Creado ",tablacargapresminc)

❗❗❗❗ Porcentajes al año, base roll year

In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargaporc}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');


---UNA VEZ BORRADOS, INSERTAR EL CALCULO


INSERT INTO {tablacargaporc}

WITH BASE_SUELDOS_PR AS (
  SELECT T0.CeCo, T0.CeBe, T0.`Ejercicio`, T0.Periodo, T0.Version, T1.DIMENSION1, T1.DIMENSION2, T1.DIMENSION3, SUM(T0.Importe) as `Importe`
  FROM {tablanomina} T0
  INNER JOIN {CECOS}  T1 ON T0.CECO=T1.CECO

  WHERE {filtro.replace("T0.","T1.")}

  GROUP BY T0.Ejercicio, T0.Periodo, T0.Version, T0.CeCo, T0.CeBe, T1.DIMENSION1, T1.DIMENSION2, T1.DIMENSION3
  ),

  CECOS AS (

-- DISTRIBUCION CECOS
  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN PRESTACIONES
  INNER JOIN {tablapres_disanioactual} T12
    ON T11.CECO=T12.CECO

),

  AREA AS (
  -- DISTRIBUCION DIR AREA

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN {tablapres_disanioactual} T12
    ON T11.DIMENSION3=T12.DIR_V AND DIMENSION6 = "DIMENSION3"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO

  WHERE CECOS.CUENTA IS NULL
  ),

  CORP AS (
  -- DISTRIBUCION DIR CORP

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN {tablapres_disanioactual} T12
    ON T11.DIMENSION2=T12.DIR_V AND DIMENSION6 = "DIMENSION2"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA
   ON T11.DIMENSION3=AREA.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL
  ),

 DIV AS (
  -- DISTRIBUCION DIV NEG

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM BASE_SUELDOS_PR T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  LEFT JOIN {tablapres_disanioactual} T12
    ON T11.DIMENSION1=T12.DIR_V AND DIMENSION6 = "DIMENSION1"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA
   ON T11.DIMENSION3=AREA.DIR_V
  LEFT JOIN CORP
   ON T11.DIMENSION2=CORP.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL AND CORP.CUENTA IS NULL
 ),

 FINAL AS (

 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CECOS
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM AREA
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CORP
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM DIV
)

SELECT Cuenta, CeCo, CeBe, Ejercicio,
CASE WHEN PERIODO <= {mesbaser} THEN {mes_captura} ELSE PERIODO END as
PERIODO, Version,
SAFE_CAST(Sueldos AS NUMERIC) AS Sueldos,
SAFE_CAST(PORC_PREST AS NUMERIC) AS PORC_PREST,
DIMENSION6, 0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE, SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc" as ORIGEN FROM FINAL

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST(PERIODO AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc} AND
  (
    REGLA3  IN
    ('PORCENTAJE REAL BASE POR DEFINIR_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)',
     'PORCENTAJE REAL BASE PRIMER SEMESTRE ACTUAL_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)'
     )
    )
    AND {filtro}

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST({mes_captura} AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) *-1 AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
  AND REGLA3  IN
    ('PORCENTAJE REAL BASE POR DEFINIR_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)',
     'PORCENTAJE REAL BASE PRIMER SEMESTRE ACTUAL_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)'
     )
    AND {filtro}
;
'''

client.query(query).result()
print("Creado ",tablacargaporc)

❗❗❗❗Replicas con y sin incremento

In [ ]:
##### SCRIPT

queryreplicas = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargapresrep}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablacargapresrep}

WITH
BPC AS (
  SELECT T0.* EXCEPT(CUENTA), T7.REGLA3, COALESCE(T1.Cuenta_reporte,T0.Cuenta) as Cuenta, T1.Clasificacion
  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen
  WHERE Ejercicio IN ( {aniobasehc} ) AND Periodo = {mesbaser} AND VERSION IN ('Reales')
  AND REGLA3 LIKE '%REPLICA%' AND IFNULL(T1.Clasificacion,'') NOT LIKE '%SUELDO%'
  AND {filtro}
),

FINAL AS (

SELECT Cuenta, CeCo, CeBe,
CASE WHEN T2.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as `Ejercicio`,
CAST(T2.MES AS INT64) as `Periodo`,
T2.Version,
CASE WHEN REGLA3 = 'REPLICA SIN INCREMENTO' THEN 0 ELSE {mes_inc_valesali} END as `MES_INC`,
CASE WHEN REGLA3 = 'REPLICA SIN INCREMENTO' THEN 0 ELSE {porc_inc} END as `PORC_INC`,
SAFE_CAST(T0.Periodo AS INT64) as `MES_BASE`,

---INCREMENTO
CASE WHEN REGLA3 = 'REPLICA SIN INCREMENTO' THEN 1 ELSE
  CASE
  -- WHEN T2.MES < {mes_inc_valesali} AND T2.VERSION IN ('PPTO_V2','PPTO_V1') AND {mes_inc_valesali} > {mesbasehc} THEN (1+{porc_inc}) --PRIMEROS MESES DOBLE INCREMENTO PPTO_V1 Y PPTO_V2
  WHEN T2.MES < {mes_inc_valesali} AND T2.VERSION IN ('PPTO_V0') THEN (1+{porc_inc}) --PRIMEROS MESES PLAN INCREMENTO ESTE AÑO
  WHEN T2.MES >= {mes_inc_valesali} AND T2.VERSION IN ('PPTO_V2','PPTO_V1') AND {mes_inc_valesali} > {mesbasehc} THEN (1+{porc_inc}) --ULTIMOS MESES ESTE AÑO INCREMNETO ESTE AÑO
  WHEN T2.MES >= {mes_inc_valesali} AND T2.VERSION IN ('PPTO_V0') THEN (1+{porc_inc})*(1+{porc_inc_sig_anio}) --PLAN JULIO EN ADELANTE
  ELSE 1 END
 END
* IFNULL(T0.Importe,0)
as `Importe`

, 'NOM_PRES_INC' as `ORIGEN`

FROM BPC T0
CROSS JOIN (
  SELECT *
  FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_MESESPRES`
  WHERE Version IN UNNEST ({params['version']})
  ) T2
)

SELECT Cuenta, CeCo, CeBe, Ejercicio, SAFE_CAST(PERIODO AS INT64) as Periodo, Version,
SAFE_CAST(0.0 AS NUMERIC) as Sueldos, SAFE_CAST(0.0 AS NUMERIC) as PORC_PREST,
'' as DIMENSION6, 0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE, SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE, "NOM_PRE_replicas" as ORIGEN FROM FINAL

WHERE (EJERCICIO = {aniobase} AND PERIODO > {mesbasehc}) OR (EJERCICIO > {aniobase})

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST(PERIODO AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_porc_ant" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
  AND REGLA3 LIKE '%REPLICA%' AND IFNULL(T1.Clasificacion,'') NOT LIKE '%SUELDO%'
    AND {filtro}

'''

client.query(queryreplicas).result()
print("Creado ",tablacargapresrep)

❗❗❗❗ Reparto garantizado

Tomar ( diciembre * 2 ) / 12
Importante, la cuenta es:

In [ ]:
##### SCRIPT

query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablacargaptugar}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablacargaptugar}

WITH
SUELDODIC AS (
  SELECT T0.CECO, T0.CEBE, EJERCICIO, VERSION, SUM(IMPORTE) as IMPORTE

  FROM `{tablanomina}` T0
  INNER JOIN {CECOS} T1 USING(CECO)

  WHERE PERIODO = 12 AND {filtro.replace("T0.","T1.")}

  GROUP BY 1,2,3,4

),

FINAL AS (
SELECT
  SUBSTRING(c.CUENTA_NOM,1,8) as Cuenta,
  b.CeCo,
  b.CeBe,
  b.Ejercicio,
  T2.MES as Periodo,
  T2.Version,
  0 as MES_INC,
  0.0 as PORC_INC,
  12 as MES_BASE,
  b.Importe as Sueldos,
  (b.Importe * 2) / 12 as Importe,
  'NOM_PRES_PTUGAR' as ORIGEN
FROM SUELDODIC b
CROSS JOIN (
  SELECT *
  FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_MESESPRES`
  WHERE Version IN UNNEST ({params['version']})
) T2
CROSS JOIN {tablareglasnom} c

WHERE REGLA3 = 'SALARIO DIC *2 / 12'
)
SELECT Cuenta, CeCo, CeBe, Ejercicio,
SAFE_CAST(
 CASE WHEN PERIODO <= {mesbaser} THEN {mes_captura} ELSE PERIODO END
   AS INT64) as Periodo,
Version,
SAFE_CAST(Sueldos AS NUMERIC) as Sueldos, SAFE_CAST(0.0 AS NUMERIC) as PORC_PREST, '' as DIMENSION6, 0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE, "NOM_PRE_ptugar" as ORIGEN
FROM FINAL

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST(PERIODO AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
"NOM_PRE_ptugar" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
  AND REGLA3 = 'SALARIO DIC *2 / 12'
    AND {filtro}

UNION ALL

SELECT T1.Cuenta_reporte as Cuenta, CeCo, CeBe,
SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
SAFE_CAST({mes_captura} AS INT64) as Periodo,
'{params['version'][0]}' as Version,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
0 as MES_INC,
0.0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(IFNULL(Importe,0) *-1 AS NUMERIC) AS IMPORTE,
"NOM_PRE_ptugar" as ORIGEN

  FROM `{ER}` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
  AND REGLA3 = 'SALARIO DIC *2 / 12'
    AND {filtro}

;

'''
client.query(query).result()
print("Creado ",tablacargaptugar)

# **❌❌❌❌  DIFERENCIA CALCULO + PLAN/REALES vs ANUAL ❌❌❌❌**
❌ No correr, validar en PPTO_V2 si se requiere

❗❗❗ Cuidado en el reproceso, afecta FORECAST

⭐ Se calcula a partir de Nómina, si se corre nómina, debe ejecutarse también este rubro

In [ ]:
##DIFF PRESTACIONES

if params['version'][0] == 'PPTO_V2' and mesbaser != mes_captura-1 :
  whereme = f'''((Periodo <= {mes_captura-1} AND Version IN ('{versionprimmeses}')) OR (Periodo = {mes_captura} AND Version IN ('PPTO') ) )'''
else:
  whereme = f'''Periodo < {mes_captura} AND Version IN ('{versionprimmeses}')'''

queryprest_diff =f"""
CREATE OR REPLACE TABLE {tablanom_diff} AS
(
WITH
--AGRUPACION -SUELDOS CALCULADOS
  SUELDOS_CALC AS
  (
    SELECT Cuenta, CeCo, CeBe, periodo, SUM(Importe) as `Importe`

  FROM {tablanomina}

  WHERE Periodo > {mesbaser} AND Version IN ('{params['version'][0]}')
  GROUP BY Cuenta, Ceco, Cebe, periodo

  ),

  --AGRUPACION -SUELDOS REALES
  SUELDOS_BASE AS
  (

  SELECT T1.Cuenta_reporte as CUENTA, CeCo, CeBe, periodo, SUM(Importe) as `Importe`

  FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE PERIODO <= {mesbaser} AND Version IN ('Reales') AND Ejercicio = {aniobase} AND T1.Clasificacion LIKE '%SUELDO%' AND T0.DIMENSION1 IN UNNEST ({direcciones_nomina})
  GROUP BY Cuenta, Ceco, Cebe, periodo

  ),

  SUELDOS_ANUAL_1 AS (
    SELECT T0.*, DIMENSION1,DIMENSION2, DIMENSION3
    FROM SUELDOS_CALC T0
    LEFT JOIN `mus_qas_drv_datos_maestros.NW_CECO3` T1 ON T0.CECO=T1.CECO
    UNION ALL
    SELECT T0.*, DIMENSION1,DIMENSION2, DIMENSION3
    FROM SUELDOS_BASE T0
    LEFT JOIN `mus_qas_drv_datos_maestros.NW_CECO3` T1 ON T0.CECO=T1.CECO
  ),

  SUELDOS_ANUAL AS (
    SELECT * EXCEPT (Importe), SUM(Importe) as Importe
    FROM SUELDOS_ANUAL_1
    Group by 1,2,3,4,5,6,7
  ),

    SUELDOS_ANUAL_SINCUENTA AS (
    SELECT * EXCEPT (Importe,periodo,CUENTA), SUM(Importe) as Importe
    FROM SUELDOS_ANUAL_1
    Group by 1,2,3,4,5
  ),

  DIST_ANIOANTERIOR AS
  (
    SELECT *EXCEPT(IMPORTE), SAFE_CAST(IMPORTE AS NUMERIC) AS IMPORTE FROM {tablapres_disanioanterior}
  ),

  DIST_ANIOACTUAL AS
  (
    SELECT *EXCEPT(IMPORTE), SAFE_CAST(IMPORTE AS NUMERIC) AS IMPORTE FROM {tablapres_disanioactual}
  ),

  ------PARA PRESTACIONES_ANTERIOR

  CECOS AS (

-- DISTRIBUCION CECOS
  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN PRESTACIONES
  INNER JOIN DIST_ANIOANTERIOR T12
    ON T11.CECO=T12.CECO

),

  AREA AS (
  -- DISTRIBUCION DIR AREA

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN DIST_ANIOANTERIOR T12
    ON T11.DIMENSION3=T12.DIR_V AND DIMENSION6 = "DIMENSION3"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO

  WHERE CECOS.CUENTA IS NULL
  ),

  CORP AS (
  -- DISTRIBUCION DIR CORP

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN DIST_ANIOANTERIOR T12
    ON T11.DIMENSION2=T12.DIR_V AND DIMENSION6 = "DIMENSION2"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA
   ON T11.DIMENSION3=AREA.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL
  ),

 DIV AS (
  -- DISTRIBUCION DIV NEG

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  LEFT JOIN DIST_ANIOANTERIOR T12
    ON T11.DIMENSION1=T12.DIR_V AND DIMENSION6 = "DIMENSION1"
  LEFT JOIN CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA
   ON T11.DIMENSION3=AREA.DIR_V
  LEFT JOIN CORP
   ON T11.DIMENSION2=CORP.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL AND CORP.CUENTA IS NULL
 ),

 PRES_ANUAL_1 AS (

 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CECOS
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM AREA
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CORP
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM DIV
),

------PARA PRESTACIONES_ACTUAL

  CECOS2 AS (

-- DISTRIBUCION CECOS
  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN PRESTACIONES
  INNER JOIN DIST_ANIOACTUAL T12
    ON T11.CECO=T12.CECO

),

  AREA2 AS (
  -- DISTRIBUCION DIR AREA

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN DIST_ANIOACTUAL T12
    ON T11.DIMENSION3=T12.DIR_V AND DIMENSION6 = "DIMENSION3"
  LEFT JOIN CECOS2 CECOS
   ON T11.CECO=CECOS.CECO

  WHERE CECOS.CUENTA IS NULL
  ),

  CORP2 AS (
  -- DISTRIBUCION DIR CORP

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6,T12.DIR_V

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  INNER JOIN DIST_ANIOACTUAL T12
    ON T11.DIMENSION2=T12.DIR_V AND DIMENSION6 = "DIMENSION2"
  LEFT JOIN CECOS2 CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA2 AREA
   ON T11.DIMENSION3=AREA.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL
  ),

 DIV2 AS (
  -- DISTRIBUCION DIV NEG

  SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`,
  CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6

  --MONTO TOTAL ANUAL DE SUELDOS
  FROM SUELDOS_ANUAL_SINCUENTA T11
  --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
  LEFT JOIN DIST_ANIOACTUAL T12
    ON T11.DIMENSION1=T12.DIR_V AND DIMENSION6 = "DIMENSION1"
  LEFT JOIN CECOS2 CECOS
   ON T11.CECO=CECOS.CECO
  LEFT JOIN AREA2 AREA
   ON T11.DIMENSION3=AREA.DIR_V
  LEFT JOIN CORP2 CORP
   ON T11.DIMENSION2=CORP.DIR_V

  WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL AND CORP.CUENTA IS NULL
 ),

 PRES_ANUAL_2 AS (

 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CECOS2
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM AREA2
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM CORP2
 UNION ALL
 SELECT Cuenta, CeCo, CeBe, Importe, ORIGEN, Sueldos, PORC_PREST, DIMENSION6
  FROM DIV2
),

     PRES_ANUAL AS (
    SELECT * EXCEPT (ORIGEN,Sueldos,PORC_PREST,DIMENSION6,Importe), SUM(CAST(Importe AS NUMERIC)) as Importe
    FROM PRES_ANUAL_1
    Group by 1,2,3
    UNION ALL
    SELECT * EXCEPT (ORIGEN,Sueldos,PORC_PREST,DIMENSION6,Importe), SUM(CAST(Importe AS NUMERIC)) as Importe
    FROM PRES_ANUAL_2
    Group by 1,2,3
  ),

  PRES_CALC AS
  (
    SELECT Cuenta, CeCo, CeBe, SUM(CAST(Importe AS NUMERIC)) as `Importe`

  FROM `{tablanompres}*` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)

  WHERE Periodo >= {mes_captura} AND Version IN ('{params['version'][0]}')
  AND
  REGLA3 IN (
    'PORCENTAJE REAL ULTIMO AÑO FLAT BASE ANUAL',
    'PORCENTAJE REAL BASE POR DEFINIR_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)',
     'PORCENTAJE REAL BASE PRIMER SEMESTRE ACTUAL_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)'
     )
  GROUP BY Cuenta, Ceco, Cebe
  ),

  PRES_BASE AS
  (
    SELECT Cuenta, CeCo, CeBe, SUM(CAST(Importe AS NUMERIC)) as `Importe`

  FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
  INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE {whereme} AND Ejercicio = {aniobase} AND
  REGLA3 IN (
    'PORCENTAJE REAL ULTIMO AÑO FLAT BASE ANUAL',
    'PORCENTAJE REAL BASE POR DEFINIR_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)',
     'PORCENTAJE REAL BASE PRIMER SEMESTRE ACTUAL_ PARTICIPACION MES (ENE-JUL REALES ACTUAL_ AGO-DIC ANTERIOR)'
     ) AND  T0.DIMENSION1 IN UNNEST ({direcciones_nomina})
  GROUP BY Cuenta, Ceco, Cebe
  ),

   PRES_ANUAL_CARGA_1 AS (
    SELECT *
    FROM PRES_CALC
    UNION ALL
    SELECT *
    FROM PRES_BASE
  ),

     PRES_ANUAL_CARGA AS (
    SELECT * EXCEPT (Importe), SUM(CAST(Importe AS NUMERIC)) as Importe
    FROM PRES_ANUAL_CARGA_1
    Group by 1,2,3
  ),


   --DIFF NO EDITABLES PRES
  PRES_DIFF_NOED AS
  (
    SELECT COALESCE(T0.Cuenta,T1.Cuenta) as CUENTA, COALESCE(T0.CeCo,T1.CECO) AS CECO, COALESCE(T0.CeBe,T1.CEBE) as CEBE,
     IFNULL(T0.IMPORTE,0)-IFNULL(T1.IMPORTE,0) as `Importe`

  FROM PRES_ANUAL T0
  FULL OUTER JOIN PRES_ANUAL_CARGA T1 ON T0.CUENTA=T1.CUENTA AND T0.CECO=T1.CECO AND T0.CEBE=T1.CEBE

  )

SELECT
CUENTA, CECO, CEBE, {aniobase} as EJERCICIO, {mes_captura} as PERIODO, '{params['version'][0]}' as VERSION,
0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE,
CAST(IMPORTE AS NUMERIC) AS IMPORTE,
'NOM_PRE_DIFF' as ORIGEN

FROM PRES_DIFF_NOED
)
"""

client.query(queryprest_diff).result()
print("Creado ",tablanom_diff)

# CARGA_INICIAL☝NÓMINA (GASTOS DE PERSONAL)
❗❗❗ Después de cargas EPM BPC ¡¡NO REPROCESAR!!, esa es la fuente, si se modifica, se "eliminan" las CARGA_INICIAL

❗❗❗ Solo ejecutar una vez, esto congela CARGA_INICIAL

❗❗❗ Cuidado en el reproceso, afecta FORECAST

In [ ]:
if ( DIMENSION6pres == "ME_PPTO" ) :
    queryCARGA_INICIAL = f'''

CREATE OR REPLACE TABLE {tablaCARGA_INICIAL}_completas AS
SELECT T0.CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, IMPORTE, 'CARGA_INICIAL' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_CARGA_INICIAL}` T3 ON T0.Cuenta=T3.CUENTA

WHERE ( T0.VERSION IN ('{params['version'][0]}') AND T0.Ejercicio IN ({aniobase}) AND Periodo > {mesbaser} )
       OR ( T0.VERSION IN ('{params['version'][1]}') AND T0.Ejercicio IN ({aniobase+1}) )

'''

else :
  queryCARGA_INICIAL = f'''
DECLARE CECOS STRING;
DECLARE CEBES STRING;
DECLARE DELETESENTENCE STRING;

----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS

{scipt_delete}

SET DELETESENTENCE = FORMAT("""

DELETE FROM {tablaCARGA_INICIAL}
WHERE (CUENTA LIKE '4%%' AND CEBE IN (%s)) OR (CUENTA LIKE '5%%' AND CECO IN (%s))
""",
CEBES,CECOS)
;

EXECUTE IMMEDIATE DELETESENTENCE;

---SE REPROCESA COMPLETO (BACKUP COMPLETO)

# CREATE OR REPLACE TABLE {tablaCARGA_INICIAL}_completas AS
# (
# SELECT T0.CUENTA, CECO, CEBE, PERIODO, EJERCICIO, '{params['version'][0]}' as VERSION, IMPORTE, 'CARGA_INICIAL' as ORIGEN

# FROM `{ER}` T0
# INNER JOIN `{CUENTAS_CARGA_INICIAL}` T3 ON T0.Cuenta=T3.CUENTA

# WHERE T0.VERSION IN ('{params['version'][0]}') AND T0.Ejercicio IN ({aniobase})
# );

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablaCARGA_INICIAL}

SELECT T0.CUENTA, CECO, CEBE, PERIODO, EJERCICIO, '{params['version'][0]}' as VERSION, IMPORTE, 'CARGA_INICIAL' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_CARGA_INICIAL}` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{CUENTAS_CED_INM_ADC}` T4 ON T4.CUENTA=T0.CUENTA AND T4.DIMENSION1 = CASE WHEN T0.DIMENSION1 IN ('NEGOCIO Y') OR T0.DIMENSION2 IN ('Fideicomisos') THEN 'NEGOCIO Y' ELSE T0.DIMENSION1 END

WHERE T4.CUENTA IS NULL AND T0.VERSION IN ('{params['version'][0]}') AND T0.Ejercicio IN ({aniobase}) AND {filtro}
;

'''


client.query(queryCARGA_INICIAL).result()
print(f"""
      # Tabla creada: {tablaCARGA_INICIAL}_completas
      Tabla creada: {tablaCARGA_INICIAL}
      """)

# BONO_ESP☝NÓMINA (GASTOS DE PERSONAL)
❗❗❗ Cuidado en el reproceso, afecta FORECAST -- requiere precargada 399

⭐ Se calcula a partir de Nómina, si se corre nómina, debe ejecutarse también este rubro

Cálculo del bono ejecutivo por resultados

1. Se toman el sueldo del mes, si es anterior a julio, se le aplica el incremento del ejercicio actual y el incremento del ejercicio siguiente, para los meses de julio en adelante, solo se aplica el incremento del ejercicio siguiente

2. Se multiplica por el parámetro de "meses bono"/12

3. Se cancela el monto acumulado hasta el mes previo



In [ ]:
### CALCULO ANUAL MENSUAL

versionprimmeses = 'Reales'

if params['version'][0] == 'PPTO_V2' and mesbasehc != mes_captura-1:
  whereme = f'''((Periodo <= {mes_captura-1} AND Version IN ('{versionprimmeses}')) OR (Periodo = {mes_captura} AND Version IN ('PPTO') ) )'''
else:
  whereme = f'''Periodo < {mes_captura} AND Version IN ('{versionprimmeses}')'''

if mesbasehc == 12 :
  messig = 1
else:
  messig = mesbasehc+1


# SUELDOS ES CALCULO NOMINA DE MESES NO REALES + CONCEPTOS PARA OBTENER DIC

query_bono = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS
{scipt_delete}

DELETE FROM `{tablabonobase}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablabonobase}

WITH

SUELDOS AS (

  --- NOMINA CALCULADA
    SELECT T0.CeCo, T0.CeBe, ORIGEN, Ejercicio, Periodo, Version,'Forecast'Posicion,  'Base' as DIMENSION6,0 Personas,0 Ejercicio_DE_INGRESO,
    0 MES_DE_INGRESO,
    CASE WHEN Periodo IN (1,11) AND T0.CUENTA = '51000041' THEN 0.5 ELSE 1 END * SUM(T0.Importe) as `Sueldo` -- SIN AGUINALDO
          FROM {tablanomina} T0
          WHERE T0.Cuenta IN ('51000001','51000002','51000040','51000041','51000044','51000045')

          GROUP BY T0.CeCo, T0.CeBe, cuenta,Version,Ejercicio, Periodo,ORIGEN
    UNION ALL
  --- NOMINA CALCULADA
    SELECT T0.CeCo, T0.CeBe, ORIGEN, Ejercicio, Periodo, Version,'Forecast'Posicion,  'Base' as DIMENSION6,0 Personas,0 Ejercicio_DE_INGRESO,
    0 MES_DE_INGRESO,
    CASE WHEN Periodo = 11 AND T0.CUENTA = '51000041' THEN 0.5 ELSE 1 END * SUM(T0.Importe) as `Sueldo` -- SIN AGUINALDO
          FROM {tablaCARGA_INICIAL} T0
          WHERE T0.Cuenta IN ('51000399')

          GROUP BY T0.CeCo, T0.CeBe, cuenta,Version,Ejercicio, Periodo,ORIGEN

    ),


    SUELDOS_AGRUP AS (

  --- NOMINA CALCULADA AGRUPADA
      SELECT T0.CeCo, T0.CeBe, Ejercicio, Periodo, Version,Posicion,DIMENSION6,SUM(Personas) AS PERSONAS,Ejercicio_DE_INGRESO,MES_DE_INGRESO,
    SUM(SUELDO)
        *
              CASE
              WHEN T0.Periodo < {mesinc} AND VERSION IN ('PPTO_V2','PPTO_V1') AND {mesinc} > {mesbaser} THEN (1+{porc_inc})*(1+{porc_inc_sig_anio}) --PRIMEROS MESES DOBLE INCREMENTO PPTO_V1 Y PPTO_V2
              WHEN T0.Periodo >= {mesinc} AND VERSION IN ('PPTO_V2','PPTO_V1') AND {mesinc} > {mesbaser} THEN (1+{porc_inc_sig_anio}) --PPTO_V1 Y PPTO_V2 JULIO EN ADELANTE
              WHEN T0.Periodo < {mesinc} AND VERSION IN ('PPTO_V0') THEN (1+{porc_inc_sig_anio})*(1+{porc_inc_seg_anio}) --PRIMEROS MESES DOBLE INCREMENTO PLAN
              WHEN T0.Periodo >= {mesinc} AND VERSION IN ('PPTO_V0') THEN (1+{porc_inc_seg_anio}) --PLAN JULIO EN ADELANTE
                END
     as `Sueldo`
          FROM SUELDOS T0


        GROUP BY 1,2,3,4,5,6,7,9,10
    ),

    BONO_PLAN AS (  -- Primeros meses

  --- BONO REALES
      SELECT T0.Cuenta, T0.CeCo, T0.CeBe, PERIODO,Ejercicio, Version,'Reales'Posicion, 'Base' as DIMENSION6,0 Personas,0 Ejercicio_DE_INGRESO,
      0 MES_DE_INGRESO,
      SUM(T0.Importe) as `Importe`

      FROM `{ER}` T0

      WHERE {whereme} AND Ejercicio = {aniobase} AND T0.Cuenta = '51000403'
      GROUP BY T0.Cuenta, T0.CeCo, T0.CeBe,Version,PERIODO, Ejercicio,DIMENSION6,Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO

      ),

        BONO AS (
  --- BONO CALCULO SOBRE NOMINA
          SELECT '51000403' as `CUENTA`, T0.CECO, T0.CEBE, T0.Periodo, T0.Ejercicio, T0.Version,T0.Posicion, T0.DIMENSION6, T0.Personas,
          T0.Ejercicio_DE_INGRESO,
          T0.MES_DE_INGRESO,
          IFNULL(
              T0.Sueldo * COALESCE(
                SAFE_CAST(BONO_ESP.Promedio_Meses AS FLOAT64)
                ,{meses_bono}
                )
              /12,0) as Importe,

              T0.Sueldo,

              COALESCE(
                SAFE_CAST(BONO_ESP.Promedio_Meses AS FLOAT64)
                ,{meses_bono}) as PARAMETROS

          FROM SUELDOS_AGRUP T0
          LEFT JOIN {CECOS} CECOS ON T0.CECO = CECOS.CECO
          LEFT JOIN (
            -- Priority 1: DIMENSION1 + DIMENSION2
            SELECT *, 1 AS prioridad
            FROM {PAR_BONO_ESP}
            WHERE DIMENSION1 IS NOT NULL
              AND DIMENSION2 IS NOT NULL

            UNION ALL

            -- Priority 2: only DIMENSION1
            SELECT *, 2 AS prioridad
            FROM {PAR_BONO_ESP}
            WHERE DIMENSION1 IS NOT NULL
              AND DIMENSION2 IS NULL

            UNION ALL

            -- Priority 3: only DIMENSION2
            SELECT *, 3 AS prioridad
            FROM {PAR_BONO_ESP}
            WHERE DIMENSION1 IS NULL
              AND DIMENSION2 IS NOT NULL
          ) BONO_ESP
          ON (
              (BONO_ESP.prioridad = 1 AND CECOS.DIMENSION1 = BONO_ESP.DIMENSION1 AND CECOS.DIMENSION2 = BONO_ESP.DIMENSION2)
            OR (BONO_ESP.prioridad = 2 AND CECOS.DIMENSION1 = BONO_ESP.DIMENSION1)
            OR (BONO_ESP.prioridad = 3 AND CECOS.DIMENSION2 = BONO_ESP.DIMENSION2)
          )
          QUALIFY ROW_NUMBONO_ESP() OVER (
            PARTITION BY T0.CECO, T0.CEBE, T0.Periodo, T0.Ejercicio, T0.Version,T0.Posicion, T0.DIMENSION6, T0.Personas,
          T0.Ejercicio_DE_INGRESO,
          T0.MES_DE_INGRESO
            ORDER BY BONO_ESP.prioridad
          ) = 1

          UNION ALL

  --- BONO REALES
          SELECT CUENTA, CECO, CEBE, PERIODO, EJERCICIO, '{params['version'][0]}' as VERSION, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,
          MES_DE_INGRESO, IMPORTE, 0 AS SUELDO, 0 as PARAMETROS

          FROM BONO_PLAN
      ),

      BONO_AGRUP AS (

  --- BONO CALCULO COMPLETO
        SELECT CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO,
        PARAMETROS,
        SUM(IMPORTE) AS IMPORTE, SUELDO AS SUELDO

        FROM BONO

        GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,14
      ),

      CALCULO AS (

      SELECT T0.* EXCEPT(PERIODO, VERSION, IMPORTE, sueldo),
      ---CASE WHEN T1.PERIODO < {mesbaser} AND T0.VERSION = '{params['version'][0]}' then {mes_captura} else t1.periodo end as
      T1.PERIODO,
      T0.PERIODO as PERIODO_BASE,
      VERSION,

      CASE WHEN T0.VERSION = '{params['version'][0]}' AND T1.PERIODO <= {mesbasehc} THEN 'AJUSTE_BONO' ELSE 'NOM_BONO' END AS ORIGEN,

    ----PARA QUE EN REPORTE APAREZCA AÑO COMPLETO DE CALCULO, HACEMOS QUE LOS CALCULOS ANTES DEL MES DE INICIO DE CAPTURA SE HAGAN "NORMALES" Y EN EL MES DE CAPTURA SE HAGA EL AJUSTE CONTRA EL MONTO QUE HAYA EN BASE
      CASE
        WHEN T1.PERIODO < {mes_captura} AND POSICION = 'Forecast' THEN 0  --- MUESTRA LOS PRIMEROS MESES QUE QUEDARAN EN BASE
        WHEN T1.PERIODO = {mes_captura} AND T0.PERIODO = {mes_captura-1}  AND POSICION = 'Forecast' THEN 0   ---PARA EL AJUSTE DEL PRIMER MES QUE TOME LO QUE QUEDA EN BASE Y NO CALCULO
        ELSE 1
      END *
      CASE
        WHEN T0.PERIODO > T1.PERIODO THEN 0  --  meses mayores al calculado SE QUITAN
        WHEN T0.PERIODO = T1.PERIODO THEN 1 * T0.PERIODO --  calculo del mes
        WHEN T0.PERIODO = T1.PERIODO-1 THEN -1 * (T0.PERIODO) -- cancela calculo acumulado mes anterior
        # WHEN T0.PERIODO <= T1.PERIODO-1 AND T0.PERIODO = {mes_captura} AND T0.VERSION = '{params['version'][0]}' THEN -1 --  cancela calculos anteriores al mes captura, para ajuste acumulado
        # WHEN T0.PERIODO = T1.PERIODO-1 AND T1.PERIODO > {mes_captura+1} AND T0.VERSION = '{params['version'][0]}' THEN -1 * (T0.PERIODO) --  mes anterior al base, hace la cancelación del calculo acumulado posterior al mes captura
      ELSE 0 END * IMPORTE
      AS IMPORTE,

      CASE WHEN T1.PERIODO = T0.PERIODO OR T1.PERIODO-1 = T0.PERIODO THEN sueldo else 0 END As SUELDO

      FROM BONO_AGRUP T0
      CROSS JOIN {project_id}.PARAMETROS_PRESUPUESTO.MESES_COMPLETOS T1

      --where  T1.PERIODO >= {mes_captura}
      )

      SELECT CUENTA, CALCULO.CECO, CALCULO.CEBE, PERIODO, EJERCICIO, VERSION,"NOM_BONO_ESP" as ORIGEN, POSICION,
      PERIODO_BASE,
      CALCULO.DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS,
      SUM(IMPORTE) as IMPORTE, SUELDO as SUELDO

      --SELECT *
      FROM CALCULO
      LEFT JOIN {CECOS} T0 ON CALCULO.CECO=T0.CECO

      WHERE {filtro}

      group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,16

'''

client.query(query_bono).result()
print("Creado ",tablabonobase)

# ⚡Base Presupuestal (Forecast)
Se realiza tabla por negocio

❗ Se puede ejecutar las veces necesarias, solo ejecutar el negocio requerido


# PPTO_V1  Base Presupuestal (Forecast)🆗
❗ Se puede ejecutar las veces necesarias, solo ejecutar el negocio requerido
Base PPTO_V1:

PLAN + headcount + CARGA_INICIAL


Nota:

  Gasto sin CARGA_INICIAL, sin gastos de personal, aplica la regla

  Si el REAL < ME then VAR% PPTO vs ME SOBRE REAL
  Si el REAL > ME then PPTO

PPTO_V1 SSCC

In [ ]:
##### SCRIPT

query_forecast = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS

{scipt_delete}

DELETE FROM `{forecast_sscc}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {forecast_sscc}

  WITH

  BASE_NONOM_NOPRE AS (
    SELECT T0.CUENTA, CECO, CEBE, PERIODO, Ejercicio, Version, Importe

    FROM `{ER}` T0
    LEFT JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')
    LEFT JOIN `{CUENTAS_CARGA_INICIAL}` T2 ON T0.Cuenta=T2.CUENTA

    WHERE (
      (T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase})) -- PLAN DE ESTE AÑO
      OR
      (T0.VERSION IN ('Reales','PPTO_V2') AND T0.Ejercicio IN ({aniobase-1}))  -- REAL Y ME AÑO ANTERIOR
    )
    AND T2.Cuenta IS NULL --- NO CARGA_INICIAL
    AND T3.CUENTA_NOM IS NULL  ---NO CUENTAS DE NOMINA CON REGLA DE CALCULO
    AND {filtro} --- A QUE NEGOCIOS SE APLICA
  ),

  ULTIMO_REAL_ANUAL AS (
    SELECT CUENTA, CECO, CEBE, SUM(Importe) as IMPORTE
    FROM BASE_NONOM_NOPRE
    WHERE VERSION = 'Reales'
    GROUP BY 1,2,3
  ),

  ME_ANUAL AS (
    SELECT CUENTA, CECO, CEBE, SUM(Importe) as IMPORTE
    FROM BASE_NONOM_NOPRE
    WHERE VERSION = 'PPTO_V2'
    GROUP BY 1,2,3
  ),

  PPTO_ANUAL AS (
    SELECT CUENTA, CECO, CEBE, SUM(Importe) as IMPORTE
    FROM BASE_NONOM_NOPRE
    WHERE VERSION = 'PPTO'
    GROUP BY 1,2,3
  ),

  ###PRIMERO IDENTIFICAR LA VAR REAL vs ME para determinar si REAL < ME
  REAL_ME AS(
    SELECT
    COALESCE(ULTIMO_REAL_ANUAL.CUENTA, ME_ANUAL.CUENTA) as CUENTA,
    COALESCE(ULTIMO_REAL_ANUAL.CECO, ME_ANUAL.CECO) as CECO,
    COALESCE(ULTIMO_REAL_ANUAL.CEBE, ME_ANUAL.CEBE) as CEBE,
    ULTIMO_REAL_ANUAL.IMPORTE as IMPORTE_REAL,
    CASE WHEN IFNULL(ULTIMO_REAL_ANUAL.IMPORTE,0) < IFNULL(ME_ANUAL.IMPORTE,0) THEN 1 ELSE 0 END as PARAMETRO

    FROM ULTIMO_REAL_ANUAL
    FULL OUTER JOIN ME_ANUAL USING(CUENTA,CECO,CEBE)
  ),

  ###DETERMINAR VARIACION PLAN vs ME ese monto se aplica a real si REAL < ME
  PPTO_O_INC AS (
    SELECT
    COALESCE(PPTO_ANUAL.CUENTA, ME_ANUAL.CUENTA) as CUENTA,
    COALESCE(PPTO_ANUAL.CECO, ME_ANUAL.CECO) as CECO,
    COALESCE(PPTO_ANUAL.CEBE, ME_ANUAL.CEBE) as CEBE,
    CASE WHEN IFNULL(ME_ANUAL.IMPORTE,0) = 0 THEN 0 ELSE PPTO_ANUAL.IMPORTE / ME_ANUAL.IMPORTE -1 END as VARIACION,
    REAL_ME.PARAMETRO

    FROM PPTO_ANUAL
    FULL OUTER JOIN ME_ANUAL USING(CUENTA,CECO,CEBE)
    FULL OUTER JOIN REAL_ME USING(CUENTA,CECO,CEBE)
  ),


  CALCULO AS (
--PPTO SIN CARGA_INICIAL, NI NOMINA, NI BONO_ESP

SELECT T0.CUENTA, T0.CECO, T0.CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
SAFE_CAST(CASE WHEN PERIODO >= {mes_captura} AND PPTO_O_INC.PARAMETRO = 1 THEN 0 ELSE IMPORTE END AS NUMERIC) as Importe,
'FORECAST' as ORIGEN

FROM BASE_NONOM_NOPRE T0
LEFT JOIN PPTO_O_INC USING(CUENTA,CECO,CEBE)

WHERE T0.VERSION IN ('PPTO')

UNION ALL

---AJUSTE MAYO DE MESES QUE NO SE MOVERAN

SELECT T0.CUENTA, T0.CECO, T0.CEBE, {mes_captura} as PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
IMPORTE * -1 as Importe,
'FORECAST' as ORIGEN

FROM BASE_NONOM_NOPRE T0
FULL OUTER JOIN PPTO_O_INC USING(CUENTA,CECO,CEBE)

WHERE T0.VERSION IN ('PPTO') AND PPTO_O_INC.PARAMETRO = 1 AND PERIODO < {mes_captura}

UNION ALL

--REAL + INC SIN CARGA_INICIAL, NI NOMINA, NI BONO_ESP

SELECT T0.CUENTA, T0.CECO, T0.CEBE,
  CASE WHEN PERIODO < {mes_captura} THEN {mes_captura} ELSE PERIODO END as PERIODO,
  {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
  Importe * (1+PPTO_O_INC.VARIACION) as Importe,
'FORECAST' as ORIGEN

FROM BASE_NONOM_NOPRE T0
LEFT JOIN PPTO_O_INC USING(CUENTA,CECO,CEBE)

WHERE T0.VERSION IN ('Reales') AND PPTO_O_INC.PARAMETRO = 1

UNION ALL

--PRIMEROS MESES NOMINA
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND {filtro}

UNION ALL

--QUITAR PRIMEROS MESES NOMINA AJUSTE PRIMER MES
SELECT T0.CUENTA, CECO, CEBE, {mes_captura} as PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe * -1 as Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND {filtro}


)

SELECT CUENTA, CECO, CEBE, PERIODO, Ejercicio, Version,
SAFE_CAST(IFNULL(IMPORTE,0) AS NUMERIC) as  Importe,
'FORECAST_SSCC' as ORIGEN
FROM CALCULO
;
'''

client.query(query_forecast).result()
print("Tabla creada: ",forecast_sscc)

PPTO_V1 NEGOCIO Z E NEGOCIO T

In [ ]:
query_forecast = f'''
CREATE OR REPLACE TABLE {forecast_fin_ifrs} AS
(
  WITH CALCULO AS (
--PPTO SIN CARGA_INICIAL NI, NOMINA, NI BONO_ESP

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
LEFT JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')
LEFT JOIN `{CUENTAS_CARGA_INICIAL}` T2 ON T0.Cuenta=T2.CUENTA

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase})
AND
T2.Cuenta IS NULL AND T3.CUENTA_NOM IS NULL AND T0.Cuenta <> '51000403' AND
DIMENSION1 IN ('NEGOCIO Z','NEGOCIO T')

UNION ALL
--PRIMEROS MESES NOMINA
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND DIMENSION1 IN ('NEGOCIO Z','NEGOCIO T')

UNION ALL

--PRIMEROS MESES NOMINA AJUSTE PRIMER MES
SELECT T0.CUENTA, CECO, CEBE, {mes_captura} as PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe * -1 as Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND DIMENSION1 IN ('NEGOCIO Z','NEGOCIO T')

)

SELECT * EXCEPT(ORIGEN,IMPORTE), SAFE_CAST(IFNULL(IMPORTE,0) AS NUMERIC) as IMPORTE, 'FORECAST_NEGOCIO Z_IFRS' as ORIGEN
FROM CALCULO
)
'''

client.query(query_forecast).result()
print("Tabla creada: ",forecast_fin_ifrs)

PPTO_V1 NEGOCIO Y

In [ ]:
query_forecast = f'''
CREATE OR REPLACE TABLE {forecast_inmob} AS
(
  WITH CALCULO AS (
--PPTO COMPLETO sin CARGA_INICIAL, NI NOMINA, NI CEDULAS, Y PRORATEO DE ALGUNAS DIMENSION5CACIONES

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
CASE WHEN T0.DIMENSION1 = 'NEGOCIO Y' AND T0.DESCDIMENSION5 IN ('Metepec II') AND PERIODO >= {mes_captura} THEN 0 ELSE Importe END as IMPORTE, 'FORECAST' as ORIGEN

FROM `{ER}` T0
LEFT JOIN `{JERARCUENTAPPTO}` T1 ON T0.Cuenta=T1.CUENTA
left JOIN `{CUENTAS_CARGA_INICIAL}` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{CUENTAS_CED_INM_ADC}` T4 ON T4.CUENTA=T0.CUENTA AND T4.DIMENSION1 = 'NEGOCIO Y'
LEFT JOIN `{CUENTAS_PRO_ADC}` T5 ON T5.CUENTA=T0.CUENTA AND T0.DIMENSION1=T5.DIMENSION1
                                    AND DescDIMENSION5 NOT IN ('Angelópolis','P Tepeyac','Satélite')
LEFT JOIN `{CUENTAS_REGLAS_NOM}` T7 ON T0.Cuenta=SUBSTRING(T7.CUENTA_NOM,1,8) AND T7.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase})
and t3.cuenta is null --CARGA_INICIAL
AND T4.CUENTA IS NULL --CEDULAS
AND T5.CUENTA IS NULL --PRORRATEO
AND T7.CUENTA_NOM IS NULL --NOM MODELO SSCC

AND (t0.DIMENSION1 IN ('NEGOCIO Y') OR DIMENSION2 = 'Fideicomisos')

UNION ALL
--PRIMEROS MESES CEDULAS
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
LEFT JOIN `{CUENTAS_CED_INM_ADC}` T4 ON T4.CUENTA=T0.CUENTA AND T4.DIMENSION1 = 'NEGOCIO Y'
LEFT JOIN `{CUENTAS_PRO_ADC}` T5 ON T5.CUENTA=T0.CUENTA AND T0.DIMENSION1=T5.DIMENSION1
                                    AND DescDIMENSION5 NOT IN ('Angelópolis','P Tepeyac','Satélite')

WHERE (T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND
(T4.CUENTA IS NOT NULL --CEDULAS INMOB
OR T5.CUENTA IS NOT NULL -- PRORRATEO
)

AND (t0.DIMENSION1 IN ('NEGOCIO Y') OR DIMENSION2 = 'Fideicomisos')
)


UNION ALL

--PRIMEROS MESES NOMINA
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND (t0.DIMENSION1 IN ('NEGOCIO Y') OR DIMENSION2 = 'Fideicomisos')

UNION ALL

--PRIMEROS MESES NOMINA AJUSTE PRIMER MES
SELECT T0.CUENTA, CECO, CEBE, {mes_captura} as PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe * -1 as Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND (t0.DIMENSION1 IN ('NEGOCIO Y') OR DIMENSION2 = 'Fideicomisos')

)

SELECT * EXCEPT(ORIGEN), 'FORECAST_NEGOCIO Y' as ORIGEN
FROM CALCULO
)
'''

client.query(query_forecast).result()
print("Tabla creada: ",forecast_inmob)

PPTO_V1 NEGOCIO X

In [ ]:
query_forecast = f'''
CREATE OR REPLACE TABLE {forecast_NEGOCIO X} AS
(WITH CALCULO AS (
--PRIMEROS MESES NOMINA
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura} AND
DIMENSION1 IN ('NEGOCIO X y CC Externos') AND DIMENSION2 <> 'Fideicomisos'

--PPTO COMPLETO sin CARGA_INICIAL, NI NOMINA, NI CEDULAS, Y PRORATEO DE ALGUNAS DIMENSION5CACIONES

UNION ALL

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
LEFT JOIN `{JERARCUENTAPPTO}` T1 ON T0.Cuenta=T1.CUENTA
left JOIN `{CUENTAS_CARGA_INICIAL}` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{CUENTAS_CED_INM_ADC}` T4 ON T4.CUENTA=T0.CUENTA AND T0.DIMENSION1=T4.DIMENSION1
LEFT JOIN `{CUENTAS_PRO_ADC}` T5 ON T5.CUENTA=T0.CUENTA AND T0.DIMENSION1=T5.DIMENSION1
                                    AND DescDIMENSION5 NOT IN ('Angelópolis','P Tepeyac','Satélite')
LEFT JOIN `{nomina_mapeo_cuentas}` T7 ON T0.Cuenta=T7.CUENTA

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase})
and t3.cuenta is null --CARGA_INICIAL
AND T4.CUENTA IS NULL --CEDULAS
AND T5.CUENTA IS NULL --PRORRATEO
AND T7.CUENTA IS NULL --NOM MODELO ETZ

AND t0.DIMENSION1 IN ('NEGOCIO X y CC Externos') AND DIMENSION2 <> 'Fideicomisos'

UNION ALL
--PRIMEROS MESES CEDULAS
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
LEFT JOIN `{CUENTAS_CED_INM_ADC}` T4 ON T4.CUENTA=T0.CUENTA AND T0.DIMENSION1=T4.DIMENSION1
LEFT JOIN `{CUENTAS_PRO_ADC}` T5 ON T5.CUENTA=T0.CUENTA AND T0.DIMENSION1=T5.DIMENSION1
                                    AND DescDIMENSION5 NOT IN ('Angelópolis','P Tepeyac','Satélite')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND
(T4.CUENTA IS NOT NULL --CEDULAS
OR T5.CUENTA IS NOT NULL -- PRORRATEO
)
AND t0.DIMENSION1 IN ('NEGOCIO X y CC Externos') AND DIMENSION2 <> 'Fideicomisos'

--------PARA EL MODELO NOMINA SSCC

UNION ALL

--PRIMEROS MESES NOMINA
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND DIMENSION1 IN ('NEGOCIO X y CC Externos')

UNION ALL

--PRIMEROS MESES NOMINA AJUSTE PRIMER MES
SELECT T0.CUENTA, CECO, CEBE, {mes_captura} as PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe * -1 as Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0
INNER JOIN `{CUENTAS_REGLAS_NOM}` T3 ON T0.Cuenta=SUBSTRING(T3.CUENTA_NOM,1,8) AND T3.REGLA3 NOT IN ('PRECARGADA','NO CONSIDERAR')

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND (t0.DIMENSION1 IN ('NEGOCIO X y CC Externos'))
)


SELECT * EXCEPT(ORIGEN), 'FORECAST_NEGOCIO X' as ORIGEN
FROM CALCULO
)
'''

client.query(query_forecast).result()
print("Tabla creada: ",forecast_NEGOCIO X)

**AJUSTE PIEDRA**

In [ ]:
query_forecast_no_mas = f'''
CREATE OR REPLACE TABLE {forecast_nomascara} AS
(
  WITH CALCULO AS (
--PPTO SIN CARGA_INICIAL NI, NOMINA, NI BONO_ESP

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND DIMENSION1 NOT IN UNNEST ({direcciones})

UNION ALL

--QUITAR PRIMEROS MESES NOMINA AJUSTE PRIMER MES
SELECT T0.CUENTA, CECO, CEBE, {mes_captura} as PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe * -1 as Importe, 'FORECAST' as ORIGEN

FROM `{ER}` T0

WHERE T0.VERSION IN ('PPTO') AND T0.Ejercicio IN ({aniobase}) AND Periodo < {mes_captura}
AND DIMENSION1 NOT IN UNNEST ({direcciones})

)

SELECT * EXCEPT(ORIGEN), 'FORECAST_NOMASCARA' as ORIGEN
FROM CALCULO
)
'''

client.query(query_forecast_no_mas).result()
print("Tabla creada: ",forecast_nomascara)

# PPTO_V2 Base Presupuestal (Forecast)
❗ Se puede ejecutar las veces necesarias, solo ejecutar el negocio requerido
Base PPTO_V2:

Reales + headcount + CARGA_INICIAL

PPTO_V2 SSCC

In [ ]:
tabla = f'''{base_presupuesta_id}_SSCC'''

query_forecast = f'''
--REALES ACTUALES
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T3 ON T0.Cuenta=T3.Cuenta_origen


WHERE T0.VERSION IN ('Reales') AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo <= {mesbaser}
AND
DIMENSION1 IN ('Servicios Compartidos')

UNION ALL

--CARGA_INICIAL EJERCICIO ACTUAL

SELECT T0.*

FROM {tablaCARGA_INICIAL} T0
LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO

WHERE T1.DIMENSION1 IN ('Servicios Compartidos')

UNION ALL

--PPTO (NO NOMINA) por REALES + PPTO

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CARGA_INICIAL` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T4 ON T0.Cuenta=T4.Cuenta_origen

WHERE T0.VERSION IN ('PPTO')
AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo > {mesbaser}
AND T3.CUENTA IS NULL AND (T4.Cuenta_origen IS NULL AND T0.Cuenta<>'51000403') --SIN CARGA_INICIAL NI NOMINA NI BONO
AND DIMENSION1 IN ('Servicios Compartidos')

'''

BASEP_FORECAST = bpd.read_gbq(query_forecast)

BASEP_FORECAST_pd = pd.DataFrame(BASEP_FORECAST)

BASEP_FORECAST_pd.columns = BASEP_FORECAST.columns

#CARGA A BIGQUERY

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    BASEP_FORECAST_pd, tabla, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tabla)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

PPTO_V2 NEGOCIO Z e NEGOCIO T

In [ ]:
tabla = f'''{base_presupuesta_id}_NEGOCIO Z_NEGOCIO T'''

query_forecast = f'''
--REALES ACTUALES
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T3 ON T0.Cuenta=T3.Cuenta_origen


WHERE T0.VERSION IN ('Reales') AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo <= {mesbaser}
AND
DIMENSION1 IN ('NEGOCIO Z','NEGOCIO T')

UNION ALL

--CARGA_INICIAL EJERCICIO ACTUAL

SELECT T0.*

FROM {tablaCARGA_INICIAL} T0
LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO

WHERE T1.DIMENSION1 IN ('NEGOCIO Z','NEGOCIO T')

UNION ALL

--PPTO (NO NOMINA) por REALES + PPTO

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CARGA_INICIAL` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T4 ON T0.Cuenta=T4.Cuenta_origen

WHERE T0.VERSION IN ('PPTO')
AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo > {mesbaser}
AND T3.CUENTA IS NULL AND (T4.Cuenta_origen IS NULL AND T0.Cuenta<>'51000403') --SIN CARGA_INICIAL NI NOMINA NI BONO
AND DIMENSION1 IN ('NEGOCIO Z','NEGOCIO T')

'''

BASEP_FORECAST = bpd.read_gbq(query_forecast)

BASEP_FORECAST_pd = pd.DataFrame(BASEP_FORECAST)

BASEP_FORECAST_pd.columns = BASEP_FORECAST.columns

#CARGA A BIGQUERY

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    BASEP_FORECAST_pd, tabla, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tabla)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

PPTO_V2 NEGOCIO Y

In [ ]:
tabla = f'''{base_presupuesta_id}_NEGOCIO Y'''

query_forecast = f'''--REALES ACTUALES
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T3 ON T0.Cuenta=T3.Cuenta_origen


WHERE T0.VERSION IN ('Reales') AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo <= {mesbaser}
AND
DIMENSION1 IN ('NEGOCIO Y')

UNION ALL

--CARGA_INICIAL EJERCICIO ACTUAL

SELECT T0.*

FROM {tablaCARGA_INICIAL} T0
LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO

WHERE T1.DIMENSION1 IN ('NEGOCIO Y')

UNION ALL

--PPTO (NO NOMINA) por REALES + PPTO NO CEDULAS

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CARGA_INICIAL` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T4 ON T0.Cuenta=T4.Cuenta_origen
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CED_NEGOCIO X_INMOB` T5 ON T0.DIMENSION1 = T5.DIMENSION1 AND T0.CUENTA = T5.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_PRORRATEO_NEGOCIO X_INMOB` T6 ON T0.DIMENSION1 = T6.DIMENSION1 AND T0.CUENTA = T6.CUENTA

WHERE T0.VERSION IN ('PPTO')
AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo > {mesbaser}
AND T3.CUENTA IS NULL AND (T4.Cuenta_origen IS NULL AND T0.Cuenta<>'51000403') --SIN CARGA_INICIAL NI NOMINA NI BONO
AND T5.CUENTA IS NULL
AND T6.CUENTA IS NULL
AND T0.DIMENSION1 IN ('NEGOCIO Y')
'''

BASEP_FORECAST = bpd.read_gbq(query_forecast)

BASEP_FORECAST_pd = pd.DataFrame(BASEP_FORECAST)

BASEP_FORECAST_pd.columns = BASEP_FORECAST.columns

#CARGA A BIGQUERY

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    BASEP_FORECAST_pd, tabla, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tabla)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

PPTO_V2 NEGOCIO X

In [ ]:
tabla = f'''{base_presupuesta_id}_NEGOCIO X'''

query_forecast = f'''
--REALES ACTUALES
SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
# LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T3 ON T0.Cuenta=T3.Cuenta_origen


WHERE T0.VERSION IN ('Reales') AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo <= {mesbaser}
AND
DIMENSION1 IN ('NEGOCIO X y CC Externos')

UNION ALL

--CARGA_INICIAL EJERCICIO ACTUAL

SELECT T0.*

FROM {tablaCARGA_INICIAL} T0
LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO

WHERE T1.DIMENSION1 IN ('NEGOCIO X y CC Externos')

UNION ALL

--PPTO (NO NOMINA) por REALES + PPTO

SELECT T0.CUENTA, CECO, CEBE, PERIODO, {aniobase} as `Ejercicio`,'{params['version'][0]}' as `Version`,
Importe, 'FORECAST' as ORIGEN

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CARGA_INICIAL` T3 ON T0.Cuenta=T3.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T4 ON T0.Cuenta=T4.Cuenta_origen
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CED_NEGOCIO X_INMOB` T5 ON T0.DIMENSION1 = T5.DIMENSION1 AND T0.CUENTA = T5.CUENTA
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_PRORRATEO_NEGOCIO X_INMOB` T6 ON T0.DIMENSION1 = T6.DIMENSION1 AND T0.CUENTA = T6.CUENTA

WHERE T0.VERSION IN ('PPTO')
AND T0.Ejercicio IN ({aniobase}) AND T0.Periodo > {mesbaser}
AND T3.CUENTA IS NULL AND (T4.Cuenta_origen IS NULL AND T0.Cuenta<>'51000403') --SIN CARGA_INICIAL NI NOMINA NI BONO
AND T5.CUENTA IS NULL
AND T6.CUENTA IS NULL
AND T0.DIMENSION1 IN ('NEGOCIO X y CC Externos')

'''

BASEP_FORECAST = bpd.read_gbq(query_forecast)

BASEP_FORECAST_pd = pd.DataFrame(BASEP_FORECAST)

BASEP_FORECAST_pd.columns = BASEP_FORECAST.columns

#CARGA A BIGQUERY

# Trabajo de carga
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )

# Carga

load_job = client.load_table_from_dataframe(
    BASEP_FORECAST_pd, tabla, job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(tabla)  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

# **Forecast Plan SSCC** ⏰

⭐ Solo en PLAN.

- Base PPTO_V2 (relaes + ppto) [cuentas que no son "NO PPTO", ni precargas ni nómina] y CN PPTO_V2 aplica inflación, lo suma a CARGA_INICIAL y nómina para FORECAST

En Stored Procedure se sustituye

In [ ]:
query_forecast_sscc = f'''CREATE OR REPLACE TABLE {base_presupuestal_sscc} AS
(
  WITH BASE_PPTO_V2 AS (

    SELECT T0.*
    FROM `{base_presupuesta_id}*` T0 --FORECAST CON REGLAS DE NEGOCIO
    LEFT JOIN `{CECOS}` T1 ON T0.CECO=T1.CECO
    LEFT JOIN `{CEBES}` T2 ON T0.CEBE=T2.CEBE
    WHERE T0.VERSION = 'PPTO_V2' AND COALESCE(T1.DIMENSION1,T2.DIMENSION1) IN ('Servicios Compartidos')


  )

  SELECT T0.Cuenta,T0.CECO, T0.CEBE,
  PERIODO,
  {aniobase+1} as EJERCICIO,
  'PPTO_V0' as VERSION
  ,SAFE_CAST(Importe*(1+{inflacionplan}) AS NUMERIC) as Importe,
  'FORECAST_SSCC' as ORIGEN

  FROM BASE_PPTO_V2 T0
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_CARGA_INICIAL` T1 ON SUBSTRING(T0.CUENTA,1,8)=T1.CUENTA
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_NOMINA` T3 ON SUBSTRING(T0.CUENTA,1,8)=T3.Cuenta_origen
  LEFT JOIN `{project_id}.mus_qas_drv_datos_maestros.Cuentas_DM` T4 ON SUBSTRING(T0.CUENTA,1,8)=T4.Cuenta  -- PENDIENTE TABLA
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T5 ON SUBSTRING(T0.CUENTA,1,8)=T5.CUENTA

  WHERE
  IMPORTE <> 0 AND
  (--NO PRECARGADA
    T1.CUENTA IS NULL
    --NO NOMINA
    AND T3.Cuenta_origen IS NULL
    --NO BONO_ESP
    AND T0.Cuenta<>'51000403'
    --CLASE CUENTA
    AND UPPER(T4.`Modificación`) IN ('USUARIOS','NEGOCIO Z')
    AND SUBSTRING(T0.CUENTA,1,8) NOT IN ('44000008','44000200'))
    )
'''

client.query(query_forecast_sscc).result()
print("Creado ",base_presupuestal_sscc)

# **ISN** ☝NÓMINA (GASTOS DE PERSONAL)
Se toma la suma de las cuentas que integran el SBC y se multiplica por el % de ISN

In [ ]:
query = f'''
----DETERMINAR CECOS Y CEBES DE LO FILTRADO PARA BORRARLOS, YA QUE LA TABLA NO TIENE ATRIBUTOS

{scipt_delete}

DELETE FROM `{tablanompres_isn}` t
WHERE t.CECO IN (SELECT id FROM filtered_ids WHERE type = 'CECO')
   OR t.CEBE IN (SELECT id FROM filtered_ids WHERE type = 'CEBE');

---UNA VEZ BORRADOS, INSERTAR EL CALCULO

INSERT INTO {tablanompres_isn}

  WITH NOMINA AS (
    SELECT CUENTA, CECO, CEBE, PERIODO, VERSION, EJERCICIO, SUM(IMPORTE) as IMPORTE

    FROM {tablanomina}

    GROUP BY 1,2,3,4,5,6

    UNION ALL

    SELECT CUENTA, CECO, CEBE, PERIODO, VERSION, EJERCICIO, SUM(IMPORTE) as IMPORTE

    FROM `{tablanompres}*`

    WHERE ORIGEN NOT IN ('NOM_PRE_ISN')
    GROUP BY 1,2,3,4,5,6

    UNION ALL

    SELECT CUENTA, CECO, CEBE, PERIODO, VERSION, EJERCICIO, SUM(IMPORTE) as IMPORTE

    FROM {tablabonobase}

    GROUP BY 1,2,3,4,5,6

    UNION ALL

    SELECT CUENTA, CECO, CEBE, PERIODO, VERSION, EJERCICIO, SUM(IMPORTE) as IMPORTE

    FROM {tablaCARGA_INICIAL}

    GROUP BY 1,2,3,4,5,6

    UNION ALL

    SELECT CUENTA, CECO, CEBE, PERIODO, VERSION, EJERCICIO, SUM(IMPORTE) as IMPORTE

    FROM `{base_presupuesta_id}*`

    GROUP BY 1,2,3,4,5,6
  ),

  SBC AS (
    SELECT CECO, CEBE, PERIODO, VERSION, EJERCICIO, SUM(IMPORTE) AS IMPORTE

    FROM NOMINA
    INNER JOIN {CUENTAS_SBC} CUENTAS_SBC ON NOMINA.CUENTA = CUENTAS_SBC.CUENTA

    GROUP BY 1,2,3,4,5
  )

  SELECT '51000053' as CUENTA, SBC.CECO, SBC.CEBE,
  SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
  SAFE_CAST(
    CASE WHEN PERIODO <= {mesbaser} AND EJERCICIO = {aniobase} THEN {mes_captura} ELSE PERIODO END
     AS INT64) AS PERIODO,
  VERSION,
  SAFE_CAST(SUM(IMPORTE) AS NUMERIC) as Sueldos, SAFE_CAST({isn_cdmx} AS NUMERIC) as PORC_PREST, 'SBC' as DIMENSION6,
  0 as MES_INC, 0.0 as PORC_INC, 0 as MES_BASE,
  SAFE_CAST(SUM(IMPORTE) AS NUMERIC) * SAFE_CAST({isn_cdmx} AS NUMERIC) AS IMPORTE, 'NOM_PRE_ISN' as ORIGEN

  FROM SBC
  INNER JOIN {CECOS} CECOS ON SBC.CECO=CECOS.CECO
  INNER JOIN {DIMENSION5S_CORP} DIMENSION5S_CORP ON CECOS.DIMENSION5=DIMENSION5S_CORP.DIMENSION5 AND DIMENSION5S_CORP.CONSIDERAR = ''

  WHERE {filtro.replace("T0.","CECOS.")}


  GROUP BY 1,2,3,4,5,6,8,9,10,11,12

  UNION ALL

  SELECT '51000053' as Cuenta, CeCo, CeBe,
  SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
  SAFE_CAST(PERIODO AS INT64) as Periodo,
  '{params['version'][0]}' as Version,
  0 as Sueldos,
  0 as PORC_PREST,
  '' as DIMENSION6,
  0 as MES_INC,
  0.0 as PORC_INC,
  0 as MES_BASE,
  SAFE_CAST(IFNULL(Importe,0) AS NUMERIC) AS IMPORTE,
  "NOM_PRE_porc_ant" as ORIGEN

    FROM `{ER}` T0
    INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
    LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

    WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
    AND CUENTA_REPORTE = '51000053'
      AND {filtro}

  UNION ALL

  SELECT '51000053' as Cuenta, CeCo, CeBe,
  SAFE_CAST(Ejercicio AS INT64) as Ejercicio,
  SAFE_CAST({mes_captura} AS INT64) as Periodo,
  '{params['version'][0]}' as Version,
  0 as Sueldos,
  0 as PORC_PREST,
  '' as DIMENSION6,
  0 as MES_INC,
  0.0 as PORC_INC,
  0 as MES_BASE,
  SAFE_CAST(IFNULL(Importe,0) *-1 AS NUMERIC) AS IMPORTE,
  "NOM_PRE_porc_ant" as ORIGEN

    FROM `{ER}` T0
    INNER JOIN {tablareglasnom} T7 ON T0.CUENTA = SUBSTRING(T7.CUENTA_NOM,1,8)
    LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

    WHERE Ejercicio IN ( {aniobase} ) AND VERSION IN ('Reales') AND PERIODO <= {mesbasehc}
    AND CUENTA_REPORTE = '51000053'
      AND {filtro}
;

'''

client.query(query).result()
print("Creado ",tablanompres_isn)

# 🚀 **Stored Procedures**
⭐ Cálculos dinámicos, porque se realizan a partir de conceptos

# **TABLAS BASE SP🆗**

⭐ Primero se crean tablas vacías, para que se puedan crear las vistas, como se reutilizará el dataset, ejercicio con ejercicipo, se "limpiarán" al inicio del ejercicio presupuestal

In [ ]:
schema_hccn= [
        bigquery.SchemaField("CECO", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("POSICION", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("SUELDO_PROMEDIO", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION6", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("PERSONAS", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("MES_DE_INGRESO", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("EJERCICIO_DE_INGRESO", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("SUELDO_A_UTILIZAR", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("COMENTARIOS", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("MANDANTE", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("TIMESTAMP", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("PATH", "STRING", mode="REQUIRED")
    ]

schema_cn = [
        bigquery.SchemaField('Validacion1', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Validacion2', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION1', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION2', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION3', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION6', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION7', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION5cacion', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Naturaleza', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Cuenta', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('CeCo_CeBe', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Ejercicio', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Mes', 'INT64', mode="REQUIRED"),
        bigquery.SchemaField('Grupo', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Concepto', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Comentarios', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Monto_CN', 'FLOAT', mode="REQUIRED"),
        bigquery.SchemaField('X', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('AUT_ARE', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('AUT_CORP', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('AUT_GEN', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Autorizado', 'FLOAT', mode="REQUIRED"),
        bigquery.SchemaField("MANDANTE", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("TIMESTAMP", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("PATH", "STRING", mode="REQUIRED")
]

schema_gen_cn= [
        bigquery.SchemaField("KEY", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION1", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION2", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION3", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION6", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION7", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("DIMENSION5cacion", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("Cuenta", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("Ejercicio", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("Version", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("Monto_CN", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("MANDANTE", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("TIMESTAMP", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("PATH", "STRING", mode="REQUIRED")
    ]

schema_cn_ger= [
        bigquery.SchemaField('Validacion1', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Validacion2', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION1', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION2', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION3', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION6', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION7', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('DIMENSION5cacion', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Naturaleza', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Cuenta', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Ejercicio', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Grupo', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Concepto', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Comentarios', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Monto_CN', 'FLOAT', mode="REQUIRED"),
        bigquery.SchemaField('X', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('AUT_ARE', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('AUT_CORP', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('AUT_GEN', 'STRING', mode="REQUIRED"),
        bigquery.SchemaField('Autorizado', 'FLOAT', mode="REQUIRED"),
        bigquery.SchemaField("MANDANTE", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("TIMESTAMP", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("PATH", "STRING", mode="REQUIRED")
    ]

schema_sp = [
    bigquery.SchemaField('Cuenta', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CeCo', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CeBe', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION1', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION2', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION3', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION6_MAESTRO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION5', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DES_DIMENSION5', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NIVEL1', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NIVEL2', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NIVEL3', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NIVEL4', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NIVEL5', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('Ejercicio', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('Periodo', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('Version', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('MES_INC', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('PORC_INC', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('MES_BASE', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('Importe', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('GRUPO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CONCEPTO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('Personas', 'FLOAT64', mode='REQUIRED'),
    bigquery.SchemaField('AREA_PERSONAL', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('POSICION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('Importe_vales', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('MONTO', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('Sueldos', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('PORC_PREST', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION6', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PERIODO_BASE', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('EJERCICIO_DE_INGRESO', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('MES_DE_INGRESO', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('PARAMETROS', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO', 'NUMERIC', mode='REQUIRED'),
    bigquery.SchemaField('ORIGEN', 'STRING', mode='REQUIRED')
    ]


schemas = [
    # schema_hccn,
    # schema_cn,  # CON CECO
    # schema_gen_cn, # DE MASCARA
    # schema_cn_ger, # SIN CECO

    # schema_sp,
    # schema_sp,
    # schema_sp,
    # schema_sp,
    # schema_sp,
    # schema_sp,
    # schema_sp,
    # schema_sp,
    # schema_sp
]

tables = [
    # tabla_base_hc
    # ,tabla_base_cn
    # ,tabla_base_gercn
    # ,tabla_base_gencn

    # ,tabla_base_gercn_sp
    # ,tabla_base_gencn_sp
    # ,tabla_base_cn_sp
    # ,tabla_base_hccn_sp
    # ,sp_CEDULAS
    # ,tabla_atributos_calc_inic
    # ,nomina_etz_cn
    # ,nomina_calculo
    # ,tabla_base_prorrateo
    # "{project_id}.intranet_bd_pruebas.SP_CNHC_2"
]


for d in range(len(schemas)) :
  try:
    table_id_update = bigquery.Table(tables[d], schema=schemas[d])
    table_id_update = client.create_table(table_id_update)  # Make an API request.
    print(
        "Created table {}.{}.{}".format(table_id_update.project, table_id_update.dataset_id, table_id_update.table_id)
    )

  except:
    client.delete_table(table_id_update, not_found_ok=True)  # Make an API request.
    print("Deleted table '{}'.".format(table_id_update))
    table_id_update = bigquery.Table(tables[d], schema=schemas[d])
    table_id_update = client.create_table(table_id_update)  # Make an API request.
    print(
        "Created table {}.{}.{}".format(table_id_update.project, table_id_update.dataset_id, table_id_update.table_id)
    )


###################################PARTICIONADAS

schema_cuotas=[
    bigquery.SchemaField('Cuenta', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Cuenta_Origen', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Ejercicio', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Version', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION6_original', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Naturaleza', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Periodo', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('DescDIMENSION5', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Importe_Original', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('Importe', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION6', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('CeBe', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('CeCo', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Markup', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO2', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO3', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO4', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO5', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('NEGOCIO Y', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DATE', 'TIMESTAMP', mode='NULLABLE'),
]

schema_prorrateo = [
    bigquery.SchemaField('Cuenta', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Cuenta_Origen', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Ejercicio', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Version', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION2', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Periodo', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('DescDIMENSION5', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Naturaleza', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION6_origen', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Gasto_Ingreso', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Importe_origen', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('Importe', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('CeBe', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('CeCo', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('Markup', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TIPO1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('NEGOCIO Y', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DATE', 'TIMESTAMP', mode='NULLABLE'),
]

schema_Posiciones_Presupuestales_HC = [
    bigquery.SchemaField('ID', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION1', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION2', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION3', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DES_DIMENSION5', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('POSICION', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('ORIGEN', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION6', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('PERSONAS', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('FECHA_INGRESO', 'DATE', mode='NULLABLE'),
    bigquery.SchemaField('CECO', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('FUNCION', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('COMENTARIOS', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('UDATE', 'TIMESTAMP', mode='NULLABLE'),
    bigquery.SchemaField('USER', 'STRING', mode='NULLABLE')
]

schemas_particionados = [
    # schema_prorrateo,
    # schema_prorrateo,
    # schema_cuotas,
    schema_Posiciones_Presupuestales_HC
]

tables_particionados = [
    # Validacion_Dist_Inmob,
    # Validacion_Prorrateo,
    # Validacion_Cuotas,
    tabla_hc_total
]

for d in range(len(tables_particionados)) :

    client = bigquery.Client()

    table_id = tables_particionados[d]

    table = bigquery.Table(table_id, schema=schemas_particionados[d])

    table.time_partitioning = bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.DAY,
        field="UDATE",
        require_partition_filter=True
    )

    table = client.create_table(table, exists_ok=True)
    print("Table ready:", table_id)

# **MODELO NOMINA**  😰 ⏰

⭐ Modelo de nómina para negocios específicos

In [ ]:
squema_detalle = [
    bigquery.SchemaField('MARCA', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION5CACION', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('POSICION', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('NUMPER', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('CECO', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('FECHAING', 'DATE', mode='NULLABLE'),
    bigquery.SchemaField('PORC_CECO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ORIGEN', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('NEGOCIO', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIMENSION6_PERSONAL', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('PORC_COMISION', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DIAS_PTU', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DIAS_AGUINALDO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('COMISION_DUMMY', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('por_incr_jul26', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('APLICA_MINIMO_GARANTIZADO', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('IMP_MINIMO_GARANTIZADO_ENE_2026', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('NOTAS', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('ESTADO', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('DIAS_PV_CORPO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SUELDO_TABULADOR', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('HC', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('VERSION', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('PERIODO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ANTIGUEDAD', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ANTIGUEDAD_MESES', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('DIAS_VACACIONES', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SUELDO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SUELDO_DIARIO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ASISTENCIA_DIAS_FESTIVOS', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DIAS_PTU_TOPE', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('UMA_DIARIA', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('FACTOR_UMA_SBC', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('MESES_BONO_ESP', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('VECES_FERIADO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PERIODOS_FERIADO', 'STRING', mode='NULLABLE'),
    bigquery.SchemaField('PLANTILLA_FERIADOS_DOM', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('MES_MIN_BONO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SAR_F00_PPTO_V1', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SAR_PLAN', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ENFERMEDAD_MATERNIDAD', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('INVALIDEZ_VIDA', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('RIESGO_TRABAJO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('GUARDERIA_PRES', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('VIVIENDA', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('MES_BASE', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PORC_PRIMA_VACACIONAL', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PORC_PRIMA_DOMINICAL', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('TOTAL_PERSONAS_MISMA_POSICION', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('PAGO_FESTIVO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('VENTA_FIS', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('VENTA_DIG', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('HC_DIMENSION5CACION', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('COMISION_ESTIMADA', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('APROVISIONAMIENTO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('VALES_DESPENSA', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PRIMA_DOMINICAL', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('DIASANIO', 'INTEGER', mode='NULLABLE'),
    bigquery.SchemaField('BONO_ESP', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ES_VACANTE', 'BOOLEAN', mode='NULLABLE'),
    bigquery.SchemaField('ES_EVENTUAL', 'BOOLEAN', mode='NULLABLE'),
    bigquery.SchemaField('SUELDO_DIC', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('AGUINALDO', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PTU', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PRIMA_VACACIONAL', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SBC', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PREMIOS_COMISIONES', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('SAR', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('IMSS', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('VIVIENDA_PATRON', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('ISN', 'FLOAT', mode='NULLABLE'),
    bigquery.SchemaField('PRIMA_ANTIGUEDAD', 'FLOAT', mode='NULLABLE'),
]

squema_nomina_detalle_cn = [
    bigquery.SchemaField('inflacionplan', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('aniobase', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('porc_inc', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('mesbaser', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('mesbasehc', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('mesinc', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('mesinc_vales', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('porc_inc_vales', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('porc_inc_vuniform', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('porc_inc_sig_anio', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('meses_bono', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('mes_captura', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('direcciones', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('porc_inc_seg_anio', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('direcciones_nomina', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('vales_via_planta', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('vales_via_planta_2', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('primera_version', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('versiones', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('divisiones_mne', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO_PROMEDIO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('COMENTARIOS', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('MANDANTE', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('TIMESTAMP', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PATH', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('SUELDOS_JUL25', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_EMERGENTE', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MONTERREY', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_FRONTERA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SUELDOS_ENE26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_EMERGENTE26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MONTERREY26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_FRONTERA26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('POR_INCR_JUL26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('APLICA_MINIMO_GARANTIZADO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('IMP_MINIMO_GARANTIZADO_ENE_2026', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('IMP_MINIMO_GARANTIZADO_2025', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('COMISION', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_PTU', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_AGUINALDO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('X', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NOTAS', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DES_DIMENSION5', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION5', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('ID', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('POSICION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NUMPER', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CECO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('AREA_PERS', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('GPOPERSDESC', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('FECHAING', 'DATE', mode='REQUIRED'),
    bigquery.SchemaField('PORC_CECO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ORIGEN', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NEGOCIO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('VALES', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION6_PERSONAL', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PORC_COMISION', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('COMISION_DUMMY', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('ESTADO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_PV_CORPO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO_TABULADOR', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('HC', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VERSION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('ANTIGUEDAD', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ANTIGUEDAD_MESES', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('ANTIGUEDAD_DIAS', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('ANTIGUEDAD_ANUAL', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('FECHA_PERIODO', 'DATE', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_VACACIONES', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PERIODO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO_DIARIO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ASISTENCIA_DIAS_FESTIVOS', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_PTU_TOPE', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('UMA_DIARIA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('FACTOR_UMA_SBC', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MESES_BONO_ESP', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VECES_FERIADO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PERIODOS_FERIADO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PLANTILLA_FERIADOS_DOM', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MES_MIN_BONO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SAR_F00_PPTO_V1', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SAR_PLAN', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ENFERMEDAD_MATERNIDAD', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('INVALIDEZ_VIDA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('RIESGO_TRABAJO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('GUARDERIA_PRES', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VIVIENDA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MES_BASE', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PORC_PRIMA_VACACIONAL', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PORC_PRIMA_DOMINICAL', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('TOTAL_PERSONAS_MISMA_POSICION', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('PAGO_FESTIVO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VENTA_FIS', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VENTA_DIG', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('HC_COMISIONABLE_DIMENSION5_POS', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('COMISION_ESTIMADA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('APROVISIONAMIENTO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VALES_DESPENSA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PRIMA_DOMINICAL', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('BONO_ESP', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ES_VACANTE', 'BOOLEAN', mode='REQUIRED'),
    bigquery.SchemaField('ES_EVENTUAL', 'BOOLEAN', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO_DIC', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('AGUINALDO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PTU', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PRIMA_VACACIONAL', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SBC', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PREMIOS_COMISIONES', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SAR', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('IMSS', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VIVIENDA_PATRON', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ISN', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PRIMA_ANTIGUEDAD', 'FLOAT', mode='REQUIRED'),
]

squema_headcount_base = [
    bigquery.SchemaField('DIVPERSONAL', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NUMPER', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('GPOPERSDESC', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('AREAPERSDESC', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('FUNCION_DESC', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CECO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('FECHAING', 'DATE', mode='REQUIRED'),
    bigquery.SchemaField('SUELDO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION6_PERSONAL', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PORC_SUELDO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MANDANTE', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('TIMESTAMP', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PATH', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NEGOCIO', 'STRING', mode='REQUIRED'),
]

squema_cecos_posiciones = [
    bigquery.SchemaField('NEGOCIO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('MARCA', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION5CACION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('POSICION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CECO', 'STRING', mode='REQUIRED'),
]
squema_parametros_nomina = [
    bigquery.SchemaField('NEGOCIO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PRIMA_DOMINICAL', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PRIMA_VACACIONAL', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DOMINGOS_PERIODO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ASISTENCIA_DIAS_FESTIVOS', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_PTU_TOPE', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('UMA_DIARIA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('FACTOR_UMA_SBC', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MESES_BONO_ESP', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VECES_FERIADO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('PERIODOS_FERIADO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PLANTILLA_FERIADOS_DOM', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MES_MIN_BONO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SAR_F00_PPTO_V1', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SAR_PLAN', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('ENFERMEDAD_MATERNIDAD', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('INVALIDEZ_VIDA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('RIESGO_TRABAJO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('GUARDERIA_PRES', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('VIVIENDA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MES_BASE', 'FLOAT', mode='REQUIRED'),
]
squema_tabuladores = [
    bigquery.SchemaField('MARCA', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('ID', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('POSICION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('SUELDOS_JUL25', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_EMERGENTE', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MONTERREY', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_FRONTERA', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('SUELDOS_ENE26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_EMERGENTE26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('MONTERREY26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION7_FRONTERA26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('POR_INCR_JUL26', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('APLICA_MINIMO_GARANTIZADO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('IMP_MINIMO_GARANTIZADO_ENE_2026', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('IMP_MINIMO_GARANTIZADO_2025', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('AREA_PERS', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('COMISION_DUMMY', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('COMISION', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_PTU', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('DIAS_AGUINALDO', 'FLOAT', mode='REQUIRED'),
    bigquery.SchemaField('X', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NOTAS', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('NEGOCIO', 'STRING', mode='REQUIRED'),
]

schema_DIMENSION5s_edos = [
    bigquery.SchemaField('DIMENSION5CACION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('ESTADO', 'STRING', mode='REQUIRED'),
]

schema_ventas_Negocio9 = [
    bigquery.SchemaField('MARCA', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION5CACION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION5', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('CANAL', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PERIODO', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('VERSION', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('DIMENSION6_V', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('EJERCICIO', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('CUENTA', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('IMPORTE', 'FLOAT', mode='REQUIRED'),
]

schema_isn = [
    bigquery.SchemaField('ESTADO', 'STRING', mode='REQUIRED'),
    bigquery.SchemaField('PORCENTAJE', 'FLOAT', mode='REQUIRED'),
]

schemas_nomina = [
    squema_detalle
    # ,squema_nomina_detalle_cn
    # ,squema_headcount_base
    # ,squema_cecos_posiciones
    # ,squema_parametros_nomina
    # ,squema_tabuladores
    # ,schema_DIMENSION5s_edos
    # ,schema_ventas_Negocio9
    # ,schema_isn
]

tablas_nomina = [
    nomina_detalle
    # ,nomina_etz_cn_detalle
    # ,nomina_headcountbase
    # ,nomina_cecos_posicion
    # ,nomina_parametros
    # ,nomina_tabulador
    # ,nomina_DIMENSION5caciones_estados
    # ,ventas_Negocio9_trans
    # ,nomina_isn
]

for schema, table_id in zip(schemas_nomina, tablas_nomina):
  try:
      client.delete_table(table_id)
      print(f"Deleted table {table_id}")
  except:
      print(f"Table {table_id} does not exist")

  table = bigquery.Table(table_id, schema=schema)
  table = client.create_table(table)

  print(
      f"Created table {table.project}.{table.dataset_id}.{table.table_id}"
  )


In [ ]:
query_nomina_base = f'''CREATE OR REPLACE PROCEDURE `{stored_procedure_MODELO_NOM_ETZ}`(negocio STRING)
OPTIONS (strict_mode=false)
BEGIN

  DECLARE dynamic_sql STRING;
  DECLARE project STRING DEFAULT "{project_datasetbases}";
  DECLARE negoci STRING DEFAULT "";
  DECLARE period STRING DEFAULT "";
  DECLARE droplines STRING DEFAULT "";
  DECLARE querytrasn STRING DEFAULT "";
  DECLARE droplines1 STRING DEFAULT "";

  SET droplines1 = FORMAT("""
  DELETE FROM `{nomina_detalle}`
  WHERE NEGOCIO = '%s'
  """,
  negocio
  );

  EXECUTE IMMEDIATE droplines1;

  SET dynamic_sql = FORMAT("""
  INSERT INTO `{nomina_detalle}`

  WITH

  BASE AS (
    SELECT * FROM `{nomina_headcountbase}`
    where negocio = '%s'
  ),

  UEST AS (
    SELECT DISTINCT * FROM {nomina_DIMENSION5caciones_estados}
  ),

  CECOS AS (
    SELECT * FROM {CECOS}
  ),

  CUENTACECOS AS (
      select T0.NEGOCIO, T1.MARCA, T1.DIMENSION5 || " " || T1.DES_DIMENSION5 AS DIMENSION5CACION,
      T0.FUNCION_DESC as POSICION,
      COUNT(DISTINCT T0.CECO) as CECOS

      FROM BASE T0
      LEFT JOIN CECOS T1 ON T0.CECO=T1.CECO

      GROUP BY 1,2,3,4

      ORDER BY CECOS DESC
      ),

  CECOSPOS AS (

    select DISTINCT T0.NEGOCIO, T1.MARCA, T1.DIMENSION5 || " " || T1.DES_DIMENSION5 AS DIMENSION5CACION,
    T0.FUNCION_DESC as POSICION,
    T1.CECO

    FROM BASE T0
    LEFT JOIN CECOS T1 ON T0.CECO=T1.CECO
    INNER JOIN CUENTACECOS ON CUENTACECOS.NEGOCIO = T0.NEGOCIO
                              AND CUENTACECOS.MARCA = T1.MARCA
                              AND CUENTACECOS.DIMENSION5CACION = T1.DIMENSION5 || " " || T1.DES_DIMENSION5
                              AND CUENTACECOS.POSICION = T0.FUNCION_DESC
                              AND CUENTACECOS.CECOS = 1
  ),

  SUELDOSN AS (
    SELECT * FROM `{nomina_tabulador}`
  ),

  pg AS (
    SELECT * FROM `{tabla_pargen}`
  ),

  pvcorpo AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VAC_CORPO`
  ),

  VENTAS_BASE AS (
    SELECT * FROM `{ventas_Negocio9_trans}`
  ),

  pv AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VACACIONAL`
  ),

  pva AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VACACIONAL_NEGOCIO X`
  ),

  v AS (
    SELECT * FROM `{nomina_parametros}`
  ),

  ---------------------------------------------------------------

 A AS (
  SELECT DISTINCT T0.NEGOCIO,
  CASE WHEN DIMENSION1 IN ('Negocio8', 'Negocio9') THEN T1.MARCA WHEN DIMENSION1 IN ('NEGOCIO Y', 'NEGOCIO X y CC Externos') THEN REPEAT("0",4-LENGTH(T0.DIVPERSONAL))||T0.DIVPERSONAL ELSE T1.CECO END AS MARCA,
  T1.DIMENSION5 || " " || T1.DES_DIMENSION5 AS DIMENSION5CACION,
  T0.FUNCION_DESC AS POSICION,

  --VALES DE DESPENSA
  IFNULL(
  CASE WHEN T0.GPOPERSDESC LIKE 'Vía Planta' THEN PARAMS.vales_via_planta
      WHEN AREAP.AREA_PERS_ID IN ('2G','2V') THEN PARAMS.vales_via_planta_2
      ELSE VALES.Importe_vales END
  ,0) as VALES

  FROM BASE T0
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.AREA_PERSONAL` AREAP ON T0.AREAPERSDESC=AREAP.AREA_PERS_DESC
  LEFT JOIN CECOS T1 ON T0.CECO = T1.CECO
  LEFT JOIN {tablavales} VALES
              ON COALESCE(AREAP.AREA_PERS_ID,T0.AREAPERSDESC)=VALES.AREA_PERSONAL
              AND REPEAT("0",CASE WHEN LENGTH(T0.DIVPERSONAL) <4 THEN 4-LENGTH(T0.DIVPERSONAL) ELSE 0 END)||T0.DIVPERSONAL=VALES.DIMENSION5
  CROSS JOIN pg PARAMS
),

CUENTA AS (
SELECT NEGOCIO, MARCA, DIMENSION5CACION, POSICION, COUNT (DISTINCT VALES) as CUENTA

FROM A

GROUP BY 1,2,3,4

ORDER BY CUENTA DESC
),

  vales AS (
    SELECT A.*

    FROM A
    INNER JOIN CUENTA
      ON A.POSICION=CUENTA.POSICION
      AND A.NEGOCIO=CUENTA.NEGOCIO
      AND A.MARCA=CUENTA.MARCA
      AND A.DIMENSION5CACION=CUENTA.DIMENSION5CACION
      AND CUENTA.CUENTA = 1
  ),

  -------------------------------------------------------------------

  ISN AS (
    SELECT * FROM `{nomina_isn}`
  ),

  SZE AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.ESTADOS_DIMENSION7_EMERGENTE`
  ),

  SZF AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.ESTADOS_DIMENSION7_FRONTERIZA`
  ),

  MESES AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.MESES_COMPLETOS`
  ),

  p AS (
    SELECT T0.*
    FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PERIODOS` T0
    INNER JOIN pg ON T0.VERSION IN UNNEST(pg.versiones)
  ),

  sueldo_base AS (
    SELECT
      CASE
        WHEN T0.NEGOCIO IN ('Negocio8', 'Negocio9') THEN
          CASE WHEN T1.MARCA = 'Apple' THEN 'Livestore' ELSE T1.MARCA END
        WHEN DIMENSION1 IN ('NEGOCIO Y', 'NEGOCIO X y CC Externos') THEN
          REPEAT("0", 4 - LENGTH(T0.DIVPERSONAL)) || T0.DIVPERSONAL
        ELSE T1.CECO
      END AS MARCA,
      T1.DIMENSION5 || " " || T1.DES_DIMENSION5 AS DIMENSION5CACION,
      REGEXP_REPLACE(NORMALIZE(T0.FUNCION_DESC, NFD), r"\\\pM", '') AS POSICION,
      NUMPER,
      T0.CECO,
      T0.FECHAING,
      IFNULL(CASE WHEN IS_NAN(T0.SUELDO) THEN 0 ELSE T0.SUELDO END, 0) AS SUELDO,
      COUNT(T0.NUMPER) AS HC,
      IFNULL(PORC_SUELDO, 1) AS PORC_CECO,
      T0.DIMENSION6_PERSONAL,
      "BASE_MODELO_ETZ" AS ORIGEN,
      NEGOCIO
    FROM BASE T0
    LEFT JOIN CECOS T1
      ON T0.CECO = T1.CECO
    WHERE MARCA <> 'BYD'
    GROUP BY 1,2,3,4,5,6,7,9,10,11,12
  ),

  DIMENSION6_PERS AS (
    SELECT DISTINCT
      T0.FUNCION_DESC AS POSICION,
      T0.DIMENSION6_Personal
    FROM BASE T0
  ),
  -- Sueldo diario
  con_sueldo_diario AS (
    SELECT
      t.* EXCEPT(SUELDO, DIMENSION6_PERSONAL),
      CASE
        WHEN DIMENSION6_PERSONAL IS NULL THEN
          CASE
            WHEN SUELDOSN.AREA_PERS IN ('1C','1E','1F','1H') THEN 'Personal Ejecutivo'
            ELSE 'Personal General'
          END
        ELSE DIMENSION6_PERSONAL
      END AS DIMENSION6_PERSONAL,
      IFNULL(SUELDOSN.COMISION, 0) AS PORC_COMISION,
      SUELDOSN.DIAS_PTU,
      SUELDOSN.DIAS_AGUINALDO,
      IFNULL(COMISION_DUMMY, "NO") AS COMISION_DUMMY,
      SUELDO,
      CASE
        WHEN UPPER(SZE.ESTADO) = 'MONTERREY' THEN COALESCE(SUELDOSN.MONTERREY26, SUELDOS_ENE26)
        WHEN SZE.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_EMERGENTE26, SUELDOS_ENE26)
        WHEN SZF.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_FRONTERA26, SUELDOS_ENE26)
        ELSE SUELDOS_ENE26
      END * IFNULL(PORC_CECO, 1) AS SUELDO_TABULADOR26,
      CASE
        WHEN UPPER(SZE.ESTADO) = 'MONTERREY' THEN COALESCE(SUELDOSN.MONTERREY, SUELDOS_JUL25)
        WHEN SZE.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_EMERGENTE, SUELDOS_JUL25)
        WHEN SZF.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_FRONTERA, SUELDOS_JUL25)
        ELSE SUELDOS_JUL25
      END * IFNULL(PORC_CECO, 1) AS SUELDO_TABULADOR25,
      SUELDOSN.por_incr_jul26,
      APLICA_MINIMO_GARANTIZADO,
      IMP_MINIMO_GARANTIZADO_ENE_2026,
      NOTAS,
      UEST.ESTADO
    FROM sueldo_base t
    LEFT JOIN UEST
      ON CASE WHEN t.NEGOCIO = 'NEGOCIO X y CC Externos' THEN t.MARCA ELSE SUBSTRING(t.DIMENSION5CACION, 1, 4) END = SUBSTRING(UEST.DIMENSION5CACION, 1, 4)
    LEFT JOIN SZE ON UEST.ESTADO = SZE.ESTADO
    LEFT JOIN SZF ON UEST.ESTADO = SZF.ESTADO
    LEFT JOIN SUELDOSN ON
      t.Marca = CASE
        WHEN SUELDOSN.NEGOCIO IN ('NEGOCIO X y CC Externos', 'NEGOCIO Y') THEN REPEAT("0", 4 - LENGTH(SUELDOSN.MARCA)) || SUELDOSN.MARCA
        ELSE SUELDOSN.MARCA
      END
      AND t.POSICION = REGEXP_REPLACE(NORMALIZE(SUELDOSN.POSICION, NFD), r"\\\pM", '')
      AND t.NEGOCIO = SUELDOSN.NEGOCIO
  ),

  antiguedad AS (
    SELECT
      SD.* EXCEPT(SUELDO, SUELDO_TABULADOR26, SUELDO_TABULADOR25, HC),
      pvcorpo.DIAS AS DIAS_PV_CORPO,
      IFNULL(
        CASE
          WHEN MES <= mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES > mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES < mesinc AND VERSION = 'PPTO_V0' THEN 1
          WHEN MES >= mesinc AND VERSION = 'PPTO_V0' THEN
            (1) * (1 + IFNULL(
              CASE
                WHEN Negocio <> 'Negocio8' THEN pg.porc_inc_sig_anio
                ELSE IFNULL(POR_INCR_JUL26, 0)
              END,
            0))
          ELSE 0
        END *
        CASE
          WHEN NUMPER <> 'Vacante' THEN SUELDO
          WHEN IFNULL(SUELDO, 0) = 0 AND VERSION IN ('PPTO_V2','PPTO_V1') THEN SUELDO_TABULADOR25
          ELSE SUELDO_TABULADOR26
        END,
      0) AS SUELDO,
      IFNULL(
        CASE
          WHEN MES <= mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES > mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES < mesinc AND VERSION = 'PPTO_V0' THEN 1
          WHEN MES >= mesinc AND VERSION = 'PPTO_V0' THEN
            (1) * (1 + IFNULL(
              CASE
                WHEN Negocio <> 'Negocio8' THEN pg.porc_inc_sig_anio
                ELSE IFNULL(POR_INCR_JUL26, 0)
              END,
            0))
          ELSE 0
        END *
        CASE
          WHEN VERSION IN ('PPTO_V2','PPTO_V1') THEN SUELDO_TABULADOR25
          ELSE SUELDO_TABULADOR26
        END,
      0) AS SUELDO_TABULADOR,
      CASE
        WHEN DIMENSION6_Personal = 'Eventuales' AND DATE_DIFF(P.FECHA, SD.FECHAING, MONTH) >= 1 THEN 0
        ELSE HC
      END AS HC,
      VERSION,
      MES AS PERIODO,
      CASE WHEN P.FECHA < SD.FECHAING THEN 0 ELSE 1 END AS FILTRO,
      DATE_DIFF(P.FECHA, SD.FECHAING, MONTH)/12 AS ANTIGUEDAD,
      DATE_DIFF(P.FECHA, SD.FECHAING, MONTH) AS ANTIGUEDAD_MESES
    FROM con_sueldo_diario SD
    CROSS JOIN P
    CROSS JOIN pg
    LEFT JOIN pvcorpo
      ON DATE_DIFF(P.FECHA, SD.FECHAING, YEAR) >= pvcorpo.MIN
      AND DATE_DIFF(P.FECHA, SD.FECHAING, YEAR) < pvcorpo.MAX
  ),

  tabuladores AS (
    SELECT
      a.* EXCEPT(FILTRO, SUELDO),
      CASE
        WHEN (VERSION = 'PPTO_V0' AND SUELDO < SUELDO_TABULADOR) OR Numper = 'Vacante'
        THEN SUELDO_TABULADOR
        ELSE SUELDO
      END AS SUELDO,
      pv.DIAS AS DIAS_VACACIONES
    FROM antiguedad a
    LEFT JOIN pv
      ON a.ANTIGUEDAD >= ROUND(pv.MIN, 0)
      AND a.ANTIGUEDAD < ROUND(pv.MAX, 0)
    WHERE FILTRO = 1
  ),

  headcount AS (
    SELECT
      T.POSICION,
      T.MARCA,
      T.DIMENSION5CACION,
      T.VERSION,
      T.PERIODO,
      SUM(HC) AS HC_DIMENSION5CACION
    FROM tabuladores T
    WHERE UPPER(IFNULL(COMISION_DUMMY, "")) = 'SI'
    GROUP BY 1,2,3,4,5
  ),

  conteo_posiciones AS (
    SELECT
      VERSION,
      PERIODO,
      DIMENSION5CACION,
      POSICION,
      COUNT(*) AS TOTAL_PERSONAS_MISMA_POSICION
    FROM tabuladores
    GROUP BY 1,2,3,4
  ),

  ptu_rangos AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PARAMETROS_PTU`
  ),

  ventas AS (
    SELECT
      CASE WHEN MARCA = 'Apple' THEN 'Livestore' ELSE MARCA END AS MARCA,
      DIMENSION5CACION,
      VERSION,
      PERIODO,
      SUM(IMPORTE) AS IMPORTE
    FROM VENTAS_BASE
    WHERE DIMENSION6_V = 'VENTA' AND CANAL = 'FISICO'
    GROUP BY 1,2,3,4
  ),

  ventasdig AS (
    SELECT
      CASE WHEN MARCA = 'Apple' THEN 'Livestore' ELSE MARCA END AS MARCA,
      DIMENSION5CACION,
      VERSION,
      PERIODO,
      SUM(IMPORTE) AS IMPORTE
    FROM VENTAS_BASE
    WHERE DIMENSION6_V = 'VENTA' AND CANAL = 'DIGITAL'
    GROUP BY 1,2,3,4
  ),

  CALCULOS AS (
    SELECT
      t.* EXCEPT(SUELDO),
      SUELDO * HC AS SUELDO,
      SUELDO / 30 AS SUELDO_DIARIO,
      v.* EXCEPT(PRIMA_VACACIONAL, PRIMA_DOMINICAL, NEGOCIO, DOMINGOS_PERIODO),
      CASE
        WHEN t.NEGOCIO IN ('Negocio8') THEN v.PRIMA_VACACIONAL
        WHEN t.NEGOCIO IN ('NEGOCIO X y CC Externos') THEN
          CASE WHEN pva.DIAS = 0.25 THEN 10 ELSE pva.DIAS END
      END AS PORC_PRIMA_VACACIONAL,
      v.PRIMA_DOMINICAL AS PORC_PRIMA_DOMINICAL,
      cp.TOTAL_PERSONAS_MISMA_POSICION,
      t.SUELDO / 30 * (v.VECES_FERIADO - 1) *
        CASE
          WHEN t.NEGOCIO = 'NEGOCIO X y CC Externos' THEN PLANTILLA_FERIADOS_DOM
          ELSE IF(IFNULL(cp.TOTAL_PERSONAS_MISMA_POSICION, 0) > 1, v.PLANTILLA_FERIADOS_DOM, 1)
        END *
        CASE WHEN CAST(T.PERIODO AS STRING) IN UNNEST(SPLIT(v.PERIODOS_FERIADO, ',')) THEN 1 ELSE 0 END * HC
        AS PAGO_FESTIVO,
      IFNULL(ve.IMPORTE, 0) AS VENTA_FIS,
      IFNULL(ventasdig.IMPORTE, 0) AS VENTA_DIG,
      HEADCOUNT.HC_DIMENSION5CACION,
      IFNULL(ve.IMPORTE, 0) * IFNULL(PORC_COMISION, 0) * HC / IFNULL(CASE WHEN HEADCOUNT.HC_DIMENSION5CACION = 0 THEN 1 ELSE HEADCOUNT.HC_DIMENSION5CACION END, 1) AS COMISION_ESTIMADA,
      CASE WHEN UPPER(IFNULL(NOTAS, "")) LIKE '%%%%APROVISIONAMIENTO%%%%' THEN
        IFNULL(ventasdig.IMPORTE, 0) * IFNULL(PORC_COMISION, 0) * HC / IFNULL(HEADCOUNT.HC_DIMENSION5CACION, 1)
      ELSE 0 END AS APROVISIONAMIENTO,
      IFNULL(vales.vales * t.HC, pg.vales_via_planta) * IFNULL(PORC_CECO, 1) *
        CASE
          WHEN t.VERSION = 'PPTO_V0' AND t.PERIODO >= pg.mesinc_vales THEN (1 + pg.porc_inc_vales)
          ELSE 1
        END AS VALES_DESPENSA,
      t.SUELDO / 30 * v.PRIMA_DOMINICAL * v.DOMINGOS_PERIODO / IF(cp.TOTAL_PERSONAS_MISMA_POSICION > 1, v.PLANTILLA_FERIADOS_DOM, 1) * HC AS PRIMA_DOMINICAL,
      DATE_DIFF(DATE(CASE WHEN t.VERSION = 'PPTO_V0' THEN 2026 ELSE 2025 END, 12, 31), FECHAING, DAY) AS DIASANIO,
      CASE WHEN t.DIMENSION6_personal LIKE '%%%%Ejecutivo%%%%' AND t.MARCA <> 'Negocio9' THEN 1 ELSE 0 END *
        t.SUELDO * v.MESES_BONO_ESP / 12 *
        CASE
          WHEN NUMPER = 'Vacante' AND EXTRACT(MONTH FROM FECHAING) > 7
            AND EXTRACT(YEAR FROM FECHAING) = CASE WHEN t.VERSION = 'PPTO_V0' THEN 2026 ELSE 2025 END
          THEN 0
          WHEN EXTRACT(YEAR FROM FECHAING) = CASE WHEN t.VERSION = 'PPTO_V0' THEN 2026 ELSE 2025 END
          THEN DATE_DIFF(DATE(CASE WHEN t.VERSION = 'PPTO_V0' THEN 2026 ELSE 2025 END, 12, 31), FECHAING, DAY) / 365
          ELSE 1
        END * HC AS BONO_ESP,
      IF(t.NUMPER = 'Vacante', TRUE, FALSE) AS ES_VACANTE,
      IF(LOWER(t.posicion) LIKE '%%%%Eventual%%%%', TRUE, FALSE) AS ES_EVENTUAL
    FROM tabuladores t
    LEFT JOIN pva ON t.ANTIGUEDAD < pva.max AND t.ANTIGUEDAD >= pva.min
    LEFT JOIN v ON t.NEGOCIO = v.NEGOCIO
    LEFT JOIN vales ON vales.DIMENSION5CACION = T.DIMENSION5CACION AND vales.posicion = t.posicion AND vales.negocio = t.negocio AND vales.marca = t.marca
    LEFT JOIN ventas ve ON t.MARCA = ve.MARCA AND SUBSTRING(t.DIMENSION5CACION, 1, 4) = SUBSTRING(ve.DIMENSION5CACION, 1, 4)
      AND t.VERSION = ve.VERSION AND t.PERIODO = ve.PERIODO
    LEFT JOIN ventasdig ON t.MARCA = ventasdig.MARCA AND SUBSTRING(t.DIMENSION5CACION, 1, 4) = SUBSTRING(ventasdig.DIMENSION5CACION, 1, 4)
      AND t.VERSION = ventasdig.VERSION AND t.PERIODO = ventasdig.PERIODO
    LEFT JOIN conteo_posiciones cp ON t.DIMENSION5CACION = cp.DIMENSION5CACION AND t.POSICION = cp.POSICION
      AND t.VERSION = cp.VERSION AND t.PERIODO = cp.PERIODO
    CROSS JOIN pg
    LEFT JOIN headcount ON T.MARCA = HEADCOUNT.MARCA AND T.DIMENSION5CACION = headcount.DIMENSION5CACION
      AND T.VERSION = headcount.VERSION AND T.PERIODO = headcount.PERIODO AND T.POSICION=headcount.posicion
    WHERE HC <> 0 AND ((t.VERSION IN ('PPTO_V2','PPTO_V1') AND t.PERIODO >= SAFE_CAST({mesbaser} AS INT64)) OR t.VERSION NOT IN ('PPTO_V2','PPTO_V1'))
  ),

  MONTOS_ANUALES AS (
    SELECT  MARCA, DIMENSION5CACION, POSICION, NUMPER, ORIGEN, NEGOCIO, VERSION, CECO, SUM(COMISION_ESTIMADA) AS COMISION_ANUAL,
    CASE WHEN PERIODO = 12 THEN SUM(CALCULOS.SUELDO) ELSE 0 END AS SUELDO_DIC

    FROM CALCULOS
    GrOUP BY 1,2,3,4,5,6,7,8,PERIODO
    ),

    GROUPANUALES AS (
      SELECT * EXCEPT(COMISION_ANUAL,SUELDO_DIC),
      SUM(COMISION_ANUAL) as COMISION_ANUAL,
      SUM(SUELDO_DIC) as SUELDO_DIC
      FROM MONTOS_ANUALES
    GROUP BY 1,2,3,4,5,6,7,8
    ),

  CALCULOS_ANUALES AS (
  SELECT CALCULOS.*,
  SUELDO_DIC,


   ((ABS(SUELDO_DIC)/30 +  IFNULL(COMISION_ANUAL /
   CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
   / 30 ,0) ) * DIAS_AGUINALDO) / 12 *HC AS AGUINALDO,

    CASE
      WHEN DIMENSION6_Personal = 'Eventuales' THEN 0
      WHEN DIAS_PTU > 1
        THEN ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
        CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
         / 30 ,0) ) * DIAS_PTU
        ELSE ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
        CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
         / 30 ,0) ) * CASE WHEN CALCULOS.Negocio = 'Negocio8' THEN 75 ELSE 0 END
    END
   /12 *HC AS PTU,


   CASE WHEN DIMENSION6_personal like '%%%%Ejecutivo%%%%' THEN DIAS_PV_CORPO
    ELSE
    CASE
      WHEN PORC_PRIMA_VACACIONAL < 1 THEN
      DIAS_VACACIONES * PORC_PRIMA_VACACIONAL
    ELSE PORC_PRIMA_VACACIONAL END END *
   ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
   CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
   / 30 ,0) )  /12 *HC AS PRIMA_VACACIONAL

  FROM CALCULOS
  LEFT JOIN GROUPANUALES ON
        CALCULOS.MARCA = GROUPANUALES.MARCA AND
        CALCULOS.DIMENSION5CACION = GROUPANUALES.DIMENSION5CACION AND
        CALCULOS.POSICION = GROUPANUALES.POSICION AND
        CALCULOS.NUMPER = GROUPANUALES.NUMPER AND
        CALCULOS.ORIGEN = GROUPANUALES.ORIGEN AND
        CALCULOS.NEGOCIO = GROUPANUALES.NEGOCIO AND
        CALCULOS.VERSION = GROUPANUALES.VERSION AND
        CALCULOS.CECO = GROUPANUALES.CECO
),

  CALCULO2 AS (

  SELECT CALCULOS_ANUALES.*,

    (
    IFNULL(SUELDO_DIARIO * HC,0) +  -- Sueldo diario
    IFNULL(COMISION_ESTIMADA / 30,0) +  -- Comisión diaria estimada
    IFNULL(PRIMA_VACACIONAL/ 30,0) +  -- Prima vacacional diaria
    IFNULL(AGUINALDO / 30,0) +    -- Aguinaldo diario
    IFNULL(PRIMA_DOMINICAL,0) / 30 +     -- Prima dominical DIARIA
    IFNULL(VALES_DESPENSA / 30,0) + -- vales diario
    IFNULL(BONO_ESP / 30,0)  --BONO_ESP DIARIO
    --IFNULL(PTU / 30,0) -- PTU  DIARIA
  )
  AS SBC,

  -- PREMIOS POR DEBAJO DEL MINIMO

    CASE WHEN UPPER(IFNULL(APLICA_MINIMO_GARANTIZADO,'')) = 'SI' AND  VERSION = 'PPTO_V0' THEN  IFNULL(IMP_MINIMO_GARANTIZADO_ENE_2026,0) - SUELDO + COMISION_ESTIMADA
    ELSE 0 END AS PREMIOS_COMISIONES

  FROM CALCULOS_ANUALES
  )


  SELECT
    b.*,
    b.SBC * ( CASE WHEN VERSION IN ('PPTO_V2','PPTO_V1') THEN SAR_F00_PPTO_V1 ELSE SAR_PLAN END ) * 30 AS SAR,

    -- IMSS (tabulado)
    (
      ENFERMEDAD_MATERNIDAD --ENFERMEDAD Y MATERNIDAD
      + INVALIDEZ_VIDA  -- INVALIDEZ Y VIDA
      + RIESGO_TRABAJO -- RIESGO DE TRABAJO
      + GUARDERIA_PRES --GUARDERIA Y PRESTACIONES
    )
  * b.SBC * 30 AS IMSS,

    -- Vivienda patrón 5
    b.SBC * VIVIENDA * 30 AS VIVIENDA_PATRON,

    -- ISN
    b.SBC * ISN.PORCENTAJE * 30 AS ISN,

    -- Prima de antigüedad (si aplica)
    CASE
      WHEN b.ANTIGUEDAD >= 3 THEN b.SUELDO_DIARIO * 12
      ELSE 0
    END AS PRIMA_ANTIGUEDAD

  FROM CALCULO2 b
  LEFT JOIN ISN ON b.ESTADO = ISN.ESTADO
  ;

  """,
  negocio
  );

  EXECUTE IMMEDIATE dynamic_sql;

  ----DROP NEGOCIO

  SET droplines = FORMAT("""
  DELETE FROM `{nomina_calculo}`
  WHERE DIMENSION1 = '%s';
  """,negocio);

  EXECUTE IMMEDIATE droplines;

  SET querytrasn = """
  INSERT INTO {nomina_calculo}

  WITH TRANSFORMACION AS (

  SELECT NEGOCIO, T1.CUENTA, T0.CECO,
      CASE WHEN VERSION IN ('PPTO_V1','PPTO_V2') THEN {aniobase} ELSE {aniobase+1} END as EJERCICIO,
      PERIODO, VERSION,
      {mesinc} as MES_INC,
      0.0 as PORC_INC,
      {mesbasehc} as MES_BASE,
      POSICION,
      SUELDO,
      SUM(HC) as HC,

      IFNULL( CASE
          WHEN RUBRO = 'Sueldo Ejecutivo' AND DIMENSION6_Personal like '%Ejecutivo%' THEN SUM(SUELDO)
          WHEN RUBRO = 'Personal General' AND DIMENSION6_Personal IN ('Personal General','Eventuales') THEN SUM(SUELDO)
          WHEN RUBRO = 'Día Festivo' THEN SUM(PAGO_FESTIVO)
          WHEN RUBRO = 'Comisiones' THEN SUM(COMISION_ESTIMADA)
          WHEN RUBRO = 'Aprovisionamiento' THEN SUM(APROVISIONAMIENTO)
          WHEN RUBRO = 'Premios y Comisiones' THEN SUM(PREMIOS_COMISIONES)
          WHEN RUBRO = 'Vales de Despensa' THEN SUM(VALES_DESPENSA)
          WHEN RUBRO = 'Prima vacacional' THEN SUM(PRIMA_VACACIONAL)
          WHEN RUBRO = 'Aguinaldo' THEN SUM(AGUINALDO)
          WHEN RUBRO = 'Prima dominical' THEN SUM(PRIMA_DOMINICAL)
          WHEN RUBRO = 'PTU' THEN SUM(PTU)
          WHEN RUBRO = 'Bono BONO_ESP' THEN SUM(BONO_ESP)
          WHEN RUBRO = 'IMSS Patronal' THEN SUM(IMSS)
          WHEN RUBRO = 'SAR' THEN SUM(SAR)
          WHEN RUBRO = 'Vivienda Patrón' THEN SUM(VIVIENDA_PATRON)
          WHEN RUBRO = 'ISN' THEN SUM(ISN)
          ELSE 0 END
          ,0)

      as IMPORTE,

      CONCAT(
        CASE
          WHEN RUBRO = 'Sueldo Ejecutivo' THEN 'NOM_SUE'
          WHEN RUBRO = 'Personal General' THEN 'NOM_SUE'
          WHEN RUBRO = 'Día Festivo' THEN 'NOM_SUE'
          WHEN RUBRO = 'Comisiones' THEN 'NOM_COM'
          WHEN RUBRO = 'Aprovisionamiento' THEN 'NOM_COM'
          WHEN RUBRO = 'Premios y Comisiones' THEN 'NOM_PRE'
          WHEN RUBRO = 'Vales de Despensa'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Prima vacacional'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Aguinaldo'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Prima dominical'  THEN 'NOM_PRE'
          WHEN RUBRO = 'PTU'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Bono BONO_ESP' THEN  'NOM_BONO_ESP'
          WHEN RUBRO = 'IMSS Patronal' THEN  'NOM_PRE'
          WHEN RUBRO = 'SAR' THEN  'NOM_PRE'
          WHEN RUBRO = 'Vivienda Patrón' THEN  'NOM_PRE'
          WHEN RUBRO = 'ISN' THEN  'NOM_PRE'
          ELSE '' END
          ,'MODELO_NOMINA_ETZ'
      ) as ORIGEN

      FROM {nomina_detalle} T0
      CROSS JOIN `{nomina_mapeo_cuentas}` T1

      group by 1,2,3,4,5,6,7,8,9,10,11, RUBRO, DIMENSION6_PERSONAL

      )

      SELECT
        TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
        T1.CECO,
        T1.CEBE,

        COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
        COALESCE(T1.DIMENSION2,T4.DIMENSION2) AS DIMENSION2,
        COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
        COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
        COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
        COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
        COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        CAST(EJERCICIO as STRING) as EJERCICIO,
        CAST(PERIODO as INT64) as PERIODO,
        VERSION,
        T0.MES_INC,
        SAFE_CAST(PORC_INC AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) as importe,
        '' as Grupo,
        '' as Concepto,
        HC as Personas,
        '' as AREA_PERSONAL,
        Posicion,
        0  as IMPORTE_VALES,
        0 as MONTO,
        SAFE_CAST(SUELDO AS NUMERIC) as SUELDOS,
        0 as PORC_PREST,
        'Base' as DIMENSION6,
        '0' as PERIODO_BASE,
        0 as EJERCICIO_DE_INGRESO,
        0 as MES_DE_INGRESO,
        0 as PARAMETROS,
        SAFE_CAST(SUELDO AS NUMERIC) as SUELDO,

        ORIGEN

        --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5


        FROM TRANSFORMACION T0
        LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
        LEFT JOIN {CEBES} T4 ON T1.CEBE=T4.CEBE
        LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA

        --WHERE ((t0.VERSION IN ('PPTO_V2','PPTO_V1') AND t0.PERIODO >= SAFE_CAST({mes_captura} AS INT64)) OR t0.VERSION NOT IN ('PPTO_V2','PPTO_V1'))
      """;

  EXECUTE IMMEDIATE querytrasn;

END
'''

client.query(query_nomina_base).result()
print("Creado SP: ",stored_procedure_MODELO_NOM_ETZ)

⭐CREAR TABLA

In [ ]:
failed = []

for division in divisiones_mne:
    all_procedure_sql = f"""
    CALL `{stored_procedure_MODELO_NOM_ETZ}`(
      "{division.strip()}"
    );
    """

    query_job = None

    try:
        query_job = client.query(all_procedure_sql)
        query_job.result()  # wait for completion

        print("✅ SP Modelo Nómina ETZ CONCLUIDO |", division)

    except BadRequest as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((division, job_id, "BadRequest"))

        print("❌ SP ERROR (BadRequest)")
        print("   División :", division)
        print("   JobId    :", job_id)

        for err in e.errors or []:
            print("   →", err.get("message"))

        continue

    except GoogleAPICallError as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((division, job_id, "APICall"))

        print("❌ SP ERROR (API Call)")
        print("   División :", division)
        print("   JobId    :", job_id)
        print("   →", str(e))

        continue

    except Exception as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((division, job_id, "Unexpected"))

        print("❌ SP ERROR (Unexpected)")
        print("   División :", division)
        print("   JobId    :", job_id)
        print("   →", str(e))

        continue


# ======================================================
# SUMMARY
# ======================================================
print("\n================ SUMMARY — FAILED DIVISIONES ================")

if not failed:
    print("🎉 Todas las divisiones procesadas correctamente.")
else:
    for division, job_id, err_type in failed:
        print(f"{division} | JobId={job_id} | Error={err_type}")

Stored Procedure para CN_HC_ modelo nómina ETZ

In [ ]:
query_nomina_cn = f'''CREATE OR REPLACE PROCEDURE `{stored_procedure_MODELO_NOM_ETZ_cn}`(path STRING, negocio STRING)
OPTIONS (strict_mode=false)
BEGIN

  DECLARE dynamic_sql STRING;
  DECLARE project STRING DEFAULT "{project_datasetbases}";
  DECLARE negoci STRING DEFAULT "";
  DECLARE period STRING DEFAULT "";
  DECLARE droplines STRING DEFAULT "";
  DECLARE querytrasn STRING DEFAULT "";
  DECLARE droplines1 STRING DEFAULT "";

  SET droplines1 = FORMAT("""
  DELETE FROM `{nomina_etz_cn_detalle}`
  WHERE ORIGEN LIKE CONCAT('%%%s%%')
  """,
  path
  );

  EXECUTE IMMEDIATE droplines1;

  SET dynamic_sql = FORMAT("""
  INSERT INTO `{nomina_etz_cn_detalle}`

  WITH

  ---CARGAR CATÁLOGOS

  UEST AS (
    SELECT DISTINCT * FROM `{project_id}.intranet_bd_pruebas.DIMENSION5CACIONES_ESTADOS`
  ),

  pg AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.PARAMETROS_GENERALES`
  ),

  pvcorpo AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VAC_CORPO`
  ),

  VENTAS_BASE AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.VENTAS_Negocio9`
  ),

  pv AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VACACIONAL`
  ),

  pva AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VACACIONAL_NEGOCIO X`
  ),

  v AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.PARAMETROS_NOMINA`
  ),

  ISN AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.ISN`
  ),

  SZE AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.ESTADOS_DIMENSION7_EMERGENTE`
  ),

  SZF AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.ESTADOS_DIMENSION7_FRONTERIZA`
  ),

  MESES AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.MESES_COMPLETOS`
  ),

  p AS (
    SELECT T0.*
    FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PERIODOS` T0
    INNER JOIN pg ON T0.VERSION IN UNNEST(pg.versiones)
  ),

  ptu_rangos AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PARAMETROS_PTU`
  ),

  CECOS AS (
    SELECT * FROM `{project_id}.mus_qas_drv_datos_maestros.NW_CECO3`
  ),

  --EXTRAER DIMENSION5CACION, NOMBRE, DE CONCEPTOS VIENE NOMBRE, CATALOGOS VIENE EN CODIGO

  NOMDIMENSION5CS AS (
    SELECT DISTINCT DIMENSION5, DES_DIMENSION5,
    FROM CECOS
  ),

    SUELDOSN AS (
    SELECT T0.* EXCEPT(MARCA), NOMDIMENSION5CS.DES_DIMENSION5 ,
        CONCAT(REPEAT("0",CASE WHEN LENGTH(T0.MARCA) <4 THEN 4-LENGTH(T0.MARCA) ELSE 0 END) , T0.MARCA) as DIMENSION5
    FROM `{project_id}.intranet_bd_pruebas.TABULADOR` T0
    LEFT JOIN NOMDIMENSION5CS ON CONCAT(REPEAT("0",4-LENGTH(T0.MARCA)),T0.MARCA) = NOMDIMENSION5CS.DIMENSION5
  ),

  ----SE CREA LA BASE DE CN CON EL TABULADOR, PORQUE SE REQUIEREN LOS CAMPOS

  BASE AS (
    SELECT T0.*, COALESCE(CECOS.DIMENSION1,'%s') as NEGOCIO, SUELDOSN.* EXCEPT(POSICION, NEGOCIO,DIMENSION5), COALESCE(CECOS.DIMENSION5,SUELDOSN.DIMENSION5) as DIMENSION5

    FROM  `{project_id}.intranet_bd_pruebas.HC_CN` T0
    LEFT JOIN CECOS ON T0.CECO = CECOS.CECO
    LEFT JOIN SUELDOSN ON T0.CECO = SUELDOSN.DES_DIMENSION5 AND T0.POSICION = SUELDOSN.POSICION AND SUELDOSN.NEGOCIO = COALESCE(CECOS.DIMENSION1,'%s')

    WHERE EJERCICIO_DE_INGRESO <> 0 AND MES_DE_INGRESO <> 0
  ),

  ---DEL HEADCOUNT BASE SE EXTRAE INFORMACION DE PRORRATEO CECOS, E INFORMACION DE POSICIONES FALTANTE EN TABULADOR

  INFODIMENSION6 AS (

    select DISTINCT T0.NEGOCIO,
      ---CECO/MARCA CODIGO DIMENSION5CACION
      CONCAT(REPEAT("0",CASE WHEN LENGTH(T0.DIVPERSONAL) <4 THEN 4-LENGTH(T0.DIVPERSONAL) ELSE 0 END) , T0.DIVPERSONAL) AS DIVPERSONAL,
      GPOPERSDESC,
      AREAPERSDESC, FUNCION_DESC as POSICION, CECO, DIMENSION6_PERSONAL, PORC_SUELDO

    FROM `intranet_bd_pruebas.HEADCOUNT_BASE` T0
  ),

  sueldo_base AS (
    SELECT
      T0.* EXCEPT(POSICION,DIMENSION6,CECO,AREA_PERS,EJERCICIO_DE_INGRESO,MES_DE_INGRESO,SUELDO_A_UTILIZAR,NEGOCIO, PERSONAS,ID),
      CASE
        WHEN T0.NEGOCIO IN ('NEGOCIO Y', 'NEGOCIO X y CC Externos') THEN
          T0.DES_DIMENSION5
        ELSE T1.CECO
      END AS ID,
      REGEXP_REPLACE(NORMALIZE(T0.POSICION, NFD), r"\\\pM", '') AS POSICION,
      CASE WHEN T0.DIMENSION6 IN ('Adicionales','Presupuestadas') THEN 'Vacante' ELSE T0.DIMENSION6 END as NUMPER,
      COALESCE(INFODIMENSION6.CECO,T0.CECO) as CECO,
      T0.AREA_PERS,
      CASE WHEN INFODIMENSION6.AREAPERSDESC NOT IN ('1C') AND T0.DIMENSION6 IN ('Adicionales','Presupuestadas') THEN 'Vía Planta' ELSE INFODIMENSION6.GPOPERSDESC END as GPOPERSDESC,
      INFODIMENSION6.DIMENSION6_PERSONAL,
      DATE(T0.EJERCICIO_DE_INGRESO, T0.MES_DE_INGRESO,1) as FECHAING,
      IFNULL(T0.SUELDO_A_UTILIZAR, 0) AS SUELDO,
      IFNULL(PORC_SUELDO, 1) AS PORC_CECO,
      CONCAT("BASE_MODELO_ETZ ",'%s ',T0.PATH) AS ORIGEN,
      T0.NEGOCIO,
      SUM(T0.PERSONAS * IFNULL(PORC_SUELDO, 1) ) AS HC

    FROM BASE T0
    LEFT JOIN CECOS T1
      ON T0.CECO = T1.CECO
    LEFT JOIN INFODIMENSION6 ON
      T0.NEGOCIO= INFODIMENSION6.NEGOCIO
      AND INFODIMENSION6.DIVPERSONAL = T0.DIMENSION5
      AND T0.POSICION= INFODIMENSION6.POSICION

    WHERE T0.NEGOCIO  = '%s'
    GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37
    ),
   ---------------------------------------------------------------

    A AS (
      SELECT PARAMS.*, T0.*,
      --VALES DE DESPENSA
      IFNULL(
      CASE WHEN T0.GPOPERSDESC LIKE 'Vía Planta' THEN PARAMS.vales_via_planta
          WHEN T0.AREA_PERS IN ('2G','2V') THEN PARAMS.vales_via_planta_2
          ELSE VALES.Importe_vales END
      ,0)
       as VALES

      FROM sueldo_base T0
      LEFT JOIN `{project_id}.intranet_bd_pruebas.VALES_DESPENSA` VALES
                  ON T0.AREA_PERS = VALES.AREA_PERSONAL
                  AND T0.DIMENSION5=VALES.DIMENSION5
      CROSS JOIN pg PARAMS
    ),


    ---EVITAR DUPLICADOS

    CUENTA AS (
    SELECT NEGOCIO, ID, DIMENSION5, POSICION, COUNT (DISTINCT VALES) as CUENTA

    FROM A

    GROUP BY 1,2,3,4

    ORDER BY CUENTA DESC
    ),

      vales AS (
        SELECT A.*

        FROM A
        INNER JOIN CUENTA
          ON A.POSICION=CUENTA.POSICION
          AND A.NEGOCIO=CUENTA.NEGOCIO
          AND A.ID=CUENTA.ID ---CECO o NOMBRE DIMENSION5CACION
          AND A.DIMENSION5=CUENTA.DIMENSION5
          AND CUENTA.CUENTA = 1
      ),

      -------------------------------------------------------------------

  -- Sueldo diario
  con_sueldo_diario AS (
    SELECT
      t.* EXCEPT(SUELDO,COMISION_DUMMY,DIMENSION6_PERSONAL),
      -- --ULTIMO DIA DEL MES
      -- DATE_SUB(
      --   DATE_ADD(FECHAING, INTERVAL 1 MONTH),
      --   INTERVAL 1 DAY
      -- ) as FECHAING,
      CASE
        WHEN DIMENSION6_PERSONAL IS NULL THEN
          CASE
            WHEN AREA_PERS IN ('1C','1E','1F','1H') THEN 'Personal Ejecutivo'
            ELSE 'Personal General'
          END
        ELSE DIMENSION6_PERSONAL
      END AS DIMENSION6_PERSONAL,
      IFNULL(COMISION, 0) AS PORC_COMISION,
      IFNULL(COMISION_DUMMY, "NO") AS COMISION_DUMMY,
      SUELDO,
      CASE
        WHEN UPPER(SZE.ESTADO) = 'MONTERREY' THEN COALESCE(MONTERREY26, SUELDOS_ENE26)
        WHEN SZE.ESTADO IS NOT NULL THEN COALESCE(DIMENSION7_EMERGENTE26, SUELDOS_ENE26)
        WHEN SZF.ESTADO IS NOT NULL THEN COALESCE(DIMENSION7_FRONTERA26, SUELDOS_ENE26)
        ELSE SUELDOS_ENE26
      END * IFNULL(PORC_CECO, 1) AS SUELDO_TABULADORSIG,
      CASE
        WHEN UPPER(SZE.ESTADO) = 'MONTERREY' THEN COALESCE(MONTERREY, SUELDOS_JUL25)
        WHEN SZE.ESTADO IS NOT NULL THEN COALESCE(DIMENSION7_EMERGENTE, SUELDOS_JUL25)
        WHEN SZF.ESTADO IS NOT NULL THEN COALESCE(DIMENSION7_FRONTERA, SUELDOS_JUL25)
        ELSE SUELDOS_JUL25
      END * IFNULL(PORC_CECO, 1) AS SUELDO_TABULADORACTUAL,
      UEST.ESTADO
    FROM vales t
    LEFT JOIN UEST
      ON t.DIMENSION5 = SUBSTRING(UEST.DIMENSION5CACION, 1, 4)
    LEFT JOIN SZE ON UEST.ESTADO = SZE.ESTADO
    LEFT JOIN SZF ON UEST.ESTADO = SZF.ESTADO
  ),

  antiguedad AS (
    SELECT
      SD.* EXCEPT(SUELDO, SUELDO_TABULADORSIG, SUELDO_TABULADORACTUAL, HC),
      IFNULL(pvcorpo.DIAS,0) AS DIAS_PV_CORPO,
      IFNULL(
        CASE
          WHEN MES <= mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES > mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES < mesinc AND VERSION = 'PPTO_V0' THEN 1
          WHEN MES >= mesinc AND VERSION = 'PPTO_V0' THEN
            (1) * (1 + IFNULL(
              CASE
                WHEN Negocio <> 'Negocio8' THEN porc_inc_sig_anio
                ELSE IFNULL(POR_INCR_JUL26, 0)
              END,
            0))
          ELSE 0
        END *
        CASE
          WHEN NUMPER <> 'Vacante' THEN SUELDO
          WHEN IFNULL(SUELDO, 0) = 0 AND VERSION IN ('PPTO_V2','PPTO_V1') THEN SUELDO_TABULADORACTUAL
          ELSE SUELDO_TABULADORSIG
        END,
      0) AS SUELDO,
      IFNULL(
        CASE
          WHEN MES <= mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES > mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES < mesinc AND VERSION = 'PPTO_V0' THEN 1
          WHEN MES >= mesinc AND VERSION = 'PPTO_V0' THEN
            (1) * (1 + IFNULL(
              CASE
                WHEN Negocio <> 'Negocio8' THEN porc_inc_sig_anio
                ELSE IFNULL(POR_INCR_JUL26, 0)
              END,
            0))
          ELSE 0
        END *
        CASE
          WHEN VERSION IN ('PPTO_V2','PPTO_V1') THEN SUELDO_TABULADORACTUAL
          ELSE SUELDO_TABULADORSIG
        END,
      0) AS SUELDO_TABULADOR,
      CASE
        WHEN DIMENSION6_Personal = 'Eventuales' AND DATE_DIFF(P.FECHA, SD.FECHAING, MONTH) >= 1 THEN 0
        ELSE HC
      END AS HC,
      VERSION,
      MES AS PERIODO,
      CASE WHEN P.FECHA < SD.FECHAING THEN 0 ELSE 1 END AS FILTRO,
      DATE_DIFF(P.FECHA, SD.FECHAING, MONTH)/12 AS ANTIGUEDAD,
      DATE_DIFF(P.FECHA, SD.FECHAING, MONTH) AS ANTIGUEDAD_MESES,
      DATE_DIFF(P.FECHA, SD.FECHAING, DAY) AS ANTIGUEDAD_DIAS,
      DATE_DIFF(DATE(EXTRACT(YEAR FROM P.FECHA),12,31), SD.FECHAING, DAY) AS ANTIGUEDAD_ANUAL,
      P.FECHA as FECHA_PERIODO

    FROM con_sueldo_diario SD
    CROSS JOIN P
    LEFT JOIN pvcorpo
      ON DATE_DIFF(P.FECHA, SD.FECHAING, YEAR) >= pvcorpo.MIN
      AND DATE_DIFF(P.FECHA, SD.FECHAING, YEAR) < pvcorpo.MAX
  ),

  tabuladores AS (
    SELECT
      a.* EXCEPT(FILTRO, SUELDO),
      CASE
        WHEN (VERSION = 'PPTO_V0' AND SUELDO < SUELDO_TABULADOR)
        THEN SUELDO_TABULADOR
        ELSE SUELDO
      END AS SUELDO,
      pv.DIAS AS DIAS_VACACIONES
    FROM antiguedad a
    LEFT JOIN pv
      ON a.ANTIGUEDAD >= ROUND(pv.MIN, 0)
      AND a.ANTIGUEDAD < ROUND(pv.MAX, 0)
    WHERE FILTRO = 1
  ),

  ---ESTO SE HACE PARA OBTENER CANTIDAD DE PERSONAS EN LA POSICION, PARA PRIMA DOMINICAL Y FESTIVOS

  headcount AS (
    SELECT
      T.POSICION,
      T.ID,
      T.DIMENSION5,
      T.VERSION,
      T.PERIODO,
      SUM(HC) AS HC_DIMENSION5CACION
    FROM tabuladores T
    WHERE UPPER(IFNULL(COMISION_DUMMY, "")) = 'SI'
    GROUP BY 1,2,3,4,5
  ),

  conteo_posiciones AS (
    SELECT
      VERSION,
      PERIODO,
      DIMENSION5,
      POSICION,
      COUNT(*) AS TOTAL_PERSONAS_MISMA_POSICION
    FROM tabuladores
    GROUP BY 1,2,3,4
  ),


  ---SOLO SE REQUIERE SI SE CALCULAN COMISIONES SOBRE VENTA

  ventas AS (
    SELECT
      CASE WHEN MARCA = 'Apple' THEN 'Livestore' ELSE MARCA END AS MARCA,
      DIMENSION5CACION,
      VERSION,
      PERIODO,
      SUM(IMPORTE) AS IMPORTE
    FROM VENTAS_BASE
    WHERE DIMENSION6_V = 'VENTA' AND CANAL = 'FISICO'
    GROUP BY 1,2,3,4
  ),

  ventasdig AS (
    SELECT
      CASE WHEN MARCA = 'Apple' THEN 'Livestore' ELSE MARCA END AS MARCA,
      DIMENSION5CACION,
      VERSION,
      PERIODO,
      SUM(IMPORTE) AS IMPORTE
    FROM VENTAS_BASE
    WHERE DIMENSION6_V = 'VENTA' AND CANAL = 'DIGITAL'
    GROUP BY 1,2,3,4
  ),

  CALCULOS AS (
    SELECT
      t.* EXCEPT(SUELDO, PERIODO),
      ---AJUSTE A PRIMER MES DE CAPTURA
      CASE WHEN t.PERIODO < 5 THEN 5 ELSE t.PERIODO END AS PERIODO,
      SUELDO * HC AS SUELDO,
      SUELDO / 30 AS SUELDO_DIARIO,
      v.* EXCEPT(PRIMA_VACACIONAL, PRIMA_DOMINICAL, DOMINGOS_PERIODO, NEGOCIO),

      CASE
        WHEN t.NEGOCIO IN ('Negocio8') THEN v.PRIMA_VACACIONAL
        WHEN t.NEGOCIO IN ('NEGOCIO X y CC Externos') THEN
          CASE WHEN pva.DIAS = 0.25 THEN 10 ELSE pva.DIAS END
      END AS PORC_PRIMA_VACACIONAL,

      v.PRIMA_DOMINICAL AS PORC_PRIMA_DOMINICAL,

      cp.TOTAL_PERSONAS_MISMA_POSICION,

      t.SUELDO / 30 * (v.VECES_FERIADO - 1) *
        CASE
          WHEN t.NEGOCIO = 'NEGOCIO X y CC Externos' THEN PLANTILLA_FERIADOS_DOM
          ELSE IF(IFNULL(cp.TOTAL_PERSONAS_MISMA_POSICION, 0) > 1, v.PLANTILLA_FERIADOS_DOM, 1)
        END *
        CASE WHEN CAST(T.PERIODO AS STRING) IN UNNEST(SPLIT(v.PERIODOS_FERIADO, ',')) THEN 1 ELSE 0 END * HC
        AS PAGO_FESTIVO,

      IFNULL(ve.IMPORTE, 0) AS VENTA_FIS,

      IFNULL(ventasdig.IMPORTE, 0) AS VENTA_DIG,

      IFNULL(HEADCOUNT.HC_DIMENSION5CACION,0) AS HC_COMISIONABLE_DIMENSION5_POS,

      IFNULL(ve.IMPORTE, 0) * IFNULL(PORC_COMISION, 0) * HC / IFNULL(CASE WHEN HEADCOUNT.HC_DIMENSION5CACION = 0 THEN 1 ELSE HEADCOUNT.HC_DIMENSION5CACION END, 1) AS COMISION_ESTIMADA,

      CASE WHEN UPPER(IFNULL(NOTAS, "")) LIKE '%%%%APROVISIONAMIENTO%%%%' THEN
        IFNULL(ventasdig.IMPORTE, 0) * IFNULL(PORC_COMISION, 0) * HC / IFNULL(HEADCOUNT.HC_DIMENSION5CACION, 1)
      ELSE 0 END AS APROVISIONAMIENTO,

      t.vales  * PORC_CECO * HC *
        CASE
          WHEN t.VERSION = 'PPTO_V0' AND t.PERIODO >= pg.mesinc_vales THEN (1 + pg.porc_inc_vales)
          ELSE 1
        END AS VALES_DESPENSA,

      --PRIMA DOMINICAL CONSIDERA QUE NO VA TODA LA PLANTILLA

      t.SUELDO / 30 * v.PRIMA_DOMINICAL * v.DOMINGOS_PERIODO / IF(cp.TOTAL_PERSONAS_MISMA_POSICION > 1, v.PLANTILLA_FERIADOS_DOM, 1) * HC AS PRIMA_DOMINICAL,

      ---BONO_ESP, Negocio9 NO TIENE, SE CALCULA SU PARTE PROPORCIONAL A LOS DIAS LABORADOS,  CUIDADO CON LOS PARAMETROS

      CASE WHEN t.DIMENSION6_personal LIKE '%%%%Ejecutivo%%%%' AND t.ID <> 'Negocio9' THEN 1 ELSE 0 END *
        t.SUELDO * v.MESES_BONO_ESP  *
        CASE
        ----VACANTES SIN 3 MESES DE PLANTA EN EL AÑO NO TIENE BONO_ESP
          WHEN NUMPER = 'Vacante' AND EXTRACT(MONTH FROM FECHAING) > 7
            AND EXTRACT(YEAR FROM FECHAING) = EXTRACT(YEAR FROM FECHA_PERIODO)
          THEN 0
        ----VACANTES CON 3 O MAS MESES DE PLANTA EN EL AÑO, PROPOCIONAL EN DIAS
          WHEN  NUMPER = 'Vacante' AND EXTRACT(YEAR FROM FECHAING) = EXTRACT(YEAR FROM FECHA_PERIODO)
          THEN ( ANTIGUEDAD_ANUAL / 365 ) / (12-EXTRACT(MONTH FROM t.FECHAING)+1)
          ELSE 1/12
        END * HC AS BONO_ESP,

      IF(t.NUMPER = 'Vacante', TRUE, FALSE) AS ES_VACANTE,
      IF(LOWER(t.posicion) LIKE '%%%%Eventual%%%%', TRUE, FALSE) AS ES_EVENTUAL

    FROM tabuladores t
    LEFT JOIN pva ON t.ANTIGUEDAD < pva.max AND t.ANTIGUEDAD >= pva.min
    LEFT JOIN v ON t.NEGOCIO = v.NEGOCIO
    LEFT JOIN ventas ve ON t.id = ve.MARCA AND SUBSTRING(t.DIMENSION5, 1, 4) = SUBSTRING(ve.DIMENSION5CACION, 1, 4)
      AND t.VERSION = ve.VERSION AND t.PERIODO = ve.PERIODO
    LEFT JOIN ventasdig ON t.id = ventasdig.MARCA AND SUBSTRING(t.DIMENSION5, 1, 4) = SUBSTRING(ventasdig.DIMENSION5CACION, 1, 4)
      AND t.VERSION = ventasdig.VERSION AND t.PERIODO = ventasdig.PERIODO
    LEFT JOIN conteo_posiciones cp ON t.DIMENSION5 = cp.DIMENSION5 AND t.POSICION = cp.POSICION
      AND t.VERSION = cp.VERSION AND t.PERIODO = cp.PERIODO
    CROSS JOIN pg
    LEFT JOIN headcount ON T.id = HEADCOUNT.id AND T.DIMENSION5 = headcount.DIMENSION5
      AND T.VERSION = headcount.VERSION AND T.PERIODO = headcount.PERIODO AND T.POSICION=headcount.posicion
    WHERE HC <> 0
  ),

  MONTOS_ANUALES AS (
    SELECT  ID, DIMENSION5, POSICION, NUMPER, ORIGEN, NEGOCIO, VERSION, CECO, SUM(COMISION_ESTIMADA) AS COMISION_ANUAL,
    CASE WHEN PERIODO = 12 THEN SUM(CALCULOS.SUELDO) ELSE 0 END AS SUELDO_DIC

    FROM CALCULOS
    GrOUP BY 1,2,3,4,5,6,7,8,PERIODO
    ),

    GROUPANUALES AS (
      SELECT * EXCEPT(COMISION_ANUAL,SUELDO_DIC),
      SUM(COMISION_ANUAL) as COMISION_ANUAL,
      SUM(SUELDO_DIC) as SUELDO_DIC
      FROM MONTOS_ANUALES
    GROUP BY 1,2,3,4,5,6,7,8
    ),

  CALCULOS_ANUALES AS (
  SELECT CALCULOS.*,
  SUELDO_DIC,


   ((ABS(SUELDO_DIC)/30 +  IFNULL(COMISION_ANUAL /
   CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
   / 30 ,0) ) * DIAS_AGUINALDO) / 12 *HC AS AGUINALDO,

    CASE
      WHEN DIMENSION6_Personal = 'Eventuales' THEN 0
      WHEN DIAS_PTU > 1
        THEN ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
        CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
         / 30 ,0) ) * DIAS_PTU
        ELSE ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
        CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
         / 30 ,0) ) * CASE WHEN CALCULOS.Negocio = 'Negocio8' THEN 75 ELSE 0 END
    END
   /12 *HC AS PTU,


   CASE WHEN DIMENSION6_personal like '%%%%Ejecutivo%%%%' THEN DIAS_PV_CORPO
    ELSE
    CASE
      WHEN PORC_PRIMA_VACACIONAL < 1 THEN
      DIAS_VACACIONES * PORC_PRIMA_VACACIONAL
    ELSE PORC_PRIMA_VACACIONAL END END *
   ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
   CASE WHEN CALCULOS.VERSION = 'PPTO_V2' THEN (12-MES_BASE) ELSE 12 END
   / 30 ,0) )  /12 *HC AS PRIMA_VACACIONAL

  FROM CALCULOS
  LEFT JOIN GROUPANUALES ON
        CALCULOS.ID = GROUPANUALES.ID AND
        CALCULOS.DIMENSION5 = GROUPANUALES.DIMENSION5 AND
        CALCULOS.POSICION = GROUPANUALES.POSICION AND
        CALCULOS.NUMPER = GROUPANUALES.NUMPER AND
        CALCULOS.ORIGEN = GROUPANUALES.ORIGEN AND
        CALCULOS.NEGOCIO = GROUPANUALES.NEGOCIO AND
        CALCULOS.VERSION = GROUPANUALES.VERSION AND
        CALCULOS.CECO = GROUPANUALES.CECO
),

  CALCULO2 AS (

  SELECT CALCULOS_ANUALES.*,

    (
    IFNULL(SUELDO_DIARIO * HC,0) +  -- Sueldo diario
    IFNULL(COMISION_ESTIMADA / 30,0) +  -- Comisión diaria estimada
    IFNULL(PRIMA_VACACIONAL/ 30,0) +  -- Prima vacacional diaria
    IFNULL(AGUINALDO / 30,0) +    -- Aguinaldo diario
    IFNULL(PRIMA_DOMINICAL,0) / 30 +     -- Prima dominical DIARIA
    IFNULL(VALES_DESPENSA / 30,0) + -- vales diario
    IFNULL(BONO_ESP / 30,0)  --BONO_ESP DIARIO
    --IFNULL(PTU / 30,0) -- PTU  DIARIA
  )
  AS SBC,

  -- PREMIOS POR DEBAJO DEL MINIMO

    CASE WHEN UPPER(IFNULL(APLICA_MINIMO_GARANTIZADO,'')) = 'SI' AND  VERSION = 'PPTO_V0' THEN  IFNULL(IMP_MINIMO_GARANTIZADO_ENE_2026,0) - SUELDO + COMISION_ESTIMADA
    ELSE 0 END AS PREMIOS_COMISIONES

  FROM CALCULOS_ANUALES
  )


  SELECT
    b.*,
    b.SBC * ( CASE WHEN VERSION IN ('PPTO_V2','PPTO_V1') THEN SAR_F00_PPTO_V1 ELSE SAR_PLAN END ) * 30 AS SAR,

    -- IMSS (tabulado)
    (
      ENFERMEDAD_MATERNIDAD --ENFERMEDAD Y MATERNIDAD
      + INVALIDEZ_VIDA  -- INVALIDEZ Y VIDA
      + RIESGO_TRABAJO -- RIESGO DE TRABAJO
      + GUARDERIA_PRES --GUARDERIA Y PRESTACIONES
    )
  * b.SBC * 30 AS IMSS,

    -- Vivienda patrón 5
    b.SBC * VIVIENDA * 30 AS VIVIENDA_PATRON,

    -- ISN
    b.SBC * ISN.PORCENTAJE * 30 AS ISN,

    -- Prima de antigüedad (si aplica)
    CASE
      WHEN b.ANTIGUEDAD >= 3 THEN b.SUELDO_DIARIO * 12
      ELSE 0
    END AS PRIMA_ANTIGUEDAD

  FROM CALCULO2 b
  LEFT JOIN ISN ON b.ESTADO = ISN.ESTADO

  ;

  """,
  negocio,
  negocio,
  negocio,
  negocio
  );

  EXECUTE IMMEDIATE dynamic_sql;

  ----DROP NEGOCIO

  SET droplines = FORMAT("""
  DELETE FROM `{nomina_etz_cn}`
  WHERE DIMENSION1 = '%s';
  """,negocio);

  EXECUTE IMMEDIATE droplines;

  SET querytrasn = FORMAT("""
  INSERT INTO {nomina_etz_cn}

  WITH TRANSFORMACION AS (

  SELECT NEGOCIO, T1.CUENTA, T0.CECO,
      CASE WHEN VERSION IN ('PPTO_V1','PPTO_V2') THEN {aniobase} ELSE {aniobase+1} END as EJERCICIO,
      PERIODO, VERSION,
      {mesinc} as MES_INC,
      0.0 as PORC_INC,
      {mesbasehc} as MES_BASE,
      POSICION,
      NUMPER as DIMENSION6,
      SUELDO,
      SUM(HC) as HC,

      IFNULL( CASE
          WHEN RUBRO = 'Sueldo Ejecutivo' AND DIMENSION6_Personal like '%Ejecutivo%' THEN SUM(SUELDO)
          WHEN RUBRO = 'Personal General' AND DIMENSION6_Personal IN ('Personal General','Eventuales') THEN SUM(SUELDO)
          WHEN RUBRO = 'Día Festivo' THEN SUM(PAGO_FESTIVO)
          WHEN RUBRO = 'Comisiones' THEN SUM(COMISION_ESTIMADA)
          WHEN RUBRO = 'Aprovisionamiento' THEN SUM(APROVISIONAMIENTO)
          WHEN RUBRO = 'Premios y Comisiones' THEN SUM(PREMIOS_COMISIONES)
          WHEN RUBRO = 'Vales de Despensa' THEN SUM(VALES_DESPENSA)
          WHEN RUBRO = 'Prima vacacional' THEN SUM(PRIMA_VACACIONAL)
          WHEN RUBRO = 'Aguinaldo' THEN SUM(AGUINALDO)
          WHEN RUBRO = 'Prima dominical' THEN SUM(PRIMA_DOMINICAL)
          WHEN RUBRO = 'PTU' THEN SUM(PTU)
          WHEN RUBRO = 'Bono BONO_ESP' THEN SUM(BONO_ESP)
          WHEN RUBRO = 'IMSS Patronal' THEN SUM(IMSS)
          WHEN RUBRO = 'SAR' THEN SUM(SAR)
          WHEN RUBRO = 'Vivienda Patrón' THEN SUM(VIVIENDA_PATRON)
          WHEN RUBRO = 'ISN' THEN SUM(ISN)
          ELSE 0 END
          ,0)

      as IMPORTE,

      CASE
          WHEN RUBRO = 'Sueldo Ejecutivo' THEN 'CN_HC_SUE'
          WHEN RUBRO = 'Personal General' THEN 'CN_HC_SUE'
          WHEN RUBRO = 'Día Festivo' THEN 'CN_HC_SUE'
          WHEN RUBRO = 'Comisiones' THEN 'CN_HC_COM'
          WHEN RUBRO = 'Aprovisionamiento' THEN 'CN_HC_COM'
          WHEN RUBRO = 'Premios y Comisiones' THEN 'CN_HC_SUE'
          WHEN RUBRO = 'Vales de Despensa'  THEN 'CN_HC_PRE'
          WHEN RUBRO = 'Prima vacacional'  THEN 'CN_HC_PRE'
          WHEN RUBRO = 'Aguinaldo'  THEN 'CN_HC_PRE'
          WHEN RUBRO = 'Prima dominical'  THEN 'CN_HC_PRE'
          WHEN RUBRO = 'PTU'  THEN 'CN_HC_PRE'
          WHEN RUBRO = 'Bono BONO_ESP' THEN  'CN_HC_BONO_ESP'
          WHEN RUBRO = 'IMSS Patronal' THEN  'CN_HC_PRE'
          WHEN RUBRO = 'SAR' THEN  'CN_HC_PRE'
          WHEN RUBRO = 'Vivienda Patrón' THEN  'CN_HC_PRE'
          WHEN RUBRO = 'ISN' THEN  'CN_HC_PRE'
          ELSE '' END as ORIGEN

      FROM {nomina_etz_cn_detalle} T0
      CROSS JOIN `{nomina_mapeo_cuentas}` T1

      group by 1,2,3,4,5,6,7,8,9,10,11,12, RUBRO, DIMENSION6_PERSONAL

      )

      SELECT
        TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
        T1.CECO,
        T1.CEBE,

        COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
        COALESCE(T1.DIMENSION2,T4.DIMENSION2) AS DIMENSION2,
        COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
        COALESCE(T1.DIMENSION6,T4  .DIMENSION6) AS DIMENSION6_MAESTRO,
        COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
        COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
        COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        CAST(EJERCICIO as STRING) as EJERCICIO,
        CAST(PERIODO as INT64) as PERIODO,
        VERSION,
        T0.MES_INC,
        SAFE_CAST(PORC_INC AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) as importe,
        '' as Grupo,
        '' as Concepto,
        HC as Personas,
        '' as AREA_PERSONAL,
        Posicion,
        0  as IMPORTE_VALES,
        0 as MONTO,
        SAFE_CAST(SUELDO AS NUMERIC) as SUELDOS,
        0 as PORC_PREST,
        T0.DIMENSION6,
        '0' as PERIODO_BASE,
        0 as EJERCICIO_DE_INGRESO,
        0 as MES_DE_INGRESO,
        0 as PARAMETROS,
        SAFE_CAST(SUELDO AS NUMERIC) as SUELDO,

        CONCAT(ORIGEN,"_MNE_",'%s') as ORIGEN

        --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5


        FROM TRANSFORMACION T0
        LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
        LEFT JOIN {CEBES} T4 ON T1.CEBE=T4.CEBE
        LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA
      """,path);

  EXECUTE IMMEDIATE querytrasn;

END
'''

client.query(query_nomina_cn).result()
print("Creado SP: ",stored_procedure_MODELO_NOM_ETZ_cn)

# **CÁLCULO NOMINA DE CONCEPTOS HEADCOUNT** 🆗

⭐ No se han actualizado reglas prestaciones

⭐ Cálculo del bono ejecutivo por resultados

1. Se toman el sueldo del mes, si es anterior a julio, se le aplica el incremento del ejercicio actual y el incremento del ejercicio siguiente, para los meses de julio en adelante, solo se aplica el incremento del ejercicio siguiente

2. Se multiplica por el parámetro de "meses bono"/12

3. Se cancela el monto acumulado hasta el mes previo

4. -Nuevos Ingresos-, si entran después de julio no aplica bono, solo no confidenciales

In [ ]:
query_cn_hc = f'''
  (WITH

  BASEHC AS (
    SELECT * FROM `{tabla_base_hc}` WHERE PATH = path_p
  ),

    TRANSFORMACION AS (
  SELECT T1.Cuenta, T5.CeCo, T5.CEBE,
  CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END as Ejercicio,
    T0.DIMENSION6,
    CASE WHEN T3.MES < PARAMS.mes_captura AND VERSION = PARAMS.primera_version THEN PARAMS.mes_captura
    ELSE T3.MES END as Periodo,
    T3.Version, T0.MES_DE_INGRESO, T0.EJERCICIO_DE_INGRESO, PARAMS.mesinc as MES_INCREMENTO,
    Personas, T0.Posicion, PARAMS.porc_inc AS PORC_INC, T0.Sueldo_a_Utilizar,
    T1.AREA_PERS,

    --SI ENTRAN ANTES DE MARZO DE PLAN ENTONCES SI APLICA INCREMENTO
    IFNULL(
      CASE
      --SI NO TIENE VALOR
      WHEN T0.MES_DE_INGRESO > T3.MES AND T3.VERSION IN('PPTO_V1','PPTO_V2') THEN 0
      WHEN T0.EJERCICIO_DE_INGRESO > CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END THEN 0
      WHEN T0.EJERCICIO_DE_INGRESO > CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END AND T3.MES>T0.MES_DE_INGRESO THEN 0
      WHEN T0.EJERCICIO_DE_INGRESO = CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END AND T3.MES<T0.MES_DE_INGRESO THEN 0

      --INCREMENTO

      ---NO APLICA INCREMENTO SI ENTRAN EN JULIO O DESPUES EN SU AÑO DE INGRESO
      WHEN T0.MES_DE_INGRESO >= PARAMS.mesinc AND T0.EJERCICIO_DE_INGRESO = CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END THEN 1
      --SIGUENTE AÑO AL DE INGRESO PLAN
      WHEN T0.EJERCICIO_DE_INGRESO < CASE WHEN T3.VERSION = "PPTO_V2" THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END AND T3.MES >= PARAMS.mesinc THEN (1+PARAMS.porc_inc_sig_anio)
      --DESPUES DE JULIO INCREMENTO A LOS MOVIEMIENTOS EN NOMINA EXISTENTE PRIMER AÑO
      WHEN T0.DIMENSION6 NOT IN ('Adicionales','Presupuestadas')
          AND T3.MES >= PARAMS.mesinc AND T3.VERSION IN ("PPTO_V1") THEN (1+PARAMS.porc_inc)
      --INCREMENTO A NUEVAS POSICIONES EN EL AÑO QUE ENTRARON
      WHEN T0.DIMENSION6 IN ('Adicionales','Presupuestadas') AND T3.MES >= PARAMS.mesinc
            AND T0.EJERCICIO_DE_INGRESO = CASE WHEN T3.VERSION = "PPTO_V2" THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END
        THEN
        --CASE WHEN T0.Posicion IN ('Director','Director Corporativo','Subdirector','Gerente')
        --          THEN
                  CASE WHEN T3.VERSION = "PPTO_V1" THEN (1+PARAMS.porc_inc) ELSE (1+PARAMS.porc_inc_sig_anio) END
         --    ELSE 1 END
        ELSE 1 END
        * T0.Sueldo_a_Utilizar * T0.PERSONAS
    ,0) as `Importe`,

    Comentarios as Concepto,

    CONCAT("CN_HC_SUE",path_p) as `ORIGEN`  --string es path parametro

    FROM BASEHC T0
    CROSS JOIN {tabla_pargen} PARAMS
    --SE ASIGNA CUENTA DE NOMINA POR POSICION
    LEFT JOIN {NOMINAPOSICION} T1 ON T0.Posicion=T1.POSICION
    --PERIODOS
    CROSS JOIN (
          SELECT *
          FROM `{PARPERIODOS}`
          CROSS JOIN {tabla_pargen} PARAMS
          WHERE Version IN UNNEST (PARAMS.versiones)
          ) T3
    --ATRIBUTOS CECO
    LEFT JOIN {CECOS} T5 ON
    CASE WHEN REGEXP_CONTAINS(T0.CECO, r'\\\d') THEN REPEAT("0",10-LENGTH(T0.CECO))||T0.CECO ELSE T0.CECO END
    = T5.Ceco

    WHERE T0.CECO IS NOT NULL AND T3.Version IN UNNEST (PARAMS.versiones)
    AND T5.DIMENSION1 <> 'NEGOCIO X y CC Externos'
    ),

    SUELDOS AS (

        SELECT
        TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
        T1.CECO,
        T1.CEBE,

        COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
        COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
        COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
        COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
        COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
        COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
        COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        CAST(EJERCICIO as STRING) as EJERCICIO,
        CAST(PERIODO as INT64) as PERIODO,
        VERSION,
        MES_INCREMENTO AS MES_INC,
        SAFE_CAST(PORC_INC AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) as importe,
        '' as Grupo,
        Concepto,
        Personas,
        AREA_PERS as AREA_PERSONAL,
        Posicion,
        0  as IMPORTE_VALES,
        0 as MONTO,
        SAFE_CAST(Sueldo_a_Utilizar AS NUMERIC) as SUELDOS,
        0 as PORC_PREST,
        T0.DIMENSION6,
        '0' as PERIODO_BASE,
        EJERCICIO_DE_INGRESO,
        MES_DE_INGRESO,
        0 as PARAMETROS,
        0 as SUELDO,
        ORIGEN

        FROM TRANSFORMACION T0
        LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
        LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
        LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA

    ),


    VALES AS (
        SELECT
        TRIM('51000032'||' '||T2.Descripcion) as Cuenta,
        CECO,
        CEBE,

        DIMENSION1,
        DIMENSION2,
        DIMENSION3,
        DIMENSION6_MAESTRO,
        DIMENSION7,
        SUELDOS.DIMENSION5,
        DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        EJERCICIO,
        PERIODO,
        VERSION,
        PARAMS.mesinc_vales AS MES_INC,
        SAFE_CAST(PARAMS.porc_inc_vales AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(
        (
          --VALES UNIFORME SE DEPOSITAN EN 8 SE PROVISIONAN 1-7
        IFNULL(CASE
                  ---SOLO DE ENERO A JULIO SE PROVISIONA
                  WHEN ( DIMENSION1 = 'Negocio11' AND PERIODO <= 7) OR PERIODO <= 7
                  THEN CAST(T6.MONTO AS INT64)/1.16 ---EL MONTO TIENE IVA
                        --SOLO LOS QUE ENTRAN ANTES DE AGOSTO
                    * CASE
                           WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO < SUELDOS.MES_DE_INGRESO THEN 0 ---POSTERIOR A LA FECHA DE INGRESO
                           WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.MES_DE_INGRESO <= 7 THEN 1 ---SI ENTRAN ANTES DE AGOSTO, LES TOCA PARA SU EJERCICIO DE INGRESO
                           WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) <> SUELDOS.EJERCICIO THEN 1  --- SI ENTRARON EN ME PARA PLAN, LES TOCA
                           ELSE 0 END  --  TODOS LOS DEMAS NO LES TOCA
                        --MESES LABORADOS ANTES DE AGOSTO
                    / CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND 7-SUELDOS.MES_DE_INGRESO > 0 THEN (7-SUELDOS.MES_DE_INGRESO+1) ELSE 7 END  --CUANTOS MESES LABORA EN SU AÑO DE INGRESO ANTES DE JULIO, ELSE 7 ES PARA PLAN CUANDO ENTRARON EN ME O PLAN
                  ELSE 0 END
        ,0)
        +
        --VALES DE DESPENSA
        IFNULL(
        CASE WHEN PERIODO >= PARAMS.mesinc_vales THEN (1+PARAMS.porc_inc_vales) ELSE 1 END * -- INCREMENTO
        CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO >= SUELDOS.MES_DE_INGRESO THEN 1 ---POSTERIOR A LA FECHA DE INGRESO
             WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) <> SUELDOS.EJERCICIO THEN 1  --- PLAN SI ENTRARON EN ME
             ELSE 0 END
        *
          ---MENOR O IGUAL A 3 MESES ANTIGÜEDAD VIA PLANTA
        CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO - SUELDOS.MES_DE_INGRESO <= 3 AND SUELDOS.AREA_PERSONAL NOT IN ('1C')
          THEN PARAMS.vales_via_planta
          ELSE T8.Importe_vales END
        ,0)
        )
          * PERSONAS

        AS NUMERIC) as importe,
        '' as Grupo,
        Concepto,
        Personas,
        SUELDOS.AREA_PERSONAL,
        SUELDOS.Posicion,
        SAFE_CAST(IFNULL(
          CASE WHEN PERIODO >= PARAMS.mesinc_vales THEN (1+PARAMS.porc_inc_vales) ELSE 1 END * -- INCREMENTO
          CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO - SUELDOS.MES_DE_INGRESO <= 3 AND SUELDOS.AREA_PERSONAL NOT IN ('1C')
          THEN PARAMS.vales_via_planta
          ELSE T8.Importe_vales END
            ,0) as NUMERIC) as IMPORTE_VALES,
        SAFE_CAST(IFNULL(CAST(T6.MONTO AS INT64)/1.16,0) as NUMERIC) as MONTO,
        SUELDOS.SUELDOS,
        PORC_PREST,
        DIMENSION6,
        PERIODO_BASE,
        EJERCICIO_DE_INGRESO,
        MES_DE_INGRESO,
        SUELDOS.PARAMETROS,
        SUELDOS.SUELDO,
        CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN


        FROM SUELDOS
        LEFT JOIN {tablavales_un} T6 ON SUELDOS.POSICION=T6.POSICION
        LEFT JOIN {tablavales} T8 ON SUELDOS.AREA_PERSONAL=T8.AREA_PERSONAL AND T8.DIMENSION5 = '0000'
        LEFT JOIN {CUENTAS} T2 ON '51000032'=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON '51000032'=T3.CUENTA
        CROSS JOIN {tabla_pargen} PARAMS

      ),


    BASE_SUELDOS_PR AS (
      SELECT T0.CeCo, T0.CeBe, T0.`Ejercicio`, T0.Periodo, T0.Version, T0.DIMENSION1, T0.DIMENSION2, T0.DIMENSION3,
      T0.DIMENSION6,Posicion, Personas,Ejercicio_DE_INGRESO,MES_DE_INGRESO,
      Concepto,
      SUM(T0.Importe) as `Importe`
      FROM SUELDOS T0

      GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13
      ),

      DIST_PREST AS (
        SELECT * FROM {tablapres_disanioanterior}
        UNION ALL
        SELECT * FROM {tablapres_disanioactual}
      ),

      CECOS AS (

      -- DISTRIBUCION CECOS
      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.CECO=T12.CECO
      ),

      AREA AS (
      -- DISTRIBUCION DIR AREA

      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6,T12.DIR_V
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.DIMENSION3=T12.DIR_V AND T12.DIMENSION6 = "DIMENSION3"
      LEFT JOIN CECOS
        ON T11.CECO=CECOS.CECO

      WHERE CECOS.CUENTA IS NULL
      ),

      CORP AS (
      -- DISTRIBUCION DIR CORP

      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6,T12.DIR_V
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.DIMENSION2=T12.DIR_V AND T12.DIMENSION6 = "DIMENSION2"
      LEFT JOIN CECOS
        ON T11.CECO=CECOS.CECO
      LEFT JOIN AREA
        ON T11.DIMENSION3=AREA.DIR_V

      WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL
      ),

      DIV AS (
      -- DISTRIBUCION DIV NEG

      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.DIMENSION1=T12.DIR_V AND T12.DIMENSION6 = "DIMENSION1"
      LEFT JOIN CECOS
        ON T11.CECO=CECOS.CECO
      LEFT JOIN AREA
        ON T11.DIMENSION3=AREA.DIR_V
      LEFT JOIN CORP
        ON T11.DIMENSION2=CORP.DIR_V

      WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL AND CORP.CUENTA IS NULL
      ),

      TRANSFORMACION_PRES AS (

      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM CECOS
      UNION ALL
      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM AREA
      UNION ALL
      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM CORP
      UNION ALL
      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM DIV
      ),

        PRESTACIONES AS (
          SELECT
              TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
              T1.CECO,
              T1.CEBE,

              COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
              COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
              COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
              COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
              COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
              COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
              COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
              T3.NIVEL1,
              T3.NIVEL2,
              T3.NIVEL3,
              T3.NIVEL4,
              T3.NIVEL5,
              CAST(EJERCICIO as STRING) as EJERCICIO,
              CAST(PERIODO as INT64) as PERIODO,
              VERSION,
              0 AS MES_INC,
              0 as PORC_INC,
              0 as MES_BASE,
              SAFE_CAST(Importe AS NUMERIC) as importe,
              DIMENSION6_PRES as Grupo,
              Concepto,
              Personas,
              '' as AREA_PERSONAL,
              Posicion,
              0  as IMPORTE_VALES,
              0 as MONTO,
              SAFE_CAST(SUELDOS AS NUMERIC) as SUELDOS,
              SAFE_CAST(PORC_PREST AS NUMERIC) as PORC_PREST,
              T0.DIMENSION6,
              '0' as PERIODO_BASE,
              EJERCICIO_DE_INGRESO,
              MES_DE_INGRESO,
              0 as PARAMETROS,
              0 as SUELDO,

              CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN --path parametro

              --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5

              FROM TRANSFORMACION_PRES T0
              LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
              LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
              LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
              LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA

      ),

      SUELDOS_BONO_ESP AS (
          SELECT T0.CeCo, T0.CeBe, ORIGEN, Ejercicio, Periodo, Version,Posicion, DIMENSION6, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO,
          CASE WHEN Periodo IN (1,11) AND T0.CUENTA = '51000041' THEN 0.5 ELSE 1 END * SUM(T0.Importe) as `Sueldo`, Concepto

                FROM SUELDOS T0
                CROSS JOIN {tabla_pargen} PARAMS

                WHERE SUBSTRING(T0.Cuenta,1,8) IN ('51000001','51000002','51000040','51000041','51000044','51000045','51000399')
                  --AND Periodo >= PARAMS.mes_captura

                GROUP BY T0.CeCo, T0.CeBe,cuenta,Version,Ejercicio, Periodo,ORIGEN,DIMENSION6,Posicion,  Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
          ),

              BONO AS (
                SELECT '51000403' as `CUENTA`, T0.CECO, T0.CEBE, T0.Periodo, SAFE_CAST(T0.Ejercicio as INT64) as EJERCICIO, T0.Version,T0.Posicion, T0.DIMENSION6, T0.Personas, T0.Ejercicio_DE_INGRESO,
                T0.MES_DE_INGRESO, Con
                IFNULL(CASE
                      WHEN (T0.MES_DE_INGRESO > 7 AND T0.DIMENSION6 IN ('Adicionales', 'Presupuestadas') AND T0.Posicion IN ('Coordinador','Jefatura') ) OR T0.POSICION IN ('Personal Gral') THEN 0 --SOLO PARA ADICIONALES Y PRESUPUESTADAS CON PLANTA DE 3 MESES
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc)*(1+PARAMS.porc_inc_sig_anio) --PRIMEROS MESES DOBLE INCREMENTO PPTO_V1 Y PPTO_V2
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_sig_anio)*(1+PARAMS.porc_inc_seg_anio) --PRIMEROS MESES DOBLE INCREMENTO PLAN
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc_sig_anio) --PPTO_V1 Y PPTO_V2 JULIO EN ADELANTE
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_seg_anio) --PLAN JULIO EN ADELANTE
                      END *
                    T0.Sueldo *
                      COALESCE(
                              SAFE_CAST(BONO_ESP.Promedio_Meses AS FLOAT64)
                              ,PARAMS.meses_bono
                              )
                    ,0) as Importe,

                    CASE
                      WHEN (T0.MES_DE_INGRESO > 7 AND T0.DIMENSION6 IN ('Adicionales', 'Presupuestadas') AND T0.Posicion IN ('Coordinador','Jefatura') ) OR T0.POSICION IN ('Personal Gral') THEN 0 --SOLO PARA ADICIONALES Y PRESUPUESTADAS CON PLANTA DE 3 MESES
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc)*(1+PARAMS.porc_inc_sig_anio) --PRIMEROS MESES DOBLE INCREMENTO PPTO_V1 Y PPTO_V2
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_sig_anio)*(1+PARAMS.porc_inc_seg_anio) --PRIMEROS MESES DOBLE INCREMENTO PLAN
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc_sig_anio) --PPTO_V1 Y PPTO_V2 JULIO EN ADELANTE
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_seg_anio) --PLAN JULIO EN ADELANTE
                      END *
                    T0.Sueldo as Sueldo,

                     COALESCE(
                              SAFE_CAST(BONO_ESP.Promedio_Meses AS FLOAT64)
                              ,PARAMS.meses_bono
                              ) as PARAMETROS

                FROM SUELDOS_BONO_ESP T0
                LEFT JOIN {CECOS} CECOS ON T0.CECO = CECOS.CECO
                CROSS JOIN {tabla_pargen} PARAMS
                LEFT JOIN (
                  -- Priority 1: DIMENSION1 + DIMENSION2
                  SELECT *, 1 AS prioridad
                  FROM {PAR_BONO_ESP}
                  WHERE DIMENSION1 IS NOT NULL
                    AND DIMENSION2 IS NOT NULL

                  UNION ALL

                  -- Priority 2: only DIMENSION1
                  SELECT *, 2 AS prioridad
                  FROM {PAR_BONO_ESP}
                  WHERE DIMENSION1 IS NOT NULL
                    AND DIMENSION2 IS NULL

                  UNION ALL

                  -- Priority 3: only DIMENSION2
                  SELECT *, 3 AS prioridad
                  FROM {PAR_BONO_ESP}
                  WHERE DIMENSION1 IS NULL
                    AND DIMENSION2 IS NOT NULL
                ) BONO_ESP
                ON (
                    (BONO_ESP.prioridad = 1 AND CECOS.DIMENSION1 = BONO_ESP.DIMENSION1 AND CECOS.DIMENSION2 = BONO_ESP.DIMENSION2)
                  OR (BONO_ESP.prioridad = 2 AND CECOS.DIMENSION1 = BONO_ESP.DIMENSION1)
                  OR (BONO_ESP.prioridad = 3 AND CECOS.DIMENSION2 = BONO_ESP.DIMENSION2)
                )
                QUALIFY ROW_NUMBONO_ESP() OVER (
                  PARTITION BY T0.CECO, T0.CEBE, T0.Periodo, T0.Ejercicio, T0.Version,T0.Posicion, T0.DIMENSION6, T0.Personas,
                T0.Ejercicio_DE_INGRESO,
                T0.MES_DE_INGRESO
                  ORDER BY BONO_ESP.prioridad
                ) = 1
            ),

            BONO_AGRUP AS (
              SELECT CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS,
              SUM(IMPORTE) AS IMPORTE, SUM(SUELDO) AS SUELDO

              FROM BONO

              GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12
            ),

            CALCULO AS (

            SELECT T0.* EXCEPT(PERIODO, VERSION, IMPORTE, sueldo),
            CASE WHEN T1.PERIODO <= PARAMS.mes_captura AND T0.VERSION = PARAMS.primera_version then PARAMS.mes_captura else t1.periodo end as PERIODO,
            T0.PERIODO as PERIODO_BASE,
            VERSION,

            CASE WHEN T0.VERSION = PARAMS.primera_version AND T1.PERIODO <= PARAMS.mes_captura AND T0.PERIODO = T1.PERIODO-1 THEN 'AJUSTE_BONO' ELSE 'NOM_BONO' END AS ORIGEN,


            CASE
              WHEN T0.PERIODO > T1.PERIODO THEN 0  --  meses mayores al calculado SE QUITAN
              WHEN T0.PERIODO = T1.PERIODO THEN 1 --  calculo del mes
              WHEN T0.PERIODO = T1.PERIODO-1 THEN -1 -- cancela calculo acumulado mes anterior
            ELSE 0 END

            *
            CASE WHEN T1.PERIODO = T0.PERIODO OR T1.PERIODO-1 = T0.PERIODO THEN sueldo else 0 END
            *
            PARAMETROS

            ---ARREGLAR CANCELACION PRIMER MES DE CAPTURA (en segundo mes)
            / CASE WHEN T1.PERIODO = PARAMS.mes_captura +1  AND T1.PERIODO-1 = T0.PERIODO AND PARAMS.mes_captura-MES_DE_INGRESO >0 THEN PARAMS.mes_captura-MES_DE_INGRESO+1 ELSE 1 END
            ---ARREGLAR EL PRIMER MES
            / CASE WHEN T1.PERIODO = PARAMS.mes_captura AND T1.PERIODO = PARAMS.mes_captura AND PARAMS.mes_captura-MES_DE_INGRESO >0 THEN PARAMS.mes_captura-MES_DE_INGRESO+1 ELSE 1 END
            --- EL MONTO QUE LE CORRESPONDE POR AÑO

            /CASE WHEN T0.EJERCICIO_DE_INGRESO = T0.EJERCICIO THEN (12-MES_DE_INGRESO+1) ELSE 12 END  --CUANTOS MESES TIENE EL BONO (PARA OBTENER MONTO MENSUAL)

            * CASE WHEN T0.EJERCICIO_DE_INGRESO = T0.EJERCICIO THEN (T0.PERIODO-MES_DE_INGRESO+1)  ELSE T0.PERIODO END --MESES QUE HAN TRANSCURRIDO (PARA ACUMULAR)

            * CASE WHEN T0.EJERCICIO_DE_INGRESO = T0.EJERCICIO THEN (12-MES_DE_INGRESO+1)/12 ELSE 1 END --- PARCIALIDAD POR TIEMPO QUE LABORÓ PARA EL ANUAL (PORCENTAJE DEL AÑO)
            AS IMPORTE,

            CASE WHEN T1.PERIODO = T0.PERIODO OR T1.PERIODO-1 = T0.PERIODO THEN sueldo else 0 END As SUELDO

            FROM BONO_AGRUP T0
            CROSS JOIN {PERIODOS} T1
            CROSS JOIN {tabla_pargen} PARAMS

            ),

            TRANSFORMACION_BONO_ESP AS (

            SELECT CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, ORIGEN, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS,
            PERIODO_BASE,
            SUM(IMPORTE) as IMPORTE, SUM(SUELDO) as SUELDOS

            FROM CALCULO

            group by CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, ORIGEN, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS, PERIODO_BASE
      ),

      BONO_ESP AS (
          SELECT
              TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
              T1.CECO,
              T1.CEBE,

              COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
              COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
              COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
              COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
              COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
              COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
              COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
              T3.NIVEL1,
              T3.NIVEL2,
              T3.NIVEL3,
              T3.NIVEL4,
              T3.NIVEL5,
              CAST(EJERCICIO as STRING) as EJERCICIO,
              CAST(PERIODO as INT64) as PERIODO,
              VERSION,
              0 AS MES_INC,
              0 AS PORC_INC,
              0 as MES_BASE,
              SAFE_CAST(Importe AS NUMERIC) as importe,
              '' as Grupo,
              '' as Concepto,
              Personas,
              '' as AREA_PERSONAL,
              Posicion,
              0  as IMPORTE_VALES,
              0 as MONTO,
              SAFE_CAST(SUELDOS AS NUMERIC) as SUELDOS,
              0 as PORC_PREST,
              T0.DIMENSION6,
              SAFE_CAST(PERIODO_BASE AS STRING) as PERIODO_BASE,
              EJERCICIO_DE_INGRESO,
              MES_DE_INGRESO,
              SAFE_CAST(PARAMETROS AS NUMERIC) as PARAMETROS,
              0 as SUELDO,

              CONCAT("CN_HC_BONO_ESP",path_p, ORIGEN) as ORIGEN ---path parametro

              --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5

              FROM TRANSFORMACION_BONO_ESP T0
              LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
              LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
              LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
              LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA
              ),

          SUELDO_DIC AS (
            SELECT SUELDOS_BONO_ESP.* EXCEPT(PERIODO) ,
            CASE WHEN T1.PERIODO <= PARAMS.mes_captura AND SUELDOS_BONO_ESP.VERSION = PARAMS.primera_version then PARAMS.mes_captura else t1.periodo end as PERIODO,
            SUELDO * 2
            * CASE WHEN SAFE_CAST(SUELDOS_BONO_ESP.EJERCICIO_DE_INGRESO as STRING) = SUELDOS_BONO_ESP.EJERCICIO THEN (12-MES_DE_INGRESO+1)/12 ELSE 1 END --- PARCIALIDAD POR TIEMPO QUE LABORÓ
            ---HASTA AQUI TENEMOS EL IMPORTE ANUAL
            / CASE WHEN SAFE_CAST(SUELDOS_BONO_ESP.EJERCICIO_DE_INGRESO as STRING) = SUELDOS_BONO_ESP.EJERCICIO THEN (12-MES_DE_INGRESO+1) ELSE 12 END  --CUANTOS MESES LABORA EN SU AÑO DE INGRESO
            * CASE WHEN SAFE_CAST(SUELDOS_BONO_ESP.EJERCICIO_DE_INGRESO as STRING) = SUELDOS_BONO_ESP.EJERCICIO AND T1.PERIODO < SUELDOS_BONO_ESP.MES_DE_INGRESO THEN 0 ELSE 1 END  --- SOLO PARA LOS PERIODOS LABORADOS

             as IMPORTE,
            '51000412' as CUENTA

            FROM SUELDOS_BONO_ESP
            CROSS JOIN {PERIODOS} T1
            CROSS JOIN {tabla_pargen} PARAMS

            WHERE SUELDOS_BONO_ESP.PERIODO = 12
          ),

          PTU_GAR AS (
            SELECT
                TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
                T1.CECO,
                T1.CEBE,

                COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
                COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
                COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
                COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
                COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
                COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
                COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
                T3.NIVEL1,
                T3.NIVEL2,
                T3.NIVEL3,
                T3.NIVEL4,
                T3.NIVEL5,
                CAST(EJERCICIO as STRING) as EJERCICIO,
                CAST(PERIODO as INT64) as PERIODO,
                VERSION,
                0 AS MES_INC,
                0 AS PORC_INC,
                0 as MES_BASE,
                SAFE_CAST(Importe AS NUMERIC) as importe,
                '' as Grupo,
                '' as Concepto,
                Personas,
                '' as AREA_PERSONAL,
                Posicion,
                0  as IMPORTE_VALES,
                0 as MONTO,
                SAFE_CAST(SUELDO AS NUMERIC) as SUELDOS,
                0 as PORC_PREST,
                T0.DIMENSION6,
                '0' as PERIODO_BASE,
                EJERCICIO_DE_INGRESO,
                MES_DE_INGRESO,
                SAFE_CAST(2 AS NUMERIC) as PARAMETROS,
                0 as SUELDO,

                CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN ---path parametro

                FROM SUELDO_DIC T0
                LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
                LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
                LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
                LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA
              ),

          CALCULOS_COMPLETOS AS (

          SELECT * FROM SUELDOS
          UNION ALL
          SELECT * FROM PRESTACIONES
          UNION ALL
          SELECT * FROM BONO_ESP
          UNION ALL
          SELECT * FROM PTU_GAR
          UNION ALL
          SELECT * FROM VALES

          ),

          SBC AS (
          SELECT * EXCEPT(CUENTA, IMPORTE), SUM(IMPORTE) AS IMPORTE

          FROM CALCULOS_COMPLETOS
          INNER JOIN {CUENTAS_SBC} CUENTAS_SBC ON SUBSTRING(CALCULOS_COMPLETOS.CUENTA,1,8) = CUENTAS_SBC.CUENTA
          GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37

          ),

          ISN AS (

            SELECT

                TRIM('51000053 '||T2.Descripcion) as Cuenta,
                CECO,
                CEBE,

                DIMENSION1,
                DIMENSION2,
                DIMENSION3,
                DIMENSION6_MAESTRO,
                DIMENSION7,
                DIMENSION5,
                DES_DIMENSION5,
                T3.NIVEL1,
                T3.NIVEL2,
                T3.NIVEL3,
                T3.NIVEL4,
                T3.NIVEL5,
                EJERCICIO,
                PERIODO,
                VERSION,
                MES_INC,
                T0.PORC_INC,
                MES_BASE,
                SAFE_CAST(IMPORTE AS NUMERIC) * SAFE_CAST(PARAMS.isn_cdmx AS NUMERIC) as importe,
                Grupo,
                Concepto,
                Personas,
                AREA_PERSONAL,
                Posicion,
                IMPORTE_VALES,
                MONTO,
                SAFE_CAST(IMPORTE AS NUMERIC) as SUELDOS,
                SAFE_CAST(PARAMS.isn_cdmx AS NUMERIC) as PORC_PREST,
                DIMENSION6,
                PERIODO_BASE,
                EJERCICIO_DE_INGRESO,
                MES_DE_INGRESO,
                PARAMETROS,
                SUELDO,

                CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN ---path parametro

                FROM SBC T0
                LEFT JOIN {CUENTAS} T2 ON '51000053'=T2.CUENTA
                LEFT JOIN {JERARCUENTAPPTO} T3 ON '51000053'=T3.CUENTA
                CROSS JOIN {tabla_pargen} PARAMS

                )

          SELECT * FROM CALCULOS_COMPLETOS
          UNION ALL
          SELECT * FROM ISN
)
'''

## CONCEPTOS NOMINA

create_procedure_cnhc = f"""
CREATE OR REPLACE PROCEDURE `{stored_procedure_cnhc}`(path STRING)

BEGIN

  DECLARE path_p STRING DEFAULT IFNULL(path,'');

  DELETE FROM {tabla_base_hccn_sp}
  WHERE ORIGEN IN
    ( CONCAT('CN_HC', path_p)  ,
      CONCAT('CN_HC_SUE', path_p)  ,
      CONCAT('CN_HC_PRESTACIONES', path_p)  ,
      CONCAT('CN_HC_BONO_ESP', path_p) ,  CONCAT('CN_HC_BONO_ESP', path_p, 'AJUSTE_BONO') ,  CONCAT('CN_HC_BONO_ESP', path_p, 'NOM_BONO'), CONCAT('CN_HC_BONO_ESP', path_p, 'BONO_CN'),
      CONCAT('CN_HC_PTU', path_p),
      CONCAT('CN_HC_PRE', path_p)

  );

   -- MERGE destination is the original table
  INSERT INTO {tabla_base_hccn_sp}
    SELECT *
    FROM (
      {query_cn_hc}
      );

END
"""
client.query(create_procedure_cnhc).result()
print("Creado ",stored_procedure_cnhc)


In [ ]:
# VERSION 2.0

query_cn_hc = f'''
  (WITH

  BASEHC AS (
    SELECT * FROM `{tabla_base_hc}` WHERE PATH = path_p
  ),

    TRANSFORMACION AS (
  SELECT T1.Cuenta, T5.CeCo, T5.CEBE,
  CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END as Ejercicio,
    T0.DIMENSION6,
    CASE WHEN T3.MES < PARAMS.mes_captura AND VERSION = PARAMS.primera_version THEN PARAMS.mes_captura
    ELSE T3.MES END as Periodo,
    T3.Version, T0.MES_DE_INGRESO, T0.EJERCICIO_DE_INGRESO, PARAMS.mesinc as MES_INCREMENTO,
    Personas, T0.Posicion, PARAMS.porc_inc AS PORC_INC, T0.Sueldo_a_Utilizar,
    T1.AREA_PERS,

    --SI ENTRAN ANTES DE MARZO DE PLAN ENTONCES SI APLICA INCREMENTO
    IFNULL(
      CASE
      --SI NO TIENE VALOR
      WHEN T0.MES_DE_INGRESO > T3.MES AND T3.VERSION IN('PPTO_V1','PPTO_V2') THEN 0
      WHEN T0.EJERCICIO_DE_INGRESO > CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END THEN 0
      WHEN T0.EJERCICIO_DE_INGRESO > CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END AND T3.MES>T0.MES_DE_INGRESO THEN 0
      WHEN T0.EJERCICIO_DE_INGRESO = CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END AND T3.MES<T0.MES_DE_INGRESO THEN 0

      --INCREMENTO

      ---NO APLICA INCREMENTO SI ENTRAN EN JULIO O DESPUES EN SU AÑO DE INGRESO
      WHEN T0.MES_DE_INGRESO >= PARAMS.mesinc AND T0.EJERCICIO_DE_INGRESO = CASE WHEN T3.VERSION IN ("PPTO_V2","PPTO_V1") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END THEN 1
      --SIGUENTE AÑO AL DE INGRESO PLAN
      WHEN T0.EJERCICIO_DE_INGRESO < CASE WHEN T3.VERSION = "PPTO_V2" THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END AND T3.MES >= PARAMS.mesinc THEN (1+PARAMS.porc_inc_sig_anio)
      --DESPUES DE JULIO INCREMENTO A LOS MOVIEMIENTOS EN NOMINA EXISTENTE PRIMER AÑO
      WHEN T0.DIMENSION6 NOT IN ('Adicionales','Presupuestadas')
          AND T3.MES >= PARAMS.mesinc AND T3.VERSION IN ("PPTO_V1") THEN (1+PARAMS.porc_inc)
      --INCREMENTO A NUEVAS POSICIONES EN EL AÑO QUE ENTRARON
      WHEN T0.DIMENSION6 IN ('Adicionales','Presupuestadas') AND T3.MES >= PARAMS.mesinc
            AND T0.EJERCICIO_DE_INGRESO = CASE WHEN T3.VERSION = "PPTO_V2" THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END
        THEN
        --CASE WHEN T0.Posicion IN ('Director','Director Corporativo','Subdirector','Gerente')
        --          THEN
                  CASE WHEN T3.VERSION = "PPTO_V1" THEN (1+PARAMS.porc_inc) ELSE (1+PARAMS.porc_inc_sig_anio) END
         --    ELSE 1 END
        ELSE 1 END
        * T0.Sueldo_a_Utilizar * T0.PERSONAS
    ,0) as `Importe`,

    Comentarios as Concepto,

    CONCAT("CN_HC_SUE",path_p) as `ORIGEN`  --string es path parametro

    FROM BASEHC T0
    CROSS JOIN {tabla_pargen} PARAMS
    --SE ASIGNA CUENTA DE NOMINA POR POSICION
    LEFT JOIN {NOMINAPOSICION} T1 ON T0.Posicion=T1.POSICION
    --PERIODOS
    CROSS JOIN (
          SELECT *
          FROM `{PARPERIODOS}`
          CROSS JOIN {tabla_pargen} PARAMS
          WHERE Version IN UNNEST (PARAMS.versiones)
          ) T3
    --ATRIBUTOS CECO
    LEFT JOIN {CECOS} T5 ON
    CASE WHEN REGEXP_CONTAINS(T0.CECO, r'\\\d') THEN REPEAT("0",10-LENGTH(T0.CECO))||T0.CECO ELSE T0.CECO END
    = T5.Ceco

    WHERE T0.CECO IS NOT NULL AND T3.Version IN UNNEST (PARAMS.versiones)
    AND T5.DIMENSION1 <> 'NEGOCIO X y CC Externos'
    ),

    SUELDOS AS (

        SELECT
        TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
        T1.CECO,
        T1.CEBE,

        COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
        COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
        COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
        COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
        COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
        COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
        COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        CAST(EJERCICIO as STRING) as EJERCICIO,
        CAST(PERIODO as INT64) as PERIODO,
        VERSION,
        MES_INCREMENTO AS MES_INC,
        SAFE_CAST(PORC_INC AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) as importe,
        '' as Grupo,
        Concepto,
        Personas,
        AREA_PERS as AREA_PERSONAL,
        Posicion,
        0  as IMPORTE_VALES,
        0 as MONTO,
        SAFE_CAST(Sueldo_a_Utilizar AS NUMERIC) as SUELDOS,
        0 as PORC_PREST,
        T0.DIMENSION6,
        '0' as PERIODO_BASE,
        EJERCICIO_DE_INGRESO,
        MES_DE_INGRESO,
        0 as PARAMETROS,
        0 as SUELDO,
        ORIGEN

        FROM TRANSFORMACION T0
        LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
        LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
        LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA

    ),


    VALES AS (
        SELECT
        TRIM('51000032'||' '||T2.Descripcion) as Cuenta,
        CECO,
        CEBE,

        DIMENSION1,
        DIMENSION2,
        DIMENSION3,
        DIMENSION6_MAESTRO,
        DIMENSION7,
        SUELDOS.DIMENSION5,
        DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        EJERCICIO,
        PERIODO,
        VERSION,
        PARAMS.mesinc_vales AS MES_INC,
        SAFE_CAST(PARAMS.porc_inc_vales AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(
        (
          --VALES UNIFORME SE DEPOSITAN EN 8 SE PROVISIONAN 1-7
        IFNULL(CASE
                  ---SOLO DE ENERO A JULIO SE PROVISIONA
                  WHEN ( DIMENSION1 = 'Negocio11' AND PERIODO <= 7) OR PERIODO <= 7
                  THEN CAST(T6.MONTO AS INT64)/1.16 ---EL MONTO TIENE IVA
                        --SOLO LOS QUE ENTRAN ANTES DE AGOSTO
                    * CASE
                           WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO < SUELDOS.MES_DE_INGRESO THEN 0 ---POSTERIOR A LA FECHA DE INGRESO
                           WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.MES_DE_INGRESO <= 7 THEN 1 ---SI ENTRAN ANTES DE AGOSTO, LES TOCA PARA SU EJERCICIO DE INGRESO
                           WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) <> SUELDOS.EJERCICIO THEN 1  --- SI ENTRARON EN ME PARA PLAN, LES TOCA
                           ELSE 0 END  --  TODOS LOS DEMAS NO LES TOCA
                        --MESES LABORADOS ANTES DE AGOSTO
                    / CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND 7-SUELDOS.MES_DE_INGRESO > 0 THEN (7-SUELDOS.MES_DE_INGRESO+1) ELSE 7 END  --CUANTOS MESES LABORA EN SU AÑO DE INGRESO ANTES DE JULIO, ELSE 7 ES PARA PLAN CUANDO ENTRARON EN ME O PLAN
                  ELSE 0 END
        ,0)
        +
        --VALES DE DESPENSA
        IFNULL(
        CASE WHEN PERIODO >= PARAMS.mesinc_vales THEN (1+PARAMS.porc_inc_vales) ELSE 1 END * -- INCREMENTO
        CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO >= SUELDOS.MES_DE_INGRESO THEN 1 ---POSTERIOR A LA FECHA DE INGRESO
             WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) <> SUELDOS.EJERCICIO THEN 1  --- PLAN SI ENTRARON EN ME
             ELSE 0 END
        *
          ---MENOR O IGUAL A 3 MESES ANTIGÜEDAD VIA PLANTA
        CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO - SUELDOS.MES_DE_INGRESO <= 3 AND SUELDOS.AREA_PERSONAL NOT IN ('1C')
          THEN PARAMS.vales_via_planta
          ELSE T8.Importe_vales END
        ,0)
        )
          * PERSONAS

        AS NUMERIC) as importe,
        '' as Grupo,
        Concepto,
        Personas,
        SUELDOS.AREA_PERSONAL,
        SUELDOS.Posicion,
        SAFE_CAST(IFNULL(
          CASE WHEN PERIODO >= PARAMS.mesinc_vales THEN (1+PARAMS.porc_inc_vales) ELSE 1 END * -- INCREMENTO
          CASE WHEN SAFE_CAST(SUELDOS.EJERCICIO_DE_INGRESO as STRING) = SUELDOS.EJERCICIO AND SUELDOS.PERIODO - SUELDOS.MES_DE_INGRESO <= 3 AND SUELDOS.AREA_PERSONAL NOT IN ('1C')
          THEN PARAMS.vales_via_planta
          ELSE T8.Importe_vales END
            ,0) as NUMERIC) as IMPORTE_VALES,
        SAFE_CAST(IFNULL(CAST(T6.MONTO AS INT64)/1.16,0) as NUMERIC) as MONTO,
        SUELDOS.SUELDOS,
        PORC_PREST,
        DIMENSION6,
        PERIODO_BASE,
        EJERCICIO_DE_INGRESO,
        MES_DE_INGRESO,
        SUELDOS.PARAMETROS,
        SUELDOS.SUELDO,
        CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN


        FROM SUELDOS
        LEFT JOIN {tablavales_un} T6 ON SUELDOS.POSICION=T6.POSICION
        LEFT JOIN {tablavales} T8 ON SUELDOS.AREA_PERSONAL=T8.AREA_PERSONAL AND T8.DIMENSION5 = '0000'
        LEFT JOIN {CUENTAS} T2 ON '51000032'=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON '51000032'=T3.CUENTA
        CROSS JOIN {tabla_pargen} PARAMS

      ),


    BASE_SUELDOS_PR AS (
      SELECT T0.CeCo, T0.CeBe, T0.`Ejercicio`, T0.Periodo, T0.Version, T0.DIMENSION1, T0.DIMENSION2, T0.DIMENSION3,
      T0.DIMENSION6,Posicion, Personas,Ejercicio_DE_INGRESO,MES_DE_INGRESO,
      Concepto,
      SUM(T0.Importe) as `Importe`
      FROM SUELDOS T0

      GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13
      ),

      DIST_PREST AS (
        SELECT * FROM {tablapres_disanioanterior}
        UNION ALL
        SELECT * FROM {tablapres_disanioactual}
      ),

      CECOS AS (

      -- DISTRIBUCION CECOS
      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.CECO=T12.CECO
      ),

      AREA AS (
      -- DISTRIBUCION DIR AREA

      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6,T12.DIR_V
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.DIMENSION3=T12.DIR_V AND T12.DIMENSION6 = "DIMENSION3"
      LEFT JOIN CECOS
        ON T11.CECO=CECOS.CECO

      WHERE CECOS.CUENTA IS NULL
      ),

      CORP AS (
      -- DISTRIBUCION DIR CORP

      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6,T12.DIR_V
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.DIMENSION2=T12.DIR_V AND T12.DIMENSION6 = "DIMENSION2"
      LEFT JOIN CECOS
        ON T11.CECO=CECOS.CECO
      LEFT JOIN AREA
        ON T11.DIMENSION3=AREA.DIR_V

      WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL
      ),

      DIV AS (
      -- DISTRIBUCION DIV NEG

      SELECT T12.Cuenta, T11.`CeCo`, T11.`CeBe`, T11.Ejercicio, T11.Periodo, T11.`Version`, T11.Posicion, T11.Personas,T11.Ejercicio_DE_INGRESO,T11.MES_DE_INGRESO,
      CAST(T11.Importe * T12.Importe AS NUMERIC) as `Importe`, "NOM_PRE" as `ORIGEN`,T11.Importe`Sueldos`, T12.Importe `PORC_PREST`, T12.DIMENSION6 as DIMENSION6_PRES, T11.DIMENSION6
      ,Concepto

      --MONTO TOTAL ANUAL DE SUELDOS
      FROM BASE_SUELDOS_PR T11
      --DISTRIBUCIÓN MENSUAL DE PRESTACIONES
      INNER JOIN DIST_PREST T12
        ON T11.DIMENSION1=T12.DIR_V AND T12.DIMENSION6 = "DIMENSION1"
      LEFT JOIN CECOS
        ON T11.CECO=CECOS.CECO
      LEFT JOIN AREA
        ON T11.DIMENSION3=AREA.DIR_V
      LEFT JOIN CORP
        ON T11.DIMENSION2=CORP.DIR_V

      WHERE CECOS.CUENTA IS NULL AND AREA.CUENTA IS NULL AND CORP.CUENTA IS NULL
      ),

      TRANSFORMACION_PRES AS (

      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM CECOS
      UNION ALL
      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM AREA
      UNION ALL
      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM CORP
      UNION ALL
      SELECT Cuenta, CeCo, CeBe, Ejercicio, Periodo, Version, Importe, Concepto, "NOM_PRE_CN" as ORIGEN, Sueldos, PORC_PREST, DIMENSION6_PRES, DIMENSION6, Posicion, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
      FROM DIV
      ),

        PRESTACIONES AS (
          SELECT
              TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
              T1.CECO,
              T1.CEBE,

              COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
              COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
              COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
              COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
              COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
              COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
              COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
              T3.NIVEL1,
              T3.NIVEL2,
              T3.NIVEL3,
              T3.NIVEL4,
              T3.NIVEL5,
              CAST(EJERCICIO as STRING) as EJERCICIO,
              CAST(PERIODO as INT64) as PERIODO,
              VERSION,
              0 AS MES_INC,
              0 as PORC_INC,
              0 as MES_BASE,
              SAFE_CAST(Importe AS NUMERIC) as importe,
              DIMENSION6_PRES as Grupo,
              Concepto,
              Personas,
              '' as AREA_PERSONAL,
              Posicion,
              0  as IMPORTE_VALES,
              0 as MONTO,
              SAFE_CAST(SUELDOS AS NUMERIC) as SUELDOS,
              SAFE_CAST(PORC_PREST AS NUMERIC) as PORC_PREST,
              T0.DIMENSION6,
              '0' as PERIODO_BASE,
              EJERCICIO_DE_INGRESO,
              MES_DE_INGRESO,
              0 as PARAMETROS,
              0 as SUELDO,

              CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN --path parametro

              --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5

              FROM TRANSFORMACION_PRES T0
              LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
              LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
              LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
              LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA

      ),

      SUELDOS_BONO_ESP AS (
          SELECT T0.CeCo, T0.CeBe, ORIGEN, Ejercicio, Periodo, Version,Posicion, DIMENSION6, Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO,
          CASE WHEN Periodo IN (1,11) AND T0.CUENTA = '51000041' THEN 0.5 ELSE 1 END * SUM(T0.Importe) as `Sueldo`, Concepto

                FROM SUELDOS T0
                CROSS JOIN {tabla_pargen} PARAMS

                WHERE SUBSTRING(T0.Cuenta,1,8) IN ('51000001','51000002','51000040','51000041','51000044','51000045','51000399')
                  --AND Periodo >= PARAMS.mes_captura

                GROUP BY T0.CeCo, T0.CeBe,cuenta,Version,Ejercicio, Periodo,ORIGEN,DIMENSION6,Posicion,  Personas, Ejercicio_DE_INGRESO, MES_DE_INGRESO
          ),

              BONO AS (
                SELECT '51000403' as `CUENTA`, T0.CECO, T0.CEBE, T0.Periodo, SAFE_CAST(T0.Ejercicio as INT64) as EJERCICIO, T0.Version,T0.Posicion, T0.DIMENSION6, T0.Personas, T0.Ejercicio_DE_INGRESO,
                T0.MES_DE_INGRESO, Con
                IFNULL(CASE
                      WHEN (T0.MES_DE_INGRESO > 7 AND T0.DIMENSION6 IN ('Adicionales', 'Presupuestadas') AND T0.Posicion IN ('Coordinador','Jefatura') ) OR T0.POSICION IN ('Personal Gral') THEN 0 --SOLO PARA ADICIONALES Y PRESUPUESTADAS CON PLANTA DE 3 MESES
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc)*(1+PARAMS.porc_inc_sig_anio) --PRIMEROS MESES DOBLE INCREMENTO PPTO_V1 Y PPTO_V2
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_sig_anio)*(1+PARAMS.porc_inc_seg_anio) --PRIMEROS MESES DOBLE INCREMENTO PLAN
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc_sig_anio) --PPTO_V1 Y PPTO_V2 JULIO EN ADELANTE
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_seg_anio) --PLAN JULIO EN ADELANTE
                      END *
                    T0.Sueldo *
                      COALESCE(
                              SAFE_CAST(BONO_ESP.Promedio_Meses AS FLOAT64)
                              ,PARAMS.meses_bono
                              )
                    ,0) as Importe,

                    CASE
                      WHEN (T0.MES_DE_INGRESO > 7 AND T0.DIMENSION6 IN ('Adicionales', 'Presupuestadas') AND T0.Posicion IN ('Coordinador','Jefatura') ) OR T0.POSICION IN ('Personal Gral') THEN 0 --SOLO PARA ADICIONALES Y PRESUPUESTADAS CON PLANTA DE 3 MESES
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc)*(1+PARAMS.porc_inc_sig_anio) --PRIMEROS MESES DOBLE INCREMENTO PPTO_V1 Y PPTO_V2
                      WHEN T0.Periodo < PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_sig_anio)*(1+PARAMS.porc_inc_seg_anio) --PRIMEROS MESES DOBLE INCREMENTO PLAN
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase THEN (1+PARAMS.porc_inc_sig_anio) --PPTO_V1 Y PPTO_V2 JULIO EN ADELANTE
                      WHEN T0.Periodo >= PARAMS.mesinc AND SAFE_CAST(T0.Ejercicio as INT64) = PARAMS.aniobase+1 THEN (1+PARAMS.porc_inc_seg_anio) --PLAN JULIO EN ADELANTE
                      END *
                    T0.Sueldo as Sueldo,

                     COALESCE(
                              SAFE_CAST(BONO_ESP.Promedio_Meses AS FLOAT64)
                              ,PARAMS.meses_bono
                              ) as PARAMETROS

                FROM SUELDOS_BONO_ESP T0
                LEFT JOIN {CECOS} CECOS ON T0.CECO = CECOS.CECO
                CROSS JOIN {tabla_pargen} PARAMS
                LEFT JOIN (
                  -- Priority 1: DIMENSION1 + DIMENSION2
                  SELECT *, 1 AS prioridad
                  FROM {PAR_BONO_ESP}
                  WHERE DIMENSION1 IS NOT NULL
                    AND DIMENSION2 IS NOT NULL

                  UNION ALL

                  -- Priority 2: only DIMENSION1
                  SELECT *, 2 AS prioridad
                  FROM {PAR_BONO_ESP}
                  WHERE DIMENSION1 IS NOT NULL
                    AND DIMENSION2 IS NULL

                  UNION ALL

                  -- Priority 3: only DIMENSION2
                  SELECT *, 3 AS prioridad
                  FROM {PAR_BONO_ESP}
                  WHERE DIMENSION1 IS NULL
                    AND DIMENSION2 IS NOT NULL
                ) BONO_ESP
                ON (
                    (BONO_ESP.prioridad = 1 AND CECOS.DIMENSION1 = BONO_ESP.DIMENSION1 AND CECOS.DIMENSION2 = BONO_ESP.DIMENSION2)
                  OR (BONO_ESP.prioridad = 2 AND CECOS.DIMENSION1 = BONO_ESP.DIMENSION1)
                  OR (BONO_ESP.prioridad = 3 AND CECOS.DIMENSION2 = BONO_ESP.DIMENSION2)
                )
                QUALIFY ROW_NUMBONO_ESP() OVER (
                  PARTITION BY T0.CECO, T0.CEBE, T0.Periodo, T0.Ejercicio, T0.Version,T0.Posicion, T0.DIMENSION6, T0.Personas,
                T0.Ejercicio_DE_INGRESO,
                T0.MES_DE_INGRESO
                  ORDER BY BONO_ESP.prioridad
                ) = 1
            ),

            BONO_AGRUP AS (
              SELECT CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS,
              SUM(IMPORTE) AS IMPORTE, SUM(SUELDO) AS SUELDO

              FROM BONO

              GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12
            ),

            CALCULO AS (

            SELECT T0.* EXCEPT(PERIODO, VERSION, IMPORTE, sueldo),
            CASE WHEN T1.PERIODO <= PARAMS.mes_captura AND T0.VERSION = PARAMS.primera_version then PARAMS.mes_captura else t1.periodo end as PERIODO,
            T0.PERIODO as PERIODO_BASE,
            VERSION,

            CASE WHEN T0.VERSION = PARAMS.primera_version AND T1.PERIODO <= PARAMS.mes_captura AND T0.PERIODO = T1.PERIODO-1 THEN 'AJUSTE_BONO' ELSE 'NOM_BONO' END AS ORIGEN,


            CASE
              WHEN T0.PERIODO > T1.PERIODO THEN 0  --  meses mayores al calculado SE QUITAN
              WHEN T0.PERIODO = T1.PERIODO THEN 1 --  calculo del mes
              WHEN T0.PERIODO = T1.PERIODO-1 THEN -1 -- cancela calculo acumulado mes anterior
            ELSE 0 END

            *
            CASE WHEN T1.PERIODO = T0.PERIODO OR T1.PERIODO-1 = T0.PERIODO THEN sueldo else 0 END
            *
            PARAMETROS

            ---ARREGLAR CANCELACION PRIMER MES DE CAPTURA (en segundo mes)
            / CASE WHEN T1.PERIODO = PARAMS.mes_captura +1  AND T1.PERIODO-1 = T0.PERIODO AND PARAMS.mes_captura-MES_DE_INGRESO >0 THEN PARAMS.mes_captura-MES_DE_INGRESO+1 ELSE 1 END
            ---ARREGLAR EL PRIMER MES
            / CASE WHEN T1.PERIODO = PARAMS.mes_captura AND T1.PERIODO = PARAMS.mes_captura AND PARAMS.mes_captura-MES_DE_INGRESO >0 THEN PARAMS.mes_captura-MES_DE_INGRESO+1 ELSE 1 END
            --- EL MONTO QUE LE CORRESPONDE POR AÑO

            /CASE WHEN T0.EJERCICIO_DE_INGRESO = T0.EJERCICIO THEN (12-MES_DE_INGRESO+1) ELSE 12 END  --CUANTOS MESES TIENE EL BONO (PARA OBTENER MONTO MENSUAL)

            * CASE WHEN T0.EJERCICIO_DE_INGRESO = T0.EJERCICIO THEN (T0.PERIODO-MES_DE_INGRESO+1)  ELSE T0.PERIODO END --MESES QUE HAN TRANSCURRIDO (PARA ACUMULAR)

            * CASE WHEN T0.EJERCICIO_DE_INGRESO = T0.EJERCICIO THEN (12-MES_DE_INGRESO+1)/12 ELSE 1 END --- PARCIALIDAD POR TIEMPO QUE LABORÓ PARA EL ANUAL (PORCENTAJE DEL AÑO)
            AS IMPORTE,

            CASE WHEN T1.PERIODO = T0.PERIODO OR T1.PERIODO-1 = T0.PERIODO THEN sueldo else 0 END As SUELDO

            FROM BONO_AGRUP T0
            CROSS JOIN {PERIODOS} T1
            CROSS JOIN {tabla_pargen} PARAMS

            ),

            TRANSFORMACION_BONO_ESP AS (

            SELECT CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, ORIGEN, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS,
            PERIODO_BASE,
            SUM(IMPORTE) as IMPORTE, SUM(SUELDO) as SUELDOS

            FROM CALCULO

            group by CUENTA, CECO, CEBE, PERIODO, EJERCICIO, VERSION, ORIGEN, POSICION, DIMENSION6, PERSONAS, EJERCICIO_DE_INGRESO,MES_DE_INGRESO, PARAMETROS, PERIODO_BASE
      ),

      BONO_ESP AS (
          SELECT
              TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
              T1.CECO,
              T1.CEBE,

              COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
              COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
              COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
              COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
              COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
              COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
              COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
              T3.NIVEL1,
              T3.NIVEL2,
              T3.NIVEL3,
              T3.NIVEL4,
              T3.NIVEL5,
              CAST(EJERCICIO as STRING) as EJERCICIO,
              CAST(PERIODO as INT64) as PERIODO,
              VERSION,
              0 AS MES_INC,
              0 AS PORC_INC,
              0 as MES_BASE,
              SAFE_CAST(Importe AS NUMERIC) as importe,
              '' as Grupo,
              '' as Concepto,
              Personas,
              '' as AREA_PERSONAL,
              Posicion,
              0  as IMPORTE_VALES,
              0 as MONTO,
              SAFE_CAST(SUELDOS AS NUMERIC) as SUELDOS,
              0 as PORC_PREST,
              T0.DIMENSION6,
              SAFE_CAST(PERIODO_BASE AS STRING) as PERIODO_BASE,
              EJERCICIO_DE_INGRESO,
              MES_DE_INGRESO,
              SAFE_CAST(PARAMETROS AS NUMERIC) as PARAMETROS,
              0 as SUELDO,

              CONCAT("CN_HC_BONO_ESP",path_p, ORIGEN) as ORIGEN ---path parametro

              --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5

              FROM TRANSFORMACION_BONO_ESP T0
              LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
              LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
              LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
              LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA
              ),

          SUELDO_DIC AS (
            SELECT SUELDOS_BONO_ESP.* EXCEPT(PERIODO) ,
            CASE WHEN T1.PERIODO <= PARAMS.mes_captura AND SUELDOS_BONO_ESP.VERSION = PARAMS.primera_version then PARAMS.mes_captura else t1.periodo end as PERIODO,
            SUELDO * 2
            * CASE WHEN SAFE_CAST(SUELDOS_BONO_ESP.EJERCICIO_DE_INGRESO as STRING) = SUELDOS_BONO_ESP.EJERCICIO THEN (12-MES_DE_INGRESO+1)/12 ELSE 1 END --- PARCIALIDAD POR TIEMPO QUE LABORÓ
            ---HASTA AQUI TENEMOS EL IMPORTE ANUAL
            / CASE WHEN SAFE_CAST(SUELDOS_BONO_ESP.EJERCICIO_DE_INGRESO as STRING) = SUELDOS_BONO_ESP.EJERCICIO THEN (12-MES_DE_INGRESO+1) ELSE 12 END  --CUANTOS MESES LABORA EN SU AÑO DE INGRESO
            * CASE WHEN SAFE_CAST(SUELDOS_BONO_ESP.EJERCICIO_DE_INGRESO as STRING) = SUELDOS_BONO_ESP.EJERCICIO AND T1.PERIODO < SUELDOS_BONO_ESP.MES_DE_INGRESO THEN 0 ELSE 1 END  --- SOLO PARA LOS PERIODOS LABORADOS

             as IMPORTE,
            '51000412' as CUENTA

            FROM SUELDOS_BONO_ESP
            CROSS JOIN {PERIODOS} T1
            CROSS JOIN {tabla_pargen} PARAMS

            WHERE SUELDOS_BONO_ESP.PERIODO = 12
          ),

          PTU_GAR AS (
            SELECT
                TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
                T1.CECO,
                T1.CEBE,

                COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
                COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
                COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
                COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
                COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
                COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
                COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
                T3.NIVEL1,
                T3.NIVEL2,
                T3.NIVEL3,
                T3.NIVEL4,
                T3.NIVEL5,
                CAST(EJERCICIO as STRING) as EJERCICIO,
                CAST(PERIODO as INT64) as PERIODO,
                VERSION,
                0 AS MES_INC,
                0 AS PORC_INC,
                0 as MES_BASE,
                SAFE_CAST(Importe AS NUMERIC) as importe,
                '' as Grupo,
                '' as Concepto,
                Personas,
                '' as AREA_PERSONAL,
                Posicion,
                0  as IMPORTE_VALES,
                0 as MONTO,
                SAFE_CAST(SUELDO AS NUMERIC) as SUELDOS,
                0 as PORC_PREST,
                T0.DIMENSION6,
                '0' as PERIODO_BASE,
                EJERCICIO_DE_INGRESO,
                MES_DE_INGRESO,
                SAFE_CAST(2 AS NUMERIC) as PARAMETROS,
                0 as SUELDO,

                CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN ---path parametro

                FROM SUELDO_DIC T0
                LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
                LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
                LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
                LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA
              ),

          CALCULOS_COMPLETOS AS (

          SELECT * FROM SUELDOS
          UNION ALL
          SELECT * FROM PRESTACIONES
          UNION ALL
          SELECT * FROM BONO_ESP
          UNION ALL
          SELECT * FROM PTU_GAR
          UNION ALL
          SELECT * FROM VALES

          ),

          SBC AS (
          SELECT * EXCEPT(CUENTA, IMPORTE), SUM(IMPORTE) AS IMPORTE

          FROM CALCULOS_COMPLETOS
          INNER JOIN {CUENTAS_SBC} CUENTAS_SBC ON SUBSTRING(CALCULOS_COMPLETOS.CUENTA,1,8) = CUENTAS_SBC.CUENTA
          GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37

          ),

          ISN AS (

            SELECT

                TRIM('51000053 '||T2.Descripcion) as Cuenta,
                CECO,
                CEBE,

                DIMENSION1,
                DIMENSION2,
                DIMENSION3,
                DIMENSION6_MAESTRO,
                DIMENSION7,
                DIMENSION5,
                DES_DIMENSION5,
                T3.NIVEL1,
                T3.NIVEL2,
                T3.NIVEL3,
                T3.NIVEL4,
                T3.NIVEL5,
                EJERCICIO,
                PERIODO,
                VERSION,
                MES_INC,
                T0.PORC_INC,
                MES_BASE,
                SAFE_CAST(IMPORTE AS NUMERIC) * SAFE_CAST(PARAMS.isn_cdmx AS NUMERIC) as importe,
                Grupo,
                Concepto,
                Personas,
                AREA_PERSONAL,
                Posicion,
                IMPORTE_VALES,
                MONTO,
                SAFE_CAST(IMPORTE AS NUMERIC) as SUELDOS,
                SAFE_CAST(PARAMS.isn_cdmx AS NUMERIC) as PORC_PREST,
                DIMENSION6,
                PERIODO_BASE,
                EJERCICIO_DE_INGRESO,
                MES_DE_INGRESO,
                PARAMETROS,
                SUELDO,

                CONCAT("CN_HC_PRESTACIONES",path_p) as ORIGEN ---path parametro

                FROM SBC T0
                LEFT JOIN {CUENTAS} T2 ON '51000053'=T2.CUENTA
                LEFT JOIN {JERARCUENTAPPTO} T3 ON '51000053'=T3.CUENTA
                CROSS JOIN {tabla_pargen} PARAMS

                )

          SELECT * FROM CALCULOS_COMPLETOS
          UNION ALL
          SELECT * FROM ISN
)
'''

## CONCEPTOS NOMINA

create_procedure_cnhc = f"""
CREATE OR REPLACE PROCEDURE `{stored_procedure_cnhc}`(path STRING)

BEGIN

  DECLARE path_p STRING DEFAULT IFNULL(path,'');

  DELETE FROM {tabla_base_hccn_sp}
  WHERE ORIGEN IN
    ( CONCAT('CN_HC', path_p)  ,
      CONCAT('CN_HC_SUE', path_p)  ,
      CONCAT('CN_HC_PRESTACIONES', path_p)  ,
      CONCAT('CN_HC_BONO_ESP', path_p) ,  CONCAT('CN_HC_BONO_ESP', path_p, 'AJUSTE_BONO') ,  CONCAT('CN_HC_BONO_ESP', path_p, 'NOM_BONO'), CONCAT('CN_HC_BONO_ESP', path_p, 'BONO_CN'),
      CONCAT('CN_HC_PTU', path_p),
      CONCAT('CN_HC_PRE', path_p)

  );

   -- MERGE destination is the original table
  INSERT INTO {tabla_base_hccn_sp}
    SELECT *
    FROM (
      {query_cn_hc}
      );

END
"""
client.query(create_procedure_cnhc).result()
print("Creado ",stored_procedure_cnhc)


In [ ]:
# ######  CALL SP

# sentencia = bpd.read_gbq(f'''
# select ARRAY_AGG(distinct path )

# from `{tabla_base_hc}`
# ''').to_pandas()

# final_string = sentencia.iloc[0, 0] if not sentencia.empty and sentencia.iloc[0,0] is not None else ""

# for tabla in final_string:
#     all_procedure_sql = f"""
#     CALL `{stored_procedure_cnhc}`(
#       "{tabla}"
#     );
#     """

#     query_job = client.query(all_procedure_sql)
#     query_job.result()
#     print("All done ",tabla)

# **TRANSFORMACIÓN CONCEPTOS DE NEGOCIO CECO🆗**


In [ ]:
query_cn = f'''
            (
                  WITH PRIMEROS AS (#
            (SELECT * FROM {project_id}.PARAMETROS_PRESUPUESTO.PAR_PERIODOS WHERE Version = 'PPTO_V1' LIMIT 1)
            UNION ALL
            (SELECT * FROM {project_id}.PARAMETROS_PRESUPUESTO.PAR_PERIODOS WHERE Version = 'PPTO_V2' LIMIT 1)
            UNION ALL
            (SELECT * FROM {project_id}.PARAMETROS_PRESUPUESTO.PAR_PERIODOS WHERE Version = 'PPTO_V0' LIMIT 1)
            ),
            TRANSFORMACION AS (

            SELECT
            Substring(Cuenta,0,8) AS Cuenta,
            CASE WHEN SUBSTRING(Cuenta,1,1) = '4' THEN 'CCINGRESOS' ELSE CECO_CEBE END AS CECO,
            CASE WHEN SUBSTRING(Cuenta,1,1) = '4' THEN CECO_CEBE ELSE CECOS.CEBE END as CEBE,
            CASE WHEN Ejercicio IN ('ME','REV') THEN PARAMS.aniobase
            WHEN Ejercicio='PPTO' THEN PARAMS.aniobase+1 END AS Ejercicio,
            CASE WHEN T0.MES < PARAMS.mes_captura THEN PARAMS.mes_captura ELSE T0.MES END AS Periodo,
            CASE WHEN Ejercicio="ME" THEN "PPTO_V2" WHEN Ejercicio="PPTO"
            THEN "PLAN" WHEN Ejercicio="REV" THEN "PPTO_V1" END AS Version,
            IFNULL(Autorizado,0) AS Importe,
            Grupo, Concepto

            FROM {tabla_base_cn} T0
            LEFT JOIN PRIMEROS T4 ON
                  CASE WHEN Ejercicio="ME" THEN "PPTO_V2"
                        WHEN Ejercicio="PPTO" THEN "PLAN"
                        WHEN Ejercicio="REV" THEN "PPTO_V1" END = T4.VERSION  --- PUEDEN CAPTURAR PERO SE CARGARA EN EL PRIMER MES

            LEFT JOIN {CECOS} CECOS ON CECO_CEBE= CECOS.CECO
            CROSS JOIN {tabla_pargen} PARAMS

            WHERE (Ejercicio IS NOT NULL OR Ejercicio <> "") AND PATH = '%s'

            )

            SELECT
              TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
              T0.CECO,
              T0.CEBE,

              COALESCE(T1.DIMENSION1,T4.DIMENSION1) AS DIMENSION1,
              COALESCE(T1.DIMENSION2,T4.DIMENSION2) AS DIMENSION2,
              COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
              COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
              COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
              COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
              COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
              T3.NIVEL1,
              T3.NIVEL2,
              T3.NIVEL3,
              T3.NIVEL4,
              T3.NIVEL5,
              CAST(EJERCICIO as STRING) as EJERCICIO,
              CAST(PERIODO as INT64) as PERIODO,
              VERSION,
              0 AS MES_INC,
              0 AS PORC_INC,
              0 as MES_BASE,
              SAFE_CAST(Importe AS NUMERIC) as importe,
              Grupo,
              Concepto,
              0 as Personas,
              '' as AREA_PERSONAL,
              '' as Posicion,
              0  as IMPORTE_VALES,
              0 as MONTO,
              0 as SUELDOS,
              0 as PORC_PREST,
              '' as DIMENSION6,
              '0' as PERIODO_BASE,
              0 as EJERCICIO_DE_INGRESO,
              0 as MES_DE_INGRESO,
              0 as PARAMETROS,
              0 as SUELDO,

              CONCAT("CN",'%s') as ORIGEN ---path parametro

              --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5

              FROM TRANSFORMACION T0
              LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
              LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
              LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
              LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA

            )
  '''

  ###CONCEPTOS CON CECO

create_procedure_cnceco = f'''

CREATE OR REPLACE PROCEDURE `{stored_procedure_conceptos}`(path STRING)

BEGIN

  DECLARE sql_delete STRING;
  DECLARE sql_write STRING;

  SET sql_delete = FORMAT("""
    DELETE FROM {tabla_base_cn_sp}
    WHERE ORIGEN LIKE CONCAT('CN', '%s');
  """,path);

  EXECUTE IMMEDIATE  sql_delete;

   -- MERGE destination is the original table
  SET sql_write =FORMAT("""
  INSERT INTO {tabla_base_cn_sp}
    SELECT *
    FROM (
      {query_cn}
      )
  """,
      path
      ,path);

    EXECUTE IMMEDIATE  sql_write;

END
'''
client.query(create_procedure_cnceco).result()
print("Creado ",stored_procedure_conceptos)



In [ ]:
# ######  CALL SP

# sentencia = bpd.read_gbq(f'''
# select ARRAY_AGG(distinct path )

# from `{tabla_base_cn}`
# ''').to_pandas()

# final_string = sentencia.iloc[0, 0] if not sentencia.empty and sentencia.iloc[0,0] is not None else ""

# for tabla in final_string:
#     all_procedure_sql = f"""
#     CALL `{stored_procedure_conceptos}`(
#       "{tabla}"
#     );
#     """

#     query_job = client.query(all_procedure_sql)
#     query_job.result()
#     print("All done ",tabla)

# **CONCEPTOS DE NEGOCIOS GENERALES (Capturas sin CECOS)🆗**


SIN CECO

In [ ]:
query_cn_gen  = f'''
(
  SELECT TRIM(SUBSTRING(T0.Cuenta,1,8)||' '||T2.Descripcion) as CUENTA,
'' as CeCo, '' as CeBe,
DIMENSION1,DIMENSION2,DIMENSION3,T0.DIMENSION6 AS DIMENSION6_MAESTRO,
T0.DIMENSION7 as DIMENSION7,
CASE WHEN REGEXP_CONTAINS(DIMENSION5cacion,  r'^\d') THEN  SUBSTRING (DIMENSION5cacion,1,4) ELSE '' END as  DIMENSION5,
CASE WHEN REGEXP_CONTAINS(DIMENSION5cacion,  r'^\d')  THEN SUBSTRING (DIMENSION5cacion,6,LENGTH(DIMENSION5cacion)) ELSE DIMENSION5cacion END AS DES_DIMENSION5,
NIVEL1,
NIVEL2,
NIVEL3,
NIVEL4,
NIVEL5,
SAFE_CAST(case when Ejercicio = "ME" THEN PARAMS.aniobase WHEN Ejercicio = 'PPTO' THEN PARAMS.aniobase+1 WHEN Ejercicio = 'REV' THEN PARAMS.aniobase END AS STRING) as Ejercicio,
0 as Periodo,
case when Ejercicio = "ME" THEN "PPTO_V2" WHEN Ejercicio = 'PPTO' THEN  'PPTO_V0' WHEN Ejercicio = 'REV' THEN 'PPTO_V1' END as Version,
0 as MES_INC,
0 as PORC_INC,
0 as MES_BASE,
SAFE_CAST(Autorizado AS NUMERIC) as Importe,
Grupo,
Concepto,
0 as Personas,
'' as AREA_PERSONAL,
'' as POSICION,
0 as Importe_vales,
0 as MONTO,
0 as Sueldos,
0 as PORC_PREST,
'' as DIMENSION6,
'0' as PERIODO_BASE,
0 as EJERCICIO_DE_INGRESO,
0 as MES_DE_INGRESO,
0 as PARAMETROS,
0 as SUELDO,
CONCAT("CN_GEN",path_p) as ORIGEN


FROM `{tabla_base_gencn}` T0
LEFT JOIN {CUENTAS} T2 ON SUBSTRING(T0.CUENTA,1,8)=T2.CUENTA
LEFT JOIN {JERARCUENTAPPTO} T3 ON SUBSTRING(T0.CUENTA,1,8)=T3.CUENTA
CROSS JOIN {tabla_pargen} PARAMS

WHERE PATH = path_p
)
'''

###CONCEPTOS GENERALES

create_procedure_cn_gen = f"""
CREATE OR REPLACE PROCEDURE `{stored_procedure_cn_gen}`(path STRING)
BEGIN

  DECLARE path_p STRING DEFAULT IFNULL(path,'');

  DELETE FROM {tabla_base_gencn_sp}
  WHERE ORIGEN = CONCAT('CN_GEN', path_p);

   -- MERGE destination is the original table
  INSERT INTO {tabla_base_gencn_sp}
    SELECT *
    FROM (
      {query_cn_gen}
      );

END;
"""
client.query(create_procedure_cn_gen).result()
print("Creado ",stored_procedure_cn_gen)

In [ ]:
# #####  CALL SP

# sentencia = bpd.read_gbq(f'''
# select ARRAY_AGG(distinct path )

# from `{tabla_base_gencn}`
# ''').to_pandas()

# final_string = sentencia.iloc[0, 0] if not sentencia.empty and sentencia.iloc[0,0] is not None else ""

# for tabla in final_string:
#     all_procedure_sql = f"""
#     CALL `{stored_procedure_cn_gen}`(
#       "{tabla}"
#     );
#     """

#     query_job = client.query(all_procedure_sql)
#     query_job.result()
#     print("All done ",tabla)

DE ER

In [ ]:

query_cn_ger = f'''
(SELECT DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6, DIMENSION7 as DIMENSION7, CASE WHEN DIMENSION1 = 'Negocio8' THEN DIMENSION7 ELSE '' END as MARCA, DIMENSION5cacion as DIMENSION5,
CASE WHEN
  DIMENSION5cacion IN ('0395', '0487', '0422' ,'0966' , '0687' , '0694' , '0691' , '0805', '0872')
  THEN 'CORP' ELSE 'OPE' END
 AS PF_CO_OP,
CAST(
  CASE WHEN EJERCICIO = 'PPTO' THEN PARAMS.aniobase+1 ELSE PARAMS.aniobase END
  as STRING) as EJERCICIO,
CASE WHEN EJERCICIO = 'PPTO' THEN 'PPTO_V0' WHEN EJERCICIO = 'REV' THEN 'PPTO_V1' WHEN EJERCICIO = 'ME' THEN 'PPTO_V2' END as VERSION,
CONCAT("CN_GEN",path_p) as ORIGEN,
TRIM(T2.Cuenta||' '||T2.Descripcion)as Cuenta,
T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5,
'CONCEPTOS GENERALES' as GRUPO, Nivel3 as CONCEPTO
,Monto_CN as Importe

FROM `%s` T0
LEFT JOIN {CUENTAS} T2 ON SUBSTRING(T0.CUENTA,1,8)=T2.CUENTA
LEFT JOIN {JERARCUENTAPPTO} T3 ON SUBSTRING(T0.CUENTA,1,8)=T3.CUENTA
CROSS JOIN {tabla_pargen} PARAMS
WHERE PATH = path_p
)
'''

###CONCEPTOS DE ER

create_procedure_cn_ger = f"""
CREATE OR REPLACE PROCEDURE `{stored_procedure_cn_ger}`(path STRING)
OPTIONS(strict_mode=false)
BEGIN

  DECLARE path_p STRING DEFAULT IFNULL(path,'');

  DELETE FROM {tabla_base_gercn_sp}
  WHERE ORIGEN = CONCAT('CN_GER', path_p);

   -- MERGE destination is the original table
  INSERT INTO {tabla_base_gercn_sp}
    SELECT *
    FROM (
      {query_cn_ger}
      );

END;
"""
client.query(create_procedure_cn_ger).result()
print("Creado ",stored_procedure_cn_ger)

# **ATRIBUTOS CALCULOS BASE** 🆗

In [ ]:
storedatributoscalcini = f'''
    -- ==============================================================
    -- ALL DECLARES FIRST (MANDATORY IN BIGQUERY)
    -- ==============================================================

    DECLARE src_fqtn STRING;
    DECLARE dst_fqtn STRING;

    DECLARE src_table_name STRING;

    DECLARE dynamic_sql STRING;
    DECLARE deletesentence STRING;

    DECLARE origen_table STRING;

    DECLARE dst_cols ARRAY<STRING>;

    DECLARE src_cols ARRAY<STRING>;

    DECLARE select_list STRING DEFAULT '';

    DECLARE i INT64 DEFAULT 0;
    DECLARE n INT64;
    DECLARE dst_col STRING;
    DECLARE dst_type STRING;

    DECLARE dst_schema ARRAY<STRUCT<
      col_name STRING,
      col_type STRING
    >>;

    ------DETECT DESTINATION COLUMNS

    EXECUTE IMMEDIATE """
      SELECT ARRAY_AGG(
              STRUCT(column_name AS col_name, data_type AS col_type)
              ORDER BY ordinal_position
            )
      FROM `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
      WHERE table_name = '{tabla_atributos_calc_inic.split(".")[2]}'
    """
    INTO dst_schema;

    SET n = ARRAY_LENGTH(dst_schema);

    SET src_table_name = REGEXP_EXTRACT(src_table, r'[^.]+$');

    ------DETECT COLUMNS IN SOURCE TABLE

    EXECUTE IMMEDIATE """
      SELECT ARRAY_AGG(UPPER(column_name))
      FROM `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
      WHERE table_name = @t
    """
    INTO src_cols
    USING src_table_name AS t;

    -- ==============================================================
    -- EXECUTABLE STATEMENTS START HERE
    -- ==============================================================

    SET src_fqtn = FORMAT(
      '`%s`',
      src_table
    );

    -- --------------------------------------------------------------
    -- Get ORIGEN safely (single value)
    -- --------------------------------------------------------------
    EXECUTE IMMEDIATE FORMAT("""
      SELECT ANY_VALUE(ORIGEN)
      FROM %s
    """, src_fqtn)
    INTO origen_table;

    -- --------------------------------------------------------------
    -- Delete previous data
    -- --------------------------------------------------------------
    SET deletesentence = FORMAT("""
      DELETE FROM `{tabla_atributos_calc_inic}`
      WHERE ORIGEN = '%s'
    """, origen_table);

    EXECUTE IMMEDIATE deletesentence;

    -- --------------------------------------------------------------
    ------BUILD SELECT LIST
    -- --------------------------------------------------------------

    WHILE i < n DO
      SET dst_col  = dst_schema[OFFSET(i)].col_name;
      SET dst_type = dst_schema[OFFSET(i)].col_type;

      SET select_list = select_list || ',\\n' ||
        CASE
          --data master

          WHEN dst_col = 'DIMENSION1'
            THEN 'COALESCE(MC.DIMENSION1, MB.DIMENSION1, CAST(K.CECO AS STRING)) AS DIMENSION1'

          WHEN dst_col = 'DIMENSION2'
            THEN 'COALESCE(MC.DIMENSION2, MB.DIMENSION2, CAST(K.CEBE AS STRING)) AS DIMENSION2'

          WHEN dst_col = 'DIMENSION3'
            THEN 'COALESCE(MC.DIMENSION3, MB.DIMENSION3) AS DIMENSION3'

          WHEN dst_col = 'DIMENSION6_MAESTRO'
            THEN 'COALESCE(MC.DIMENSION6, MB.DIMENSION6) AS DIMENSION6_MAESTRO'

          WHEN dst_col = 'DIMENSION7'
            THEN 'COALESCE(MC.DIMENSION7, MB.DIMENSION7) AS DIMENSION7'

          WHEN dst_col = 'DIMENSION5'
            THEN 'COALESCE(MC.DIMENSION5, MB.DIMENSION5) AS DIMENSION5'

          WHEN dst_col = 'DES_DIMENSION5'
            THEN 'COALESCE(MC.DES_DIMENSION5, MB.DES_DIMENSION5) AS DES_DIMENSION5'

          WHEN dst_col = 'NIVEL1' THEN 'CAST(IFNULL(JCP.NIVEL1,"") AS STRING) AS NIVEL1'
          WHEN dst_col = 'NIVEL2' THEN 'CAST(IFNULL(JCP.NIVEL2,"") AS STRING) AS NIVEL2'
          WHEN dst_col = 'NIVEL3' THEN 'CAST(IFNULL(JCP.NIVEL3,"") AS STRING) AS NIVEL3'
          WHEN dst_col = 'NIVEL4' THEN 'CAST(IFNULL(JCP.NIVEL4,"") AS STRING) AS NIVEL4'
          WHEN dst_col = 'NIVEL5' THEN 'CAST(IFNULL(JCP.NIVEL5,"") AS STRING) AS NIVEL5'

          WHEN upper(dst_col) = 'CUENTA' THEN "TRIM(CONCAT(CAST(K.CUENTA AS STRING), CHR(32), COALESCE(CNT.DESCRIPCION, ''))) AS CUENTA"

          -- explicit overrides
          WHEN dst_col = 'ORIGEN'    THEN 'CAST(K.ORIGEN AS STRING) AS ORIGEN'

          -- source column exists → cast to destination type
          WHEN UPPER(dst_col) IN UNNEST(src_cols)
            THEN FORMAT(
              'SAFE_CAST(K.%s AS %s) AS %s',
              dst_col,
              dst_type,
              dst_col
            )

          -- source column missing → NULL of destination type
          ELSE FORMAT(
            'CAST(%s AS %s) AS %s',
            CASE
              WHEN dst_type = 'STRING'  THEN "''"
              WHEN dst_type = 'INT64'   THEN '0'
              WHEN dst_type = 'NUMERIC' THEN '0'
              WHEN dst_type = 'FLOAT64' THEN '0'
              WHEN dst_type = 'BOOL'    THEN 'FALSE'
              ELSE 'NULL'
            END,
            dst_type,
            dst_col
          )
        END;

      SET i = i + 1;
    END WHILE;

    -- SELECT select_list;

    -- --------------------------------------------------------------
    -- Insert normalized data
    -- --------------------------------------------------------------
    SET dynamic_sql = FORMAT("""
      INSERT INTO `{tabla_atributos_calc_inic}`
      SELECT %s
      FROM %s AS K
      LEFT JOIN `{CECOS}` MC
        ON CAST(K.CECO AS STRING) = MC.CECO
      LEFT JOIN `{CEBES}` MB
        ON CAST(K.CEBE AS STRING) = MB.CEBE
      INNER JOIN `{CUENTAS}` CNT ON SUBSTRING(K.CUENTA,1,8) = CNT.CUENTA
      LEFT JOIN `{JERARCUENTAPPTO}` JCP ON SUBSTRING(K.CUENTA,1,8) = JCP.CUENTA
      WHERE K.CUENTA IS NOT NULL
  """,
      SUBSTR(select_list, 3),  -- remove leading comma
      src_fqtn
    );

    -- SELECT dynamic_sql;

    EXECUTE IMMEDIATE dynamic_sql;

'''

storedprocedure_master = f'''
CREATE OR REPLACE PROCEDURE `{stored_procedure_atributos}`(src_table STRING)
OPTIONS (strict_mode = false)
BEGIN
  {storedatributoscalcini}
END;
'''

client.query(storedprocedure_master).result()
print("Creado: ",stored_procedure_atributos)

# **CEDULAS🆗**

La creación de las cédulas y su carga es automática en el desarrollo de [Sheets]()


In [ ]:
# #Se genera una tabla en blanco para poder generar la vista

# if DIMENSION6pres == 'ME_PPTO':

#   schema_cedula= [
#       bigquery.SchemaField("CUENTA", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("CECOCEBE", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("DIMENSION5", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("DESC_DIMENSION5", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("ENE", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("FEB", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MAR", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("ABR", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MAY", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("JUN", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("JUL", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("AGO", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("SEP", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("OCT", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("NOV", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("DIC", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("TOTAL", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("x", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("ENE2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("FEB2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MAR2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("ABR2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MAY2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("JUN2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("JUL2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("AGO2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("SEP2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("OCT2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("NOV2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("DIC2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("TOTAL2", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MANDANTE", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("TIMESTAMP", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("PATH", "STRING", mode="REQUIRED"),

#   ]

# else:

#   schema_cedula= [
#       bigquery.SchemaField("CUENTA", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("CECOCEBE", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("DIMENSION5", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("DESC_DIMENSION5", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("ENE", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("FEB", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MAR", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("ABR", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MAY", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("JUN", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("JUL", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("AGO", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("SEP", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("OCT", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("NOV", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("DIC", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("TOTAL", "FLOAT", mode="REQUIRED"),
#       bigquery.SchemaField("MANDANTE", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("TIMESTAMP", "STRING", mode="REQUIRED"),
#       bigquery.SchemaField("PATH", "STRING", mode="REQUIRED"),

#   ]

# try:
#   table_cedula = bigquery.Table(cedulablancaid)
#   table_cedula = client.create_table(table_cedula, schema=schema_cedula)  # Make an API request.
#   print(
#       "Created table {}.{}.{}".format(table_cedula.project, table_cedula.dataset_id, table_cedula.table_id)
#   )

# except:
#   client.delete_table(table_cedula, not_found_ok=True)  # Make an API request.
#   print("Deleted table '{}'.".format(table_cedula))
#   table_cedula = bigquery.Table(table_cedula, schema=schema_cedula)
#   table_cedula = client.create_table(table_cedula)  # Make an API request.
#   print(
#       "Created table {}.{}.{}".format(table_cedula.project, table_cedula.dataset_id, table_cedula.table_id)
#   )

In [ ]:
if DIMENSION6pres == "ME_PPTO":
  querycedulas =  f'''SELECT

CUENTA,
CASE WHEN CUENTA LIKE "4%" THEN
  CASE WHEN LENGTH(CECOCEBE)<10 THEN CONCAT(REPEAT("0",10-LENGTH(CECOCEBE),CECOCEBE) END
   ELSE '' END as CEBE,
CASE WHEN CUENTA LIKE "4%" THEN "CCINGRESOS" ELSE
  CASE WHEN LENGTH(CECOCEBE)<10 THEN CONCAT(REPEAT("0",10-LENGTH(CECOCEBE),CECOCEBE) END
 END as CECO,
CASE WHEN Version IN ("PPTO_V2") THEN PARAMS.aniobase ELSE PARAMS.aniobase+1 END as `Ejercicio`,
Version,
Mes as `Periodo`,
IFNULL(CASE
WHEN Mes = 1 AND Version IN ("PPTO_V2") THEN ENE
WHEN Mes = 2 AND Version IN ("PPTO_V2") THEN FEB
WHEN Mes = 3 AND Version IN ("PPTO_V2") THEN MAR
WHEN Mes = 4 AND Version IN ("PPTO_V2") THEN ABR
WHEN Mes = 5 AND Version IN ("PPTO_V2") THEN MAY
WHEN Mes = 6 AND Version IN ("PPTO_V2") THEN JUN
WHEN Mes = 7 AND Version IN ("PPTO_V2") THEN JUL
WHEN Mes = 8 AND Version IN ("PPTO_V2") THEN AGO
WHEN Mes = 9 AND Version IN ("PPTO_V2") THEN SEP
WHEN Mes = 10 AND Version IN ("PPTO_V2") THEN OCT
WHEN Mes = 11 AND Version IN ("PPTO_V2") THEN NOV
WHEN Mes = 12 AND Version IN ("PPTO_V2") THEN DIC

WHEN Mes = 1 AND Version = "PLAN" THEN ENE2
WHEN Mes = 2 AND Version = "PLAN" THEN FEB2
WHEN Mes = 3 AND Version = "PLAN" THEN MAR2
WHEN Mes = 4 AND Version = "PLAN" THEN ABR2
WHEN Mes = 5 AND Version = "PLAN" THEN MAY2
WHEN Mes = 6 AND Version = "PLAN" THEN JUN2
WHEN Mes = 7 AND Version = "PLAN" THEN JUL2
WHEN Mes = 8 AND Version = "PLAN" THEN AGO2
WHEN Mes = 9 AND Version = "PLAN" THEN SEP2
WHEN Mes = 10 AND Version = "PLAN" THEN OCT2
WHEN Mes = 11 AND Version = "PLAN" THEN NOV2
WHEN Mes = 12 AND Version = "PLAN" THEN DIC2 END
,0) as `Importe`,

MANDANTE, TIMESTAMP, PATH,

FROM `{cedulablancaid}` T0
CROSS JOIN {PARPERIODOS} T2
CROSS JOIN {tabla_pargen} PARAMS

WHERE T2.Version IN UNNEST (PARAMS.versiones)

'''

else:
  querycedulas =  f'''SELECT

CUENTA,
CASE WHEN CUENTA LIKE "4%" THEN
  CASE WHEN LENGTH(CECOCEBE)<10 THEN CONCAT(REPEAT("0",10-LENGTH(CECOCEBE),CECOCEBE) END
ELSE "" END as CEBE,
CASE WHEN CUENTA LIKE "4%" THEN "CCINGRESOS" ELSE
  CASE WHEN LENGTH(CECOCEBE)<10 THEN CONCAT(REPEAT("0",10-LENGTH(CECOCEBE),CECOCEBE) END
END as CECO,
PARAMS.aniobase as `Ejercicio`,
PARAMS.primera_version as Version,
T2.PERIODO,
CASE
WHEN T2.PERIODO = 1 THEN ENE
WHEN T2.PERIODO = 2 THEN FEB
WHEN T2.PERIODO = 3 THEN MAR
WHEN T2.PERIODO = 4 THEN ABR
WHEN T2.PERIODO = 5 THEN MAY
WHEN T2.PERIODO = 6 THEN JUN
WHEN T2.PERIODO = 7 THEN JUL
WHEN T2.PERIODO = 8 THEN AGO
WHEN T2.PERIODO = 9 THEN SEP
WHEN T2.PERIODO = 10 THEN OCT
WHEN T2.PERIODO = 11 THEN NOV
WHEN T2.PERIODO = 12 THEN DIC

END as `Importe`,

MANDANTE, TIMESTAMP, PATH,

FROM `{cedulablancaid}` T0
CROSS JOIN {PERIODOS} T2
CROSS JOIN {tabla_pargen} PARAMS

--WHERE T2.Version IN UNNEST (PARAMS.versiones)

WHERE T2.PERIODO >= PARAMS.mes_captura AND PATH LIKE CONCAT('%', path_p, '%')

'''

create_procedure_cedulas = f"""
CREATE OR REPLACE PROCEDURE `{stored_procedure_cedulas}`(path STRING)

BEGIN

  DECLARE path_p STRING DEFAULT IFNULL(path,'');

  DELETE FROM {sp_CEDULAS}
  WHERE ORIGEN LIKE CONCAT('%', path_p, '%');

   -- MERGE destination is the original table
  INSERT INTO {sp_CEDULAS}
    WITH TRANSFORMACION AS ( {querycedulas} )

    SELECT
              TRIM(T0.Cuenta||' '||T2.Descripcion) as Cuenta,
              IFNULL(T0.CECO,""),
              COALESCE(T0.CEBE,T4.CEBE,""),

              COALESCE(T1.DIMENSION1,T4.DIMENSION1,T0.CECO) AS DIMENSION1,
              COALESCE(T1.DIMENSION2,T4.DIMENSION2,T0.CEBE) AS DIMENSION2,
              COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
              COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
              COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
              COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
              COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
              T3.NIVEL1,
              T3.NIVEL2,
              T3.NIVEL3,
              T3.NIVEL4,
              T3.NIVEL5,
              CAST(EJERCICIO as STRING) as EJERCICIO,
              CAST(PERIODO as INT64) as PERIODO,
              VERSION,
              0 AS MES_INC,
              0 AS PORC_INC,
              0 as MES_BASE,
              SAFE_CAST(Importe AS NUMERIC) as importe,
              '' as Grupo,
              '' as Concepto,
              0 as Personas,
              '' as AREA_PERSONAL,
              '' as Posicion,
              0  as IMPORTE_VALES,
              0 as MONTO,
              0 as SUELDOS,
              0 as PORC_PREST,
              '' as DIMENSION6,
              '0' as PERIODO_BASE,
              0 as EJERCICIO_DE_INGRESO,
              0 as MES_DE_INGRESO,
              0 as PARAMETROS,
              0 as SUELDO,

              PATH as ORIGEN ---path parametro

              --T3.NIVEL1, T3.NIVEL2, T3.NIVEL3, T3.NIVEL4, T3.NIVEL5

              FROM TRANSFORMACION T0
              LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
              LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
              LEFT JOIN {CUENTAS} T2 ON T0.CUENTA=T2.CUENTA
              LEFT JOIN {JERARCUENTAPPTO} T3 ON T0.CUENTA=T3.CUENTA
;

END
"""
client.query(create_procedure_cedulas).result()
print("Creado ",stored_procedure_cedulas)

In [ ]:
######  CALL SP

# sentencia = bpd.read_gbq(f'''
# select ARRAY_AGG(distinct path )

# from `intranet_bd_pruebas.CEDULAS`
# ''').to_pandas()

# final_string = sentencia.iloc[0, 0] if not sentencia.empty and sentencia.iloc[0,0] is not None else ""

# for tabla in final_string:
    # all_procedure_sql = f"""
    # CALL `{stored_procedure_cedulas}`(
    #   "{tabla}"
    # );
    # """

    # query_job = client.query(all_procedure_sql)
    # query_job.result()
    # print("All done ",tabla)

# ⛳ **PRODUCTIVA**

⭐ Se crea una tabla fija con los "cálculos de una vez", para eficientizar la consulta en desarrollo

❗❗❗ Nuevo proceso, Stored Procedure

❗❗❗❗❗❗❗ **SE REQUIERE ELEGIR TABLAS**

In [ ]:
####LISTADO DE TABLAS QUE CONFORMAN CALCULOS INICIALES, FORECAST E HISTORICOS

tablas_convertir = [
    #  base_hist ##HISTORICOS BASE       OK MATRICIALES
    # ,base_hist_acum ##HISTORICOS ACUMULADOS  OK MATRICIALES

    # ,tablanomina ##SUELDOS   OK MATRICIALES
    # ,tablanomina_vales ##VALES DESPENSA Y UNIFORMES #DIFF
    # ,tablabonobase ##BONO_ESP OK MATRICIALES

    # ,tablacargapresrep ##PRESTACIONES MES BASE REPLICADO  OK MATRICIALES
    # ,tablacargapresccero ##PRESTACIONES CRECIMIENTO CERO      OK MATRICIALES
    # ,tablacargapresccero_sinout ##PRESTACIONES CRECIMIENTO CERO SIN OUTLIERS    MATRICIALES SIN CAMBIOS; VACIAS
    # ,tablacargapresminc ##PRESTACIONES MES DE INCREMENTO DETERMINADO    MATRICIALES SIN CAMBIOS; VACIAS
    # ,tablacargaporc ##PRESTACIONES BASE PORCENTAJE ROLL YEAR
    # ,tablacargaporc_ant ##PRESTACIONES BASE PORCENTAJE AÑO ANTERIOR
    # ,tablacargaptugar ##PRESTACIONES PTU REPARTO GARANTIZADO
    # ,tablacargapinfl ##PRESTACIONES INFLACION      MATRICIALES SIN CAMBIOS; VACIAS
    # ,tablanompres_anioantinf ##PRESTACIONES REPLICA AÑO ANTERIOR 1-12 + INFLACION          OK MATRICIALES
    # ,tablanompres_isn ##PRESTACIONES ISN

    # ,tablaCARGA_INICIAL ##CARGA_INICIAL

    # ,forecast_sscc ##FORECAST SSCC
    # ,forecast_fin_ifrs ##FORECAST NEGOCIO Z,NEGOCIO T
    # ,forecast_inmob ##FORECAST NEGOCIO Y
    # ,forecast_NEGOCIO X ##FORECAST NEGOCIO X
    # ,forecast_nomascara  ###FORECAST DIVISION NO EN MASCARA, DEJA 4 MESES PPTO + 5 NEGATIVO DE LOS PRIMEROS MESES, CON ESTO METEMOS AJUSTE EN MAYO SI METEN REALES "MAL"
]

from google.api_core.exceptions import BadRequest, GoogleAPICallError

failed = []

for tabla in tablas_convertir:
    all_procedure_sql = f"""
    CALL `{stored_procedure_atributos}`(
      "{tabla}"
    );
    """

    query_job = None

    try:
        query_job = client.query(all_procedure_sql)
        query_job.result()  # wait for completion

        print("✅ SP Atributos CONCLUIDO:", stored_procedure_atributos, "|", tabla)

    except BadRequest as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "BadRequest"))

        print("❌ SP ERROR (BadRequest)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)

        for err in e.errors or []:
            print("   →", err.get("message"))

        continue

    except GoogleAPICallError as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "APICall"))

        print("❌ SP ERROR (API Call)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)
        print("   →", str(e))

        continue

    except Exception as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "Unexpected"))

        print("❌ SP ERROR (Unexpected)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)
        print("   →", str(e))

        continue


# ======================================================
# SUMMARY
# ======================================================
print("\n================ SUMMARY — FAILED TABLES ================")

if not failed:
    print("🎉 All tables processed successfully.")
else:
    for tabla, job_id, err_type in failed:
        print(f"{tabla} | JobId={job_id} | Error={err_type}")


⚓ PRIMEROS MESES REPORTE NOMINA

In [ ]:
queryPrim_real = f'''CREATE OR REPLACE TABLE {base_primerosmesesrealesnom} AS
(
  WITH CUENTAS AS (
    SELECT DISTINCT T0.REGLA3, SUBSTRING(T0.CUENTA_NOM,1,8) as CUENTA
    FROM {CUENTAS_REGLAS_NOM} T0
    WHERE REGLA3 NOT IN ('NO CONSIDERAR','PRECARGADA')
  ),

  ER AS (
     SELECT * EXCEPT(DIMENSION2, DIMENSION3, DESCDIMENSION5), DIMENSION2 AS DIMENSION2, DIMENSION3 AS DIMENSION3, DESCDIMENSION5 AS DES_DIMENSION5
     FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
     WHERE Version IN ('Reales') AND EJERCICIO IN ({aniobase-1},{aniobase})
  )
--PRIMEROS MESES REALES

  SELECT T0.Cuenta,
  DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6 AS DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5,
  T0.Ejercicio, T0.Periodo, T0.Version, SUM(Importe) as importe,
  0 as Personas, 0 as Ejercicio_DE_INGRESO, 0 as MES_DE_INGRESO,
  CASE
    WHEN T0.CUENTA = '51000403' THEN 'Bono'
    WHEN  Clasificacion LIKE '%SUELDO%' THEN 'Sueldo'
    ELSE 'Prestaciones'
    END as
  ORIGEN, 'Base' as Posicion, '' as Grupo, '' as Concepto,
  'Base' as DIMENSION6HC

  FROM ER T0
  INNER JOIN CUENTAS ON T0.CUENTA=CUENTAS.CUENTA
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Ejercicio = {aniobase} AND PERIODO < {mesbasehc}
  GROUP BY T0.Cuenta, DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5, DES_DIMENSION5, T0.Ejercicio, T0.Periodo, T0.Version, Clasificacion

  --REALES PARA COMPARACION

  UNION ALL

  SELECT T0.Cuenta,
  DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6 AS DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5,
  T0.Ejercicio, T0.Periodo, T0.Version, SUM(Importe) as importe,
  0 as Personas, 0 as Ejercicio_DE_INGRESO, 0 as MES_DE_INGRESO,
   CASE
    WHEN T0.CUENTA = '51000403' THEN 'Bono'
    WHEN  Clasificacion LIKE '%SUELDO%' THEN 'Sueldo'
    ELSE 'Prestaciones'
    END as
  ORIGEN, 'Base' as Posicion, '' as Grupo, '' as Concepto,
  'Base' as DIMENSION6HC

  FROM ER T0
  INNER JOIN CUENTAS ON T0.CUENTA=CUENTAS.CUENTA
  LEFT JOIN `{CUENTAS_NOM}` T1 ON T0.Cuenta=T1.Cuenta_origen

  WHERE Version IN ('Reales') AND Ejercicio = {aniobase-1}
  GROUP BY T0.Cuenta, DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5, DES_DIMENSION5, T0.Ejercicio, T0.Periodo, T0.Version, Clasificacion
)
'''

client.query(queryPrim_real).result()
print("Creado ",base_primerosmesesrealesnom)

# ⭕ **REPORTES**

# ⚡**Base Completa**

⏰ Se unen todas las tablas normalizadas con los SP

En términos simples, simple vista UNION ALL, se deben "enlistar" las tablas que se **requieren**

In [ ]:
##Lista de tablas que se contemplan en la vista

tablas = [
  #CALCULOS INICIALES (SP ATRIBUTOS) HISTORICOS, CALCULO SUELDOS,PRESTACIONES, BONO_ESP, VALES, FORECAST
    tabla_atributos_calc_inic,
  #MODELO NOMINA ETZ (SE APAGA EN REV26)
    # nomina_calculo,
  #CEDULAS
    sp_CEDULAS,
  #CALCULOS DINAMICOS, CADA ENVIO DE INFORMACION LAS EDITA
    tabla_base_gercn_sp, #CN GER
    tabla_base_gencn_sp, #CN GEN
    tabla_base_cn_sp,    #CN CECO
    tabla_base_hccn_sp,  #MODELO "tradicional" de nómina, apartir de los conceptos headcount
    # nomina_etz_cn, (SE APAGA EN REV26)

  #PRORRATEO Y DIFF
    tabla_base_prorrateo
]

###TABLAS CON AJUSTE, LOS MESES PIEDRA LOS ENVIA AL PRIMER MES DE CAPTURA, EN FORECAST TENEMOS LA CALCELACION DE LOS PRIMEROS MESES, ASI 4 MESES PIEDRA EN NEGATIVO + 4 MESES CALCULO, GENERAN EL AJUSTE

tablas_con_filtro = [
    tabla_atributos_calc_inic,
    nomina_calculo
]

## Función que genera query

def union_all_query(values):
    parts = []

    for v in values:

        # Tablas que requieren filtro especial
        if v in tablas_con_filtro:
            parts.append(f"""
              SELECT * EXCEPT(PERIODO),
              CASE
                WHEN NOT EXISTS (
                  SELECT 1
                  FROM UNNEST(['HISTORICOS','FORECAST','CARGA_INICIAL']) p
                  WHERE STARTS_WITH(ORIGEN, p)
                ) AND VERSION = 'PPTO_V1' AND PERIODO <= {mes_captura} --AND PERIODO > {mesbaser}

                THEN {mes_captura}

                ELSE PERIODO
              END as PERIODO
              FROM `{v}`

              WHERE CUENTA NOT LIKE
              -----ORIGEN NOT IN ('') OR (ORIGEN IN ('') AND PERIODO >= {mes_captura}) ----SI SE REQUIERE, POR EL MOMENTO PPTO_V1 NO LO REQUIERE, YA QUE EL AJUSTE SE HACE CON FORECAST
              """)

                      # Resto de tablas sin filtro

        else:
                          parts.append(f"""
              SELECT * EXCEPT(PERIODO), PERIODO
              FROM `{v}`
              """)

    return "\nUNION ALL\n".join(parts)


#Generar query
query_unionall = union_all_query(tablas)
# query_unionall
vista_forecast_v = bigquery.Table(vista_forecast)
vista_forecast_v.view_query = query_unionall


##Ejecutar creación-actualización de vista
try:
  view = client.create_table(vista_forecast_v)
  print(f"Created {view.table_type}: {str(view.reference)}")
except:
  view = client.update_table(vista_forecast_v, ["view_query"])
  print(f"Updated {view.table_type}: {str(view.reference)}")

# ⏰ **PRORRATEO Y CUOTAS** ✈

❗ Se generó Stored Procedure 😀

In [ ]:
#Se genera una tabla en blanco para poder generar la vista

schema_prorr= [
      bigquery.SchemaField("Cuenta", "STRING", mode="REQUIRED"),
      bigquery.SchemaField("CeCo", "STRING", mode="REQUIRED"),
      bigquery.SchemaField("CeBe", "STRING", mode="REQUIRED"),
      bigquery.SchemaField("Ejercicio", "INT64", mode="REQUIRED"),
      bigquery.SchemaField("Version", "STRING", mode="REQUIRED"),
      bigquery.SchemaField("Periodo", "FLOAT", mode="REQUIRED"),
      bigquery.SchemaField("Importe", "FLOAT", mode="REQUIRED")
  ]


try:
  table_pror = bigquery.Table(vista_prorrycuot, schema=schema_prorr)
  tableprorrycuot = client.create_table(table_pror)  # Make an API request.
  print(
      "Created table {}.{}.{}".format(tableprorrycuot.project, tableprorrycuot.dataset_id, tableprorrycuot.table_id)
  )

except:
  client.delete_table(table_pror, not_found_ok=True)  # Make an API request.
  print("Deleted table '{}'.".format(table_pror))
  table_pror = bigquery.Table(vista_prorrycuot, schema=schema_prorr)
  tableprorrycuot = client.create_table(table_pror)  # Make an API request.
  print(
      "Created table {}.{}.{}".format(tableprorrycuot.project, tableprorrycuot.dataset_id, tableprorrycuot.table_id)
  )

**Stored Procedure**

⭐ Se realizará ajuste y transformación juntos

In [ ]:
#####CALCULO DE DIFERENCIAS
query_sp_prorrateov1 = f'''
CREATE OR REPLACE TABLE {tabla_base_prorrateo} AS
(---TRANSFORMACION PRORRATEO PARA MASCARA SIN FILTRO
  SELECT
        TRIM(SUBSTRING(T0.Cuenta,1,8)||' '||T2.Descripcion) as CUENTA,
        CASE WHEN T0.CUENTA LIKE '4%%' THEN 'CCINGRESOS' ELSE T0.CECO END AS CECO,
        COALESCE(T0.CEBE,T1.CEBE,"") AS CEBE,
        COALESCE(T1.DIMENSION1,T4.DIMENSION1) AS DIMENSION1,
        COALESCE(T1.DIMENSION2,T4.DIMENSION2) AS DIMENSION2,
        COALESCE(T1.DIMENSION3,T4.DIMENSION3) AS DIMENSION3,
        COALESCE(T1.DIMENSION6,T4.DIMENSION6) AS DIMENSION6_MAESTRO,
        COALESCE(T1.DIMENSION7,T4.DIMENSION7) AS DIMENSION7,
        COALESCE(T1.DIMENSION5,T4.DIMENSION5) AS DIMENSION5,
        COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) as DES_DIMENSION5,
        T3.NIVEL1,
        T3.NIVEL2,
        T3.NIVEL3,
        T3.NIVEL4,
        T3.NIVEL5,
        CAST(EJERCICIO as STRING) as EJERCICIO,
        CAST(PERIODO as INT64) as PERIODO,
        VERSION,
        0 AS MES_INC,
        SAFE_CAST(0 AS NUMERIC) AS PORC_INC,
        0 as MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) as importe,
        '' as Grupo,
        '' as Concepto,
        0 as Personas,
        '' as AREA_PERSONAL,
        '' as Posicion,
        0.0  as IMPORTE_VALES,
        0.0 as MONTO,
        0.0 as SUELDOS,
        0.0 as PORC_PREST,
        '' as DIMENSION6,
        '0' as PERIODO_BASE,
        0 as EJERCICIO_DE_INGRESO,
        0 as MES_DE_INGRESO,
        0.0 as PARAMETROS,
        0.0 as SUELDO,

        ORIGEN

        FROM {vista_prorrycuot} T0
        LEFT JOIN {CECOS} T1 ON T0.CECO=T1.CECO
        LEFT JOIN {CEBES} T4 ON T0.CEBE=T4.CEBE
        LEFT JOIN  {CUENTAS} T2 ON SUBSTRING(T0.CUENTA,1,8)=T2.CUENTA
        LEFT JOIN {JERARCUENTAPPTO} T3 ON SUBSTRING(T0.CUENTA,1,8)=T3.CUENTA
        CROSS JOIN {tabla_pargen} PARAMS

        WHERE COALESCE(T1.DES_DIMENSION5,T4.DES_DIMENSION5) NOT IN ('Angelópolis','P Tepeyac','Satélite')
);

INSERT INTO {tabla_base_prorrateo}
(WITH

PARAMS AS (SELECT * FROM {tabla_pargen}),

PRIMEROS_MESES_PROR_CUOTAS_CALCULO AS (
  SELECT
        T0.Cuenta,
        T0.CECO,
        T0.CEBE,
        T0.DIMENSION1,
        T0.DIMENSION2,
        DIMENSION3,
        DIMENSION6_MAESTRO,
        DIMENSION7,
        DIMENSION5,
        DES_DIMENSION5,
        NIVEL1,
        NIVEL2,
        NIVEL3,
        NIVEL4,
        NIVEL5,
        EJERCICIO,
        PARAMS.mes_captura as PERIODO,
        VERSION,
        MES_INC,
        T0.PORC_INC,
        MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) as importe,
        Grupo,
        Concepto,
        0 as Personas,
        AREA_PERSONAL,
        Posicion,
        IMPORTE_VALES,
        MONTO,
        SUELDOS,
        PORC_PREST,
        T0.DIMENSION6,
        PERIODO_BASE,
        EJERCICIO_DE_INGRESO,
        MES_DE_INGRESO,
        PARAMETROS,
        SUELDO,

        'Ajuste Prorrateo y Cuotas' ORIGEN

  FROM {tabla_base_prorrateo} T0
  CROSS JOIN PARAMS
  WHERE PERIODO < PARAMS.mes_captura AND PERIODO > 0)
,

---DE LA BASE OBTENER PRIMEROS MESES NO MODIFICABLES, PERIODO PRIMERO DE CAPTURA, SIGNO CONTRARIO

PRIMEROS_MESES_PROR_CUOTAS_REALES AS (

    SELECT
        T0.Cuenta,
        T0.CECO,
        T0.CEBE,
        T0.DIMENSION1,
        T0.DIMENSION2,
        DIMENSION3,
        DIMENSION6_MAESTRO,
        DIMENSION7,
        DIMENSION5,
        DES_DIMENSION5,
        NIVEL1,
        NIVEL2,
        NIVEL3,
        NIVEL4,
        NIVEL5,
        EJERCICIO,
        PARAMS.mes_captura as PERIODO,
        VERSION,
        MES_INC,
        T0.PORC_INC,
        MES_BASE,
        SAFE_CAST(Importe AS NUMERIC) * -1 as importe,
        Grupo,
        Concepto,
        0 as Personas,
        AREA_PERSONAL,
        Posicion,
        IMPORTE_VALES,
        MONTO,
        SUELDOS,
        PORC_PREST,
        T0.DIMENSION6,
        PERIODO_BASE,
        EJERCICIO_DE_INGRESO,
        MES_DE_INGRESO,
        PARAMETROS,
        SUELDO,

        'Ajuste Prorrateo y Cuotas' ORIGEN


      FROM {vista_forecast} T0
      LEFT JOIN {CUENTAS_PRO_ADC} T3 ON T0.DIMENSION1 = T3.DIMENSION1 AND SUBSTRING(T0.CUENTA,1,8)=T3.CUENTA
      CROSS JOIN PARAMS

      WHERE
      PERIODO < PARAMS.mes_captura
      AND PERIODO > 0
      AND T0.VERSION = '{params['version'][0]}'
      AND EJERCICIO = '{aniobase}'
      AND ORIGEN NOT IN ('PRORRATEO','DISTRIBUCION', 'CUOTAS','Ajuste Prorrateo y Cuotas')
      AND T0.DIMENSION2 <> 'Inmob CC Ext'
      AND (
        (T3.CUENTA IS NOT NULL AND T0.DES_DIMENSION5 NOT IN ('Angelópolis','P Tepeyac','Satélite') ) ---PARA LAS CUENTAS DE PRORRATEO Y CUOTAS
      OR (T0.DIMENSION1 = 'NEGOCIO Y' AND T0.DES_DIMENSION5 IN ('Metepec II')) --PARA OBTENER LA DIFERENCIA DE LOS PRIMEROS MESES DE FIDEICOMISOS
          )
)

SELECT *
FROM PRIMEROS_MESES_PROR_CUOTAS_REALES

UNION ALL

SELECT *
FROM PRIMEROS_MESES_PROR_CUOTAS_CALCULO
);

DELETE FROM `{tabla_base_prorrateo}`
WHERE PERIODO < {mes_captura} AND PERIODO > 0;


'''



create_procedure_prorrateo = f"""

CREATE OR REPLACE PROCEDURE `{stored_procedure_DIFF_PRORRATEO}`()

BEGIN

  {query_sp_prorrateov1}

END

"""
client.query(create_procedure_prorrateo).result()
print("Creado ",stored_procedure_DIFF_PRORRATEO)

# client.query("CALL `{project_id}.intranet_bd_pruebas.DIFF_PRORRATEO_sp`()").result()
# print("Procesado")

In [ ]:
# ######  CALL SP

# all_procedure_sql = f"""
# CALL `{stored_procedure_DIFF_PRORRATEO}`(
# );
# """

# query_job = None

# query_job = client.query(all_procedure_sql)
# query_job.result()
# print("All done")

# **⚓ BACKUP BASE COMPLETA_LOG**

❗❗❗❗ Este proceso es manual, solo se ejecuta cuando se requiere realizar backup

**Stored Procedure, base completa mes, solo la final**

In [ ]:
# client = bigquery.Client(project="{project_id}")
# from google.cloud.bigquery import SchemaField

# table_id = vista_forecast

# table = client.get_table(table_id)

# schema_log = []

# for field in table.schema:
#     schema_log.append(SchemaField(field.name, field.field_type, field.mode))

# schema_log.append(SchemaField("BACKUP_DATE", "TIMESTAMP", mode="REQUIRED"))


# table = bigquery.Table(log_audit_base, schema=schema_log)

# # 🔹 Partition by BACKUP_DATE (daily)
# table.time_partitioning = bigquery.TimePartitioning(
#     type_=bigquery.TimePartitioningType.DAY,
#     field="BACKUP_DATE",
#     require_partition_filter=True
# )

# # 🔹 Cluster by common filter keys (tune these to your queries)
# table.clustering_fields = ["CECO", "CUENTA", "VERSION"]

# try:
#     table = client.create_table(table)
#     print(f"Created table {table.full_table_id}")
# except Exception:
#     client.delete_table(table, not_found_ok=True)
#     table = client.create_table(table)
#     print(f"Recreated table {table.full_table_id}")

In [ ]:
# create_procedure_backup = f"""
# CREATE OR REPLACE PROCEDURE `{stored_procedure_BACKUP_BASE}`()

# BEGIN

#   INSERT INTO {log_audit_base}
#    ----CREATE OR REPLACE TABLE {log_audit_base} AS ---- SOLO SI NO EXISTE TABLA
#     SELECT *,
#     TIMESTAMP(FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', CURRENT_TIMESTAMP(), 'America/Mexico_City')) as BACKUP_DATE,
#     FROM {vista_forecast};

# END
# """
# client.query(create_procedure_backup).result()
# print("Creado ",stored_procedure_BACKUP_BASE)


**Stored Procedure, base raw, cálculos**

**Stored Procedure Distribucion**

In [ ]:
from google.cloud import bigquery
from google.cloud.bigquery import SchemaField

client = bigquery.Client(project="{project_id}")

# 1. Definimos la lista de tablas a procesar
tablas_a_respaldar = [
    tabladistpres,
    tablapres_disanioanterior,
    tablapres_disanioactual
]

def crear_log_y_procedimiento(tabla_origen):
    # Definimos el nombre de la tabla de LOG (ejemplo: DISTPRESTACIONES_LOG)
    tabla_log_id = f"{tabla_origen}_LOG"
    sp_name = f"SP_BACKUP_{tabla_origen.split('.')[-1]}"

    print(f"--- Procesando: {tabla_origen} ---")

    # A. Obtener esquema de la tabla origen y añadir BACKUP_DATE
    try:
        table_source = client.get_table(tabla_origen)
        schema_log = [SchemaField(f.name, f.field_type, f.mode) for f in table_source.schema]
        schema_log.append(SchemaField("BACKUP_DATE", "TIMESTAMP", mode="REQUIRED"))
    except Exception as e:
        print(f"Error al obtener tabla {tabla_origen}: {e}")
        return

    # B. Configurar la tabla de destino (Log)
    table_log_obj = bigquery.Table(tabla_log_id, schema=schema_log)

    # Particionamiento por día
    table_log_obj.time_partitioning = bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.DAY,
        field="BACKUP_DATE",
        require_partition_filter=True
    )

    # Clustering por las columnas que mencionaste
    # (Asegúrate de que los nombres coincidan exactamente en mayúsculas/minúsculas)
    table_log_obj.clustering_fields = ["CeCo", "Cuenta", "DIMENSION6"]

    # C. Crear o recrear la tabla
    client.delete_table(tabla_log_id, not_found_ok=True)
    table_log_obj = client.create_table(table_log_obj)
    print(f"Tabla de log creada: {table_log_obj.full_table_id}")

    # D. Crear el Stored Procedure
    create_sp_sql = f"""
    CREATE OR REPLACE PROCEDURE `{project_datasetbases}.{sp_name}`()
    BEGIN
      INSERT INTO `{tabla_log_id}`
      SELECT *,
      TIMESTAMP(FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', CURRENT_TIMESTAMP(), 'America/Mexico_City')) as BACKUP_DATE
      FROM `{tabla_origen}`;
    END
    """
    client.query(create_sp_sql).result()
    print(f"Procedimiento {sp_name} creado con éxito.\n")

# 2. Ejecutar el ciclo para las 3 tablas
for t in tablas_a_respaldar:
    crear_log_y_procedimiento(t)

**Stored Procedure Nómina**

**GENERAR RAW DATA BACKUP**

In [ ]:
# schema_raw_data = [
#     bigquery.SchemaField('ABR', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('AGO', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('AREA_PERSONAL', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('AUT_ARE', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('AUT_CORP', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('AUT_GEN', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Autorizado', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('CEBE', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('CECO', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('CeCo_CeBe', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('CECOCEBE', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('COMENTARIOS', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Concepto', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Cuenta', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DESC_DIMENSION5', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DIC', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION3', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION2', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION1', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Ejercicio', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('EJERCICIO_DE_INGRESO', 'INT64', mode='NULLABLE'),
#     bigquery.SchemaField('ENE', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('FEB', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('Grupo', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Importe', 'NUMERIC', mode='NULLABLE'),
#     bigquery.SchemaField('Importe_vales', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('JUL', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('JUN', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('KEY', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('MANDANTE', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('MAR', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('MAY', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('Mes', 'INT64', mode='NULLABLE'),
#     bigquery.SchemaField('MES_BASE', 'NUMERIC', mode='NULLABLE'),
#     bigquery.SchemaField('MES_DE_INGRESO', 'INT64', mode='NULLABLE'),
#     bigquery.SchemaField('MES_INC', 'INT64', mode='NULLABLE'),
#     bigquery.SchemaField('MONTO', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('Monto_CN', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('Naturaleza', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('NOV', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('OCT', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('ORIGEN', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('PARAMETROS', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('PATH', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Periodo', 'NUMERIC', mode='NULLABLE'),
#     bigquery.SchemaField('PERIODO_BASE', 'NUMERIC', mode='NULLABLE'),
#     bigquery.SchemaField('PERSONAS', 'INT64', mode='NULLABLE'),
#     bigquery.SchemaField('PORC_INC', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('PORC_PREST', 'NUMERIC', mode='NULLABLE'),
#     bigquery.SchemaField('POSICION', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('SEP', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('SUELDO', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('SUELDO_A_UTILIZAR', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('SUELDO_PROMEDIO', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('SUELDOS', 'NUMERIC', mode='NULLABLE'),
#     bigquery.SchemaField('TIMESTAMP', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION6', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('TOTAL', 'FLOAT64', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION5', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION5cacion', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Validacion1', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('Validacion2', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('VERSION', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('X', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('DIMENSION7', 'STRING', mode='NULLABLE'),
#     bigquery.SchemaField('BACKUP_DATE', 'TIMESTAMP', mode='NULLABLE'),
#     ]

# table = bigquery.Table(backupk_raw_data, schema=schema_raw_data)

# # 🔹 Partition by BACKUP_DATE (daily)
# table.time_partitioning = bigquery.TimePartitioning(
#     type_=bigquery.TimePartitioningType.DAY,
#     field="BACKUP_DATE",
#     require_partition_filter=True
# )

# # 🔹 Cluster by common filter keys (tune these to your queries)
# table.clustering_fields = ["CECO", "CUENTA", "VERSION"]

# try:
#     table = client.create_table(table)
#     print(f"Created table {table.full_table_id}")
# except Exception:
#     client.delete_table(backupk_raw_data, not_found_ok=True)
#     table = client.create_table(table)
#     print(f"Recreated table {table.full_table_id}")

In [ ]:
# query_raw_data = f'''
#     -- ==============================================================
#     -- ALL DECLARES FIRST (MANDATORY IN BIGQUERY)
#     -- ==============================================================

#     ---GUARD TO KNOW IF THE BACKUP IT IS NEEDED CONTROL VARIABLES

#     DECLARE src_last_modified TIMESTAMP;
#     DECLARE dst_last_backup   TIMESTAMP;

#     DECLARE src_fqtn STRING;
#     DECLARE dst_fqtn STRING;

#     DECLARE src_table_name STRING;

#     DECLARE dynamic_sql STRING;
#     DECLARE deletesentence STRING;

#     DECLARE origen_table STRING;

#     DECLARE dst_cols ARRAY<STRING>;

#     DECLARE src_cols ARRAY<STRING>;

#     DECLARE select_list STRING DEFAULT '';

#     DECLARE i INT64 DEFAULT 0;
#     DECLARE n INT64;
#     DECLARE dst_col STRING;
#     DECLARE dst_type STRING;

#     DECLARE dst_schema ARRAY<STRUCT<
#       col_name STRING,
#       col_type STRING
#     >>;

#     DECLARE query_ts TIMESTAMP;

#     SET src_fqtn = FORMAT(
#       '`{project_id}.intranet_bd_pruebas.%s`',
#       src_table
#     );


#     ------DETECT DESTINATION COLUMNS

#     EXECUTE IMMEDIATE """
#       SELECT ARRAY_AGG(
#               STRUCT(column_name AS col_name, data_type AS col_type)
#               ORDER BY ordinal_position
#             )
#       FROM
#        `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
#       WHERE table_name = '{backupk_raw_data.split(".")[2]}'
#     """
#     INTO dst_schema;

#     SET n = ARRAY_LENGTH(dst_schema);

#     SET src_table_name = REGEXP_EXTRACT(src_table, r'[^.]+$');


#     --------

#     -- Get source table last_modified_time
#     EXECUTE IMMEDIATE FORMAT("""
#       SELECT TIMESTAMP_MILLIS(last_modified_time)
#       FROM `{project_id}.data_warehouse_finance.__TABLES__`
#       WHERE table_id = '%s'
#     """, src_table_name)
#     INTO src_last_modified;

#     -- Get last BACKUP_DATE for this table (partition-safe)
#     EXECUTE IMMEDIATE FORMAT("""
#       SELECT MAX(BACKUP_DATE)
#       FROM `{project_id}.intranet_bd_pruebas.BACKUP_RAW_DATA`
#       WHERE ORIGEN = '%s'
#         AND BACKUP_DATE >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
#     """, src_table_name)
#     INTO dst_last_backup;

#     -- Stop if source not newer
#     IF dst_last_backup IS NOT NULL AND src_last_modified <= dst_last_backup THEN
#       SELECT FORMAT(
#         'SKIPPED: %s | src_last_modified=%s | last_backup=%s',
#         src_table_name,
#         CAST(src_last_modified AS STRING),
#         CAST(dst_last_backup AS STRING)
#       ) AS status;
#       RETURN;
#     END IF;


#     ------DETECT COLUMNS IN SOURCE TABLE

#     EXECUTE IMMEDIATE """
#       SELECT ARRAY_AGG(UPPER(column_name))
#       FROM `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
#       WHERE table_name = @t
#     """
#     INTO src_cols
#     USING src_table_name AS t;

#     -- --------------------------------------------------------------
#     ------BUILD SELECT LIST
#     -- --------------------------------------------------------------

#     SET query_ts = CURRENT_TIMESTAMP();

#     WHILE i < n DO
#       SET dst_col  = dst_schema[OFFSET(i)].col_name;
#       SET dst_type = dst_schema[OFFSET(i)].col_type;

#       SET select_list = select_list || ',\\n' ||
#         CASE
#           WHEN dst_col = 'BACKUP_DATE'
#             THEN FORMAT(
#               "TIMESTAMP('%s') AS BACKUP_DATE",
#               FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', query_ts, 'America/Mexico_City')
#             )

#           WHEN dst_col = 'ORIGEN' THEN FORMAT ("'%s'as ORIGEN" ,  src_table)

#           -- source column exists → cast to destination type
#           WHEN UPPER(dst_col) IN UNNEST(src_cols)
#             THEN FORMAT(
#               'SAFE_CAST(K.%s AS %s) AS %s',
#               dst_col,
#               dst_type,
#               dst_col
#             )

#           -- source column missing → NULL of destination type
#           ELSE FORMAT(
#             'CAST(%s AS %s) AS %s',
#             CASE
#               WHEN dst_type = 'STRING'  THEN "''"
#               WHEN dst_type = 'INT64'   THEN '0'
#               WHEN dst_type = 'NUMERIC' THEN '0'
#               WHEN dst_type = 'FLOAT64' THEN '0'
#               WHEN dst_type = 'BOOL'    THEN 'FALSE'
#               ELSE 'NULL'
#             END,
#             dst_type,
#             dst_col
#           )
#         END;

#       SET i = i + 1;
#     END WHILE;

#     SELECT select_list;

#     -- --------------------------------------------------------------
#     -- Insert data
#     -- --------------------------------------------------------------
#     SET dynamic_sql = FORMAT("""
#       INSERT INTO `{project_id}.intranet_bd_pruebas.BACKUP_RAW_DATA`
#       SELECT %s
#       FROM %s AS K
#   """,
#       SUBSTR(select_list, 3),  -- remove leading comma
#       src_fqtn
#     );

#     SELECT dynamic_sql;

#     EXECUTE IMMEDIATE dynamic_sql;

# '''

# storedprocedure_master_raw = f'''
# CREATE OR REPLACE PROCEDURE `{stored_procedure_BACKUP_BASE_raw}`(src_table STRING)
# OPTIONS (strict_mode = false)
# BEGIN
#   {query_raw_data}
# END;
# '''

# client.query(storedprocedure_master_raw).result()
# print("Creado: ",stored_procedure_BACKUP_BASE_raw)

💡 **Ejecutar SP de BACKUPS arriba, base completa, abajo por tabla "inicial" RAW DATA**

In [ ]:
##1 BACKUP BASE FINAL ACTUAL ( en parametros sheets se ejecuta 3 veces al día)

# callspbackup = f"""
# CALL `{stored_procedure_BACKUP_BASE}`();
# """

# query_job = client.query(callspbackup)
# query_job.result()
# print("Backup realizado")

## BACKUP RAW DATA, requiere listado de tablas

tablas_backup = [
    # "{project_id}.intranet_bd_pruebas.BASEP_CARGA_INICIAL_completas"
    # base_hist ##HISTORICOS BASE
    # ,base_hist_acum ##HISTORICOS ACUMULADOS

    # ,tablanomina ##SUELDOS
    # ,tablanomina_vales ##VALES DESPENSA Y UNIFORMES
    # ,tablabonobase ##BONO_ESP

    # ,tablacargapresrep ##PRESTACIONES MES BASE REPLICADO
    # ,tablacargapresccero ##PRESTACIONES CRECIMIENTO CERO
    # ,tablacargapresccero_sinout ##PRESTACIONES CRECIMIENTO CERO SIN OUTLIERS
    # ,tablacargapresminc ##PRESTACIONES MES DE INCREMENTO DETERMINADO
    # ,tablacargaporc ##PRESTACIONES BASE PORCENTAJE ROLL YEAR
    # ,tablacargaporc_ant ##PRESTACIONES BASE PORCENTAJE AÑO ANTERIOR
    # ,tablacargaptugar ##PRESTACIONES PTU REPARTO GARANTIZADO
    # ,tablacargapinfl ##PRESTACIONES INFLACION
    # ,tablanompres_anioantinf ##PRESTACIONES REPLICA 1-12 AÑO ANTERIOR + INFLACION
    # ,tablanompres_isn ##PRESTACIONES ISN

    # ,tablaCARGA_INICIAL ##CARGA_INICIAL

    # ,forecast_sscc ##FORECAST SSCC
    # ,forecast_fin_ifrs ##FORECAST NEGOCIO Z,NEGOCIO T
    # ,forecast_inmob ##FORECAST NEGOCIO Y
    # ,forecast_NEGOCIO X ##FORECAST NEGOCIO X
    # ,forecast_nomascara  ###FORECAST DIVISION NO EN MASCARA, DEJA 4 MESES PPTO + 5 NEGATIVO DE LOS PRIMEROS MESES, CON ESTO METEMOS AJUSTE EN MAYO SI METEN REALES "MAL"

    # ,tabla_base_hc ### CONCEPTOS DE NEGOCIO HEADCOUNT
    # ,tabla_base_cn ### CONCEPTOS DE NEGOCIO CECO
    # ,tabla_base_gercn ### CONCEPTOS DE NEGOCIO ER
    # ,tabla_base_gencn ### CONCEPTOS DE NEGOCIO GENERALES

    # ,cedulablancaid ### CEDULAS
    # ,vista_prorrycuot ### RESULTADO PRORRATEO
]

tables = []
for tabla in tablas_backup:
    tables.append(tabla.split(".")[2])

# print("Query elaborar tabla base: ", f"""
#  SELECT distinct column_name, data_type

# FROM `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
#       WHERE table_name IN UNNEST ({tables})
# .
# """)


from google.api_core.exceptions import BadRequest, GoogleAPICallError

failed = []

for tabla in tablas_backup:
    table = tabla.split(".")[2]

    callspbackup2 = f"""
    CALL `{stored_procedure_BACKUP_BASE_raw}`('{table}');
    """

    try:
        query_job = client.query(callspbackup2)
        query_job.result()  # wait for completion

        print("✅ SP Atributos CONCLUIDO:", stored_procedure_BACKUP_BASE_raw, "|", tabla)

    except BadRequest as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "BadRequest"))

        print("❌ SP ERROR (BadRequest)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)

        for err in e.errors or []:
            print("   →", err.get("message"))

        continue

    except GoogleAPICallError as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "APICall"))

        print("❌ SP ERROR (API Call)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)
        print("   →", str(e))

        continue

    except Exception as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "Unexpected"))

        print("❌ SP ERROR (Unexpected)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)
        print("   →", str(e))

        continue


# ======================================================
# SUMMARY
# ======================================================
print("\n================ SUMMARY — FAILED TABLES ================")

if not failed:
    print("🎉 All tables processed successfully.")
else:
    for tabla, job_id, err_type in failed:
        print(f"{tabla} | JobId={job_id} | Error={err_type}")

⏰🛕❣🦺🥵   **Reestablecer backups**

In [ ]:
#### RESTORE BACKUP

# consultar_fechas = f'''
# SELECT MAX(BACKUP_DATE)
#       FROM `{project_id}.intranet_bd_pruebas.LOG_AUDITORIA_BASE_PRESUPUESTAL`
#       WHERE
#         BACKUP_DATE >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
# '''

# ####EJECUTA ESTE QUERY EN BQ PARA OBTENER UNA LISTA DE TODOS LOS ORIGENES EN LA TABLA ---
# consultar_origenes = f'''
# select string_agg(distinct origen, "','")

# from `intranet_bd_pruebas.CALCULOS_INCIALES`
# '''

#######SELECCIONAR FECHA BACKUP EJEMPLO DE ESTRUCTURA '2026-02-16 05:53:27 UTC'
fechabackup = '2026-02-16 05:53:27 UTC'

filtros = f'''
ORIGEN IN (
'HISTORICOS',
'HISTORICOS_ACUM',

'FORECAST_NEGOCIO X',
'FORECAST_NEGOCIO Y',
'FORECAST_NOMASCARA',
'FORECAST_SSCC',
'FORECAST_NEGOCIO Z_IFRS',

'NOM_PRES_AÑO_ANTERIOR_INFLACION',
'NOM_PRE_cecimientocero_sinout',
'NOM_PRE_ptugar',
'NOM_BONO_ESP',
'D',
'NOM_PRE_porc_ant',
'NOM_SUE',
'NOM_VAL',
'NOM_PRE_crecimientocero',
'NOM_PRE_replicas',
'NOM_PRE_ISN',
'NOM_PRE_inflacion',
'NOM_PRE_mesinc',

'CARGA_INICIAL'
)
'''

####LOS FILTROS SE PUEDEN HACER A PLACER AQUI SE MUESTRAN TODOS LOS QUE EXISTEN EN ESA BASE

backup = f'''
delete from `{tabla_atributos_calc_inic}`

WHERE {filtros};

INSERT INTO `{tabla_atributos_calc_inic}`

select Cuenta,CeCo,CeBe,DIMENSION1,DIMENSION2,DIMENSION3,DIMENSION6_MAESTRO,DIMENSION7,DIMENSION5,DES_DIMENSION5,NIVEL1,NIVEL2,NIVEL3,NIVEL4,NIVEL5,Ejercicio,Periodo,Version,MES_INC,PORC_INC,MES_BASE,Importe,GRUPO,CONCEPTO,Personas,AREA_PERSONAL,POSICION,
SAFE_CAST(IFNULL(Importe_vales,0)AS NUMERIC) as IMPORTE_VALES,
SAFE_CAST(IFNULL(MONTO,0)AS NUMERIC) as MONTO,
SAFE_CAST(IFNULL(SUELDOS,0)AS NUMERIC) as SUELDOS,
SAFE_CAST(IFNULL(PORC_PREST,0)AS NUMERIC) as PORC_PREST,DIMENSION6,PERIODO_BASE,EJERCICIO_DE_INGRESO,MES_DE_INGRESO,SAFE_CAST(IFNULL(PARAMETROS,0)AS NUMERIC) as PARAMETROS,SAFE_CAST(IFNULL(SUELDO,0)AS NUMERIC) as SUELDO,ORIGEN


from `{log_audit_base}`

WHERE BACKUP_DATE = '{fechabackup}' and
 {filtros};

'''

# 🔧 **PRUEBAS**

⭐ Aplica atributos, da estructura de base desarrollo en otra tabla CALCULOS_INICIALES_pruebas, con una query, se compara contra la productiva CALCULOS_INICIALES y se valida que se obtenga en resultado deseado, si es el correcto, se procede a ejecutar "Asignar Atributos"

😀 Toma las tablas RAW y las envía a otra base, así podemos, literal, probar cambios, nuevos cálculos, etc!!!

♐ **Ejecutar Stored Procedure**

In [ ]:
tabla_pruebas = f'{project_datasetbases}.PRUEBAS_{tabla_atributos_calc_inic.split(".")[2]}'
sp_atributos_pruebas = f'{stored_procedure_atributos}_pruebas'

####LISTADO DE TABLAS QUE CONFORMAN CALCULOS INICIALES, FORECAST E HISTORICOS

tablas_convertir = [
    # base_hist ##HISTORICOS BASE
    # ,base_hist_acum ##HISTORICOS ACUMULADOS

    # ,tablanomina ##SUELDOS
    # ,tablanomina_vales ##VALES DESPENSA Y UNIFORMES
    # ,tablabonobase ##BONO_ESP

    # ,tablacargaporc_ant ##PRESTACIONES BASE PORCENTAJE AÑO ANTERIOR
    # ,tablacargapresccero ##PRESTACIONES CRECIMIENTO CERO
    # ,tablacargapresccero_sinout ##PRESTACIONES CRECIMIENTO CERO SIN OUTLIERS
    # ,tablacargapinfl ##PRESTACIONES INFLACION
    # ,tablanompres_anioantinf ##PRESTACIONES REPLICA AÑO ANTERIOR 1-12 + INFLACION
    # ,tablacargapresminc ##PRESTACIONES MES DE INCREMENTO DETERMINADO
    # ,tablacargaporc ##PRESTACIONES BASE PORCENTAJE ROLL YEAR
    # ,tablacargapresrep ##PRESTACIONES MES BASE REPLICADO
    # ,tablacargaptugar ##PRESTACIONES PTU REPARTO GARANTIZADO

    # ,tablanompres_isn ##PRESTACIONES ISN

    # ,tablaCARGA_INICIAL ##CARGA_INICIAL

    # ,forecast_sscc ##FORECAST SSCC
    # ,forecast_fin_ifrs ##FORECAST NEGOCIO Z,NEGOCIO T
    # ,forecast_inmob ##FORECAST NEGOCIO Y
    # ,forecast_NEGOCIO X ##FORECAST NEGOCIO X
    # ,forecast_nomascara  ###FORECAST DIVISION NO EN MASCARA, DEJA 4 MESES PPTO + 5 NEGATIVO DE LOS PRIMEROS MESES, CON ESTO METEMOS AJUSTE EN MAYO SI METEN REALES "MAL"
]

from google.api_core.exceptions import BadRequest, GoogleAPICallError

failed = []

for tabla in tablas_convertir:
    all_procedure_sql = f"""
    CALL `{sp_atributos_pruebas}`(
      "{tabla}"
    );
    """

    query_job = None

    try:
        query_job = client.query(all_procedure_sql)
        query_job.result()  # wait for completion

        print("✅ SP Atributos CONCLUIDO:", stored_procedure_atributos, "|", tabla)

    except BadRequest as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "BadRequest"))

        print("❌ SP ERROR (BadRequest)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)

        for err in e.errors or []:
            print("   →", err.get("message"))

        continue

    except GoogleAPICallError as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "APICall"))

        print("❌ SP ERROR (API Call)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)
        print("   →", str(e))

        continue

    except Exception as e:
        job_id = query_job.job_id if query_job else "N/A"
        failed.append((tabla, job_id, "Unexpected"))

        print("❌ SP ERROR (Unexpected)")
        print("   Tabla :", tabla)
        print("   JobId :", job_id)
        print("   →", str(e))

        continue


# ======================================================
# SUMMARY
# ======================================================
print("\n================ SUMMARY — FAILED TABLES ================")

if not failed:
    print("🎉 All tables processed successfully.")
else:
    for tabla, job_id, err_type in failed:
        print(f"{tabla} | JobId={job_id} | Error={err_type}")


⭐ **Validar en BQ**


WITH BKP AS (

select DIMENSION1, DIMENSION2, DIMENSION3, ceco, version, ejercicio,origen, sum(importe) as importe_bkp

from `intranet_bd_pruebas.CALCULOS_INCIALES`

group by 1,2,3,4,5,6,7
),

ACT AS (

select DIMENSION1, DIMENSION2, DIMENSION3, ceco, version, ejercicio,origen, sum(importe) as importe_act

from `intranet_bd_pruebas.PRUEBAS_CALCULOS_INCIALES`

group by 1,2,3,4,5,6,7

)

SELECT
COALESCE(BKP.DIMENSION1,ACT.DIMENSION1) as DIMENSION1,
COALESCE(BKP.DIMENSION2,ACT.DIMENSION2) as DIMENSION2,
COALESCE(BKP.DIMENSION3,ACT.DIMENSION3) as DIMENSION3,
COALESCE(BKP.origen,ACT.origen) as origen,
COALESCE(BKP.ceco,ACT.ceco) as ceco,
COALESCE(BKP.version,ACT.version) as version,
COALESCE(BKP.ejercicio,ACT.ejercicio) as ejercicio,
importe_bkp,
importe_act,
round(ifnull(bkp.importe_bkp,0)-ifnull(act.importe_act,0),2) as diff

FROM BKP
FULL OUTER JOIN ACT USING (DIMENSION1
,DIMENSION2
,DIMENSION3
,VERSION
,EJERCICIO
,CECO
,origen
)

where round(ifnull(bkp.importe_bkp,0)-ifnull(act.importe_act,0),2) <> 0 and COALESCE(BKP.origen,ACT.origen) = 'NOM_BONO_ESP'

order by 3,4



Crear Tabla Base y Stored Procedure

In [ ]:
# # CREAR TABLA COMO COPIA DE LA PRODUCTIVA

# crear_tabla_pruebas = f'''
# CREATE OR REPLACE TABLE {tabla_pruebas} as
# SELECT * FROM {tabla_atributos_calc_inic}
# '''

# client.query(crear_tabla_pruebas).result()
# print("Creado ",tabla_pruebas)


# ###GENERAR STORED PROCEDURE

# storedatributoscalcini_pruebas = f'''
#     -- ==============================================================
#     -- ALL DECLARES FIRST (MANDATORY IN BIGQUERY)
#     -- ==============================================================

#     DECLARE src_fqtn STRING;
#     DECLARE dst_fqtn STRING;

#     DECLARE src_table_name STRING;

#     DECLARE dynamic_sql STRING;
#     DECLARE deletesentence STRING;

#     DECLARE origen_table STRING;

#     DECLARE dst_cols ARRAY<STRING>;

#     DECLARE src_cols ARRAY<STRING>;

#     DECLARE select_list STRING DEFAULT '';

#     DECLARE i INT64 DEFAULT 0;
#     DECLARE n INT64;
#     DECLARE dst_col STRING;
#     DECLARE dst_type STRING;

#     DECLARE dst_schema ARRAY<STRUCT<
#       col_name STRING,
#       col_type STRING
#     >>;

#     ------DETECT DESTINATION COLUMNS

#     EXECUTE IMMEDIATE """
#       SELECT ARRAY_AGG(
#               STRUCT(column_name AS col_name, data_type AS col_type)
#               ORDER BY ordinal_position
#             )
#       FROM `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
#       WHERE table_name = '{tabla_atributos_calc_inic.split(".")[2]}'
#     """
#     INTO dst_schema;

#     SET n = ARRAY_LENGTH(dst_schema);

#     SET src_table_name = REGEXP_EXTRACT(src_table, r'[^.]+$');

#     ------DETECT COLUMNS IN SOURCE TABLE

#     EXECUTE IMMEDIATE """
#       SELECT ARRAY_AGG(UPPER(column_name))
#       FROM `{project_id}.intranet_bd_pruebas.INFORMATION_SCHEMA.COLUMNS`
#       WHERE table_name = @t
#     """
#     INTO src_cols
#     USING src_table_name AS t;

#     -- ==============================================================
#     -- EXECUTABLE STATEMENTS START HERE
#     -- ==============================================================

#     SET src_fqtn = FORMAT(
#       '`%s`',
#       src_table
#     );

#     -- --------------------------------------------------------------
#     -- Get ORIGEN safely (single value)
#     -- --------------------------------------------------------------
#     EXECUTE IMMEDIATE FORMAT("""
#       SELECT ANY_VALUE(ORIGEN)
#       FROM %s
#     """, src_fqtn)
#     INTO origen_table;

#     -- --------------------------------------------------------------
#     -- Delete previous data
#     -- --------------------------------------------------------------
#     SET deletesentence = FORMAT("""
#       DELETE FROM `{tabla_pruebas}`
#       WHERE ORIGEN = '%s'
#     """, origen_table);

#     EXECUTE IMMEDIATE deletesentence;

#     -- --------------------------------------------------------------
#     ------BUILD SELECT LIST
#     -- --------------------------------------------------------------

#     WHILE i < n DO
#       SET dst_col  = dst_schema[OFFSET(i)].col_name;
#       SET dst_type = dst_schema[OFFSET(i)].col_type;

#       SET select_list = select_list || ',\\n' ||
#         CASE
#           --data master

#           WHEN dst_col = 'DIMENSION1'
#             THEN 'COALESCE(MC.DIMENSION1, MB.DIMENSION1, CAST(K.CECO AS STRING)) AS DIMENSION1'

#           WHEN dst_col = 'DIMENSION2'
#             THEN 'COALESCE(MC.DIMENSION2, MB.DIMENSION2, CAST(K.CEBE AS STRING)) AS DIMENSION2'

#           WHEN dst_col = 'DIMENSION3'
#             THEN 'COALESCE(MC.DIMENSION3, MB.DIMENSION3) AS DIMENSION3'

#           WHEN dst_col = 'DIMENSION6_MAESTRO'
#             THEN 'COALESCE(MC.DIMENSION6, MB.DIMENSION6) AS DIMENSION6_MAESTRO'

#           WHEN dst_col = 'DIMENSION7'
#             THEN 'COALESCE(MC.DIMENSION7, MB.DIMENSION7) AS DIMENSION7'

#           WHEN dst_col = 'DIMENSION5'
#             THEN 'COALESCE(MC.DIMENSION5, MB.DIMENSION5) AS DIMENSION5'

#           WHEN dst_col = 'DES_DIMENSION5'
#             THEN 'COALESCE(MC.DES_DIMENSION5, MB.DES_DIMENSION5) AS DES_DIMENSION5'

#           WHEN dst_col = 'NIVEL1' THEN 'CAST(IFNULL(JCP.NIVEL1,"") AS STRING) AS NIVEL1'
#           WHEN dst_col = 'NIVEL2' THEN 'CAST(IFNULL(JCP.NIVEL2,"") AS STRING) AS NIVEL2'
#           WHEN dst_col = 'NIVEL3' THEN 'CAST(IFNULL(JCP.NIVEL3,"") AS STRING) AS NIVEL3'
#           WHEN dst_col = 'NIVEL4' THEN 'CAST(IFNULL(JCP.NIVEL4,"") AS STRING) AS NIVEL4'
#           WHEN dst_col = 'NIVEL5' THEN 'CAST(IFNULL(JCP.NIVEL5,"") AS STRING) AS NIVEL5'

#           WHEN upper(dst_col) = 'CUENTA' THEN "TRIM(CONCAT(CAST(K.CUENTA AS STRING), CHR(32), COALESCE(CNT.DESCRIPCION, ''))) AS CUENTA"

#           -- explicit overrides
#           WHEN dst_col = 'ORIGEN'    THEN 'CAST(K.ORIGEN AS STRING) AS ORIGEN'

#           -- source column exists → cast to destination type
#           WHEN UPPER(dst_col) IN UNNEST(src_cols)
#             THEN FORMAT(
#               'SAFE_CAST(K.%s AS %s) AS %s',
#               dst_col,
#               dst_type,
#               dst_col
#             )

#           -- source column missing → NULL of destination type
#           ELSE FORMAT(
#             'CAST(%s AS %s) AS %s',
#             CASE
#               WHEN dst_type = 'STRING'  THEN "''"
#               WHEN dst_type = 'INT64'   THEN '0'
#               WHEN dst_type = 'NUMERIC' THEN '0'
#               WHEN dst_type = 'FLOAT64' THEN '0'
#               WHEN dst_type = 'BOOL'    THEN 'FALSE'
#               ELSE 'NULL'
#             END,
#             dst_type,
#             dst_col
#           )
#         END;

#       SET i = i + 1;
#     END WHILE;

#     -- SELECT select_list;

#     -- --------------------------------------------------------------
#     -- Insert normalized data
#     -- --------------------------------------------------------------
#     SET dynamic_sql = FORMAT("""
#       INSERT INTO `{tabla_pruebas}`
#       SELECT %s
#       FROM %s AS K
#       LEFT JOIN `{CECOS}` MC
#         ON CAST(K.CECO AS STRING) = MC.CECO
#       LEFT JOIN `{CEBES}` MB
#         ON CAST(K.CEBE AS STRING) = MB.CEBE
#       INNER JOIN `{CUENTAS}` CNT ON SUBSTRING(K.CUENTA,1,8) = CNT.CUENTA
#       LEFT JOIN `{JERARCUENTAPPTO}` JCP ON SUBSTRING(K.CUENTA,1,8) = JCP.CUENTA
#       WHERE K.CUENTA IS NOT NULL
#   """,
#       SUBSTR(select_list, 3),  -- remove leading comma
#       src_fqtn
#     );

#     -- SELECT dynamic_sql;

#     EXECUTE IMMEDIATE dynamic_sql;

# '''

# storedprocedure_master_pruebas = f'''
# CREATE OR REPLACE PROCEDURE `{sp_atributos_pruebas}`(src_table STRING)
# OPTIONS (strict_mode = false)
# BEGIN
#   {storedatributoscalcini_pruebas}
# END;
# '''

# client.query(storedprocedure_master_pruebas).result()
# print("Creado: ",sp_atributos_pruebas)

# **VISTA BUILDING BLOCK**

⭐ Reporte en desarrollo

Estructura los conceptos de negocio, para mostrar el detalle de estos en el Building Block


In [ ]:
bbreport_v = bigquery.Table(bbreport)
querybb = f'''--

WITH BASES AS (
  #NOMINA CN
  SELECT *
  FROM `{tabla_base_hccn_sp}`

  UNION ALL

  #NOMINA ETZ CN
  SELECT *
  FROM `{nomina_etz_cn}`

  UNION ALL

  #CONCEPTOS
  SELECT *
  FROM `{tabla_base_cn_sp}`

  UNION ALL

  #CONCEPTOS
  SELECT *
  FROM `{tabla_base_gencn_sp}`

  UNION ALL

  #CONCEPTOS
  SELECT *
  FROM `{tabla_base_gercn_sp}`
),

REPORTE AS (

SELECT DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5, DES_DIMENSION5, CEBE,DIMENSION6_MAESTRO,DIMENSION7, Nivel1, 'GASTO NETO' AS DIMENSION6BB,
CASE WHEN Version = 'PPTO_V0' THEN 'PPTO' WHEN Version = 'PPTO_V2' THEN 'ME' WHEN Version = 'PPTO_V1' THEN 'REV' END AS `ORDEN`,
'HEADCOUNT' AS `GRUPO`,
DIMENSION6||" "||CASE WHEN EJERCICIO_DE_INGRESO = {aniobase+1} THEN 'PPTO_V0'
WHEN EJERCICIO_DE_INGRESO = {aniobase} THEN
CASE WHEN Version = 'PPTO_V2' THEN 'ME' WHEN Version = 'PPTO_V1' THEN 'REV' END END AS `CONCEPTO`,
Importe AS `Monto`

FROM BASES T0

WHERE ORIGEN LIKE "CN_HC%"

UNION ALL

SELECT DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5, DES_DIMENSION5, CEBE, DIMENSION6_MAESTRO, DIMENSION7,Nivel1, CASE WHEN T0.CUenta LIKE '4%' THEN 'INGRESOS' ELSE 'GASTO NETO' END AS DIMENSION6BB,
CASE WHEN Version = 'PPTO_V0' THEN 'PPTO' WHEN Version = 'PPTO_V2' THEN 'ME' WHEN Version = 'PPTO_V1' THEN 'REV' END AS `ORDEN`,
GRUPO, CONCEPTO,
Importe AS `Monto`

FROM BASES T0

WHERE ORIGEN LIKE "CN%" AND ORIGEN NOT LIKE "CN_HC%"

)

SELECT * EXCEPT(DIMENSION5, DES_DIMENSION5), DES_DIMENSION5

FROM REPORTE

'''

bbreport_v.view_query = querybb

try:
    view = client.create_table(bbreport_v)
    print(f"Created {view.table_type}: {str(view.reference)}")
except:
  view = client.update_table(bbreport_v, ["view_query"])
  print(f"Updated {view.table_type}: {str(view.reference)}")

# **VISTA CARATULA DE VENTAS**

❗❗❗❗ Reporte pausado, probablemente en desuso


Tabla históricos

In [ ]:
udate = datetime.now().strftime(dateformat)

query_historicos = f''' SELECT
DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, DIMENSION88, DIMENSION7, T0.Cuenta, Periodo, Ejercicio, Version,
'VENTAS_HISTORICO' as ORIGEN,
'{udate}' as UDATE,

SUM(Importe) as Importe

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

WHERE Version IN ('Reales','PPTO','PPTO_V2') AND IMPORTE <>0 AND T1.P_L IN ('Costo de Ventas','Ventas Totales') AND Ejercicio = {aniobase}

GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13

UNION ALL

SELECT
DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, DIMENSION88, DIMENSION7, T0.Cuenta, Periodo, Ejercicio, Version,
'VENTAS_HISTORICO' as ORIGEN,
'{udate}' as UDATE,

SUM(Importe) as Importe

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

WHERE Version IN ('PPTO_V0') AND IMPORTE <>0 AND T1.P_L IN ('Costo de Ventas','Ventas Totales') AND Ejercicio = {aniobase+1}

GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13

'''
bigquery_result_hist_ventas = bpd.read_gbq(query_historicos)
bigquery_result_hist_ventas_header = bigquery_result_hist_ventas.columns
df_bigquery_result_hist_ventas = pd.DataFrame(bigquery_result_hist_ventas)
df_bigquery_result_hist_ventas.columns = bigquery_result_hist_ventas_header

df_bigquery_result_hist_ventas['UDATE'] = pd.to_datetime(df_bigquery_result_hist_ventas['UDATE'],format=dateformat)

df_bigquery_result_hist_ventas = df_bigquery_result_hist_ventas[df_bigquery_result_hist_ventas['DIMENSION1'] != np.empty]

# print(carga['UDATE'].dtypes, carga['UDATE'].unique().tolist())

job_config = bigquery.LoadJobConfig(
write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_dataframe(
    df_bigquery_result_hist_ventas, f"{tablahistventas}_{aniobase}", job_config=job_config
    )

load_job.result()  # Espera que la carga termine
destination_table = client.get_table(f"{tablahistventas}_{aniobase}")  # Revisando si la tabla existe
print("Loaded {} rows.".format(destination_table.num_rows))
job = client.get_job(load_job)
print(f"State: {job.state}")

**Vista Final**

In [ ]:
ventasreportcalc_v = bigquery.Table(ventasreport)

query_ventasreport = f'''
SELECT
DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, DIMENSION88, DIMENSION7, T0.Cuenta, Periodo, Ejercicio, Version,
ORIGEN,
cast(format_date('%d%m%Y', udate) as int64) as UDATE,
Importe*-1 as Importe

FROM `{tablahistventas}_*` T0

UNION ALL

SELECT
DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, DIMENSION88, DIMENSION7, T0.Cuenta, Periodo, Ejercicio, Version,
ORIGEN,
cast(format_date('%d%m%Y', udate) as int64) as UDATE,
Importe*-1 as Importe

FROM `{ventas_id}` T0

'''

ventasreportcalc_v.view_query = query_ventasreport

try:
    view = client.create_table(ventasreportcalc_v)
    print(f"Created {view.table_type}: {str(view.reference)}")
except:
  view = client.update_table(ventasreportcalc_v, ["view_query"])
  print(f"Updated {view.table_type}: {str(view.reference)}")

# **VISTA HC (reporte)**

⭐ Comparativo de headcount, periodos cerrados más conceptos de negocio HC

✈ Primero creamos los históricos que se utilizarán en el reporte

In [ ]:
hc_PLAN25 = '`{project_id}.ME24_PPTO25.HC_ME_PPTO2025_DIMENSION5`'
hc_REV25 = '`{project_id}.REV2025.vHC_r`'
hc_PLAN26 = '`{project_id}.ME_PPTO2026.BACKUP_REPORTE_HEADCOUNT_161225_1256`'

if aniobase == 2025 :
  hc_PLAN = hc_PLAN25
else :
  hc_PLAN = hc_PLAN26

if DIMENSION6pres == 'ME_PPTO' :
  hc_1 = hc_REV25
  filtro_nuevamedidia = 6
else :
  hc_1 = hc_PLAN26
  filtro_nuevamedidia = 1

from dateutil.relativedelta import relativedelta

d = date(aniobasehc, mesbasehc, 1)
prev_date = (d - relativedelta(months=1)).replace(day=1)

mesbasereportehc = prev_date.month
aniobasereportehc = prev_date.year


query_hc_hist = f'''
CREATE OR REPLACE TABLE
{headcount_historicos}
AS
WITH
CECO_DIM AS (
  SELECT
    CECO,
    DIMENSION1,
    DIMENSION2,
    DIMENSION3,
    PF_CO_OP,
    CASE
      WHEN DIMENSION1 IN ('NEGOCIO X y CC Externos','NEGOCIO Y')
        THEN DES_DIMENSION5
      ELSE CONCAT(DIMENSION5, ' ', DES_DIMENSION5)
    END AS DIMENSION5CACION
  FROM `{CECOS}`
),
POS_DIM AS (
  SELECT POSICION, Orden
  FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA`
),
-- HISTDIC
HIST_DIC AS (
  SELECT
    DIMENSION1, DIMENSION2, DIMENSION3,
    CASE WHEN DIMENSION1 IN ('NEGOCIO X y CC Externos','NEGOCIO Y')
      THEN DES_DIMENSION5 ELSE CONCAT(DIMENSION5,' ',DES_DIMENSION5) END AS DIMENSION5CACION,
      CECO,
    Posicion, p.Orden, PF_CO_OP AS DIMENSION6_Nom,
    '2' AS Periodo,
    'Reales' AS DIMENSION6,
    COUNT(num_per) AS HC
  FROM `{project_id}.human_capital.HC_HISTORICO_ATRIBUTOS_CECO`
  LEFT JOIN POS_DIM p USING (Posicion)
  WHERE ANIO IN (DATE({aniobase-1},12,1))
  GROUP BY 1,2,3,4,5,6,7,8,9
),
-- HIST
HIST_EXP AS (
  SELECT
    DIMENSION1, DIMENSION2, DIMENSION3,
    CASE
      WHEN DIMENSION1 IN ('NEGOCIO X y CC Externos','NEGOCIO Y')
        THEN DES_DIMENSION5 ELSE CONCAT(DIMENSION5,' ',DES_DIMENSION5) END AS DIMENSION5CACION,
        CECO,
    Posicion, p.Orden, PF_CO_OP AS DIMENSION6_Nom,
    CASE
      WHEN ANIO = DATE({aniobasehc},{mesbasehc},1)  THEN '4'
      WHEN ANIO = DATE({aniobasereportehc},{mesbasereportehc},1)  THEN '3'
    END AS Periodo,
    'Reales' AS DIMENSION6,
    COUNT(num_per) AS HC
  FROM `{project_id}.human_capital.HC_HISTORICO_ATRIBUTOS_CECO`
  LEFT JOIN POS_DIM p USING (Posicion)
  WHERE ANIO IN (DATE({aniobasereportehc},{mesbasereportehc},1), DATE({aniobasehc},{mesbasehc},1))
  GROUP BY 1,2,3,4,5,6,7,8,9
),
-- PPTO
PPTO_EXP AS (
  SELECT DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5CACION, '' as CECO,
         Posicion, Orden, DIMENSION6_Nom,
         '6' AS Periodo, DIMENSION6, HC
  FROM {hc_PLAN}
  WHERE Periodo LIKE '7.1%'
),
-- PRIMER MEDIDA (BASE)
REV_EXP AS (
  SELECT DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5CACION, '' as CECO,
         Posicion, Orden, DIMENSION6_Nom,
         '{filtro_nuevamedidia}' AS Periodo, DIMENSION6, HC
  FROM {hc_1}
  WHERE PERIODO LIKE '5.1%'
)
SELECT * FROM HIST_EXP
UNION ALL SELECT * FROM HIST_DIC
UNION ALL SELECT * FROM PPTO_EXP
UNION ALL SELECT * FROM REV_EXP;
'''

client.query(query_hc_hist).result()
print("Creado ",headcount_historicos)

In [ ]:
if DIMENSION6pres == 'ME_PPTO' :
  columns_CN = f'''
    SUM(`5_Presupuestadas`) as `5_Presupuestadas`,
    SUM(`5_Retabulaciones`) as `5_Retabulaciones`,
    SUM(`5_Adicionales`) as `5_Adicionales`,
    SUM(`5_Promociones`) as `5_Promociones`,
    SUM(`5_Movimientos`) as `5_Movimientos`,

    SUM(`7_Presupuestadas`) as `7_Presupuestadas`,
    SUM(`7_Retabulaciones`) as `7_Retabulaciones`,
    SUM(`7_Adicionales`) as `7_Adicionales`,
    SUM(`7_Promociones`) as `7_Promociones`,
    SUM(`7_Movimientos`) as `7_Movimientos`
  '''
else :
  columns_CN = f'''
    SUM(`5_Presupuestadas`) as `5_Presupuestadas`,
    SUM(`5_Retabulaciones`) as `5_Retabulaciones`,
    SUM(`5_Adicionales`) as `5_Adicionales`,
    SUM(`5_Promociones`) as `5_Promociones`,
    SUM(`5_Movimientos`) as `5_Movimientos`
  '''

query_hc_report_dyn = f'''
WITH
MASCARA_AGG AS (
  SELECT
    CECO,
    DIMENSION1, DIMENSION2, DIMENSION3,
    CASE WHEN DIMENSION1 IN ('NEGOCIO X y CC Externos','NEGOCIO Y')
      THEN DES_DIMENSION5 ELSE CONCAT(DIMENSION5,' ',DES_DIMENSION5) END AS DIMENSION5CACION,
    Posicion,
    DIMENSION6,
    VERSION,
    SUM(IFNULL(Personas,0)) AS HC
  FROM {tabla_base_hccn_sp}
  WHERE
    CECO IS NOT NULL
    AND CECO <> ''
    AND PERIODO = 12
    AND VERSION IN ('PPTO_V2','PPTO_V0','PPTO_V1')
    and SUBSTRING(CUENTA,1,8)
 in ('51000001','51000002','51000003')
  GROUP BY 1,2,3,4,5,6,7,8
),
MASCARA_EXP AS (
  SELECT
    d.DIMENSION1,
    d.DIMENSION2,
    d.DIMENSION3,
    CASE WHEN d.DIMENSION1 IN ('NEGOCIO X y CC Externos','NEGOCIO Y')
      THEN d.DES_DIMENSION5 ELSE CONCAT(d.DIMENSION5,' ',d.DES_DIMENSION5) END AS DIMENSION5CACION,
    m.CECO,
    m.Posicion,
    0 AS Orden,
    d.PF_CO_OP AS DIMENSION6_Nom,
    IF(m.VERSION IN ('PPTO_V2','PPTO_V1'),'5','7') AS Periodo,
    m.DIMENSION6,
    m.HC
  FROM MASCARA_AGG m
  JOIN {CECOS} d
    USING (CECO)
),
BASE AS (
  SELECT *, '' as metric FROM {headcount_historicos}
  UNION ALL
  SELECT *, CONCAT(PERIODO,"_",DIMENSION6) as metric FROM MASCARA_EXP
),
TOTALES AS (
  SELECT
    DIMENSION1, DIMENSION2, DIMENSION3,
    Posicion,DIMENSION5CACION,CECO, DIMENSION6_Nom,
    SUM(IF(Periodo='1',HC,0)) AS `Total 1`,
    SUM(IF(Periodo='2',HC,0)) AS `Total 2`,
    SUM(IF(Periodo='3',HC,0)) AS `Total 3`,
    SUM(IF(Periodo='4',HC,0)) AS `Total 4`,
    SUM(IF(Periodo='5',HC,0)) AS `Total 5`, --CONCEPTOS
    SUM(IF(metric='5_Presupuestadas',HC,0)) AS `5_Presupuestadas`,
    SUM(IF(metric='5_Retabulaciones',HC,0)) AS `5_Retabulaciones`,
    SUM(IF(metric='5_Adicionales',HC,0)) AS `5_Adicionales`,
    SUM(IF(metric='5_Promociones',HC,0)) AS `5_Promociones`,
    SUM(IF(metric='5_Movimientos de Área',HC,0)) AS `5_Movimientos`,
    SUM(IF(Periodo='6',HC,0)) AS `Total 6`,
    SUM(IF(Periodo='7',HC,0)) AS `Total 7`, --CONCEPTOS
    SUM(IF(metric='7_Presupuestadas',HC,0)) AS `7_Presupuestadas`,
    SUM(IF(metric='7_Retabulaciones',HC,0)) AS `7_Retabulaciones`,
    SUM(IF(metric='7_Adicionales',HC,0)) AS `7_Adicionales`,
    SUM(IF(metric='7_Promociones',HC,0)) AS `7_Promociones`,
    SUM(IF(metric='7_Movimientos de Área',HC,0)) AS `7_Movimientos`,

  FROM BASE
  GROUP BY 1,2,3,4,5,6,7
)

SELECT
DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5CACION as DES_DIMENSION5, CECO,
    Posicion, DIMENSION6_Nom,
    SUM(`Total 1`) as `Total 1`,
      SUM(`Total 1`-`Total 2`) as `Total 2_1`,
      SUM(`Total 2`) as `Total 2`,
      SUM(`Total 3`) as `Total 3`,
      SUM(`Total 4`) as `Total 4`,
      SUM(`Total 5`) as `Total 5`,
      SUM(`Total 4`+`Total 5`) as `Total 5_1`,
      SUM(`Total 6`) as `Total 6`,
      SUM(`Total 4`+`Total 5`-`Total 6`) as `Total 6_1`,
      SUM(`Total 7`) as `Total 7`,
      SUM(`Total 4`+`Total 5`+`Total 7`) as `Total 7_1`,
      SUM(`Total 7`) as `Total 7_2`,

    {columns_CN}

FROM TOTALES

GROUP BY DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5CACION,
    Posicion, DIMENSION6_Nom,CECO

ORDER BY DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION5CACION,
    Posicion

'''

hcreport_v = bigquery.Table(hcreport)
hcreport_v.view_query = query_hc_report_dyn

try:
    view = client.create_table(hcreport_v)
    print(f"Created {view.table_type}: {str(view.reference)}")
except:
  view = client.update_table(hcreport_v, ["view_query"])
  print(f"Updated {view.table_type}: {str(view.reference)}")

In [ ]:
############# DEPRECATED ######################


# hcreport_v = bigquery.Table(hcreport)

# if DIMENSION6pres == "ME_PPTO":
#   if params['aniobase']-1 == 2024 :
#     tabla_hcejer_ant = "{project_id}.ME24_PPTO25.HC_ME_PPTO2025_DIMENSION5"
#   else:
#     tabla_hcejer_ant = f"{client.project}.ME_PPTO{str(aniobase-1)}.vHC_r"
#   tabla_hcrev_eract = f"{client.project}.REV{str(aniobase)}.vHC_r"
#   queryhcr = f'''--
#   WITH HC1 AS (
#     --ME
#     SELECT T1.DIMENSION1, T1.DIMENSION2, T1.DIMENSION3, CASE WHEN T1.DIMENSION1 IN ("NEGOCIO X y CC Externos","NEGOCIO Y") THEN T1.DES_DIMENSION5 ELSE T1.DIMENSION5 || " " || T1.DES_DIMENSION5 END as DIMENSION5CACION,
#     T0.Posicion, T2.Orden, T1.PF_CO_OP as `DIMENSION6_Nom`, '5 CAMBIOS PPTO ME' as `Periodo`,
#     T0.DIMENSION6, T0.Personas as `HC`

#     FROM (SELECT * FROM `{tabla_base_hc}*` WHERE CECO IS NOT NULL AND CECO <> "") T0
#     LEFT JOIN `{project_id}.mus_qas_drv_datos_maestros.NW_CECO3` T1 ON T0.CECO=T1.CECO
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T0.POSICION=T2.POSICION

#     WHERE T0.EJERCICIO_DE_INGRESO = {aniobase}

#     UNION ALL

#     --PLAN
#     SELECT T1.DIMENSION1, T1.`DIMENSION2`, T1.`DIMENSION3`, CASE WHEN T1.DIMENSION1 IN ("NEGOCIO X y CC Externos","NEGOCIO Y") THEN T1.DES_DIMENSION5 ELSE T1.DIMENSION5 || " " || T1.DES_DIMENSION5 END as DIMENSION5CACION,
#     T0.Posicion, T2.Orden, T1.PF_CO_OP as `DIMENSION6_Nom`, '7 CAMBIOS PLAN' as `Periodo`,
#     T0.DIMENSION6, T0.Personas as `HC`

#     FROM (SELECT * FROM `{tabla_base_hc}*` WHERE CECO IS NOT NULL AND CECO <> "") T0
#     LEFT JOIN `{project_id}.mus_qas_drv_datos_maestros.NW_CECO3` T1 ON T0.CECO=T1.CECO
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T0.POSICION=T2.POSICION

#     WHERE T0.EJERCICIO_DE_INGRESO = {aniobase+1}

#     UNION ALL

#     --ULTIMO MES DEL AÑO CORRIENTE
#     SELECT `DIMENSION1`, `DIMENSION2`,  `DIMENSION3`, CASE WHEN T0.DIMENSION1 IN ("NEGOCIO X y CC Externos","NEGOCIO Y") THEN T0.DES_DIMENSION5 ELSE T0.DIMENSION5 || " " || T0.DES_DIMENSION5 END as DIMENSION5CACION,
#     T0.`Posicion`,  T2.Orden, PF_CO_OP as `DIMENSION6_Nom`, '4 HC JUL{str(aniobase)[-2:]}' as `Periodo`,
#     'Reales' as `DIMENSION6`, COUNT(num_per)`HC`

#     FROM `{project_id}.human_capital.HC_HISTORICO_ATRIBUTOS_CECO` T0
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T0.POSICION=T2.POSICION

#     WHERE EXTRACT(MONTH from ANIO) = {mesbasehc} AND EXTRACT(YEAR from ANIO) = {aniobase}

#     GROUP BY 1,2,3,4,5,6,7

#     UNION ALL

#     --PRESUPUESTO AÑO CORRIENTE
#     SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION,
#     T0.Posicion, Orden,
#     DIMENSION6_Nom, '6 APROB {str(aniobase)[-2:]}' `Periodo`, `DIMENSION6`, `HC`

#     FROM {tabla_hcejer_ant} T0

#     WHERE Periodo = "7.1 PLAN{str(aniobase)[-2:]}"

#     UNION ALL

#     --RERVISION 24
#     SELECT DIMENSION1, DIMENSION2, `DIMENSION3`,DIMENSION5CACION,
#     Posicion, Orden,
#     `DIMENSION6_Nom`, '1 APROB REV{str(aniobase)[-2:]}' as `Periodo`, DIMENSION6, `HC`

#     FROM {tabla_hcrev_eract}

#     WHERE PERIODO = '5.1 REV{str(aniobase)[-2:]}'

#     UNION ALL

#     --Últimos dos periodos
#     SELECT `DIMENSION1`, `DIMENSION2`, `DIMENSION3`,CASE WHEN T0.DIMENSION1 IN ("NEGOCIO X y CC Externos","NEGOCIO Y") THEN T0.DES_DIMENSION5 ELSE T0.DIMENSION5 || " " || T0.DES_DIMENSION5 END as DIMENSION5CACION,
#     T0.`Posicion`,
#     T2.Orden,PF_CO_OP as `DIMENSION6_Nom`,
#     CASE WHEN Anio = DATE({aniobase-1},12,1) THEN '2 REAL {str(aniobase-1)[-2:]}'
#         WHEN Anio = DATE({aniobase},6,1) THEN '3 HC JUN{str(aniobase)[-2:]}' END as
#     `Periodo`,
#     'Reales' as `DIMENSION6`,
#     COUNT(num_per)`HC`

#     FROM `{project_id}.human_capital.HC_HISTORICO_ATRIBUTOS_CECO` T0
#     CROSS JOIN (SELECT MAX(anio)`Ult_Fecha` FROM `{project_id}.human_capital.FAC_FYA_HC_TRN` )
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T0.POSICION=T2.POSICION

#     WHERE
#     (Anio = DATE({aniobase-1},12,1) OR Anio = DATE({aniobase},6,1))

#     GROUP BY 1,2,3,4,5,6,7,Anio
# ),

# REPORTE AS (

# SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION, POSICION, ORDEN, DIMENSION6_NOM,

# CASE WHEN PERIODO = '1 APROB REV{str(aniobase)[-2:]}' THEN '1 REV{str(aniobase)[-2:]}'
#     WHEN PERIODO = '2 REAL {str(aniobase-1)[-2:]}' THEN '2 REAL{str(aniobase-1)[-2:]}'
#     WHEN PERIODO = '3 HC JUN{str(aniobase)[-2:]}' THEN '3 JUN{str(aniobase)[-2:]}'
#     WHEN PERIODO = '4 HC JUL{str(aniobase)[-2:]}' THEN '4 JUL{str(aniobase)[-2:]}'
#     WHEN PERIODO = '5 CAMBIOS PPTO ME' THEN '5 CN_HC ME'
#     WHEN PERIODO = '6 APROB {str(aniobase)[-2:]}' THEN '6 PPTO{str(aniobase)[-2:]}'
#     WHEN PERIODO = '7 CAMBIOS PLAN' THEN '7 CN_HC PLAN'  END as PERIODO,

# DIMENSION6, IFNULL(HC,0) as HC

# FROM HC1

# UNION ALL

# --DIFF ANIO ANT

# SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION, POSICION, ORDEN, DIMENSION6_NOM,
# '2.1 REAL{str(aniobase-1)[-2:]} vs REV{str(aniobase)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('1 APROB REV{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END -
#  CASE WHEN PERIODO IN ('2 REAL {str(aniobase-1)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# --FINAL ME

# UNION ALL

# SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION, POSICION, ORDEN, DIMENSION6_NOM,
# '5.1 ME{str(aniobase)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('4 HC JUL{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END +
#  CASE WHEN PERIODO IN ('5 CAMBIOS PPTO ME') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# -- DIF APROB ME

# UNION ALL

# SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION, POSICION, ORDEN, DIMENSION6_NOM,
# '6.1 ME{str(aniobase)[-2:]} vs REV{str(aniobase)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('4 HC JUL{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END +
#  CASE WHEN PERIODO IN ('5 CAMBIOS PPTO ME') THEN IFNULL(HC,0) ELSE 0 END -
#  CASE WHEN PERIODO IN ('1 APROB REV{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# --FINAL PLAN

# UNION ALL

# SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION, POSICION, ORDEN, DIMENSION6_NOM,
# '7.1 PLAN {str(aniobase+1)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('4 HC JUL{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END +
#  CASE WHEN PERIODO IN ('5 CAMBIOS PPTO ME') THEN IFNULL(HC,0) ELSE 0 END +
#  CASE WHEN PERIODO IN ('7 CAMBIOS PLAN') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# -- DIF APROB PLAN

# UNION ALL

# SELECT DIMENSION1, DIMENSION2, DIMENSION3,DIMENSION5CACION, POSICION, ORDEN, DIMENSION6_NOM,
# '7.2 PLAN{str(aniobase+1)[-2:]} vs ME{str(aniobase)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('7 CAMBIOS PLAN') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# )

# SELECT *

# FROM REPORTE

# WHERE HC <> 0

# '''

# else:
#   if params['aniobase']-1 == 2024 :
#     tabla_hcejer_ant = "{project_id}.ME24_PPTO25.HC_ME_PPTO2025"
#   else:
#     tabla_hcejer_ant = f"{client.project}.ME_PPTO{str(params['aniobase']-1)}.vHC_r"
#   tabla_hcrev_eract = f"{client.project}.REV{str(params['aniobase'])}.vHC_r"
#   queryhcr = f'''--
#   WITH HC1 AS (
#     --REV
#     SELECT T1.DIMENSION1, T1.DIMENSION2, T1.DIMENSION3,
#     T0.Posicion, T2.Orden, T1.PF_CO_OP as `DIMENSION6_Nom`, '5 CAMBIOS PLAN' as `Periodo`,
#     T0.DIMENSION6, T0.Personas as `HC`

#     FROM (SELECT * FROM `{tabla_base_hc}*` WHERE CECO IS NOT NULL AND CECO <> "") T0
#     LEFT JOIN `{project_id}.mus_qas_drv_datos_maestros.NW_CECO3` T1 ON T0.CECO=T1.CECO
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T0.POSICION=T2.POSICION

#     UNION ALL

#     --ULTIMO MES DEL AÑO CORRIENTE
#     SELECT `DIMENSION1`, `DIMENSION2`, `DIMENSION3`,
#     T0.`Posicion`,  T2.Orden, PF_CO_OP as `DIMENSION6_Nom`, '4 HC FEB{str(aniobase)[-2:]}' as `Periodo`,
#     'Reales' as `DIMENSION6`, COUNT(num_per)`HC`

#     FROM `{project_id}.human_capital.HC_HISTORICO_ATRIBUTOS_CECO` T0
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T0.POSICION=T2.POSICION

#     WHERE EXTRACT(MONTH from ANIO) = {mesbasehc} AND EXTRACT(YEAR from ANIO) = {aniobase}

#     GROUP BY 1,2,3,4,5,6

#     UNION ALL

#     --PRESUPUESTO AÑO CORRIENTE
#     SELECT DIMENSION1, DIMENSION2, DIMENSION3,
#     T0.Posicion, Orden,
#     DIMENSION6_Nom, '6 APROB {str(aniobase)[-2:]}' `Periodo`, `DIMENSION6`, `HC`

#     FROM {tabla_hcejer_ant} T0

#     WHERE Periodo = "7.1 PLAN {str(aniobase)[-2:]}"

#     UNION ALL

#     --ME AÑO ANTERIOR
#     SELECT DIMENSION1, DIMENSION2, DIMENSION3,
#     T0.Posicion, Orden,
#     DIMENSION6_Nom, '1 APROB ME{str(aniobase-1)[-2:]}' `Periodo`, `DIMENSION6`, `HC`

#     FROM {tabla_hcejer_ant} T0

#     WHERE Periodo = "5.1 ME{str(aniobase-1)[-2:]}"

#     UNION ALL

#     --Últimos dos periodos
#     SELECT `DIMENSION1`,`DIMENSION2`, `DIMENSION3`,
#     T1.`Posicion`,
#     T2.Orden,PF_CO_OP as `DIMENSION6_Nom`,
#     CASE WHEN Anio = DATE({aniobase-1},12,1) THEN '2 REAL {str(aniobase-1)[-2:]}'
#         WHEN Anio = DATE({aniobase},1,1) THEN '3 HC ENE{str(aniobase)[-2:]}' END as
#     `Periodo`,
#     'Reales' as `DIMENSION6`,
#     COUNT(num_per)`HC`

#     FROM `{project_id}.human_capital.HC_HISTORICO_ATRIBUTOS_CECO` T1
#     CROSS JOIN (SELECT MAX(anio)`Ult_Fecha` FROM `{project_id}.human_capital.FAC_FYA_HC_TRN` )
#     LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_NOMINA_POSICION_CUENTA` T2 ON T1.POSICION=T2.POSICION

#     WHERE
#     Anio = DATE({aniobase-1},12,1) OR Anio = DATE({aniobase},1,1)

#     GROUP BY 1,2,3,4,5,ANIO
# )

# SELECT DIMENSION1, DIMENSION2, DIMENSION3, POSICION, ORDEN, DIMENSION6_NOM,

# CASE WHEN PERIODO = '1 APROB ME{str(aniobase-1)[-2:]}' THEN '1 ME{str(aniobase-1)[-2:]}'
#     WHEN PERIODO = '2 REAL {str(aniobase-1)[-2:]}' THEN '2 REAL{str(aniobase-1)[-2:]}'
#     WHEN PERIODO = '3 HC ENE{str(aniobase)[-2:]}' THEN '3 ENE{str(aniobase)[-2:]}'
#     WHEN PERIODO = '4 HC FEB{str(aniobase)[-2:]}' THEN '4 FEB{str(aniobase)[-2:]}'
#     WHEN PERIODO = '5 CAMBIOS PLAN' THEN '5 CN_HC'
#     WHEN PERIODO = '6 APROB {str(aniobase)[-2:]}' THEN '6 PPTO{str(aniobase)[-2:]}'
# END as PERIODO,

# DIMENSION6, IFNULL(HC,0) as HC

# FROM HC1

# UNION ALL

# --DIFF ANIO ANT

# SELECT DIMENSION1, DIMENSION2, DIMENSION3, POSICION, ORDEN, DIMENSION6_NOM,
# '2.1 REAL{str(aniobase-1)[-2:]} vs ME{str(aniobase-1)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('1 APROB ME{str(aniobase-1)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END -
#  CASE WHEN PERIODO IN ('2 REAL {str(aniobase-1)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# --FINAL ME

# UNION ALL

# SELECT DIMENSION1, DIMENSION2, DIMENSION3, POSICION, ORDEN, DIMENSION6_NOM,
# '5.1 REV{str(aniobase)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('4 HC FEB{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END +
#  CASE WHEN PERIODO IN ('5 CAMBIOS PLAN') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1

# -- DIF APROB ME

# UNION ALL

# SELECT DIMENSION1, DIMENSION2, DIMENSION3, POSICION, ORDEN, DIMENSION6_NOM,
# '6.1 REV{str(aniobase)[-2:]} vs PPTO{str(aniobase)[-2:]}' as PERIODO,

#  DIMENSION6,

#  CASE WHEN PERIODO IN ('4 HC FEB{str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END +
#  CASE WHEN PERIODO IN ('5 CAMBIOS PLAN') THEN IFNULL(HC,0) ELSE 0 END -
#  CASE WHEN PERIODO IN ('6 APROB {str(aniobase)[-2:]}') THEN IFNULL(HC,0) ELSE 0 END
#   HC

# FROM HC1
# '''

# hcreport_v.view_query = queryhcr

# try:
#     view = client.create_table(hcreport_v)
#     print(f"Created {view.table_type}: {str(view.reference)}")
# except:
#   view = client.update_table(hcreport_v, ["view_query"])
#   print(f"Updated {view.table_type}: {str(view.reference)}")

# **VISTA REPORTE CALCULOS HEADCOUNT**

⭐ Reporte que muestra el cálculo de nómina, por "capas", base - conceptos.

In [ ]:
query_hc_calculos = f''' WITH
  CALCULOS_NOM AS (
  #NOMINA CN
  SELECT T0.Cuenta,
  DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5, CECO,
  T0.Ejercicio, T0.Periodo, T0.Version, Importe,
  Personas,Ejercicio_DE_INGRESO, MES_DE_INGRESO,
     ORIGEN, Posicion, '' as Grupo, '' as Concepto,
  DIMENSION6 as DIMENSION6HC

  FROM {tabla_base_hccn_sp} T0

  UNION ALL

  SELECT
  T0.Cuenta,
  DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5, CECO,
  T0.Ejercicio, T0.Periodo, T0.Version, Importe,
  Personas,Ejercicio_DE_INGRESO, MES_DE_INGRESO,
  CASE
    WHEN T0.CUENTA LIKE '51000403%' THEN 'Bono'
    WHEN  Clasificacion LIKE '%SUELDO%' THEN 'Sueldo'
    ELSE 'Prestaciones'
    END as   ORIGEN, Posicion, '' as Grupo, '' as Concepto,
  DIMENSION6 as DIMENSION6HC
  FROM {nomina_etz_cn} T0
  LEFT JOIN `{CUENTAS_NOM}` T1 ON SUBSTRING(T0.Cuenta,1,8)=T1.Cuenta_origen

  UNION ALL

  SELECT T0.Cuenta,
  DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5, CECO,
  T0.Ejercicio, T0.Periodo, T0.Version, Importe,
  Personas,Ejercicio_DE_INGRESO, MES_DE_INGRESO,
     ORIGEN, Posicion, '' as Grupo, '' as Concepto,
  DIMENSION6 as DIMENSION6HC

  FROM {nomina_calculo} T0

  UNION ALL

  SELECT
  T0.Cuenta,
  DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5, CECO,
  T0.Ejercicio, T0.Periodo, T0.Version, Importe,
  0 as Personas,0 as Ejercicio_DE_INGRESO, 0 as MES_DE_INGRESO,

    ORIGEN, 'Base' Posicion, '' as Grupo, '' as Concepto,
  'Base' as DIMENSION6HC
  FROM {tabla_atributos_calc_inic} T0
  WHERE ORIGEN LIKE 'NOM_%'
  )

  SELECT T0.*

  FROM CALCULOS_NOM T0

  --WHERE DIMENSION1 IN UNNEST ({direcciones_nomina})

  # UNION ALL

  # SELECT T0.Cuenta,
  # DIMENSION1, DIMENSION2, DIMENSION3, DIMENSION6_MAESTRO, DIMENSION5,	DES_DIMENSION5, CECO,
  # SAFE_CAST(T0.Ejercicio AS STRING) AS EJERCICIO, T0.Periodo, CASE WHEN T0.Ejercicio = {aniobase} THEN '{params['version'][0]}' ELSE T0.Version END as Version, Importe,
  # 0 as Personas,0 as Ejercicio_DE_INGRESO, 0 as MES_DE_INGRESO,
  # ORIGEN,
  # 'Base' as Posicion, '' as Grupo, '' as Concepto,
  # 'Base' as DIMENSION6HC
  # FROM {base_primerosmesesrealesnom} T0

  # WHERE DIMENSION1 IN UNNEST ({direcciones_nomina})
'''

hcreportcalc_v = bigquery.Table(hcreportcalc)
hcreportcalc_v.view_query = query_hc_calculos

try:
    view = client.create_table(hcreportcalc_v)
    print(f"Created {view.table_type}: {str(view.reference)}")
except:
  view = client.update_table(hcreportcalc_v, ["view_query"])
  print(f"Updated {view.table_type}: {str(view.reference)}")

# **VISTA FINAL (STAKEHOLDERS)**

# ⏰ **MODELO DE LARGO PLAZO**
⭐ El modelo de largo plazo debe ser una herramienta que permita a los diferentes negocios, inputar variables, que generen el ER a 10 años, basado en premisas.

La "última capa" es la correspondiente a aperturas.

# ⏰ **BALANCE GENERAL**
⭐ Se requieren las balanzas de comprobación, y la jerarquía de cuentas de balance

Se genera forecast para cada rubro, algunos son proporcionados por otras áreas

# ⏰ **FLUJO DE EFECTIVO**
⭐ Una vez generado ER y BG, se realiza el flujo de efectivo, el cuál muestra las entradas y salidas de efectivo

# ▶ **Automatización de procesos "Adicionales"**

# **VENTAS**

⭐ Se proporcionará en Excel, formato*

 * Trabajan en Sheets y GCP


❗ Se corre en dos partes:
  1. Generación de archivos para enviar a negocios
  2. Carga de archivos compartidos por negocios (respetar carpetas)

In [ ]:
# %%bigquery borrado


# CREATE OR REPLACE TABLE {project_id}.DATASET_XXX.Negocio9_v2
# AS SELECT DIMENSION1,DIMENSION3,DIMENSION5,DESCDIMENSION5,DIMENSION6,MT,DIMENSION7,Periodo,Importe,Cuenta,Ejercicio,VERSION,UDATE
#  FROM  {project_id}.DATASET_XXX.Negocio9;


# update {project_id}.DATASET_XXX.Negocio9_v2
# SET VERSION = 'PPTO_V2_v2'
# WHERE EJERCICIO = 2025;


# update {project_id}.DATASET_XXX.Negocio9_v2
# SET VERSION = 'PLAN_v2'
# WHERE EJERCICIO = 2026;


# CREATE OR REPLACE TABLE {project_id}.DATASET_XXX.Negocio8_v2
# AS SELECT DIMENSION1,DIMENSION3,DIMENSION5,DESCDIMENSION5,DIMENSION6,MT,DIMENSION7,Periodo,Importe,Cuenta,Ejercicio,VERSION,UDATE
#  FROM  {project_id}.DATASET_XXX.Negocio8;


# update {project_id}.DATASET_XXX.Negocio8_v2
# SET VERSION = 'PPTO_V2_v2'
# WHERE EJERCICIO = 2025;


# update {project_id}.DATASET_XXX.Negocio8_v2
# SET VERSION = 'PLAN_v2'
# WHERE EJERCICIO = 2026;

# CREATE OR REPLACE TABLE {project_id}.DATASET_XXX.Negocio10
# AS SELECT DIMENSION1,DIMENSION3,DIMENSION5,DESCDIMENSION5,DIMENSION6,MT,DIMENSION7,Periodo,Importe,Cuenta,Ejercicio,VERSION,UDATE
#  FROM  {project_id}.DATASET_XXX.Negocio10;


# update {project_id}.DATASET_XXX.Negocio10
# SET VERSION = 'PPTO_V2_v2'
# WHERE EJERCICIO = 2025;


# update {project_id}.DATASET_XXX.Negocio10
# SET VERSION = 'PLAN_v2'
# WHERE EJERCICIO = 2026;

#  CREATE OR REPLACE TABLE {project_id}.DATASET_XXX.Negocio0_v2
#  AS SELECT DIMENSION1,DIMENSION3,DIMENSION5,DESCDIMENSION5,DIMENSION6,MT,DIMENSION7,Periodo,Importe,Cuenta,Ejercicio,VERSION,UDATE
#  FROM  {project_id}.DATASET_XXX.Negocio0;


#  update {project_id}.DATASET_XXX.Negocio0_v2
#  SET VERSION = 'PPTO_V2_v2'
#  WHERE EJERCICIO = 2025;


#  update {project_id}.DATASET_XXX.Negocio0_v2
#  SET VERSION = 'PLAN_v2'
#  WHERE EJERCICIO = 2026;

In [ ]:
archivo_ventas = os.path.join(rutaexcelventas, "Ventas.xlsx")
# names = ["Negocio8","Negocio9","Negocio10"]

names = ["Negocio0","Negocio11","Negocio8","Negocio9","Negocio12"]
#names = ["Negocio8"]
# filtros = [["Negocio0"],["Negocio11"],["Negocio8","Negocio9","Negocio10"]]

my_red = openpyxl.styles.colors.Color(rgb='999999')
my_fill = openpyxl.styles.fills.PatternFill(patternType='solid', fgColor=my_red)

In [ ]:
####MAJOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
names = ["Negocio0"]#,"Negocio11","Negocio8","Negocio9","Negocio12"]

for i in range(0,len(names)):

  if names[i] == 'Negocio0' :
    filtro = f" AND DIMENSION2 IN ('QQQQ','WWWW','FFFF') AND DIMENSION6 IN ('HHHH','XXXXX','GGGGG','CCCC','VVVV')"
  else :
    filtro = f" AND DIMENSION6 IN ('AAAA','HH','Negocio8')"

  if DIMENSION6pres == "REV":

    query_vts = f'''WITH VTS AS (
    SELECT DIMENSION1,
    CASE WHEN DIMENSION6 = 'VAD' THEN 'Digital' ELSE DIMENSION2 END AS DIMENSION2,
    CASE WHEN MT = '2023' THEN 'MT' ELSE MT END AS MT,
    CASE WHEN Periodo = 1 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ENE,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as FEB,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAR,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ABR,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAY,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUN,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUL,
    CASE WHEN Periodo = 8 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as AGO,
    CASE WHEN Periodo = 9 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as SEP,
    CASE WHEN Periodo = 10 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as OCT,
    CASE WHEN Periodo = 11 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as NOV,
    CASE WHEN Periodo = 12 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as DIC,
    CASE WHEN T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as VENTAS,

    CASE WHEN Periodo = 1 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ENE_c,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as FEB_c,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAR_c,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ABR_c,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAY_c,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUN_c,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUL_c,
    CASE WHEN Periodo = 8 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as AGO_c,
    CASE WHEN Periodo = 9 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as SEP_c,
    CASE WHEN Periodo = 10 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as OCT_c,
    CASE WHEN Periodo = 11 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as NOV_c,
    CASE WHEN Periodo = 12 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as DIC_c,
    CASE WHEN T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as COSTO

    FROM {project_id}.data_warehouse_finance.accounting_journal T0
    LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

    WHERE Version = 'PPTO' AND Ejercicio = {aniobase} AND DIMENSION1 IN ('{names[i]}') AND IMPORTE <>0
    AND T1.P_L IN ('Costo de Ventas','Ventas Totales') {filtro}
    )

    SELECT DIMENSION1, DIMENSION2, MT,
    SUM(VENTAS) as VENTA

    FROM VTS

    GROUP BY 1,2,3
    ORDER BY 2,3
    '''
  else:
    query_vts = f'''WITH VTS AS (
    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    CASE WHEN Periodo = 1 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ENE,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as FEB,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAR,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ABR,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAY,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUN,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUL,

    CASE WHEN Periodo = 1 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ENE_c,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as FEB_c,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAR_c,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ABR_c,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAY_c,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUN_c,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUL_c

    FROM {project_id}.data_warehouse_finance.accounting_journal T0
    LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

    WHERE Version = 'Reales' AND Ejercicio = {aniobase} AND Periodo <= {mesbaser} AND DIMENSION1 IN ('{names[i]}') AND IMPORTE <>0
    AND T1.P_L IN ('Costo de Ventas','Ventas Totales') {filtro}
    )

    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    SUM(ENE) as ENE,
    SUM(FEB) as FEB,
    SUM(MAR) as MAR,
    SUM(ABR) as ABR,
    SUM(MAY) as MAY,
    SUM(JUN) as JUN,
    SUM(JUL) as JUL,
    "" as AGO,
    "" as SEP,
    "" as OCT,
    "" as NOV,
    "" as DIC,
    "" as SUBTOTAL1,
    SUM(ENE_c) as ENE_c,
    SUM(FEB_c) as FEB_c,
    SUM(MAR_c) as MAR_c,
    SUM(ABR_c) as ABR_c,
    SUM(MAY_c) as MAY_c,
    SUM(JUN_c) as JUN_c,
    SUM(JUL_c) as JUL_c,
    "" as AGO_c,
    "" as SEP_c,
    "" as OCT_c,
    "" as NOV_c,
    "" as DIC_c,
    "" as SUBTOTAL2

    FROM VTS

    GROUP BY 1,2,3,4,5,6,7
    ORDER BY 2,3
    '''

  print(query_vts)


❗❗❗❗❗❗ IGNORA!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

❗NO ME CORRAS ❗

Generación de archivos "carátulas de ventas"

In [ ]:
for i in range(0,len(names)):
  newfile = f"{rutaventas}/Carátula de Ventas {names[i]}.xlsx"
  try:
    archivo = load_workbook(filename=newfile)
    sheet = archivo.active
    print("Archivo cargado.")
    maxrow = 0
    for cell in sheet["A"]:
      if not cell.value is np.empty:
        maxrow = maxrow +1
    sheet.delete_rows(idx=3, amount=maxrow-2)
    print(f"{maxrow-2} filas borradas")
  except FileNotFoundError:
    shutil.copy(archivo_ventas,newfile)
    print("Archivo creado.")

  if names[i] == 'Negocio0' :
    filtro = f" AND DIMENSION2 IN ('TTTT','TTTTTTT','TTTTT') AND DIMENSION6 IN ('FFFFFF','FFFFFFF','FFFFFFFFFF','FFFFFFFFFF','FFFFFF')"
  else :
    filtro = f" AND DIMENSION6 IN ('FFFFFF','FFFFFFFFFF','Negocio8')"

  if DIMENSION6pres == "REV":

    query_vts = f'''WITH VTS AS (
    SELECT DIMENSION1,
    CASE WHEN DIMENSION6 = 'DSFS' THEN 'FSFSFSFS' ELSE DIMENSION2 END AS DIMENSION2,
    CASE WHEN DIMENSION88 = 'SFSFFS' THEN 'FSFSFS' ELSE DIMENSION88 END AS DIMENSION88,
    CASE WHEN Periodo = 1 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ENE,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as FEB,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAR,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ABR,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAY,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUN,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUL,
    CASE WHEN Periodo = 8 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as AGO,
    CASE WHEN Periodo = 9 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as SEP,
    CASE WHEN Periodo = 10 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as OCT,
    CASE WHEN Periodo = 11 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as NOV,
    CASE WHEN Periodo = 12 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as DIC,

    CASE WHEN Periodo = 1 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ENE_c,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as FEB_c,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAR_c,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ABR_c,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAY_c,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUN_c,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUL_c,
    CASE WHEN Periodo = 8 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as AGO_c,
    CASE WHEN Periodo = 9 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as SEP_c,
    CASE WHEN Periodo = 10 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as OCT_c,
    CASE WHEN Periodo = 11 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as NOV_c,
    CASE WHEN Periodo = 12 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as DIC_c

    FROM {project_id}.data_warehouse_finance.accounting_journal T0
    LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

    WHERE Version = 'PPTO' AND Ejercicio = {aniobase} AND DIMENSION1 IN ('{names[i]}') AND IMPORTE <>0
    AND T1.P_L IN ('Costo de Ventas','Ventas Totales') {filtro}
    )

    SELECT DIMENSION1, DIMENSION2, MT,
    SUM(ENE) as ENE,
    SUM(FEB) as FEB,
    SUM(MAR) as MAR,
    SUM(ABR) as ABR,
    SUM(MAY) as MAY,
    SUM(JUN) as JUN,
    SUM(JUL) as JUL,
    SUM(AGO) as AGO,
    SUM(SEP) as SEP,
    SUM(OCT) as OCT,
    SUM(NOV) as NOV,
    SUM(DIC) as DIC,
    "" as SUBTOTAL1,
    SUM(ENE_c) as ENE_c,
    SUM(FEB_c) as FEB_c,
    SUM(MAR_c) as MAR_c,
    SUM(ABR_c) as ABR_c,
    SUM(MAY_c) as MAY_c,
    SUM(JUN_c) as JUN_c,
    SUM(JUL_c) as JUL_c,
    SUM(AGO_c) as AGO_c,
    SUM(SEP_c) as SEP_c,
    SUM(OCT_c) as OCT_c,
    SUM(NOV_c) as NOV_c,
    SUM(DIC_c) as DIC_c,
    "" as SUBTOTAL2

    FROM VTS

    GROUP BY 1,2,3
    ORDER BY 2,3
    '''
  else:
    query_vts = f'''WITH VTS AS (
    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    CASE WHEN Periodo = 1 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ENE,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as FEB,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAR,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ABR,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAY,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUN,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUL,

    CASE WHEN Periodo = 1 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ENE_c,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as FEB_c,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAR_c,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ABR_c,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAY_c,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUN_c,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUL_c

    FROM {project_id}.data_warehouse_finance.accounting_journal T0
    LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

    WHERE Version = 'Reales' AND Ejercicio = {aniobase} AND Periodo <= {mesbaser} AND DIMENSION1 IN ('{names[i]}') AND IMPORTE <>0
    AND T1.P_L IN ('Costo de Ventas','Ventas Totales') {filtro}
    )

    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    SUM(ENE) as ENE,
    SUM(FEB) as FEB,
    SUM(MAR) as MAR,
    SUM(ABR) as ABR,
    SUM(MAY) as MAY,
    SUM(JUN) as JUN,
    SUM(JUL) as JUL,
    "" as AGO,
    "" as SEP,
    "" as OCT,
    "" as NOV,
    "" as DIC,
    "" as SUBTOTAL1,
    SUM(ENE_c) as ENE_c,
    SUM(FEB_c) as FEB_c,
    SUM(MAR_c) as MAR_c,
    SUM(ABR_c) as ABR_c,
    SUM(MAY_c) as MAY_c,
    SUM(JUN_c) as JUN_c,
    SUM(JUL_c) as JUL_c,
    "" as AGO_c,
    "" as SEP_c,
    "" as OCT_c,
    "" as NOV_c,
    "" as DIC_c,
    "" as SUBTOTAL2

    FROM VTS

    GROUP BY 1,2,3,4,5,6,7
    ORDER BY 2,3
    '''

  # print(query_vts)
  bigquery_result = bpd.read_gbq(query_vts)
  df_bigquery_result = pd.DataFrame(bigquery_result)


  if len(df_bigquery_result) > 0 :
    df_bigquery_result_sorted = df_bigquery_result.sort_values(by=[1, 2])
    for row_num, row_data in enumerate(df_bigquery_result_sorted.itertuples(index=False), start=3):
              for col_num, cell_value in enumerate(row_data, start=1):
                  sheet.cell(row=row_num, column=col_num, value=cell_value)
                  sheet.cell(row=row_num, column=col_num).numBONO_ESP_format = '#,##0.00'
                  if col_num < mes_captura+3 and col_num > 3 or col_num < mes_captura+16 and col_num > 16:
                    sheet.cell(row=row_num, column=col_num).fill = my_fill
                  sheet[f"P{row_num}"] = f"=SUM(D{row_num}:O{row_num})"
                  sheet[f"AC{row_num}"] = f"=SUM(Q{row_num}:AA{row_num})"
                  sheet[f"AD{row_num}"] = f"=IFERROR(1-(AC{row_num}/P{row_num})*-1,0)"
                  sheet.cell(row=row_num, column=34).numBONO_ESP_format = '0.00%'

  print(f'''Información pegada, {len(df_bigquery_result)}''')
  if DIMENSION6pres == "REV":
    sheet.delete_cols(idx=35, amount=27)
    sheet["T1"] = "REV"
  archivo.save(filename=newfile)

❗ CÓRREME ❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗

⭐ Lectura, validación y carga

In [ ]:
# names = ["Negocio0"]
drive.mount('/content/drive', force_remount=True)

for i in range(0,len(names)):
  newfile = f"{rutaventas}/Archivos correctos/Carátula de Ventas {names[i]}.xlsx"
  try:
    dfarchivo = pd.read_excel(newfile, sheet_name="Captura", header=1, dtype=str, engine='openpyxl')
    print("Archivo leído", newfile)
    print(len(dfarchivo))
  except FileNotFoundError:
    print("Archivo no encontrado.")
    continue

  if names[i] == 'Negocio0' :
    filtro = f" AND DIMENSION2 IN ('Operaciones','Restaurante','Digital') AND DIMENSION6 IN ('BARRA','EXP GOURMET','RESTAURANTE','TIENDA','VAD')"
  else :
    filtro = f" AND DIMENSION6 IN ('TIENDA','VAD','Negocio8')"

  if DIMENSION6pres == "REV":
    query_vts = f'''WITH VTS AS (
    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    CASE WHEN Periodo = 1 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ENE,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as FEB,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAR,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ABR,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAY,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUN,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUL,
    CASE WHEN Periodo = 8 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as AGO,
    CASE WHEN Periodo = 9 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as SEP,
    CASE WHEN Periodo = 10 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as OCT,
    CASE WHEN Periodo = 11 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as NOV,
    CASE WHEN Periodo = 12 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as DIC,

    CASE WHEN Periodo = 1 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ENE_c,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as FEB_c,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAR_c,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ABR_c,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAY_c,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUN_c,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUL_c,
    CASE WHEN Periodo = 8 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as AGO_c,
    CASE WHEN Periodo = 9 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as SEP_c,
    CASE WHEN Periodo = 10 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as OCT_c,
    CASE WHEN Periodo = 11 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as NOV_c,
    CASE WHEN Periodo = 12 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as DIC_c

    FROM {project_id}.data_warehouse_finance.accounting_journal T0
    LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

    WHERE Version = 'PPTO' AND Ejercicio = {aniobase} AND DIMENSION1 IN ('{names[i]}') AND IMPORTE <>0
    AND T1.P_L IN ('Costo de Ventas','Ventas Totales') {filtro} AND Periodo <= {mes_captura}
    )

    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    SUM(ENE) as ENE,
    SUM(FEB) as FEB,
    SUM(MAR) as MAR,
    SUM(ABR) as ABR,
    SUM(MAY) as MAY,
    SUM(JUN) as JUN,
    SUM(JUL) as JUL,
    SUM(AGO) as AGO,
    SUM(SEP) as SEP,
    SUM(OCT) as OCT,
    SUM(NOV) as NOV,
    SUM(DIC) as DIC,
    SUM(ENE_c) as ENE_c,
    SUM(FEB_c) as FEB_c,
    SUM(MAR_c) as MAR_c,
    SUM(ABR_c) as ABR_c,
    SUM(MAY_c) as MAY_c,
    SUM(JUN_c) as JUN_c,
    SUM(JUL_c) as JUL_c,
    SUM(AGO_c) as AGO_c,
    SUM(SEP_c) as SEP_c,
    SUM(OCT_c) as OCT_c,
    SUM(NOV_c) as NOV_c,
    SUM(DIC_c) as DIC_c

    FROM VTS

    GROUP BY 1,2,3,4,5,6,7
    ORDER BY 2,3
    '''
  else:
    query_vts = f'''WITH VTS AS (
    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    CASE WHEN Periodo = 1 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ENE,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as FEB,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAR,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as ABR,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as MAY,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUN,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Ventas Totales' THEN IMPORTE ELSE 0 END as JUL,

    CASE WHEN Periodo = 1 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ENE_c,
    CASE WHEN Periodo = 2 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as FEB_c,
    CASE WHEN Periodo = 3 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAR_c,
    CASE WHEN Periodo = 4 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as ABR_c,
    CASE WHEN Periodo = 5 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as MAY_c,
    CASE WHEN Periodo = 6 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUN_c,
    CASE WHEN Periodo = 7 AND T1.P_L = 'Costo de Ventas' THEN IMPORTE ELSE 0 END as JUL_c

    FROM {project_id}.data_warehouse_finance.accounting_journal T0
    LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.Cuentas T1 ON T0.CUENTA=T1.CUENTA

    WHERE Version = 'Reales' AND Ejercicio = {aniobase} AND Periodo <= {mesbaser} AND DIMENSION1 IN ('{names[i]}') AND IMPORTE <>0
    AND T1.P_L IN ('Costo de Ventas','Ventas Totales') {filtro}
    )

    SELECT DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    SUM(ENE) as ENE,
    SUM(FEB) as FEB,
    SUM(MAR) as MAR,
    SUM(ABR) as ABR,
    SUM(MAY) as MAY,
    SUM(JUN) as JUN,
    SUM(JUL) as JUL,

    SUM(ENE_c) as ENE_c,
    SUM(FEB_c) as FEB_c,
    SUM(MAR_c) as MAR_c,
    SUM(ABR_c) as ABR_c,
    SUM(MAY_c) as MAY_c,
    SUM(JUN_c) as JUN_c,
    SUM(JUL_c) as JUL_c

    FROM VTS

    GROUP BY 1,2,3,4,5,6,7
    ORDER BY 2,3
    '''

  # bigquery_result = bpd.read_gbq(query_vts)
  # bigquery_result_header = bigquery_result.columns
  # df_bigquery_result = pd.DataFrame(bigquery_result)
  # df_bigquery_result.columns = bigquery_result_header

  # dfarchivo.columns = dfarchivo.columns.astype(str)

  # dfarchivodf = dfarchivo.loc[:, ['División','Dir. Área','DIMENSION5cación','Descripción DIMENSION5cación','DIMENSION6','MT','DIMENSION7'
  #               ,'1','2','3','1.1','2.1','3.1']]

  # dfarchivodf = dfarchivodf.rename(columns={'División':'DIMENSION1','Dir. Área':'DIMENSION3','DIMENSION5cación':'DIMENSION5','Descripción DIMENSION5cación':'DESCDIMENSION5',
  #                                       '1':'ENE','2':'FEB','3':'MAR','1.1':'ENE_c','2.1':'FEB_c','3.1':'MAR_c','DIMENSION6':'DIMENSION6','DIMENSION7':'DIMENSION7'})

  # df = dfarchivodf.merge(df_bigquery_result, on=['DIMENSION1','DIMENSION3','DIMENSION5','DIMENSION6','MT','DIMENSION7'], how='outer')
  # def same_merge(x): return ','.join(x[x.notnull()].astype(str))
  # df_new = df.groupby(level=0, axis=1).apply(lambda x: x.apply(same_merge, axis=1))

  # for k in range (0,6):
  #   df.iloc[:,14+k] = df.iloc[:,14+k].astype(float)
  #   df.iloc[:,7+k] = df.iloc[:,7+k].astype(float)
  #   df.loc[:,f"Diff {str(df.columns[k+7])[0:5]}"] = df.iloc[:,7+k]-df.iloc[:,14+k]
  # for k in range (0,6):
  #   if k == 0:
  #     df.loc[:,f"Diff"] = df.iloc[:,20+k]
  #   else:
  #     df.loc[:,f"Diff"] = df.loc[:,f"Diff"]+df.iloc[:,20+k]

  # print(f"VALIDACION {names[i]}: ",round(df['Diff'].sum(),0))

  # if round(df['Diff'].sum(),0) == np.empty :
  if len(dfarchivo) < 0 :
  #  round(df['Diff'].sum(),0) == 0 :
    print(f"{names[i]}, no cargado, hay diferencia en reales")
    continue
  else:
    if DIMENSION6pres == "REV":
      dfarchivo.columns = dfarchivo.columns.astype(str)
      carga = pd.melt(dfarchivo, id_vars=['División','Dir. Área','DIMENSION5cación','Descripción DIMENSION5cación','DIMENSION6','MT','DIMENSION7'],
                    value_vars=['1','2','3','4','5','6','7','8','9','10','11','12'
                    ,'1.1','2.1','3.1','4.1','5.1','6.1','7.1','8.1','9.1','10.1','11.1','12.1'])
    else:
      dfarchivo.columns = dfarchivo.columns.astype(str)
      carga = pd.melt(dfarchivo, id_vars=['División','Dir. Área','DIMENSION5cación','Descripción DIMENSION5cación','DIMENSION6','MT','DIMENSION7'],
                    value_vars=['1','2','3','4','5','6','7','8','9','10','11','12'
                    ,'1.1','2.1','3.1','4.1','5.1','6.1','7.1','8.1','9.1','10.1','11.1','12.1'
                    ,'1.2','2.2','3.2','4.2','5.2','6.2','7.2','8.2','9.2','10.2','11.2','12.2'
                    ,'1.3','2.3','3.3','4.3','5.3','6.3','7.3','8.3','9.3','10.3','11.3','12.3'])

    print(carga['value'].fillna(0).astype(float).sum())

    carga = carga.rename(columns={'División':'DIMENSION1','Dir. Área':'DIMENSION3','DIMENSION5cación':'DIMENSION5','Descripción DIMENSION5cación':'DESCDIMENSION5',
                                  'DIMENSION6':'DIMENSION6','DIMENSION7':'DIMENSION7'})

    carga.loc[carga['variable'].str.find(".1") != -1,"Cuenta"] = "COSTO"
    carga.loc[carga['variable'].str.find(".") == -1,"Cuenta"] = "VENTA"
    carga.loc[carga['variable'].str.find(".3") != -1,"Cuenta"] = "COSTO"
    carga.loc[carga['variable'].str.find(".2") != -1,"Cuenta"] = "VENTA"

    carga.loc[carga['variable'].str.find(".1") != -1,"Ejercicio"] = aniobase
    carga.loc[carga['variable'].str.find(".") == -1,"Ejercicio"] = aniobase
    carga.loc[carga['variable'].str.find(".3") != -1,"Ejercicio"] = aniobase +1
    carga.loc[carga['variable'].str.find(".2") != -1,"Ejercicio"] = aniobase +1

    carga['variable'] = carga['variable'].str.split(".").str[0]
    carga = carga.rename(columns={'variable':'Periodo','value':'Importe'})
    carga['Importe'] = carga['Importe'].fillna(0)
    carga['Importe'] = carga['Importe'].astype(float)
    carga['Periodo'] = carga['Periodo'].astype(np.int64)
    carga['Ejercicio'] = carga['Ejercicio'].astype(np.int64)

    carga.loc[(carga['Ejercicio'] == aniobase), 'Version'] = params['version'][0]
    carga.loc[(carga['Ejercicio'] == aniobase+1), 'Version'] = params['version'][1]

    ctimecarga = os.path.getmtime(newfile)
    udatecarga = datetime.fromtimestamp(ctimecarga, tz=timezone)
    udatecarga = udatecarga.strftime(dateformat)
    print(udatecarga)

    carga['UDATE'] = udatecarga
    carga['UDATE'] = pd.to_datetime(carga['UDATE'],format=dateformat)

    # print(carga['UDATE'].dtypes, carga['UDATE'].unique().tolist())
    print(carga['Importe'].sum())

    # carga.dtypes

    job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Reemplazar la tabla
    )
    tablacarga = f"{project_datasetbasesventas}.{names[i]}"

    # Carga

    load_job = client.load_table_from_dataframe(
        carga, tablacarga, job_config=job_config
        )

    load_job.result()  # Espera que la carga termine
    destination_table = client.get_table(tablacarga)  # Revisando si la tabla existe
    print("Loaded {} rows.".format(destination_table.num_rows))
    job = client.get_job(load_job)
    print(f"State: {job.state}")

# **QUERY VENTAS**

❗❗❗❗❗❗No correr, solo si se requiere, se debe agregar manualmente a vista final

Vista para desarrollo

In [ ]:
if DIMENSION6pres == "ME_PPTO" :
  query_vts = f'''
    SELECT
    DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    CASE WHEN Cuenta = 'VENTA' THEN
        CASE WHEN DIMENSION1 = 'Negocio0' AND DIMENSION3 = 'VAD Negocio0' OR 'DIMENSION6' = 'VAD' THEN '40000005'
            WHEN DIMENSION1 IN ('Suburiba','Negocio10') AND DIMENSION6 <> 'VAD' THEN '40000000'
            ELSE '40000003' END
        WHEN Cuenta = 'COSTO' THEN
        CASE WHEN DIMENSION1 = 'Negocio0' AND DIMENSION3 = 'VAD Negocio0' OR DIMENSION6 = 'VAD' THEN '50200020'
            WHEN DIMENSION1 = 'Negocio0' AND DIMENSION3 = 'Restaurante' THEN '50200012'
            ELSE '50200013' END
    END as Cuenta,

    "" as Ceco, "" as Cebe, Periodo, Ejercicio,
    --CASE WHEN Ejercicio = {aniobase} THEN '{params['version'][0]}'
    --    WHEN Ejercicio = {aniobase+1} THEN '{params['version'][1]}'
    --    END AS
         Version,
    IMPORTE,
    'CAR_VEN' as ORIGEN, UDATE

    FROM `{project_datasetbasesventas}.*`
    WHERE Importe <>0

  '''

else:
  query_vts = f'''
    SELECT
    DIMENSION1, DIMENSION3, DIMENSION5, DESCDIMENSION5, DIMENSION6, MT, DIMENSION7,
    CASE WHEN Cuenta = 'VENTA' THEN
        CASE WHEN DIMENSION1 = 'Negocio0' AND DIMENSION3 = 'VAD Negocio0' OR 'DIMENSION6' = 'VAD' THEN '40000005'
            WHEN DIMENSION1 IN ('Suburiba','Negocio10') AND DIMENSION6 <> 'VAD' THEN '40000000'
            ELSE '40000003' END
        WHEN Cuenta = 'COSTO' THEN
        CASE WHEN DIMENSION1 = 'Negocio0' AND DIMENSION3 = 'VAD Negocio0' OR DIMENSION6 = 'VAD' THEN '50200020'
            WHEN DIMENSION1 = 'Negocio0' AND DIMENSION3 = 'Restaurante' THEN '50200012'
            ELSE '50200013' END
    END as Cuenta,

    "" as Ceco, "" as Cebe, Periodo, Ejercicio,
    '{params['version'][0]}' AS Version,
    IMPORTE,
    'CAR_VEN' as ORIGEN, UDATE

    FROM `{project_datasetbasesventas}.*`
    WHERE Importe <>0

  '''

In [ ]:
ventas_table = bigquery.Table(ventas_id)
ventas_table.view_query  = query_vts

try:
  view = client.create_table(ventas_table)
  print(f"Created {view.table_type}: {str(view.reference)}")
except:
  view = client.update_table(ventas_table, ["view_query"])
  print(f"Updated {view.table_type}: {str(view.reference)}")

# ➕ **CARGAS FUERA DE MASCARA**

# **Adaptación negocio A**

Base BI

⭐ **TABLAS BASE**

In [ ]:
################TABLA BASE


#Se genera una tabla en blanco para poder generar la vista

schema_Negocio9= [
      bigquery.SchemaField('JERARQUIA', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('EXPECTATIVAS', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('Cuenta', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('DIMENSION1', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('EJERCICIO', 'NUMERIC', mode='NULLABLE'),
      bigquery.SchemaField('PERIODO', 'NUMERIC', mode='NULLABLE'),
      bigquery.SchemaField('VERSION', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('ORIGEN', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('UDATE', 'TIMESTAMP', mode='NULLABLE'),
      bigquery.SchemaField('IDUB', 'STRING', mode='NULLABLE'),
      bigquery.SchemaField('IMPORTE', 'FLOAT', mode='NULLABLE'),
  ]


try:
  table = bigquery.Table(Negocio9_cedulas_sp, schema=schema_Negocio9)
  table_Negocio9_ = client.create_table(table)  # Make an API request.
  print(
      "Created table {}.{}.{}".format(table_Negocio9_.project, table_Negocio9_.dataset_id, table_Negocio9_.table_id)
  )

except:
  client.delete_table(table, not_found_ok=True)  # Make an API request.
  print("Deleted table '{}'.".format(table))
  table = bigquery.Table(Negocio9_cedulas_sp, schema=schema_Negocio9)
  table_Negocio9_ = client.create_table(table)  # Make an API request.
  print(
      "Created table {}.{}.{}".format(table_Negocio9_.project, table_Negocio9_.dataset_id, table_Negocio9_.table_id)
  )

In [ ]:
# Initialize empty fields list
fields = []

cednom = ['Negocio9_CEDULAS']

# --- Logic for Schema Selection ---
for tablaa in cednom :
  table_full_id = f"{project_datasetbases}.{tablaa}"

  if DIMENSION6pres == 'REV':
      if "VENTAS" in cednom:
          fields = [
              {"name": "MARCA", "type": "STRING"}, {"name": "DIMENSION5", "type": "STRING"},
              {"name": "DESC_DIMENSION5", "type": "STRING"}, {"name": "ESTATUS", "type": "STRING"},
              *[{"name": f"{m}_VTA_FIS", "type": "FLOAT"} for m in ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]],
              *[{"name": f"{m}_COS_FIS", "type": "FLOAT"} for m in ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]],
              *[{"name": f"{m}_VTA_DIG", "type": "FLOAT"} for m in ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]],
              *[{"name": f"{m}_COS_DIG", "type": "FLOAT"} for m in ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]],
              {"name": "MANDANTE", "type": "STRING"}, {"name": "TIMESTAMP", "type": "TIMESTAMP"},
              {"name": "PATH", "type": "STRING"}, {"name": "CEDULA", "type": "STRING"}
          ]
      else:
          fields = [
              {"name": "COMENTARIOS", "type": "STRING"},
              {"name": "ORIGEN", "type": "STRING"},
              {"name": "DIMENSION6", "type": "STRING"},
              {"name": "PREMISA", "type": "STRING"},
               {"name": "DIMENSION5", "type": "STRING"},
                {"name": "LASTLEVEL", "type": "STRING"},
              *[{"name": m, "type": "FLOAT"} for m in ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]],
              {"name": "REV", "type": "FLOAT"},
               {"name": "MANDANTE", "type": "STRING"},
              {"name": "TIMESTAMP", "type": "TIMESTAMP"},
               {"name": "PATH", "type": "STRING"},
                {"name": "CEDULA", "type": "STRING"}
          ]
      print("Schema creado PPTO_V1")

  else: # if DIMENSION6pres != 'REV'
      if "VENTAS" in cednom:
          # Complex schema for non-REV Ventas (ME and PLAN fields)
          months = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]
          fields = [{"name": "MARCA", "type": "STRING"}, {"name": "DIMENSION5", "type": "STRING"}, {"name": "DESC_DIMENSION5", "type": "STRING"}, {"name": "ESTATUS", "type": "STRING"}]
          for suffix in ["VTA_FIS_ME", "COS_FIS_ME", "VTA_DIG_ME", "COS_DIG_ME", "VTA_FIS_PLAN", "COS_FIS_PLAN", "VTA_DIG_PLAN", "COS_DIG_PLAN"]:
              fields.extend([{"name": f"{m}_{suffix}", "type": "FLOAT"} for m in months])
          fields.extend([{"name": "MANDANTE", "type": "STRING"}, {"name": "TIMESTAMP", "type": "STRING"}, {"name": "PATH", "type": "STRING"}, {"name": "CEDULA", "type": "STRING"}])

      elif "REV" in cednom:
          months_sp = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"]
          fields = [
              {"name": "COMENTARIOS", "type": "STRING"}, {"name": "ORIGEN", "type": "STRING"},
              {"name": "PREMISA", "type": "STRING"}, {"name": "UB", "type": "STRING"}, {"name": "RUBRO", "type": "STRING"},
              {"name": "2025_PPTO", "type": "FLOAT"},
              *[{"name": m, "type": "FLOAT"} for m in months_sp],
              {"name": "ME_2025", "type": "FLOAT"}, {"name": "X", "type": "STRING"},
              *[{"name": f"{m}2", "type": "FLOAT"} for m in months_sp],
              {"name": "PLAN_2025", "type": "FLOAT"}, {"name": "VAR_POR", "type": "FLOAT"}, {"name": "POR", "type": "FLOAT"},
              {"name": "MANDANTE", "type": "STRING"}, {"name": "TIMESTAMP", "type": "TIMESTAMP"},
              {"name": "PATH", "type": "STRING"}, {"name": "CEDULA", "type": "STRING"}
          ]

      else: # Default for ME_PPTO
          months = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]
          fields = [
              {"name": "DIMENSION5", "type": "STRING"}, {"name": "LASTLEVEL", "type": "STRING"},
              *[{"name": m, "type": "FLOAT"} for m in months],
              {"name": "TOTAL", "type": "FLOAT"}, {"name": "x", "type": "STRING"},
              *[{"name": f"{m}2", "type": "FLOAT"} for m in months],
              {"name": "TOTAL2", "type": "FLOAT"},
              {"name": "MANDANTE", "type": "STRING"}, {"name": "TIMESTAMP", "type": "TIMESTAMP"},
              {"name": "PATH", "type": "STRING"}, {"name": "CEDULA", "type": "STRING"}
          ]
          print("Schema creado ME_PPTO")

  # --- BigQuery Execution ---

  schema = [bigquery.SchemaField(f['name'], f['type']) for f in fields]

  try:
      client.get_table(table_full_id)
      print("Existe tabla")
      client.delete_table(table_full_id, not_found_ok=True)  # Make an API request.
      print(f"Table deleted: {table.table_id}")

  except exceptions.NotFound:
      table = bigquery.Table(table_full_id, schema=schema)
      table = client.create_table(table)
      print(f"Table created: {table.table_id}")
  except Exception as e:
      print(f"An error occurred: {e}")

⭐ **GENERAR BASE INICIAL**

In [ ]:
query_datos_iniciales = f'''
WITH HISTORICOS AS (
  ##REALES ULTIMOS DOS EJERCICIOS CERRADOS

SELECT T0.`Cuenta`, `Ejercicio`,
T0.DIMENSION1, T0.DIMENSION2 as DIMENSION2,
CASE WHEN Marca IN ('Negocio0','Liv Express') THEN 'APP-CORNERS' ELSE DIMENSION5 END ||
INITCAP(T0.DIMENSION1) as IDUB,
T0.Periodo,
T0.VERSION,
'HISTORICOS' as `ORIGEN`, FUNCION
,SUM(IMPORTE)as IMPORTE

FROM `{ER}` T0
LEFT JOIN `{JERARCUENTAPPTO}` T1 ON T0.Cuenta=T1.CUENTA

WHERE T0.VERSION = 'Reales' AND (T0.Ejercicio IN ({aniobase}-1,{aniobase}-2,{aniobase}-3)) AND DIMENSION1 IN UNNEST ({divisiones_Negocio9})
GROUP BY 1,2,3,4,5,6,7,8,9

UNION ALL

##PRESUPUESTO Y ME DEL ULTIMO EJERCICIO

SELECT T0.`Cuenta`, `Ejercicio`,
T0.DIMENSION1, T0.DIMENSION2 as DIMENSION2,
CASE WHEN Marca IN ('Negocio0','Liv Express') THEN 'APP-CORNERS' ELSE DIMENSION5 END ||
INITCAP(T0.DIMENSION1) as IDUB,
T0.Periodo,
T0.VERSION,
'HISTORICOS' as `ORIGEN`, FUNCION
,SUM(IMPORTE)as IMPORTE

FROM `{ER}` T0
LEFT JOIN `{JERARCUENTAPPTO}` T1 ON T0.Cuenta=T1.CUENTA

WHERE T0.VERSION IN ('PPTO', 'PPTO_V2') AND (T0.Ejercicio IN ({aniobase}-1)) AND DIMENSION1 IN UNNEST ({divisiones_Negocio9})

GROUP BY 1,2,3,4,5,6,7,8,9

UNION ALL

##EJERCICIO ACTUAL

SELECT T0.`Cuenta`, `Ejercicio`,
T0.DIMENSION1, T0.DIMENSION2 as DIMENSION2,
CASE WHEN Marca IN ('Negocio0','Liv Express') THEN 'APP-CORNERS' ELSE DIMENSION5  END ||
INITCAP(T0.DIMENSION1) as IDUB,
T0.Periodo,
T0.VERSION,
'HISTORICOS' as `ORIGEN`, FUNCION
,SUM(IMPORTE)as IMPORTE

FROM `{project_id}.data_warehouse_finance.accounting_journal` T0
LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` T1 ON T0.Cuenta=T1.CUENTA

WHERE T0.VERSION IN ('Reales','PPTO','PPTO_V0') AND (T0.Ejercicio IN ({aniobase})) AND DIMENSION1 IN UNNEST ({divisiones_Negocio9})

GROUP BY 1,2,3,4,5,6,7,8,9
),

BASE AS (
SELECT
T3.Lvl5 as JERARQUIA, T1.P_L as EXPECTATIVAS,T0.Cuenta, INITCAP(DIMENSION1) as DIMENSION1, DIMENSION2, Ejercicio, Periodo, VERSION, 'HISTORICOS' ORIGEN,
IDUB
, IMPORTE

FROM HISTORICOS T0
LEFT JOIN {project_datasetreportesbi}.Jerarquia_Negocio9_HFT T1 ON T0.Cuenta = T1.Cuenta AND T0.DIMENSION1=T1.DIV
LEFT JOIN {project_datasetreportesbi}.JERARQUIA_Negocio9_INT T3 ON T0.Cuenta = T3.Cuenta AND T3.FUN=T0.Funcion

UNION ALL

SELECT
T3.Lvl5 as JERARQUIA, T1.P_L as EXPECTATIVAS,T0.Cuenta, INITCAP(DIMENSION1) as DIMENSION1, '' as DIMENSION2, CAST(SUBSTRING(T0.Version,1,4) AS INT64) as Ejercicio,
CAST(Periodo AS INT) as PERIODO,
CASE WHEN SUBSTRING(T0.Version,6,LENGTH(T0.Version)) = "REAL" THEN 'Reales'
ELSE SUBSTRING(T0.Version,6,LENGTH(T0.Version)) END as VERSION,
'OT' as ORIGEN,
--CASE WHEN Marca IN ('Negocio0','Liv Express') THEN 'APP-CORNERS' ELSE UB END
UB||
INITCAP(T0.DIMENSION1) as IDUB,
MONTO as IMPORTE

FROM {project_datasetreportesbi}.Negocio9_ONE_TIMERS T0
LEFT JOIN {project_datasetreportesbi}.Jerarquia_Negocio9_HFT T1 ON T0.Cuenta = T1.Cuenta AND UPPER(T0.DIMENSION1)=UPPER(T1.DIV)
LEFT JOIN {project_datasetreportesbi}.JERARQUIA_Negocio9_INT T3 ON T0.Cuenta = T3.Cuenta AND T3.FUN=SUBSTRING(T0.Lvl5,12,3)
),
ORIGINAL AS (
SELECT
CASE WHEN JERARQUIA = 'Ventas Netas' AND  ORIGEN IN('HISTORICOS','OT') THEN
    CASE
         WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
            THEN 'Ventas Digitales'
         ELSE 'Ventas Físicas'
        END
      WHEN JERARQUIA = 'Costo de Ventas' AND  ORIGEN IN('HISTORICOS','OT') THEN
    CASE
         WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
            THEN 'Costo de Ventas Digitales'
         ELSE 'Costo de Ventas Físicas'
             END
ELSE JERARQUIA END as JERARQUIA,
CASE WHEN EXPECTATIVAS = 'Ventas Netas' AND ORIGEN IN('HISTORICOS','OT') THEN
    CASE
         WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
            THEN 'Ventas Digitales'
            ELSE 'Ventas Físicas'
            END
     WHEN EXPECTATIVAS = 'Costo de ventas' AND  ORIGEN IN('HISTORICOS','OT') THEN
    CASE
         WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
            THEN 'Costo de Ventas Digitales'
            ELSE 'Costo de Ventas Físicas'
            END
      WHEN JERARQUIA IN ('Ventas Digitales','Ventas Físicas') THEN JERARQUIA
      WHEN IFNULL(EXPECTATIVAS ,"") = "" THEN T1.PL
ELSE EXPECTATIVAS END as EXPECTATIVAS, Cuenta, DIMENSION1,DIMENSION2,
EJERCICIO, PERIODO, VERSION, ORIGEN, IDUB, IMPORTE
FROM BASE T0
LEFT JOIN `REPORTES_BI.MAPEO_Negocio9_HFT` T1 ON T0.JERARQUIA=T1.LASTLEVEL
)

-- SOLO ajustes Negocio10 sobre EXPECTATIVAS. Todo lo demás intacto.
SELECT
  JERARQUIA,
  CASE
    WHEN (UPPER(COALESCE(DIMENSION1,''))='Negocio10' OR UPPER(IDUB) LIKE '%Negocio10%')
         AND JERARQUIA='Costo de Ventas Físicas' THEN 'Costo de Ventas Físicas'
    WHEN (UPPER(COALESCE(DIMENSION1,''))='Negocio10' OR UPPER(IDUB) LIKE '%Negocio10%')
         AND JERARQUIA='Ventas Físicas' THEN 'Ventas Físicas'
    ELSE EXPECTATIVAS
  END AS EXPECTATIVAS,
  Cuenta, DIMENSION1, DIMENSION2,EJERCICIO, PERIODO, VERSION, ORIGEN,
  CAST(DATETIME(CURRENT_TIMESTAMP(), 'America/Mexico_City') AS TIMESTAMP) as UDATE,
   IDUB, IMPORTE
FROM ORIGINAL;

 '''

storedprocedure_Negocio9_historicos_query = f'''
CREATE OR REPLACE PROCEDURE `{storedprocedure_Negocio9_historicos}`()
OPTIONS (strict_mode = false)
BEGIN
  DELETE FROM {Negocio9_cedulas_sp}
  WHERE ORIGEN IN ('HISTORICOS','OT');

   -- MERGE destination is the original table
  INSERT INTO {Negocio9_cedulas_sp}

       {query_datos_iniciales}

END;
'''

client.query(storedprocedure_Negocio9_historicos_query).result()
print("Creado: ",storedprocedure_Negocio9_historicos)

⭐ **TRANSFORMAR CARGAS**

In [ ]:
conversion = f'''
WITH BASE AS (
SELECT
   CASE
       WHEN CONTAINS_SUBSTR(T0.LASTLEVEL,"CV Físico") THEN "Costo de Ventas Físicas"
       WHEN CONTAINS_SUBSTR(T0.LASTLEVEL,"CV Digital") THEN "Costo de Ventas Digitales"
       ELSE T0.LASTLEVEL END as JERARQUIA,
   '' as EXPECTATIVAS,
   '' as Cuenta,
   CASE WHEN T0.DIMENSION5 like '%395%' THEN 'Negocio9' WHEN UPPER(MARCA) = 'Negocio9' THEN 'Negocio9' WHEN UPPER(MARCA) IN ('RED DE CARGA','BYD') OR UPPER(T0.DIMENSION5) like '%CORPORATIVO BYD%' THEN 'Negocio10' ELSE 'Negocio8' END as DIMENSION1,
   CASE WHEN UPPER(MARCA) = 'BYD' THEN 'Negocio10'
        WHEN UPPER(T0.DIMENSION6) LIKE '%DIGITAL%' THEN 'Digital'
        WHEN T0.DIMENSION5 like '%395%' THEN 'Negocio9'
        WHEN UPPER(MARCA) = 'Negocio9' THEN 'Negocio9'
        WHEN UPPER(MARCA) = 'RED DE CARGA' THEN 'Red de Carga'
        ELSE 'Negocio8' END as DIMENSION2,
   CASE WHEN VERSIONES.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as EJERCICIO,
   PERIODOS.PERIODO as PERIODO,
   VERSIONES.VERSION,
   PATH AS ORIGEN,
   INITCAP(MATRIZ.NOMBRE) as DIMENSION5,
   INITCAP(MATRIZ.MARCA) as MARCA,

    CASE WHEN
      CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5
            ELSE UPPER(T0.DIMENSION5) END

            like '%CORPORATIVO Negocio9%'
      OR
      CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5
            ELSE UPPER(T0.DIMENSION5) END

            like '%BYD%'
      THEN '0487'

            ELSE T0.DIMENSION5 END
      ||
      CASE
        WHEN UPPER(MARCA) = 'BYD' OR UPPER(T0.DIMENSION5) like '%CORPORATIVO BYD%' THEN 'Negocio10'
        WHEN
        CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5
            ELSE UPPER(T0.DIMENSION5) END

            like '%CORPORATIVO Negocio9%' THEN 'Negocio8'
        WHEN T0.DIMENSION5 = 'DIR 10' THEN 'Negocio8'
        WHEN T0.DIMENSION5 IN('APPNegocio8','CORNERSNegocio8','LIV.COMNegocio8','APP-CORNERSNegocio8') THEN ""
        WHEN T0.DIMENSION5 like '%395%' THEN 'Negocio9'
        WHEN INITCAP(MARCA) = 'Negocio9' THEN INITCAP(MARCA)
        ELSE 'Negocio8' END
      as IDUB,

   CASE
     WHEN VERSIONES.VERSION IN ('PPTO_V2','PPTO_V1') THEN
       CASE
         WHEN PERIODOS.PERIODO = 1 THEN ENE
         WHEN PERIODOS.PERIODO = 2 THEN FEB
         WHEN PERIODOS.PERIODO = 3 THEN MAR
         WHEN PERIODOS.PERIODO = 4 THEN ABR
         WHEN PERIODOS.PERIODO = 5 THEN MAY
         WHEN PERIODOS.PERIODO = 6 THEN JUN
         WHEN PERIODOS.PERIODO = 7 THEN JUL
         WHEN PERIODOS.PERIODO = 8 THEN AGO
         WHEN PERIODOS.PERIODO = 9 THEN SEP
         WHEN PERIODOS.PERIODO = 10 THEN OCT
         WHEN PERIODOS.PERIODO = 11 THEN NOV
         WHEN PERIODOS.PERIODO = 12 THEN DIC
       END
    --  ELSE
    --    CASE
    --      WHEN PERIODOS.PERIODO = 1 THEN ENE2
    --      WHEN PERIODOS.PERIODO = 2 THEN FEB2
    --      WHEN PERIODOS.PERIODO = 3 THEN MAR2
    --      WHEN PERIODOS.PERIODO = 4 THEN ABR2
    --      WHEN PERIODOS.PERIODO = 5 THEN MAY2
    --      WHEN PERIODOS.PERIODO = 6 THEN JUN2
    --      WHEN PERIODOS.PERIODO = 7 THEN JUL2
    --      WHEN PERIODOS.PERIODO = 8 THEN AGO2
    --      WHEN PERIODOS.PERIODO = 9 THEN SEP2
    --      WHEN PERIODOS.PERIODO = 10 THEN OCT2
    --      WHEN PERIODOS.PERIODO = 11 THEN NOV2
    --      WHEN PERIODOS.PERIODO = 12 THEN DIC2
    --    END
     END * -1000 AS
   IMPORTE

    ,TIMESTAMP


    FROM `{project_datasetbases}.Negocio9_CEDULAS` T0
    CROSS JOIN `{PERIODOS}` PERIODOS
    CROSS JOIN (SELECT DISTINCT VERSION FROM `{PARPERIODOS}`) VERSIONES
    LEFT JOIN `{project_datasetreportesbi}.FILTRO` FILtrO ON T0.LASTLEVEL = FILTRO.RUBRO
    LEFT JOIN `{project_datasetreportesbi}.Negocio9_Matriz_DIMENSION5caciones` MATRIZ ON
      CASE WHEN
      CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5 ELSE UPPER(T0.DIMENSION5) END like '%CORPORATIVO Negocio9%' THEN '0487Negocio8'
        WHEN T0.DIMENSION5 = 'DIR 10' THEN 'APP-CORNERSNegocio8'
        WHEN T0.DIMENSION5 = '0395' THEN '0395Negocio9'
         ELSE T0.DIMENSION5 END  =

      CASE WHEN T0.DIMENSION5 like '%CORPORATIVO Negocio9%' OR T0.DIMENSION5 = 'DIR 10' OR T0.DIMENSION5 like '%0395%' THEN MATRIZ.NOMUB ELSE MATRIZ.UB END

    WHERE VERSIONES.VERSION IN UNNEST ({params['version']})  AND CARGA <> "" AND PATH = path_p

  ),
  ORIGINAL AS (
    SELECT
    CASE WHEN JERARQUIA = 'Ventas Netas' AND  ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Ventas Digitales'
            ELSE 'Ventas Físicas'
            END
          WHEN JERARQUIA = 'Costo de Ventas' AND  ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Costo de Ventas Digitales'
            ELSE 'Costo de Ventas Físicas'
                END
    ELSE JERARQUIA END as JERARQUIA,
    CASE WHEN EXPECTATIVAS = 'Ventas Netas' AND ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Ventas Digitales'
                ELSE 'Ventas Físicas'
                END
        WHEN EXPECTATIVAS = 'Costo de ventas' AND  ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Costo de Ventas Digitales'
                ELSE 'Costo de Ventas Físicas'
                END
          WHEN JERARQUIA IN ('Ventas Digitales','Ventas Físicas') THEN JERARQUIA
          WHEN IFNULL(EXPECTATIVAS ,"") = "" THEN T1.PL
    ELSE EXPECTATIVAS END as EXPECTATIVAS, Cuenta, DIMENSION1, DIMENSION2,
    EJERCICIO, PERIODO, VERSION, ORIGEN, IDUB, IMPORTE
    ,T0.TIMESTAMP
    FROM BASE T0
    LEFT JOIN `REPORTES_BI.MAPEO_Negocio9_HFT` T1 ON T0.JERARQUIA=T1.LASTLEVEL
    )

    -- SOLO ajustes Negocio10 sobre EXPECTATIVAS. Todo lo demás intacto.
    SELECT
      JERARQUIA,
      CASE
        WHEN (UPPER(COALESCE(DIMENSION1,''))='Negocio10' OR UPPER(IDUB) LIKE '%Negocio10%')
            AND JERARQUIA='Costo de Ventas Físicas' THEN 'Costo de Ventas Físicas'
        WHEN (UPPER(COALESCE(DIMENSION1,''))='Negocio10' OR UPPER(IDUB) LIKE '%Negocio10%')
            AND JERARQUIA='Ventas Físicas' THEN 'Ventas Físicas'
        ELSE EXPECTATIVAS
      END AS EXPECTATIVAS,
      Cuenta, DIMENSION1, DIMENSION2, EJERCICIO, SAFE_CAST(PERIODO AS INT64) AS PERIODO, VERSION, ORIGEN,
      TIMESTAMP as UDATE,
      IDUB, IMPORTE
    FROM ORIGINAL
 '''

storedprocedure_Negocio9_cedulas = f'''
CREATE OR REPLACE PROCEDURE `{stored_procedure_Negocio9}`(path STRING)
OPTIONS (strict_mode = false)
BEGIN
  DECLARE path_p STRING DEFAULT IFNULL(path,'');
  DECLARE dst_last_backup TIMESTAMP;
  DECLARE dst_actual_backup TIMESTAMP;

  DELETE FROM {Negocio9_cedulas_sp}
  WHERE ORIGEN = CONCAT(path_p) AND VERSION IN UNNEST ({params['version']});

   -- MERGE destination is the original table
  INSERT INTO {Negocio9_cedulas_sp}
       {conversion}
       ;

      -- Get last BACKUP_DATE for this table (partition-safe)
      EXECUTE IMMEDIATE FORMAT("""
        SELECT MAX(UDATE)
        FROM `{project_id}.intranet_bd_pruebas.LOG_BASE_Negocio8_PPTO`
        WHERE ORIGEN = '%s'
          AND UDATE >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
      """, path_p)
      INTO dst_last_backup;

      -- Get last ACTUAL DATE
      EXECUTE IMMEDIATE FORMAT("""
        SELECT MAX(UDATE)
        FROM `{Negocio9_cedulas_sp}`
        WHERE ORIGEN = '%s'
      """, path_p)
      INTO dst_actual_backup;

      -- Stop if source not newer
      IF dst_last_backup IS NOT NULL AND dst_actual_backup <= dst_last_backup THEN
        SELECT FORMAT(
          'SKIPPED: %s | dst_actual_backup=%s | last_backup=%s',
          path_p,
          CAST(dst_actual_backup AS STRING),
          CAST(dst_last_backup AS STRING)
        ) AS status;
        RETURN;
      END IF;


    INSERT INTO `{project_id}.intranet_bd_pruebas.LOG_BASE_Negocio8_PPTO`
       {conversion}
       ;


END;
'''

client.query(storedprocedure_Negocio9_cedulas).result()
print("Creado: ",stored_procedure_Negocio9)

Version 2

In [ ]:
versiones2 = [f"{item}_2" for item in params['version']]
sp2Negocio9 = f"{stored_procedure_Negocio9}_2"

conversion2 = f'''
WITH BASE AS (
SELECT
   CASE
       WHEN CONTAINS_SUBSTR(T0.LASTLEVEL,"CV Físico") THEN "Costo de Ventas Físicas"
       WHEN CONTAINS_SUBSTR(T0.LASTLEVEL,"CV Digital") THEN "Costo de Ventas Digitales"
       ELSE T0.LASTLEVEL END as JERARQUIA,
   '' as EXPECTATIVAS,
   '' as Cuenta,
   CASE WHEN T0.DIMENSION5 like '%395%' THEN 'Negocio9' WHEN UPPER(MARCA) = 'Negocio9' THEN 'Negocio9' WHEN UPPER(MARCA) IN ('RED DE CARGA','BYD') OR UPPER(T0.DIMENSION5) like '%CORPORATIVO BYD%' THEN 'Negocio10' ELSE 'Negocio8' END as DIMENSION1,
   CASE WHEN UPPER(MARCA) = 'BYD' THEN 'Negocio10'
        WHEN UPPER(T0.DIMENSION6) LIKE '%DIGITAL%' THEN 'Digital'
        WHEN T0.DIMENSION5 like '%395%' THEN 'Negocio9'
        WHEN UPPER(MARCA) = 'Negocio9' THEN 'Negocio9'
        WHEN UPPER(MARCA) = 'RED DE CARGA' THEN 'Red de Carga'
        ELSE 'Negocio8' END as DIMENSION2,
   CASE WHEN VERSIONES.VERSION IN ('PPTO_V2','PPTO_V1') THEN {aniobase} ELSE {aniobase+1} END as EJERCICIO,
   PERIODOS.PERIODO as PERIODO,
   CONCAT(VERSIONES.VERSION,"_2" as VERSION,
   PATH AS ORIGEN,
   INITCAP(MATRIZ.NOMBRE) as DIMENSION5,
   INITCAP(MATRIZ.MARCA) as MARCA,

    CASE WHEN
      CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5
            ELSE UPPER(T0.DIMENSION5) END

            like '%CORPORATIVO Negocio9%'
      OR
      CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5
            ELSE UPPER(T0.DIMENSION5) END

            like '%BYD%'
      THEN '0487'

            ELSE T0.DIMENSION5 END
      ||
      CASE
        WHEN UPPER(MARCA) = 'BYD' OR UPPER(T0.DIMENSION5) like '%CORPORATIVO BYD%' THEN 'Negocio10'
        WHEN
        CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5
            ELSE UPPER(T0.DIMENSION5) END

            like '%CORPORATIVO Negocio9%' THEN 'Negocio8'
        WHEN T0.DIMENSION5 = 'DIR 10' THEN 'Negocio8'
        WHEN T0.DIMENSION5 IN('APPNegocio8','CORNERSNegocio8','LIV.COMNegocio8','APP-CORNERSNegocio8') THEN ""
        WHEN T0.DIMENSION5 like '%395%' THEN 'Negocio9'
        WHEN INITCAP(MARCA) = 'Negocio9' THEN INITCAP(MARCA)
        ELSE 'Negocio8' END
      as IDUB,

   CASE
     WHEN VERSIONES.VERSION IN ('PPTO_V2','PPTO_V1') THEN
       CASE
         WHEN PERIODOS.PERIODO = 1 THEN ENE
         WHEN PERIODOS.PERIODO = 2 THEN FEB
         WHEN PERIODOS.PERIODO = 3 THEN MAR
         WHEN PERIODOS.PERIODO = 4 THEN ABR
         WHEN PERIODOS.PERIODO = 5 THEN MAY
         WHEN PERIODOS.PERIODO = 6 THEN JUN
         WHEN PERIODOS.PERIODO = 7 THEN JUL
         WHEN PERIODOS.PERIODO = 8 THEN AGO
         WHEN PERIODOS.PERIODO = 9 THEN SEP
         WHEN PERIODOS.PERIODO = 10 THEN OCT
         WHEN PERIODOS.PERIODO = 11 THEN NOV
         WHEN PERIODOS.PERIODO = 12 THEN DIC
       END
    --  ELSE
    --    CASE
    --      WHEN PERIODOS.PERIODO = 1 THEN ENE2
    --      WHEN PERIODOS.PERIODO = 2 THEN FEB2
    --      WHEN PERIODOS.PERIODO = 3 THEN MAR2
    --      WHEN PERIODOS.PERIODO = 4 THEN ABR2
    --      WHEN PERIODOS.PERIODO = 5 THEN MAY2
    --      WHEN PERIODOS.PERIODO = 6 THEN JUN2
    --      WHEN PERIODOS.PERIODO = 7 THEN JUL2
    --      WHEN PERIODOS.PERIODO = 8 THEN AGO2
    --      WHEN PERIODOS.PERIODO = 9 THEN SEP2
    --      WHEN PERIODOS.PERIODO = 10 THEN OCT2
    --      WHEN PERIODOS.PERIODO = 11 THEN NOV2
    --      WHEN PERIODOS.PERIODO = 12 THEN DIC2
    --    END
     END * -1000 AS
   IMPORTE

    ,TIMESTAMP


    FROM `{project_datasetbases}.Negocio9_CEDULAS` T0
    CROSS JOIN `{PERIODOS}` PERIODOS
    CROSS JOIN (SELECT DISTINCT VERSION FROM `{PARPERIODOS}`) VERSIONES
    LEFT JOIN `{project_datasetreportesbi}.FILTRO` FILtrO ON T0.LASTLEVEL = FILTRO.RUBRO
    LEFT JOIN `{project_datasetreportesbi}.Negocio9_Matriz_DIMENSION5caciones` MATRIZ ON
      CASE WHEN
      CASE WHEN LENGTH(T0.DIMENSION5) <4 THEN REPEAT("0",4-LENGTH(T0.DIMENSION5))||T0.DIMENSION5 ELSE UPPER(T0.DIMENSION5) END like '%CORPORATIVO Negocio9%' THEN '0487Negocio8'
        WHEN T0.DIMENSION5 = 'DIR 10' THEN 'APP-CORNERSNegocio8'
        WHEN T0.DIMENSION5 = '0395' THEN '0395Negocio9'
         ELSE T0.DIMENSION5 END  =

      CASE WHEN T0.DIMENSION5 like '%CORPORATIVO Negocio9%' OR T0.DIMENSION5 = 'DIR 10' OR T0.DIMENSION5 like '%0395%' THEN MATRIZ.NOMUB ELSE MATRIZ.UB END

    WHERE VERSIONES.VERSION IN UNNEST ({params['version']})  AND CARGA <> "" AND PATH = path_p

  ),
  ORIGINAL AS (
    SELECT
    CASE WHEN JERARQUIA = 'Ventas Netas' AND  ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Ventas Digitales'
            ELSE 'Ventas Físicas'
            END
          WHEN JERARQUIA = 'Costo de Ventas' AND  ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Costo de Ventas Digitales'
            ELSE 'Costo de Ventas Físicas'
                END
    ELSE JERARQUIA END as JERARQUIA,
    CASE WHEN EXPECTATIVAS = 'Ventas Netas' AND ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Ventas Digitales'
                ELSE 'Ventas Físicas'
                END
        WHEN EXPECTATIVAS = 'Costo de ventas' AND  ORIGEN IN('HISTORICOS','OT') THEN
        CASE
            WHEN T0.DIMENSION2 = 'Digital' OR IDUB LIKE '%490%'  OR IDUB LIKE '%436%'
                THEN 'Costo de Ventas Digitales'
                ELSE 'Costo de Ventas Físicas'
                END
          WHEN JERARQUIA IN ('Ventas Digitales','Ventas Físicas') THEN JERARQUIA
          WHEN IFNULL(EXPECTATIVAS ,"") = "" THEN T1.PL
    ELSE EXPECTATIVAS END as EXPECTATIVAS, Cuenta, DIMENSION1, DIMENSION2,
    EJERCICIO, PERIODO, VERSION, ORIGEN, IDUB, IMPORTE
    ,T0.TIMESTAMP
    FROM BASE T0
    LEFT JOIN `REPORTES_BI.MAPEO_Negocio9_HFT` T1 ON T0.JERARQUIA=T1.LASTLEVEL
    )

    -- SOLO ajustes Negocio10 sobre EXPECTATIVAS. Todo lo demás intacto.
    SELECT
      JERARQUIA,
      CASE
        WHEN (UPPER(COALESCE(DIMENSION1,''))='Negocio10' OR UPPER(IDUB) LIKE '%Negocio10%')
            AND JERARQUIA='Costo de Ventas Físicas' THEN 'Costo de Ventas Físicas'
        WHEN (UPPER(COALESCE(DIMENSION1,''))='Negocio10' OR UPPER(IDUB) LIKE '%Negocio10%')
            AND JERARQUIA='Ventas Físicas' THEN 'Ventas Físicas'
        ELSE EXPECTATIVAS
      END AS EXPECTATIVAS,
      Cuenta, DIMENSION1, DIMENSION2, EJERCICIO, SAFE_CAST(PERIODO AS INT64) AS PERIODO, VERSION, ORIGEN,
      TIMESTAMP as UDATE,
      IDUB, IMPORTE
    FROM ORIGINAL
 '''

storedprocedure_Negocio9_cedulas2 = f'''
CREATE OR REPLACE PROCEDURE `{sp2Negocio9}`(path STRING)
OPTIONS (strict_mode = false)
BEGIN
  DECLARE path_p STRING DEFAULT IFNULL(path,'');
  DECLARE dst_last_backup TIMESTAMP;
  DECLARE dst_actual_backup TIMESTAMP;

  DELETE FROM {Negocio9_cedulas_sp}
  WHERE ORIGEN = CONCAT(path_p) AND VERSION IN UNNEST ({versiones2});

   -- MERGE destination is the original table
  INSERT INTO {Negocio9_cedulas_sp}
       {conversion2}
       ;

      -- Get last BACKUP_DATE for this table (partition-safe)
      EXECUTE IMMEDIATE FORMAT("""
        SELECT MAX(UDATE)
        FROM `{project_id}.intranet_bd_pruebas.LOG_BASE_Negocio8_PPTO`
        WHERE ORIGEN = '%s'
          AND UDATE >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
      """, path_p)
      INTO dst_last_backup;

      -- Get last ACTUAL DATE
      EXECUTE IMMEDIATE FORMAT("""
        SELECT MAX(UDATE)
        FROM `{Negocio9_cedulas_sp}`
        WHERE ORIGEN = '%s'
      """, path_p)
      INTO dst_actual_backup;

      -- Stop if source not newer
      IF dst_last_backup IS NOT NULL AND dst_actual_backup <= dst_last_backup THEN
        SELECT FORMAT(
          'SKIPPED: %s | dst_actual_backup=%s | last_backup=%s',
          path_p,
          CAST(dst_actual_backup AS STRING),
          CAST(dst_last_backup AS STRING)
        ) AS status;
        RETURN;
      END IF;


    INSERT INTO `{project_id}.intranet_bd_pruebas.LOG_BASE_Negocio8_PPTO`
       {conversion2}
       ;


END;
'''

client.query(storedprocedure_Negocio9_cedulas2).result()
print("Creado: ",sp2Negocio9)

⭐ **Estructura Reales para reporte Nómina BI**


In [ ]:
realesnom = f'''
-- CREATE OR REPLACE TABLE `intranet_bd_pruebas.Negocio9_MONTO_NOMINA_REALES` AS

WITH
REPNOM AS (
 SELECT * FROM {project_id}.ME_PPTO2026_NOMINA.PAR_ESTRUCTURA_REP_NOM
),

REALES AS (
  SELECT T0.*, T1.*EXCEPT(CUENTA)
  FROM `data_warehouse_finance.accounting_journal` T0
  LEFT JOIN `PARAMETROS_PRESUPUESTO.PAR_JERARQUIA_GENERAL` t1 using(cuenta)
  WHERE DIMENSION1 IN ('Negocio8','Negocio9') AND version = 'Reales' AND ejercicio = 2025
  ),

  CALCULO AS(

SELECT 'Negocio8' as NEGOCIO, MARCA, PERIODO,'' as POSICION,EJERCICIO,VERSION,CUENTA,RUBRO,
DIMENSION5||DIMENSION1 as IDDIMENSION5, 0 as HC,
SUM(
CASE
  WHEN ORDEN = 0 AND NIVEL2 = 'TOTAL VENTAS NETAS' AND ((DIMENSION1 = 'Negocio8' AND DIMENSION2 <> 'Digital') OR DIMENSION1 <> 'Negocio8') THEN (IMPORTE*-1)
  WHEN ORDEN = 1 AND NIVEL5 = 'Sueldo Ejecutivo' THEN (IMPORTE)
  WHEN ORDEN = 2 AND NIVEL5 = 'Personal General' THEN (IMPORTE)
  WHEN ORDEN = 3 AND NIVEL4 = 'Sueldos' THEN (IMPORTE)
  WHEN ORDEN = 4  THEN 0
  WHEN ORDEN = 5 AND NIVEL4 = 'Sueldos'  THEN (IMPORTE)
  WHEN ORDEN = 21 AND NIVEL3 = 'COMISIONES SOBRE VENTAS'  THEN (IMPORTE)
  WHEN ORDEN = 7 AND NIVEL5 = 'Despensa Y Vales'  THEN (IMPORTE)
  WHEN ORDEN = 8 AND NIVEL5 = 'Vacaciones' THEN (IMPORTE)
  WHEN ORDEN = 9 AND NIVEL5 = 'Aguinaldo' THEN (IMPORTE)
  WHEN ORDEN = 10 AND CUENTA = '51000007' THEN (IMPORTE)
  WHEN ORDEN = 13 AND CUENTA IN ('54000002','51000021','51000412')  THEN (IMPORTE)
  WHEN ORDEN = 14 AND CUENTA = '51000403'  THEN (IMPORTE)
  WHEN ORDEN = 16 AND NIVEL5 = 'Cuotas Imss'  THEN (IMPORTE)
  WHEN ORDEN = 17 AND NIVEL5 LIKE '%Sar Retiro%'  THEN (IMPORTE)
  WHEN ORDEN = 18 AND NIVEL5 = 'Sar Vivienda' THEN (IMPORTE)
  WHEN ORDEN = 19 AND NIVEL5 = 'Impto. Sobre Nominas' THEN (IMPORTE)
  WHEN ORDEN = 23 AND CUENTA = '51000006'  THEN (IMPORTE)
  WHEN ORDEN = 22 AND CUENTA = '51000057'  THEN (IMPORTE)

  WHEN ORDEN = 15
       AND ( NIVEL5 in ( 'Despensa Y Vales', 'Vacaciones' , 'Aguinaldo' )
       OR CUENTA IN ('51000007','54000002','51000021','51000412', '51000403') ) THEN (IMPORTE)
       ELSE 0
  END
  ) as IMPORTE

FROM REALES
CROSS JOIN REPNOM

WHERE ORDEN IN (0,1,2,3,4,5,6,7,8,9,10,13,14,15,16,17,18,19,21,22,23)

GROUP BY 1,2,3,4,5,6,7,8,9,10)


SELECT NEGOCIO, MARCA, PERIODO, POSICION,EJERCICIO,VERSION,CUENTA,RUBRO, IDDIMENSION5, HC, SUM(IMPORTE) as IMPORTE

FROM CALCULO

WHERE IMPORTE <> 0

GROUP BY 1,2,3,4,5,6,7,8,9,10

'''

⭐ **Transformar a desarrollo** ⏰

In [ ]:
query_Negocio9bi = f'''
WITH BASE AS (
  SELECT
   *EXCEPT(JERARQUIA),
   CASE WHEN JERARQUIA IN ('Ventas Físicas', 'Ventas Digitales') THEN 'Ventas Netas'
        WHEN JERARQUIA IN ('Costo de Ventas Físicas','Costo de Ventas Digitales') THEN 'Costo de Ventas'
        ELSE JERARQUIA END as JERARQUIA
  FROM `{project_datasetreportesbi}.SP_Negocio9_CEDULAS` T0
  INNER JOIN `{project_datasetreportesbi}.Negocio9_Matriz_DIMENSION5caciones` MATRIZ ON T0.IDUB = MATRIZ.NOMUB
  WHERE VERSION IN UNNEST ({params['version']}) AND ejercicio >= {aniobase}
  ),

    CUENTAS_LVL5 AS (
      select 'Negocio9' as DIMENSION1, T0.Lvl5, SAFE_CAST(min(safe_cast(T0.Cuenta AS INT64)) AS STRING) as CUENTA

      from `{project_datasetreportesbi}.JERARQUIA_Negocio9_INT` T0
      INNER JOIN `{project_id}.data_warehouse_finance.jerarquias` T1 ON T0.CUENTA=T1.CUENTA
      WHERE ifnull(Negocio9,"")<>""
      group by 1,2
      UNION ALL
      select 'Negocio8' as DIMENSION1, T0.Lvl5, SAFE_CAST(min(safe_cast(T0.Cuenta AS INT64)) AS STRING) as CUENTA

      from `{project_datasetreportesbi}.JERARQUIA_Negocio9_INT` T0
      INNER JOIN `{project_id}.data_warehouse_finance.jerarquias` T1 ON T0.CUENTA=T1.CUENTA
      WHERE ifnull(Negocio8,"")<>""
      group by 1,2
      UNION ALL
      select 'Negocio10' as DIMENSION1, T0.Lvl5, SAFE_CAST(min(safe_cast(T0.Cuenta AS INT64)) AS STRING) as CUENTA

      from `{project_datasetreportesbi}.JERARQUIA_Negocio9_INT` T0
      INNER JOIN `{project_id}.data_warehouse_finance.jerarquias` T1 ON T0.CUENTA=T1.CUENTA
      WHERE ifnull(Negocio10,"")<>""
      group by 1,2
    )

    SELECT
    CASE WHEN BASE.DIMENSION2 = 'Negocio10' THEN 'Negocio10' ELSE BASE.DIMENSION1 END as DIMENSION1,
    BASE.DIMENSION2,
    MATRIZ.MARCA,
    EJERCICIO,
    CASE WHEN PERIODO < {mes_captura} THEN {mes_captura} ELSE PERIODO END as PERIODO,
    VERSION,
    'Negocio9' as ORIGEN,
    TRIM(CUENTAS_LVL5.CUENTA) as Cuenta,
    sum(ifnull(Importe,0)) AS IMPORTE

    FROM BASE
    LEFT JOIN CUENTAS_LVL5 ON
      --CASE WHEN JERARQUIA = 'Amortización' THEN 'Renta Fija' ELSE JERARQUIA END ***Validar si se requiere, cuadrando EBITDA, al parecer no es necesario***
      JERARQUIA= CUENTAS_LVL5.LVL5 AND BASE.DIMENSION1=CUENTAS_LVL5.DIMENSION1
    LEFT JOIN {CUENTAS} CUENTAS ON CUENTAS_LVL5.CUENTA = CUENTAS.CUENTA
    LEFT JOIN `{JERARCUENTAPPTO}` T3 ON CUENTAS_LVL5.CUENTA=T3.CUENTA
    LEFT JOIN `{project_datasetreportesbi}.Negocio9_Matriz_DIMENSION5caciones` MATRIZ ON BASE.IDUB=MATRIZ.NOMUB

    GROUP BY 1,2,3,4,5,6,7,8
'''
storedprocedure_Negocio9_mascara = f'''
CREATE OR REPLACE PROCEDURE `{stored_procedure_Negocio9_mascara}`()
BEGIN
  CREATE OR REPLACE TABLE {tablacargaNegocio9} AS
  {query_Negocio9bi};
END;
'''

client.query(storedprocedure_Negocio9_mascara).result()
print("Creado: ",stored_procedure_Negocio9_mascara)

⭐ **Stored Procedure Modelo Nómina**  ###FALTAAAAAA APLICAR VARIABLES

In [ ]:
querymnb = f'''
CREATE OR REPLACE PROCEDURE `{project_id}.intranet_bd_pruebas.MODELO_NOMINA_Negocio9_sp`()
--OPTIONS (strict_mode=false)
BEGIN

  DELETE FROM `{project_id}.intranet_bd_pruebas.MODELO_ETZ_BASE`
  WHERE NEGOCIO = 'Negocio8'
  ;

  INSERT INTO `{project_id}.intranet_bd_pruebas.MODELO_ETZ_BASE`

  WITH

   CECOS AS (
    SELECT * FROM {project_id}.mus_qas_drv_datos_maestros.NW_CECO3
  ),

  BASE AS (
    SELECT T0.* ,CECOS.MARCA, CECOS.DIMENSION5|| " " ||CECOS.DES_DIMENSION5 as DIMENSION5CACION
    FROM `{project_id}.intranet_bd_pruebas.HEADCOUNT_BASE` T0
    INNER JOIN CECOS USING (CECO)

    where negocio = 'Negocio8'
  ),

  UEST AS (
    SELECT DISTINCT * FROM {project_id}.intranet_bd_pruebas.DIMENSION5CACIONES_ESTADOS
  ),

  CUENTACECOS AS (
      select T0.NEGOCIO, T1.MARCA, T1.DIMENSION5 || " " || T1.DES_DIMENSION5 AS DIMENSION5CACION,
      T0.FUNCION_DESC as POSICION,
      COUNT(DISTINCT T0.CECO) as CECOS

      FROM BASE T0
      LEFT JOIN CECOS T1 ON T0.CECO=T1.CECO

      GROUP BY 1,2,3,4

      ORDER BY CECOS DESC
      ),

  CECOSPOS AS (

    select DISTINCT T0.NEGOCIO, T0.MARCA, T0.DIMENSION5CACION,
    T0.FUNCION_DESC as POSICION,
    T0.CECO

    FROM BASE T0
    INNER JOIN CUENTACECOS ON CUENTACECOS.NEGOCIO = T0.NEGOCIO
                              AND CUENTACECOS.MARCA = T0.MARCA
                              AND CUENTACECOS.DIMENSION5CACION = T0.DIMENSION5CACION
                              AND CUENTACECOS.POSICION = T0.FUNCION_DESC
                              AND CUENTACECOS.CECOS = 1
  ),

  SUELDOSN AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.TABULADOR`
    WHERE NEGOCIO = 'Negocio8'
  ),

  pg AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.PARAMETROS_GENERALES`
  ),

  pvcorpo AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VAC_CORPO`
  ),

  VENTAS_BASE AS (
    SELECT INITCAP(MARCA) AS MARCA,
    -- SPLIT(T0.IDUB," ")[0]
    CASE WHEN LENGTH(T1.UB1) < 4 THEN REPEAT("0",4-LENGTH(T1.UB1)) ELSE T1.UB1 END
    as DIMENSION5CACION,
    JERARQUIA
    VERSION, PERIODO, SUM(IMPORTE) as IMPORTE

    FROM {project_id}.REPORTES_BI.SP_Negocio9_CEDULAS T0
    LEFT JOIN `REPORTES_BI.Negocio9_Matriz_DIMENSION5caciones` T1 ON T0.IDUB= T1.NOMUB

    WHERE EJERCICIO = 2026 AND VERSION = 'PPTO_V1' AND JERARQUIA LIKE 'Ventas%'

    GROUP BY 1,2,3,4
  ),

  pv AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VACACIONAL`
  ),

  pva AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PRIMA_VACACIONAL_NEGOCIO X`
  ),

  v AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.PARAMETROS_NOMINA`
    WHERE NEGOCIO = 'Negocio8'
  ),

  ---------------------------------------------------------------

 A AS (
  SELECT DISTINCT T0.NEGOCIO,
  T0.MARCA,
  T0.DIMENSION5CACION,
  T0.FUNCION_DESC AS POSICION,

  --VALES DE DESPENSA
  IFNULL(
  CASE WHEN T0.GPOPERSDESC LIKE 'Vía Planta' THEN PARAMS.vales_via_planta
      WHEN T2.AREA_PERS_ID IN ('2G','2V') THEN PARAMS.vales_via_planta_mt
      ELSE VALES.Importe_vales END
  ,0) as VALES

  FROM BASE T0
  LEFT JOIN `{project_id}.PARAMETROS_PRESUPUESTO.AREA_PERSONAL` T2 ON T0.AREAPERSDESC=T2.AREA_PERS_DESC
  LEFT JOIN {project_id}.intranet_bd_pruebas.VALES_DESPENSA VALES
              ON T2.AREA_PERS_ID=VALES.AREA_PERSONAL
              AND REPEAT("0",CASE WHEN LENGTH(T0.DIVPERSONAL) <4 THEN 4-LENGTH(T0.DIVPERSONAL) ELSE 0 END)||T0.DIVPERSONAL=VALES.DIMENSION5
  CROSS JOIN pg PARAMS
),

CUENTA AS (
SELECT NEGOCIO, MARCA, DIMENSION5CACION, POSICION, COUNT (DISTINCT VALES) as CUENTA

FROM A

GROUP BY 1,2,3,4

ORDER BY CUENTA DESC
),

  vales AS (
    SELECT A.*

    FROM A
    INNER JOIN CUENTA
      ON A.POSICION=CUENTA.POSICION
      AND A.NEGOCIO=CUENTA.NEGOCIO
      AND A.MARCA=CUENTA.MARCA
      AND A.DIMENSION5CACION=CUENTA.DIMENSION5CACION
      AND CUENTA.CUENTA = 1
  ),

  -------------------------------------------------------------------

  ISN AS (
    SELECT * FROM `{project_id}.intranet_bd_pruebas.ISN`
  ),

  SZE AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.ESTADOS_DIMENSION7_EMERGENTE`
  ),

  SZF AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.ESTADOS_DIMENSION7_FRONTERIZA`
  ),

  MESES AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.MESES_COMPLETOS`
  ),

  p AS (
    SELECT T0.*
    FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PERIODOS` T0
    INNER JOIN pg ON T0.VERSION IN UNNEST(pg.versiones)
  ),

  sueldo_base AS (
    SELECT
      T0.MARCA,
      T0.DIMENSION5CACION,
      REGEXP_REPLACE(NORMALIZE(T0.POSICION, NFD), r"\\pM", '') AS POSICION,
      'Vacante' as NUMPER,
      IFNULL(CECOSPOS.CECO,"") as CECO,
      DATE(2026,MESES.Periodo,1) as FECHAING,
      SUM(IFNULL(ADICIONAL_REV,0)) AS HC,
      1 AS PORC_CECO,
      "NOMINA_Negocio9" AS ORIGEN,
      NEGOCIO
    FROM `{project_id}.intranet_bd_pruebas.CAPTURA_HC_Negocio9` T0
    LEFT JOIN `PARAMETROS_PRESUPUESTO.MESES_COMPLETOS` MESES ON UPPER(T0.MES_ING_REV) = MESES.NOMBRE
    LEFT JOIN CECOSPOS USING (MARCA,DIMENSION5CACION,NEGOCIO,POSICION)

    WHERE IFNULL(ADICIONAL_REV,0) <> 0

    GROUP BY 1,2,3,4,5,6,8,9,10
  ),

  DIMENSION6_PERS AS (
    SELECT DISTINCT
      T0.FUNCION_DESC AS POSICION,
      T0.DIMENSION6_Personal
    FROM BASE T0
  ),

  -- Sueldo diario
  con_sueldo_diario AS (
    SELECT
      t.* ,
      CASE
        WHEN DIMENSION6_PERSONAL IS NULL THEN
          CASE
            WHEN SUELDOSN.AREA_PERS IN ('1C','1E','1F','1H') THEN 'Personal Ejecutivo'
            ELSE 'Personal General'
          END
        ELSE DIMENSION6_PERSONAL
      END AS DIMENSION6_PERSONAL,
      IFNULL(SUELDOSN.COMISION, 0) AS PORC_COMISION,
      SUELDOSN.DIAS_PTU,
      SUELDOSN.DIAS_AGUINALDO,
      IFNULL(COMISION_DUMMY, "NO") AS COMISION_DUMMY,
      0 as SUELDO,
      CASE
        WHEN UPPER(SZE.ESTADO) = 'MONTERREY' THEN COALESCE(SUELDOSN.MONTERREY26, SUELDOS_ENE26)
        WHEN SZE.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_EMERGENTE26, SUELDOS_ENE26)
        WHEN SZF.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_FRONTERA26, SUELDOS_ENE26)
        ELSE SUELDOS_ENE26
      END * IFNULL(PORC_CECO, 1) AS SUELDO_TABULADOR26,
      CASE
        WHEN UPPER(SZE.ESTADO) = 'MONTERREY' THEN COALESCE(SUELDOSN.MONTERREY, SUELDOS_JUL25)
        WHEN SZE.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_EMERGENTE, SUELDOS_JUL25)
        WHEN SZF.ESTADO IS NOT NULL THEN COALESCE(SUELDOSN.DIMENSION7_FRONTERA, SUELDOS_JUL25)
        ELSE SUELDOS_JUL25
      END * IFNULL(PORC_CECO, 1) AS SUELDO_TABULADOR25,
      SUELDOSN.por_incr_jul26,
      APLICA_MINIMO_GARANTIZADO,
      IMP_MINIMO_GARANTIZADO_ENE_2026,
      NOTAS,
      UEST.ESTADO
    FROM sueldo_base t
    LEFT JOIN DIMENSION6_PERS USING(POSICION)
    LEFT JOIN UEST
      ON CASE WHEN t.NEGOCIO = 'NEGOCIO X y CC Externos' THEN t.MARCA ELSE SUBSTRING(t.DIMENSION5CACION, 1, 4) END = SUBSTRING(UEST.DIMENSION5CACION, 1, 4)
    LEFT JOIN SZE ON UEST.ESTADO = SZE.ESTADO
    LEFT JOIN SZF ON UEST.ESTADO = SZF.ESTADO
    LEFT JOIN SUELDOSN ON
      t.Marca = CASE
        WHEN SUELDOSN.NEGOCIO IN ('NEGOCIO X y CC Externos', 'NEGOCIO Y') THEN REPEAT("0", 4 - LENGTH(SUELDOSN.MARCA)) || SUELDOSN.MARCA
        ELSE SUELDOSN.MARCA
      END
      AND t.POSICION = REGEXP_REPLACE(NORMALIZE(SUELDOSN.POSICION, NFD), r"\\pM", '')
      AND t.NEGOCIO = SUELDOSN.NEGOCIO
  ),

  antiguedad AS (
    SELECT
      SD.* EXCEPT(SUELDO, SUELDO_TABULADOR26, SUELDO_TABULADOR25, HC),
      pvcorpo.DIAS AS DIAS_PV_CORPO,
      IFNULL(
        CASE
          WHEN MES <= mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES > mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES < mesinc AND VERSION = 'PPTO_V0' THEN 1
          WHEN MES >= mesinc AND VERSION = 'PPTO_V0' THEN
            (1) * (1 + IFNULL(
              CASE
                WHEN Negocio <> 'Negocio8' THEN pg.porc_inc_sig_anio
                ELSE IFNULL(POR_INCR_JUL26, 0)
              END,
            0))
          ELSE 0
        END *
        CASE
          WHEN NUMPER <> 'Vacante' THEN SUELDO
          WHEN IFNULL(SUELDO, 0) = 0 AND VERSION IN ('PPTO_V2','PPTO_V1') THEN SUELDO_TABULADOR26
          ELSE SUELDO_TABULADOR26
        END,
      0) AS SUELDO,
      IFNULL(
        CASE
          WHEN MES <= mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES > mesinc AND VERSION IN ('PPTO_V2','PPTO_V1') THEN 1
          WHEN MES < mesinc AND VERSION = 'PPTO_V0' THEN 1
          WHEN MES >= mesinc AND VERSION = 'PPTO_V0' THEN
            (1) * (1 + IFNULL(
              CASE
                WHEN Negocio <> 'Negocio8' THEN pg.porc_inc_sig_anio
                ELSE IFNULL(POR_INCR_JUL26, 0)
              END,
            0))
          ELSE 0
        END *
        CASE
          WHEN VERSION IN ('PPTO_V2','PPTO_V1') THEN SUELDO_TABULADOR26
          ELSE SUELDO_TABULADOR26
        END,
      0) AS SUELDO_TABULADOR,
      CASE
        WHEN DIMENSION6_Personal = 'Eventuales' AND DATE_DIFF(P.FECHA, SD.FECHAING, MONTH) >= 1 THEN 0
        ELSE HC
      END AS HC,
      VERSION,
      MES AS PERIODO,
      CASE WHEN P.FECHA < SD.FECHAING THEN 0 ELSE 1 END AS FILTRO,
      DATE_DIFF(P.FECHA, SD.FECHAING, MONTH)/12 AS ANTIGUEDAD,
      DATE_DIFF(P.FECHA, SD.FECHAING, MONTH) AS ANTIGUEDAD_MESES
    FROM con_sueldo_diario SD
    CROSS JOIN P
    CROSS JOIN pg
    LEFT JOIN pvcorpo
      ON DATE_DIFF(P.FECHA, SD.FECHAING, YEAR) >= pvcorpo.MIN
      AND DATE_DIFF(P.FECHA, SD.FECHAING, YEAR) < pvcorpo.MAX
  ),

  tabuladores AS (
    SELECT
      a.* EXCEPT(FILTRO, SUELDO),
      CASE
        WHEN (VERSION = 'PPTO_V0' AND SUELDO < SUELDO_TABULADOR) OR Numper = 'Vacante'
        THEN SUELDO_TABULADOR
        ELSE SUELDO
      END AS SUELDO,
      pv.DIAS AS DIAS_VACACIONES
    FROM antiguedad a
    LEFT JOIN pv
      ON a.ANTIGUEDAD >= ROUND(pv.MIN, 0)
      AND a.ANTIGUEDAD < ROUND(pv.MAX, 0)
    WHERE FILTRO = 1
  ),

  headcount AS (
    SELECT
      T.POSICION,
      T.MARCA,
      T.DIMENSION5CACION,
      T.VERSION,
      T.PERIODO,
      SUM(HC) AS HC_DIMENSION5CACION
    FROM tabuladores T
    WHERE UPPER(IFNULL(COMISION_DUMMY, "")) = 'SI'
    GROUP BY 1,2,3,4,5
  ),

  conteo_posiciones AS (
    SELECT
      VERSION,
      PERIODO,
      DIMENSION5CACION,
      POSICION,
      COUNT(*) AS TOTAL_PERSONAS_MISMA_POSICION
    FROM tabuladores
    GROUP BY 1,2,3,4
  ),

  ptu_rangos AS (
    SELECT * FROM `{project_id}.PARAMETROS_PRESUPUESTO.PAR_PARAMETROS_PTU`
  ),

  ventas AS (
    SELECT
      CASE WHEN MARCA = 'Apple' THEN 'Livestore' ELSE MARCA END AS MARCA,
      DIMENSION5CACION,
      VERSION,
      PERIODO,
      SUM(IMPORTE) AS IMPORTE
    FROM VENTAS_BASE
    WHERE DIMENSION6_V = 'VENTA' AND CANAL = 'FISICO'
    GROUP BY 1,2,3,4
  ),

  ventasdig AS (
    SELECT
      CASE WHEN MARCA = 'Apple' THEN 'Livestore' ELSE MARCA END AS MARCA,
      DIMENSION5CACION,
      VERSION,
      PERIODO,
      SUM(IMPORTE) AS IMPORTE
    FROM VENTAS_BASE
    WHERE DIMENSION6_V = 'VENTA' AND CANAL = 'DIGITAL'
    GROUP BY 1,2,3,4
  ),

  CALCULOS AS (
    SELECT
      t.* EXCEPT(SUELDO),
      SUELDO * HC AS SUELDO,
      SUELDO / 30 AS SUELDO_DIARIO,
      v.* EXCEPT(PRIMA_VACACIONAL, PRIMA_DOMINICAL, NEGOCIO, DOMINGOS_PERIODO),
      CASE
        WHEN t.NEGOCIO IN ('Negocio8') THEN v.PRIMA_VACACIONAL
        WHEN t.NEGOCIO IN ('NEGOCIO X y CC Externos') THEN
          CASE WHEN pva.DIAS = 0.25 THEN 10 ELSE pva.DIAS END
      END AS PORC_PRIMA_VACACIONAL,
      v.PRIMA_DOMINICAL AS PORC_PRIMA_DOMINICAL,
      cp.TOTAL_PERSONAS_MISMA_POSICION,
      t.SUELDO / 30 * (v.VECES_FERIADO - 1) *
        CASE
          WHEN t.NEGOCIO = 'NEGOCIO X y CC Externos' THEN PLANTILLA_FERIADOS_DOM
          ELSE IF(IFNULL(cp.TOTAL_PERSONAS_MISMA_POSICION, 0) > 1, v.PLANTILLA_FERIADOS_DOM, 1)
        END *
        CASE WHEN CAST(T.PERIODO AS STRING) IN UNNEST(SPLIT(v.PERIODOS_FERIADO, ',')) THEN 1 ELSE 0 END * HC
        AS PAGO_FESTIVO,
      IFNULL(ve.IMPORTE, 0) AS VENTA_FIS,
      IFNULL(ventasdig.IMPORTE, 0) AS VENTA_DIG,
      HEADCOUNT.HC_DIMENSION5CACION,
      IFNULL(ve.IMPORTE, 0) * IFNULL(PORC_COMISION, 0) * HC / IFNULL(CASE WHEN HEADCOUNT.HC_DIMENSION5CACION = 0 THEN 1 ELSE HEADCOUNT.HC_DIMENSION5CACION END, 1) AS COMISION_ESTIMADA,
      CASE WHEN UPPER(IFNULL(NOTAS, "")) LIKE '%%%%APROVISIONAMIENTO%%%%' THEN
        IFNULL(ventasdig.IMPORTE, 0) * IFNULL(PORC_COMISION, 0) * HC / IFNULL(HEADCOUNT.HC_DIMENSION5CACION, 1)
      ELSE 0 END AS APROVISIONAMIENTO,
      IFNULL(vales.vales * t.HC, pg.vales_via_planta) * IFNULL(PORC_CECO, 1) *
        CASE
          WHEN t.VERSION = 'PPTO_V0' AND t.PERIODO >= pg.mesinc_vales THEN (1 + pg.porc_inc_vales)
          ELSE 1
        END AS VALES_DESPENSA,
      t.SUELDO / 30 * v.PRIMA_DOMINICAL * v.DOMINGOS_PERIODO / IF(cp.TOTAL_PERSONAS_MISMA_POSICION > 1, v.PLANTILLA_FERIADOS_DOM, 1) * HC AS PRIMA_DOMINICAL,
      DATE_DIFF(DATE(CASE WHEN t.VERSION = 'PPTO_V0' THEN 2027 ELSE 2026 END, 12, 31), FECHAING, DAY) AS DIASANIO,
      CASE WHEN t.DIMENSION6_personal LIKE '%%%%Ejecutivo%%%%' AND t.MARCA <> 'Negocio9' THEN 1 ELSE 0 END *
        t.SUELDO * v.MESES_BONO_ESP / 12 *
        CASE
          WHEN NUMPER = 'Vacante' AND EXTRACT(MONTH FROM FECHAING) > 7
            AND EXTRACT(YEAR FROM FECHAING) = CASE WHEN t.VERSION = 'PPTO_V0' THEN 2027 ELSE 2026 END
          THEN 0
          WHEN EXTRACT(YEAR FROM FECHAING) = CASE WHEN t.VERSION = 'PPTO_V0' THEN 2027 ELSE 2026 END
          THEN DATE_DIFF(DATE(CASE WHEN t.VERSION = 'PPTO_V0' THEN 2027 ELSE 2026 END, 12, 31), FECHAING, DAY) / 365
          ELSE 1
        END * HC AS BONO_ESP,
      IF(t.NUMPER = 'Vacante', TRUE, FALSE) AS ES_VACANTE,
      IF(LOWER(t.posicion) LIKE '%%%%Eventual%%%%', TRUE, FALSE) AS ES_EVENTUAL
    FROM tabuladores t
    LEFT JOIN pva ON t.ANTIGUEDAD < pva.max AND t.ANTIGUEDAD >= pva.min
    LEFT JOIN v ON t.NEGOCIO = v.NEGOCIO
    LEFT JOIN vales ON vales.DIMENSION5CACION = T.DIMENSION5CACION AND vales.posicion = t.posicion AND vales.negocio = t.negocio AND vales.marca = t.marca
    LEFT JOIN ventas ve ON t.MARCA = ve.MARCA AND SUBSTRING(t.DIMENSION5CACION, 1, 4) = SUBSTRING(ve.DIMENSION5CACION, 1, 4)
      AND t.VERSION = ve.VERSION AND t.PERIODO = ve.PERIODO
    LEFT JOIN ventasdig ON t.MARCA = ventasdig.MARCA AND SUBSTRING(t.DIMENSION5CACION, 1, 4) = SUBSTRING(ventasdig.DIMENSION5CACION, 1, 4)
      AND t.VERSION = ventasdig.VERSION AND t.PERIODO = ventasdig.PERIODO
    LEFT JOIN conteo_posiciones cp ON t.DIMENSION5CACION = cp.DIMENSION5CACION AND t.POSICION = cp.POSICION
      AND t.VERSION = cp.VERSION AND t.PERIODO = cp.PERIODO
    CROSS JOIN pg
    LEFT JOIN headcount ON T.MARCA = HEADCOUNT.MARCA AND T.DIMENSION5CACION = headcount.DIMENSION5CACION
      AND T.VERSION = headcount.VERSION AND T.PERIODO = headcount.PERIODO AND T.POSICION=headcount.posicion
    WHERE HC <> 0 --AND ((t.VERSION IN ('PPTO_V2','PPTO_V1') AND t.PERIODO >= SAFE_CAST(1 AS INT64)) OR t.VERSION NOT IN ('PPTO_V2','PPTO_V1'))
  ),

  MONTOS_ANUALES AS (
    SELECT  MARCA, DIMENSION5CACION, POSICION, NUMPER, ORIGEN, NEGOCIO, VERSION, CECO, SUM(COMISION_ESTIMADA) AS COMISION_ANUAL,
    SUM(IFNULL(CALCULOS.SUELDO,0)) AS SUELDO_DIC

    FROM CALCULOS

    WHERE PERIODO = 12
    GrOUP BY 1,2,3,4,5,6,7,8
    ),

  CALCULOS_ANUALES AS (
  SELECT CALCULOS.*,
  -- SAFE_CAST(IFNULL(SUELDO_DIC,0) AS FLOAT64) AS
  SUELDO_DIC,


   ((ABS(SUELDO_DIC)/30 +  IFNULL(COMISION_ANUAL /
   CASE WHEN CALCULOS.VERSION IN ('PPTO_V2','PPTO_V1') THEN (12-EXTRACT(MONTH FROM FECHAING)) ELSE 12 END
   / 30 ,0) ) * DIAS_AGUINALDO) / 12 *HC AS AGUINALDO,

    CASE
      WHEN DIMENSION6_Personal = 'Eventuales' THEN 0
      WHEN DIAS_PTU > 1
        THEN ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
        CASE WHEN CALCULOS.VERSION IN ('PPTO_V2','PPTO_V1') THEN (12-EXTRACT(MONTH FROM FECHAING)) ELSE 12 END
         / 30 ,0) ) * DIAS_PTU
        ELSE ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
        CASE WHEN CALCULOS.VERSION IN ('PPTO_V2','PPTO_V1') THEN (12-EXTRACT(MONTH FROM FECHAING)) ELSE 12 END
         / 30 ,0) ) * CASE WHEN CALCULOS.Negocio = 'Negocio8' THEN 75 ELSE 0 END
    END
   /12 *HC AS PTU,


   CASE WHEN DIMENSION6_personal like '%%%%Ejecutivo%%%%' THEN DIAS_PV_CORPO
    ELSE
    CASE
      WHEN PORC_PRIMA_VACACIONAL < 1 THEN
      DIAS_VACACIONES * PORC_PRIMA_VACACIONAL
    ELSE PORC_PRIMA_VACACIONAL END END *
   ( ABS(SUELDO_DIC)/30 + IFNULL(COMISION_ANUAL /
   CASE WHEN CALCULOS.VERSION IN ('PPTO_V2','PPTO_V1') THEN (12-EXTRACT(MONTH FROM FECHAING)) ELSE 12 END
   / 30 ,0) )  /12 *HC AS PRIMA_VACACIONAL

  FROM CALCULOS
  LEFT JOIN MONTOS_ANUALES USING(MARCA,DIMENSION5CACION,POSICION,NUMPER,ORIGEN,NEGOCIO,VERSION,CECO)
),

  CALCULO2 AS (

  SELECT CALCULOS_ANUALES.*,

    (
    IFNULL(SUELDO_DIARIO * HC,0) +  -- Sueldo diario
    IFNULL(COMISION_ESTIMADA / 30,0) +  -- Comisión diaria estimada
    IFNULL(PRIMA_VACACIONAL/ 30,0) +  -- Prima vacacional diaria
    IFNULL(AGUINALDO / 30,0) +    -- Aguinaldo diario
    IFNULL(PRIMA_DOMINICAL,0) / 30 +     -- Prima dominical DIARIA
    IFNULL(VALES_DESPENSA / 30,0) + -- vales diario
    IFNULL(BONO_ESP / 30,0)  --BONO_ESP DIARIO
    --IFNULL(PTU / 30,0) -- PTU  DIARIA
  )
  AS SBC,

  -- PREMIOS POR DEBAJO DEL MINIMO

    CASE WHEN UPPER(IFNULL(APLICA_MINIMO_GARANTIZADO,'')) = 'SI' AND  VERSION = 'PPTO_V0' THEN  IFNULL(IMP_MINIMO_GARANTIZADO_ENE_2026,0) - SUELDO + COMISION_ESTIMADA
    ELSE 0 END AS PREMIOS_COMISIONES

  FROM CALCULOS_ANUALES
  )


  SELECT
    b.*,
    b.SBC * ( CASE WHEN VERSION IN ('PPTO_V2','PPTO_V1') THEN SAR_F00_PPTO_V1 ELSE SAR_PLAN END ) * 30 AS SAR,

    -- IMSS (tabulado)
    (
      ENFERMEDAD_MATERNIDAD --ENFERMEDAD Y MATERNIDAD
      + INVALIDEZ_VIDA  -- INVALIDEZ Y VIDA
      + RIESGO_TRABAJO -- RIESGO DE TRABAJO
      + GUARDERIA_PRES --GUARDERIA Y PRESTACIONES
    )
  * b.SBC * 30 AS IMSS,

    -- Vivienda patrón 5
    b.SBC * VIVIENDA * 30 AS VIVIENDA_PATRON,

    -- ISN
    b.SBC * ISN.PORCENTAJE * 30 AS ISN,

    -- Prima de antigüedad (si aplica)
    CASE
      WHEN b.ANTIGUEDAD >= 3 THEN b.SUELDO_DIARIO * 12
      ELSE 0
    END AS PRIMA_ANTIGUEDAD

  FROM CALCULO2 b
  LEFT JOIN ISN ON b.ESTADO = ISN.ESTADO
  ;



  ----DROP NEGOCIO

  -- DELETE FROM `{project_id}.intranet_bd_pruebas.BASE_MODELO_NOMINA_FIJO`
  -- WHERE DIMENSION1 = 'Negocio8';


  -- INSERT INTO
  CREATE OR REPLACE TABLE `{project_id}.intranet_bd_pruebas.Negocio9_MONTO_NOMINA` as

  WITH TRANSFORMACION AS (

  SELECT NEGOCIO, RUBRO, T1.CUENTA, T0.CECO,
      CASE WHEN VERSION IN ('PPTO_V1','PPTO_V2') THEN 2026 ELSE 2027 END as EJERCICIO,
      PERIODO, VERSION,
      7 as MES_INC,
      0.0 as PORC_INC,
      1 as MES_BASE,
      POSICION,
      DIMENSION5CACION,
      SUELDO,
      SUM(HC) as HC,

      SUM(IFNULL( CASE
          WHEN RUBRO = 'Sueldo Ejecutivo' AND DIMENSION6_Personal like '%Ejecutivo%' THEN (SUELDO)
          WHEN RUBRO = 'Personal General' AND DIMENSION6_Personal IN ('Personal General','Eventuales') THEN (SUELDO)
          WHEN RUBRO = 'Día Festivo' THEN (PAGO_FESTIVO)
          WHEN RUBRO = 'Comisiones' THEN (COMISION_ESTIMADA)
          WHEN RUBRO = 'Aprovisionamiento' THEN (APROVISIONAMIENTO)
          WHEN RUBRO = 'Premios y Comisiones' THEN (PREMIOS_COMISIONES)
          WHEN RUBRO = 'Vales de Despensa' THEN (VALES_DESPENSA)
          WHEN RUBRO = 'Prima vacacional' THEN (PRIMA_VACACIONAL)
          WHEN RUBRO = 'Aguinaldo' THEN (AGUINALDO)
          WHEN RUBRO = 'Prima dominical' THEN (PRIMA_DOMINICAL)
          WHEN RUBRO = 'PTU' THEN (PTU)
          WHEN RUBRO = 'Bono BONO_ESP' THEN (BONO_ESP)
          WHEN RUBRO = 'IMSS Patronal' THEN (IMSS)
          WHEN RUBRO = 'SAR' THEN (SAR)
          WHEN RUBRO = 'Vivienda Patrón' THEN (VIVIENDA_PATRON)
          WHEN RUBRO = 'ISN' THEN (ISN)
          ELSE 0 END
          ,0)
      )

      as IMPORTE,

      CONCAT(
        CASE
          WHEN RUBRO = 'Sueldo Ejecutivo' THEN 'NOM_SUE'
          WHEN RUBRO = 'Personal General' THEN 'NOM_SUE'
          WHEN RUBRO = 'Día Festivo' THEN 'NOM_SUE'
          WHEN RUBRO = 'Comisiones' THEN 'NOM_COM'
          WHEN RUBRO = 'Aprovisionamiento' THEN 'NOM_COM'
          WHEN RUBRO = 'Premios y Comisiones' THEN 'NOM_PRE'
          WHEN RUBRO = 'Vales de Despensa'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Prima vacacional'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Aguinaldo'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Prima dominical'  THEN 'NOM_PRE'
          WHEN RUBRO = 'PTU'  THEN 'NOM_PRE'
          WHEN RUBRO = 'Bono BONO_ESP' THEN  'NOM_BONO_ESP'
          WHEN RUBRO = 'IMSS Patronal' THEN  'NOM_PRE'
          WHEN RUBRO = 'SAR' THEN  'NOM_PRE'
          WHEN RUBRO = 'Vivienda Patrón' THEN  'NOM_PRE'
          WHEN RUBRO = 'ISN' THEN  'NOM_PRE'
          ELSE '' END
          ,'MODELO_NOMINA_ETZ'
      ) as ORIGEN

      FROM {project_id}.intranet_bd_pruebas.MODELO_ETZ_BASE T0
      CROSS JOIN `{project_id}.PARAMETROS_PRESUPUESTO.PAR_CUENTAS_MODELO_NOMINA` T1

      group by 1,2,3,4,5,6,7,8,9,10,11,12,13

      )

      SELECT
        NEGOCIO,
        PERIODO,
        EJERCICIO,
        VERSION,
        CUENTA,
        RUBRO,
        COALESCE(T1.DIMENSION5,SPLIT(DIMENSION5CACION," ")[0])
        ||
        CASE WHEN UPPER(MARCA) = 'Negocio9' THEN 'Negocio9' ELSE 'Negocio8' END
            as IDDIMENSION5,
        SUM(HC) AS HC,
        SUM(IMPORTE) as IMPORTE

        FROM TRANSFORMACION T0
        LEFT JOIN {project_id}.mus_qas_drv_datos_maestros.NW_CECO3 T1 ON T0.CECO=T1.CECO
        GROUP BY 1,2,3,4,5,6,7
        ;

END
'''

# client.query(querymnb).result()
# print("Creado: ",stored_procedure_nomina_Negocio9)

# **NOTAS QUERYS**

⭐ Querys que ya no se utilizan, pero pueden llegar a requerirse

-- ❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗❗PARA VALIDAR DIFERENCIAS VS BACKUP


-- SELECT MAX(BACKUP_DATE)
--       FROM `{project_id}.intranet_bd_pruebas.LOG_AUDITORIA_BASE_PRESUPUESTAL`
--       WHERE
--         BACKUP_DATE >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)

WITH BKP AS (

select DIMENSION1, DIMENSION2, DIMENSION3, ceco, version, origen, ejercicio,cuenta, sum(importe) as importe_bkp

from `intranet_bd_pruebas.LOG_AUDITORIA_BASE_PRESUPUESTAL`

WHERE BACKUP_DATE = '2026-02-23 05:53:26 UTC'

group by 1,2,3,4,5,6,7,8
),

ACT AS (

select DIMENSION1, DIMENSION2, DIMENSION3, ceco, version, origen, ejercicio,cuenta, sum(importe) as importe_act

from `intranet_bd_pruebas.BASE_PRESUPUESTAL_COMPLETA_MES`


group by 1,2,3,4,5,6,7,8

)

SELECT
COALESCE(BKP.DIMENSION1,ACT.DIMENSION1) as DIMENSION1,
COALESCE(BKP.DIMENSION2,ACT.DIMENSION2) as DIMENSION2,
COALESCE(BKP.DIMENSION3,ACT.DIMENSION3) as DIMENSION3,
COALESCE(BKP.cuenta,ACT.cuenta) as cuenta,
COALESCE(BKP.ceco,ACT.ceco) as ceco,
COALESCE(BKP.version,ACT.version) as version,
COALESCE(BKP.ejercicio,ACT.ejercicio) as ejercicio,
importe_bkp,
importe_act,
round(ifnull(bkp.importe_bkp,0)-ifnull(act.importe_act,0),2) as diff

FROM BKP
FULL OUTER JOIN ACT USING (DIMENSION1
,DIMENSION2
,DIMENSION3
,VERSION
,EJERCICIO
,CECO
,CUENTA
,ORIGEN
)

where round(ifnull(bkp.importe_bkp,0)-ifnull(act.importe_act,0),2) <> 0 AND ORIGEN IN ('NOM_BONO_ESP')

order by 3,4


WITH

MAXC1 as (
  select Cuenta, DIMENSION1, DIMENSION2, SUM(IMPORTE) AS IMPORTE

from `data_warehouse_finance.accounting_journal`

where ejercicio = 2025 and version IN ('PPTO_V0','Reales') and DIMENSION1 = 'Negocio0' and round(importe,0)<>0 -- and cuenta = '51000247'

GROUP BY 1,2,3
),

MAXC2 AS (
  SELECT CUENTA, DIMENSION1,MAX(IMPORTE) AS IMPORTE

  FROM MAXC1

  GROUP BY 1,2
),

MAXCF as (
  SELECT T0.Cuenta, T0.DIMENSION1, T0.DIMENSION2

  FROM MAXC1 T0
  INNER JOIN MAXC2 T1 ON T0.CUENTA=T1.CUENTA AND T0.DIMENSION1=T1.DIMENSION1  AND t0.iMPORTE=T1.IMPORTE

),

MAXI as (
  select Cuenta, DIMENSION1, DIMENSION2, DIMENSION3, SUM(IMPORTE) AS IMPORTE

from `data_warehouse_finance.accounting_journal`

where ejercicio = 2025 and version IN ('PPTO_V0','Reales') and DIMENSION1 = 'Negocio0' and round(importe,0)<>0 -- and cuenta = '51000247'

GROUP BY 1,2,3,4
),

MAXI2 AS (
  SELECT CUENTA, DIMENSION1, DIMENSION2, MAX(IMPORTE) AS IMPORTE

  FROM MAXI

  GROUP BY 1,2,3
),

MAXI3 AS (
  SELECT T0.Cuenta, T0.DIMENSION1, T0.DIMENSION2, T0.DIMENSION3

  FROM MAXI T0
  INNER JOIN MAXI2 T1 ON T0.CUENTA=T1.CUENTA AND T0.DIMENSION1=T1.DIMENSION1 AND T0.DIMENSION2=T1.DIMENSION2 AND t0.iMPORTE=T1.IMPORTE
),

MAXFF AS (

  SELECT T0.CUENTA, T0.DIMENSION1, T0.DIMENSION2, T1.DIMENSION3

  FROM MAXCF T0
  LEFT JOIN MAXI3 T1 ON T0.CUENTA=T1.CUENTA AND T0.DIMENSION1=T1.DIMENSION1 AND T0.DIMENSION2=T1.DIMENSION2
)

SELECT DISTINCT * FROM MAXFF


WITH MASCARA AS (
  select DIMENSION1, sum(importe) as importe
from `REV2025.vER`

where version = "PPTO_V1" and ejercicio = '2025'
group by 1
),

BASECONS AS (
  
  SELECT Division_de_Negocio_CEBE as DIMENSION1, SUM(IMporte) as importe

FROM {project_id}.REV2025.BASE_CONSOLIDADA

where Division_de_Negocio_CEBE NOT IN ('','CCINGRESOS') AND Division_de_Negocio_CEBE IS NOT NULL AND NOT REGEXP_CONTAINS(Division_de_Negocio_CEBE, r'^[0-9]')

group by 1
)

SELECT T0.DIMENSION1, T0.importe as ImporteMascara, T1.importe as Importebase, T0.importe-T1.importe as DIFF

FROM MASCARA T0
LEFT JOIN BASECONS T1 ON T0.DIMENSION1= T1.DIMENSION1